# **`README.md`**

# A Distributed Method for Cooperative Transaction Cost Mitigation

<!-- PROJECT SHIELDS -->
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](https://opensource.org/licenses/MIT)
[![Python Version](https://img.shields.io/badge/python-3.9%2B-blue.svg)](https://www.python.org/)
[![arXiv](https://img.shields.io/badge/arXiv-2603.07881-b31b1b.svg)](https://arxiv.org/abs/2603.07881)
[![Journal](https://img.shields.io/badge/Journal-ArXiv%20Preprint-003366)](https://arxiv.org/abs/2603.07881)
[![Year](https://img.shields.io/badge/Year-2026-purple)](https://github.com/cooperative_transaction_cost_mitigation)
[![Discipline](https://img.shields.io/badge/Discipline-Quantitative%20Finance%20%7C%20Distributed%20Optimization-00529B)](https://github.com/cooperative_transaction_cost_mitigation)
[![Data Sources](https://img.shields.io/badge/Data-LSEG%20%7C%20FRED-lightgrey)](https://www.lseg.com/)
[![Core Method](https://img.shields.io/badge/Method-ADMM%20%7C%20Convex%20Optimization-orange)](https://github.com/cooperative_transaction_cost_mitigation)
[![Analysis](https://img.shields.io/badge/Analysis-3%2F2--Power%20TC%20%7C%20VAR%20Alpha-red)](https://github.com/cooperative_transaction_cost_mitigation)
[![Validation](https://img.shields.io/badge/Validation-Primal%2FDual%20Residuals-green)](https://github.com/cooperative_transaction_cost_mitigation)
[![Robustness](https://img.shields.io/badge/Robustness-Walk--Forward%20Simulation-yellow)](https://github.com/cooperative_transaction_cost_mitigation)
[![Code style: black](https://img.shields.io/badge/code%20style-black-000000.svg)](https://github.com/psf/black)
[![Type Checking: mypy](https://img.shields.io/badge/type%20checking-mypy-blue)](http://mypy-lang.org/)
[![NumPy](https://img.shields.io/badge/numpy-%23013243.svg?style=flat&logo=numpy&logoColor=white)](https://numpy.org/)
[![Pandas](https://img.shields.io/badge/pandas-%23150458.svg?style=flat&logo=pandas&logoColor=white)](https://pandas.pydata.org/)
[![SciPy](https://img.shields.io/badge/SciPy-%230C55A5.svg?style=flat&logo=scipy&logoColor=white)](https://scipy.org/)
[![CVXPY](https://img.shields.io/badge/CVXPY-Optimization-brightgreen)](https://www.cvxpy.org/)
[![YAML](https://img.shields.io/badge/YAML-%23CB171E.svg?style=flat&logo=yaml&logoColor=white)](https://yaml.org/)
[![Jupyter](https://img.shields.io/badge/Jupyter-%23F37626.svg?style=flat&logo=Jupyter&logoColor=white)](https://jupyter.org/)
[![Open Source](https://img.shields.io/badge/Open%20Source-%E2%9D%A4-brightgreen)](https://github.com/cooperative_transaction_cost_mitigation)

**Repository:** `https://github.com/cooperative_transaction_cost_mitigation`

**Owner:** 2026 Craig Chirinda (Open Source Projects)

This repository contains an **independent**, professional-grade Python implementation of the research methodology from the 2026 paper entitled **"A Distributed Method for Cooperative Transaction Cost Mitigation"** by:

*   **Nikhil Devanathan**
*   **Logan Bell**
*   **Dylan Rueter**
*   **Stephen Boyd**

The project provides a complete, end-to-end computational framework for replicating the paper's findings. It delivers a modular, highly optimized pipeline that executes the entire research workflow: from the ingestion and cleansing of raw market data to the econometric simulation of synthetic alpha signals, culminating in the rigorous execution of a privacy-preserving, distributed convex optimization protocol that mitigates institutional market impact.

## Table of Contents

- [Introduction](#introduction)
- [Theoretical Background](#theoretical-background)
- [Features](#features)
- [Methodology Implemented](#methodology-implemented)
- [Core Components (Notebook Structure)](#core-components-notebook-structure)
- [Key Callable: `run_distributed_cooperative_optimization_pipeline`](#key-callable-run_distributed_cooperative_optimization_pipeline)
- [Prerequisites](#prerequisites)
- [Installation](#installation)
- [Input Data Structure](#input-data-structure)
- [Usage](#usage)
- [Output Structure](#output-structure)
- [Project Structure](#project-structure)
- [Customization](#customization)
- [Contributing](#contributing)
- [Recommended Extensions](#recommended-extensions)
- [License](#license)
- [Citation](#citation)
- [Acknowledgments](#acknowledgments)

## Introduction

This project provides a Python implementation of the analytical framework presented in Devanathan, Bell, Rueter, and Boyd (2026). The core of this repository is the iPython Notebook `cooperative_transaction_cost_mitigation_draft.ipynb`, which contains a comprehensive suite of 35+ orchestrated tasks to replicate the paper's findings.

The pipeline addresses a critical "tragedy of the commons" in multi-manager quantitative hedge funds: individual Portfolio Managers (PMs) optimize their sleeves independently, but their aggregated trades incur non-linear market impact costs that erode firm-wide returns. Solving this centrally requires PMs to divulge their proprietary alpha models and constraints, which violates institutional privacy firewalls.

The codebase operationalizes the proposed solution:
-   **Simulates** a realistic market environment using a low-rank factor risk model and a VAR(1) noise process calibrated to specific Information Coefficients (IC).
-   **Models** execution friction using a rigorous 3/2-power transaction cost function.
-   **Coordinates** autonomous PMs using the Alternating Direction Method of Multipliers (ADMM), broadcasting a synthetic "tax/subsidy" signal that internalizes the firm's marginal execution costs.
-   **Evaluates** the protocol via a 25-year walk-forward backtest, demonstrating that just 5 iterations of ADMM can capture ~75% of the savings of a fully centralized (but privacy-violating) joint optimization.

## Theoretical Background

The implemented methods combine techniques from Distributed Convex Optimization, Market Microstructure, and Financial Econometrics.

**1. The Joint Firm Problem:**
The theoretical optimum minimizes the NAV-weighted sum of PM objectives plus the aggregate transaction costs on the net trade $z$:
$$ \text{minimize } \sum_{i=1}^{M} \lambda_i f_i(x_i) + \gamma_{\text{tc}}\phi_{\text{tc}(z)} \quad \text{subject to } z = \sum_{i=1}^{M} \lambda_i x_i $$

**2. Non-Linear Transaction Cost Modeling:**
The pipeline implements the 3/2-power model, accounting for both fixed bid-ask spreads and temporary market impact scaled by volatility $\nu_j$ and dollar volume $\omega_j$:
$$ \phi_{\text{tc}}(z) = \frac{1}{2}\kappa_{\text{spread}}^T |z| + \kappa_{\text{impact}}^T |z|^{3/2} \quad \text{where} \quad (\kappa_{\text{impact}})_j = \frac{b_j \nu_j}{\sqrt{\omega_j/V}} $$

**3. The ADMM Protocol (Algorithm 1):**
To decouple the joint problem, the Central Planner broadcasts a sharing signal $\ell^k$. Each PM then solves a proximally regularized local problem:
$$ \ell^k = u^k + \frac{\rho}{M} \left( -Dz_{\text{sum}}^k + D \sum_{j=1}^M \lambda_j x^{j,k} \right) $$
$$ x^{i,k+1} = \arg\min_x \left( \lambda^i f^i(x) + \lambda^i (\ell^k)^T Dx + \frac{\rho}{2} \|\lambda^i D(x - x^{i,k})\|_2^2 \right) $$
The diagonal scaling matrix $D$ is heuristically set to the Hessian of the cost approximation: $D_{jj} = \sqrt{2(\kappa_{\text{impact}})_j}$.

**4. Econometric Simulation (VAR(1) Alpha):**
To test the protocol, synthetic alphas are generated using a stationary Vector Autoregressive process, calibrated via the discrete-time Lyapunov equation:
$$ E_t = \Phi E_{t-1} + U_t \quad \text{where} \quad \Sigma_E = \Phi \Sigma_E \Phi^T + \Sigma_U $$


## Features

The provided iPython Notebook (`cooperative_transaction_cost_mitigation_draft.ipynb`) implements the full research pipeline, including:

-   **Disciplined Convex Programming:** Utilizes `CVXPY` to construct and solve complex PM local objectives featuring leverage, concentration, shorting, and turnover constraints.
-   **Factored Risk Constraints:** Implements the risk constraint $\|\Sigma^{1/2}w\|_2 \le \sigma_{\text{target}}$ using a highly optimized stacked vector approach $y = [F^T w; \text{diag}(\sqrt{D^{\text{idio}}}) w]$ to avoid forming dense $N \times N$ covariance matrices.
-   **Configuration-Driven Design:** All study parameters (hyperparameters, institutional limits, econometric targets) are managed in an external, cryptographically hashed `config.yaml` file.
-   **Rigorous State Isolation:** Employs deep-copying and strict object immutability to ensure that the endogenous state trajectories of the four compared protocols (Independent, Cooperative, ADMM-2, ADMM-5) never contaminate each other during the walk-forward simulation.
-   **Cryptographic Archival:** Automatically serializes all artifacts (Parquet, JSON, NPZ) and generates an immutable `.tar.gz` tarball with a master SHA-256 fingerprint for absolute reproducibility.

## Methodology Implemented

The core analytical steps directly implement the methodology from the paper:

1.  **Data Engineering (Tasks 1-14):** Ingests raw market data, applies structural filters, forward-fills missing prices, constructs the $N=434$ asset universe based on end-of-sample market cap, and computes daily dollar volumes and forward returns.
2.  **Risk & Alpha Modeling (Tasks 15-21):** Estimates a $J=15$ low-rank factor covariance matrix, solves the Lyapunov equation using Kronecker properties, and simulates VAR(1) noise paths to generate synthetic alphas calibrated to specific IC targets.
3.  **Parameterization (Tasks 22-23):** Dynamically computes the endogenous market impact coefficients $\kappa_{\text{impact},t}$ and the ADMM scaling matrix $D_t^{\text{scale}}$ at each time step.
4.  **Optimization Solvers (Tasks 24-29):** Defines the CVXPY closures for the independent PM problem, the massive centralized cooperative problem, and the iterative ADMM distributed updates.
5.  **Walk-Forward Simulation (Tasks 30-32):** Executes the daily rebalancing loop, rigorously accounting for post-trade weight drift, cash returns, and transaction cost attribution.
6.  **Evaluation & Archival (Tasks 33-35):** Computes annualized performance metrics, verifies the manuscript's qualitative claims regarding PM-level outcomes, and freezes the research archive.

## Core Components (Notebook Structure)

The notebook is structured as a logical pipeline with modular orchestrator functions for each of the 35 major tasks. All functions are self-contained, fully documented with type hints and docstrings, and designed for professional-grade execution.

## Key Callable: `run_distributed_cooperative_optimization_pipeline`

The project is designed around a single, top-level user-facing interface function:

-   **`run_distributed_cooperative_optimization_pipeline`:** This master orchestrator function runs the entire automated research pipeline from end-to-end. A single call to this function reproduces the entire computational portion of the project, managing data validation, econometric simulation, convex optimization, and deterministic state reconstruction.

## Prerequisites

-   Python 3.9+
-   Core dependencies: `pandas`, `numpy`, `scipy`, `cvxpy`, `pyyaml`.
-   Recommended Solver: `mosek` (requires license) or `scs`/`ecos` (open-source alternatives supported via config).

## Installation

1.  **Clone the repository:**
    ```sh
    git clone https://github.com/cooperative_transaction_cost_mitigation.git
    cd cooperative_transaction_cost_mitigation
    ```

2.  **Create and activate a virtual environment (recommended):**
    ```sh
    python -m venv venv
    source venv/bin/activate  # On Windows, use `venv\Scripts\activate`
    ```

3.  **Install Python dependencies:**
    ```sh
    pip install pandas numpy scipy cvxpy pyyaml
    ```

## Input Data Structure

The pipeline requires four primary data structures, strictly validated at runtime:

1.  **`raw_market_df` (pd.DataFrame):** A MultiIndex `["date", "asset_id"]` panel containing `adjusted_trade_price`, `adjusted_bid_price`, `adjusted_ask_price`, `share_volume`, and `market_cap_usd`.
2.  **`risk_free_rate_series` (pd.Series):** A DatetimeIndex series containing the 3-month U.S. Treasury Bill rate.
3.  **`master_trading_calendar` (pd.DatetimeIndex):** The canonical sequence of valid business days.
4.  **`asset_identifier_map` (pd.DataFrame):** A mapping table linking internal `asset_id`s to vendor tickers.

*Note: The pipeline includes a synthetic data generator for testing purposes if access to proprietary LSEG/CRSP data is unavailable.*

## Usage

The notebook provides a complete, step-by-step guide. The primary workflow is to execute the final cell, which demonstrates how to load the configuration, generate synthetic data, and use the top-level orchestrator:

```python
import os
import yaml
import pandas as pd
import numpy as np

# 1. Load the master configuration from the YAML file.
# (Assumes config.yaml is in the working directory)
with open("config.yaml", "r") as f:
    study_config = yaml.safe_load(f)

# 2. Load raw datasets (Example using synthetic generator provided in the notebook)
# In production, load from Parquet: pd.read_parquet("lseg_market_data.parquet")
(
    raw_market_df,
    risk_free_rate_series,
    master_trading_calendar,
    asset_identifier_map
) = generate_synthetic_market_environment()

# 3. Execute the entire replication study.
pipeline_summary = run_distributed_cooperative_optimization_pipeline(
    raw_market_df=raw_market_df,
    risk_free_rate_series=risk_free_rate_series,
    master_trading_calendar=master_trading_calendar,
    asset_identifier_map=asset_identifier_map,
    study_config=study_config,
    output_base_dir="./institutional_research_archive"
)

# 4. Access results
if pipeline_summary.get("status") == "SUCCESS":
    print("\n[*] Final Reproduction Fidelity:")
    print(f"    Classification: {pipeline_summary['fidelity_classification']}")
    
    print("\n[*] Firm-Level Performance Metrics:")
    firm_table = pipeline_summary["reproduction_package"]["performance_metrics"]["firm_table"]
    print(firm_table.to_string())
    
    print(f"\n[*] Archive frozen at: {pipeline_summary['archive_path']}")
```

## Output Structure

The pipeline returns a master dictionary containing:
-   **`manifest`**: The cryptographic provenance record, including environment versions, random seeds, input hashes, and all manuscript-unspecified placeholder assumptions.
-   **`exogenous_artifacts`**: The cleansed market panel, frozen universe IDs, risk models, and synthetic alpha vectors.
-   **`protocol_results`**: The raw historical trajectories (NAVs, trades, costs) and ADMM convergence traces for each of the four protocols.
-   **`performance_metrics`**: The finalized firm-level and PM-level tables (Return, Volatility, Sharpe) and cumulative paths.
-   **`evaluation_summary`**: The formal verification of the manuscript's qualitative claims regarding PM-level outcomes.

## Project Structure

```
cooperative_transaction_cost_mitigation/
│
├── cooperative_transaction_cost_mitigation_draft.ipynb   # Main implementation notebook
├── config.yaml                                           # Master configuration file
├── requirements.txt                                      # Python package dependencies
│
├── LICENSE                                               # MIT Project License File
└── README.md                                             # This file
```

## Customization

The pipeline is highly customizable via the `config.yaml` file. Users can modify study parameters such as:
-   **Institutional Constraints:** Adjust leverage ($L$), concentration ($C$), and turnover ($T$) limits to reflect different fund mandates.
-   **ADMM Hyperparameters:** Tune the penalty parameter $\rho$ or test different iteration counts ($K$).
-   **Econometrics:** Alter the target Information Coefficient (IC) range or the VAR(1) temporal autocorrelation to simulate different market regimes.

## Contributing

Contributions are welcome. Please fork the repository, create a feature branch, and submit a pull request with a clear description of your changes. Adherence to PEP 8, strict type hinting, and comprehensive docstrings is required.

## Recommended Extensions

Future extensions could include:
-   **Alternative Cost Models:** Replacing the 3/2-power model with piecewise-affine or quadratic transaction cost models.
-   **Asynchronous ADMM:** Implementing asynchronous updates where PMs solve their local problems at different frequencies.
-   **Live Execution Integration:** Adapting the state-transition layer to ingest real-time FIX execution reports rather than simulated walk-forward accounting.

## License

This project is licensed under the MIT License. See the `LICENSE` file for details.

## Citation

If you use this code or the methodology in your research, please cite the original paper:

```bibtex
@article{devanathan2026distributed,
  title={A Distributed Method for Cooperative Transaction Cost Mitigation},
  author={Devanathan, Nikhil and Bell, Logan and Rueter, Dylan and Boyd, Stephen},
  journal={arXiv preprint arXiv:2603.07881},
  year={2026}
}
```

For the implementation itself, you may cite this repository:
```
Chirinda, C. (2026). A Distributed Method for Cooperative Transaction Cost Mitigation: An Open Source Implementation.
GitHub repository: https://github.com/cooperative_transaction_cost_mitigation
```

## Acknowledgments

-   Credit to **Nikhil Devanathan, Logan Bell, Dylan Rueter, and Stephen Boyd** for the foundational research that forms the entire basis for this computational replication.
-   This project is built upon the exceptional tools provided by the open-source community. Sincere thanks to the developers of the scientific Python ecosystem, particularly the **CVXPY** contributors for enabling robust Disciplined Convex Programming.

--

*This README was generated based on the structure and content of the `cooperative_transaction_cost_mitigation_draft.ipynb` notebook and follows best practices for research software documentation.*

# Paper

Title: "*A Distributed Method for Cooperative Transaction Cost Mitigation*"

Authors: Nikhil Devanathan, Logan Bell, Dylan Rueter, Stephen Boyd

E-Journal Submission Date: 9 March 2026

Link: https://arxiv.org/abs/2603.07881

Abstract:

Funds at large portfolio management firms may consist of many portfolio managers (PMs), each managing a portion of the fund and optimizing a distinct objective. Although the PMs determine their trades independently, the trade lists may be netted and executed by the firm. These net trades may be sufficiently large to impact the market prices, so the PMs may realize prices on their trades that are different from the observed midpoint price of the assets before execution. These transaction costs generally reduce the returns of a portfolio over time. We propose a simple protocol, based on methods from distributed convex optimization, by which a firm can communicate estimated transaction costs to its PMs, and the PMs can potentially revise their trades to realize reduced transaction costs. This protocol does not require the PMs to disclose their method of determining trades to the firm or to each other, nor does it require the PMs to communicate their trade lists with each other. As the number of adjustment rounds grows, the trades converge to the ones that are optimal for the firm. As a practical matter we observe that even just a few rounds of adjustment lead to substantial savings for the firm and the PMs

# Summary

### Rigorous Synthesis of Distributed Cooperative Transaction Cost Mitigation

#### Introduction: The Coordination Problem in Multi-Manager Firms
In the contemporary landscape of systematic investment management, a fundamental friction exists between the autonomy of individual portfolio managers (PMs) and the aggregate efficiency of the firm. A systematic firm often employs multiple PMs, each overseeing an independent "sleeve" or sub-portfolio within a unified fund. While these PMs typically optimize their specific objectives—factoring in proprietary alpha signals, risk constraints, and local transaction costs—their collective actions are coupled at the firm level. This coupling is most pronounced in the context of **netted transaction costs**.

When a firm aggregates the independent trade lists of its PMs to execute a single net trade vector, the resulting market impact and execution costs are a function of the aggregate volume. Large, uncoordinated trades can "eat through" the order book, leading to realized prices that deviate significantly from the mid-price observed at the time of decision-making. Conversely, when PMs trade in opposite directions, the firm can "cross" these trades internally, effectively eliminating market impact and bid-ask spreads for those specific shares.

The paper at hand addresses this **coordination problem**: how can a firm synchronize the actions of autonomous PMs to optimize a global objective (minimizing total transaction costs) without compromising the privacy of their proprietary strategies or requiring them to disclose their internal optimization logic to one another?.

#### Mathematical Formulation of the Joint Problem
To understand the solution, we must first define the firm's environment mathematically. We consider a firm with $M$ portfolio managers and $N$ assets. Each PM $i$ manages a portfolio with a net asset value (NAV) denoted as $V_i$, contributing to the firm’s total NAV, $V = \sum V_i$. The relative weight of each PM’s portfolio is $\lambda_i = V_i / V$.

**A. Portfolio Manager Objectives**
Each PM $i$ aims to minimize a convex objective function $f_i(x_i)$, where $x_i \in \mathbb{R}^N$ represents their trade weight vector. This objective function is typically defined as the negative of the PM's risk-adjusted expected return, net of fees. Crucially, $f_i$ can incorporate indicator functions for constraints such as:
*   **Risk Limits:** Bounding portfolio volatility.
*   **Turnover Limits:** Restricting the magnitude of trading.
*   **Leverage and Concentration Constraints:** Bounding absolute weights or positions in specific assets.

**B. The Net Trade and Transaction Costs**
The firm’s net trade weight vector $z$ is the NAV-weighted sum of individual trade weights: $z = \sum_{i=1}^{M} \lambda_i x_i$. The transaction cost $\phi_{tc}(z)$ is modeled as a non-linear function of this net trade. The sources highlight a common 3/2-power model:
$$\phi_{tc}(z) = (1/2)\kappa_{spread}^T |z| + \kappa_{impact}^T |z|^{3/2}$$
where $\kappa_{spread}$ represents the bid-ask spread and $\kappa_{impact}$ accounts for market impact based on returns volatility and dollar volume.

**C. The Firm's Global Objective**
The firm’s goal is to minimize the NAV-weighted sum of PM objectives plus the aggregate transaction costs:
$$\text{minimize } \sum_{i=1}^{M} \lambda_i f_i(x_i) + \gamma_{tc}\phi_{tc}(z)$$
$$\text{subject to } z = \sum_{i=1}^{M} \lambda_i x_i$$
where $\gamma_{tc}$ is a scaling parameter to adjust the priority of transaction cost mitigation. This "joint problem" couples the PMs because the optimal trade for PM $i$ now depends on the trades of all other PMs through the $z$ term in the global objective.

#### Distributed Solution via ADMM
Solving the joint problem centrally would require every PM to send their entire objective function $f_i$ (including proprietary alpha models and risk parameters) to a central entity. For reasons of **data privacy and autonomy**, this is often untenable. The paper proposes a distributed iterative method based on the **Alternating Direction Method of Multipliers (ADMM)**.

**A. Reformulation for Decentralization**
The researchers transform the global problem by introducing NAV-scaled trade weights $\tilde{x}_i = \lambda_i x_i$ and auxiliary "consensus" variables $z_i$. This allows the coupling constraint to be handled through a dual variables approach. By applying ADMM, the problem is decomposed into individual PM updates and a central "sharing" update.

**B. The Logic of the Sharing Signal**
A central planner (or a simple automated process within the firm) coordinates the PMs by broadcasting a sharing update signal $\ell^k$ in each iteration $k$. This signal is derived from the net trade and the dual variable $u^k$, which tracks the deviation between the PMs' proposed trades and the firm's optimal net trade.

#### Step-by-Step Protocol: Algorithm 1
The core contribution of the paper is a practical protocol that proceeds in rounds. Even a few rounds can yield significant cost savings.

**Step 1: Initialization**
*   The firm fixes the iteration count $K$, a step-size parameter $\phi$, a penalty parameter $\rho$, and a diagonal scaling matrix $D$.
*   Each PM $i$ initializes their trade list $x_{i,0}$ by solving their local optimization problem independently: $x_{i,0} = \text{argmin } f_i(x)$.
*   The central planner receives the initial aggregate trade $\sum \lambda_i x_{i,0}$ and initializes the dual and consensus variables to zero.

**Step 2: Distributed Update (The PM Round)**
In each iteration $k$, the central planner broadcasts the signal $\ell^k$. Each PM then updates their trade list by solving a modified version of their original problem:
$$x_{i,k+1} = \text{argmin}_x \left( \lambda_i f_i(x) + \lambda_i (\ell^k)^T D x + \frac{\rho}{2} \|\lambda_i D (x - x_{i,k})\|_2^2 \right)$$
This update contains two critical modifications to the PM's original objective:
1.  **Linear Price Adjustment $(\ell^k)^T D x$:** This term acts as a synthetic tax or subsidy. Assets that contribute to high aggregate transaction costs receive a "premium" (increasing the cost to buy), while assets that offset the firm's net trade receive a "discount".
2.  **Quadratic Stability Penalty $\frac{\rho}{2} \|\dots\|_2^2$:** This discourages the PM from making excessively large changes from their previous trade list $x_{i,k}$ in a single round, ensuring the iterative process remains stable.

**Step 3: Gathered Update (The Firm Round)**
The central planner collects the new NAV-weighted trade lists $\sum \lambda_i x_{i,k+1}$. It then solves a central optimization problem to update the firm-wide net trade $z_{sum}^{k+1}$ and the dual variable $u^{k+1}$. The dual update is:
$$u^{k+1} = u^k + \phi \rho \frac{1}{M} (D \sum \lambda_i x_{i,k+1} - D z_{sum}^{k+1})$$
This update effectively "learns" the marginal transaction costs and feeds them back into the next round's signal $\ell^{k+1}$.

**Step 4: Convergence**
The process repeats for $K$ rounds. Mathematically, as $K \to \infty$, the trades $x_i$ converge to the global optimum of the joint problem. Practically, the paper finds that $K=2$ or $K=5$ provides substantial benefits.

#### Hyperparameter Tuning and Extensions
The effectiveness of the ADMM protocol depends on several parameters:
*   **Scaling Matrix $D$:** The authors suggest setting the diagonal elements $D_{jj} = \sqrt{2(\kappa_{impact})_j}$ to align the quadratic penalty with the curvature of the transaction cost function.
*   **Penalty Parameter $\rho$:** Controls the strength of the consensus constraint; a value of $\rho = 10.0$ was found to be effective in backtests.
*   **Firmwide Constraints:** The protocol can easily incorporate constraints on the *aggregate* portfolio, such as total leverage limits or "box" constraints on the net trade due to market liquidity, by modifying the central $g(z)$ function.
*   **Shorting Costs:** If the firm manages borrow costs centrally (e.g., netting long and short positions across PMs to reduce the net borrow), these costs can be included as a coupling term in the joint objective.

#### Empirical Validation: The Backtest
To demonstrate the protocol's efficacy, the authors conducted a comprehensive backtest using historical U.S. equities data (S&P 500 constituents) from July 2000 to April 2025.

**A. Backtest Setup**
*   **Universe:** 434 assets.
*   **PM Configuration:** $M=4$ PMs, each assigned a random 75% subset of the universe.
*   **Synthetic Alphas:** To isolate the impact of the protocol, the authors used synthetic alpha signals calibrated to a specific Information Coefficient (IC) using a VAR(1) noise process.
*   **Risk Model:** A common low-rank factor model (15 factors) estimated with a 42-day forward-looking window.

**B. Protocols Compared**
1.  **Independent:** PMs ignore firm-level netting and optimize alone.
2.  **Full Cooperative:** A theoretical benchmark solving the joint problem centrally.
3.  **ADMM (2 and 5 iterations):** The proposed distributed protocol.

**C. Results and Performance Statistics**
The results clearly indicate the superiority of cooperation.
*   **Sharpe Ratio:** The independent protocol achieved a Sharpe ratio of **1.60**, whereas the full cooperative and ADMM (5-iter) protocols achieved **2.05** and **2.08**, respectively.
*   **Volatility Reduction:** Cooperative methods showed lower volatility (approx. 8.6% - 9.2%) compared to the independent case (9.77%).
*   **Transaction Cost Savings:** The ADMM 5-iter protocol captured approximately **75% of the total savings** achievable by the full cooperative solution. Figure 2 in the source document illustrates a dramatic divergence in cumulative transaction costs, with the cooperative lines remaining significantly below the independent baseline over the 25-year period.

#### 7. Critical Analysis: Autonomy vs. Performance
One of the most nuanced findings in the paper involves the impact on individual PMs. While the firm’s total returns and Sharpe ratio improve, the outcome for an individual PM is **indeterminate**.

The sources note that it is "demonstrably not the case that every PM will realize greater performance by cooperating". For instance, in the backtest, PM 2 actually experienced worse performance after 2010 when cooperating compared to trading independently. This occurs because the coordination process might lead a PM to alter their trades in a way that reduces firm-wide costs but marginally degrades their local alpha capture or risk profile.

However, from the perspective of a firm that views its PMs as "investible assets," the global improvement in frictional costs justifies the protocol. The method effectively transforms transaction cost management from a series of isolated, often-competing trades into a **principled, cooperative mechanism** for preserving the firm's capital.

#### 8. Conclusion and Future Implications
The "Distributed Method for Cooperative Transaction Cost Mitigation" provides a mathematically rigorous yet practically simple framework for multi-manager firms. By utilizing ADMM to create a synthetic price adjustment signal, firms can:
1.  **Reduce Frictional Costs:** Substantially lowering the impact of spreads and market price moves.
2.  **Preserve PM Privacy:** Avoiding the need to share alpha signals or proprietary optimization constraints.
3.  **Maintain Operational Flexibility:** Working with as few as two rounds of communication.

In conclusion, this research marks a significant step forward in applying distributed convex optimization to finance. It offers a solution to a pervasive "tragedy of the commons" in portfolio management, where individual optimization leads to collective inefficiency. For firms structured as collections of independent sleeves, this protocol offers a path to institutional-grade execution efficiency while respecting the decentralized nature of their talent.

**

**Professor's Post-Script (Information outside the sources):**
*While the paper focuses on the mathematical and computational aspects of the protocol, practitioners should be aware that the "cooperative" nature assumed here requires institutional alignment. In a real-world setting, PM compensation structures would likely need to be adjusted to account for the "firm-first" trade adjustments, as a PM whose local performance suffers for the sake of firm-wide cost savings might otherwise be disincentivized to participate in the protocol.*

# Import Essential Modules

In [ ]:
#!/usr/bin/env python3
# ==============================================================================#
#
#  A Distributed Method for Cooperative Transaction Cost Mitigation
#
#  This module provides a complete, production-grade implementation of the
#  analytical framework presented in "A Distributed Method for Cooperative
#  Transaction Cost Mitigation" by Devanathan, Bell, Rueter, and Boyd (2026).
#  It delivers a computationally tractable system for the decentralized
#  optimization of institutional externalities, enabling multiple autonomous
#  portfolio managers to cooperatively minimize firm-wide market impact
#  without compromising the privacy of their proprietary alpha signals or
#  local risk constraints.
#
#  Core Methodological Components:
#  • Distributed convex optimization via Alternating Direction Method of Multipliers (ADMM)
#  • Non-linear 3/2-power transaction cost modeling for market impact
#  • Proximal operator calculus for constrained portfolio optimization
#  • Synthetic alpha generation via Kronecker-structured VAR(1) noise processes
#  • Low-rank factor covariance estimation with forward-looking volatility
#  • Walk-forward simulation with rigorous state-transition accounting
#
#  Technical Implementation Features:
#  • Disciplined Convex Programming (DCP) via CVXPY
#  • High-performance vectorized tensor operations via NumPy
#  • Memory-efficient MultiIndex panel data manipulation via pandas
#  • Discrete-time Lyapunov equation solvers for stationary covariance calibration
#  • Deterministic cryptographic fingerprinting for absolute reproducibility
#  • Immutable, strictly typed data structures for state isolation
#
#  Paper Reference:
#  Devanathan, N., Bell, L., Rueter, D., & Boyd, S. (2026). A Distributed Method
#  for Cooperative Transaction Cost Mitigation. arXiv preprint arXiv:2603.07881.
#  https://arxiv.org/abs/2603.07881
#
#  Author: CS Chirinda
#  License: MIT
#  Version: 1.0.0
#
# ==============================================================================#

# Standard Library Imports
import sys
import os
import time
import math
import json
import hashlib
import tarfile
import logging
import platform
import subprocess
import importlib.metadata
from pathlib import Path
from dataclasses import dataclass, field
from enum import Enum
from typing import (
    Dict,
    List,
    Tuple,
    Set,
    Any,
    Union,
    Optional,
    Callable,
    NamedTuple
)

# Third-Party Scientific and Optimization Stack
import numpy as np
import pandas as pd
import scipy.linalg
import cvxpy as cp

# Configure module-level logger for institutional-grade transparency
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


# Implementation

# Draft 1

## **Discussion of the Inputs-Processes-Outputs (IPO) of Key Callables**

Below is a clinical and granular exposition of the 36 orchestrator callables that constitute the computational architecture of the "Distributed Method for Cooperative Transaction Cost Mitigation" pipeline.

### 1. `task_1_orchestrator` (Structural Validation)
*   **Inputs:** `raw_market_df`, `risk_free_rate_series`, `master_trading_calendar`, `asset_identifier_map`, `study_config`.
*   **Processes:** Validates the MultiIndex schema, enforces strict data types, normalizes timezone-aware datetimes to UTC, and verifies the presence of required columns.
*   **Outputs:** Structurally validated and normalized versions of the five input objects.
*   **Data Transformation:** Transforms potentially malformed, timezone-inconsistent raw data into a strictly typed, lexicographically sorted panel format.
*   **Role & Equations:** Implements the foundational data engineering prerequisites. While it contains no explicit equations, it ensures the structural integrity required to compute the transaction cost model $\phi_{\text{tc}}(z)$ and the risk model $\Sigma_t$.

### 2. `task_2_orchestrator` (Semantic Validation)
*   **Inputs:** `study_config`.
*   **Processes:** Extracts and verifies exact scalar values for global firm constants ($M=4$, $N=434$), institutional constraints ($L=1.5$, $C=0.2$), and ADMM hyperparameters ($\rho=10.0$, $\varphi=1.0$).
*   **Outputs:** The semantically validated configuration dictionary.
*   **Data Transformation:** Maps raw configuration strings/floats into validated, immutable execution parameters.
*   **Role & Equations:** Enforces the exact parameterization of the study, specifically ensuring that the ADMM penalty parameter $\rho$ and dual extrapolation parameter $\varphi$ match the LaTeX context: "Empirically, we find the choices $\rho = 10.0$ and $\varphi = 1$ to work well."

### 3. `task_3_orchestrator` (Numerical & Cross-Input Consistency)
*   **Inputs:** Validated `raw_market_df`, `asset_identifier_map`, `master_trading_calendar`, `risk_free_rate_series`.
*   **Processes:** Verifies strict positivity of prices, non-negativity of volumes, referential integrity of asset IDs, and ensures the calendar length $\ge 43$.
*   **Outputs:** A strictly typed `ValidationReport` dataclass.
*   **Data Transformation:** Reduces massive panel datasets into a discrete set of boolean validation flags and violation counts.
*   **Role & Equations:** Prevents mathematical singularities. By enforcing price positivity, it guarantees that the fractional spread $\kappa_{\text{spread}}$ and the reference price $p_{\text{ref}}$ do not trigger division-by-zero errors.

### 4. `task_4_orchestrator` (Environment Freezing & Hashing)
*   **Inputs:** The five primary input objects.
*   **Processes:** Records OS/package versions, instantiates isolated `numpy.random.Generator` objects, and computes deterministic SHA-256 hashes of all inputs.
*   **Outputs:** The provenance `manifest` and a dictionary of isolated random generators.
*   **Data Transformation:** Transforms in-memory DataFrames and dictionaries into cryptographic byte-streams to generate immutable hex digests.
*   **Role & Equations:** Implements the reproducibility framework. It ensures that the stochastic components of the synthetic alpha generation ($E_t = \Phi E_{t-1} + U_t$) are perfectly deterministic across execution environments.

### 5. `task_5_orchestrator` (Identifier Integrity)
*   **Inputs:** `raw_market_df`, `asset_identifier_map`, `master_trading_calendar`, `study_config`, `manifest`.
*   **Processes:** Verifies 1-to-1 mapping of `asset_id` to `vendor_id` and records the exact temporal boundaries of the calendar.
*   **Outputs:** The augmented provenance `manifest`.
*   **Data Transformation:** Extracts boundary metadata from the DatetimeIndex and injects it into the JSON-serializable manifest.
*   **Role & Equations:** Ensures longitudinal panel continuity, which is critical for the 42-day forward-looking window calculations required by the risk model $\Sigma_t$.

### 6. `task_6_orchestrator` (Calendar Freezing)
*   **Inputs:** `master_trading_calendar`, `risk_free_rate_series`, `manifest`.
*   **Processes:** Validates monotonic increase, absence of weekends, and computes the subset of valid decision dates $\mathcal{T}_{\text{valid}}$.
*   **Outputs:** The frozen `valid_decision_dates` DatetimeIndex and the augmented `manifest`.
*   **Data Transformation:** Slices the master calendar array `[:-42]` to isolate dates capable of supporting a full forward window.
*   **Role & Equations:** Defines the temporal domain of the walk-forward simulation. It guarantees that for every $t \in \mathcal{T}_{\text{valid}}$, the forward return vector $R_t$ can be computed without out-of-bounds indexing.

### 7. `task_7_orchestrator` (Structural Filtering)
*   **Inputs:** `raw_market_df`, `study_config`.
*   **Processes:** Applies 8 conjunctive boolean masks (e.g., primary observation, USD currency, positive prices) and verifies the absence of duplicate index pairs.
*   **Outputs:** The structurally filtered DataFrame and an audit log.
*   **Data Transformation:** Reduces the raw panel row-count via vectorized boolean indexing, mapping dirty data to a structurally sound subset.
*   **Role & Equations:** Cleanses the data required for the transaction cost model. It ensures that the inputs to $(\kappa_{\text{impact}})_j = b_j \nu_j / \sqrt{\omega_j/V}$ are strictly valid.

### 8. `task_8_orchestrator` (Missing Data Imputation)
*   **Inputs:** Filtered DataFrame, `master_trading_calendar`, `study_config`, audit log.
*   **Processes:** Unstacks the panel, forward-fills bid/ask prices along the temporal axis, drops missing trade prices/volumes, and restacks.
*   **Outputs:** The finalized `clean_market_df` and a comprehensive row-count summary.
*   **Data Transformation:** Pivots long-format data to wide-format to apply vectorized temporal forward-filling without cross-asset contamination, then pivots back.
*   **Role & Equations:** Implements the manuscript's explicit instruction: "Missing values are forward-filled." This guarantees that $\kappa_{\text{spread}}$ can be computed continuously.

### 9. `task_9_orchestrator` (Candidate Set Construction)
*   **Inputs:** `clean_market_df`, `master_trading_calendar`, `study_config`, `manifest`.
*   **Processes:** Identifies historical S&P 500 constituents and applies an 80% temporal coverage threshold filter.
*   **Outputs:** A list of candidate `asset_id`s, the augmented `manifest`, and a funnel log.
*   **Data Transformation:** Aggregates boolean flags across the temporal axis to filter the cross-sectional asset dimension.
*   **Role & Equations:** Implements the universe selection criteria: "Our universe consists of $N = 434$ assets drawn from historical S&P 500 constituents, filtered to those with sufficient data history..."

### 10. `task_10_orchestrator` (End-of-Sample Ranking)
*   **Inputs:** `clean_market_df`, `master_trading_calendar`, candidate IDs, `study_config`, `manifest`.
*   **Processes:** Extracts market caps on the ranking date, sorts descending, truncates to $N=434$, and restricts the panel.
*   **Outputs:** The frozen `universe_asset_ids` tuple, the `universe_market_df`, and the augmented `manifest`.
*   **Data Transformation:** Sorts and slices the cross-sectional dimension, establishing the canonical $N$-dimensional vector order for all downstream linear algebra.
*   **Role & Equations:** Finalizes the universe: "...and sorted by market capitalization at the end of the sample period." This fixes the dimensionality $N=434$ for vectors $x^i, z \in \mathbb{R}^N$.

### 11. `task_11_orchestrator` (Macro Scalar Alignment)
*   **Inputs:** `risk_free_rate_series`, `master_trading_calendar`, `valid_decision_dates`, `study_config`, `manifest`.
*   **Processes:** Reindexes the FRED series to the calendar, forward-fills, normalizes percentages, and divides by 252.
*   **Outputs:** The `rf_daily_series` and the augmented `manifest`.
*   **Data Transformation:** Transforms an irregular, annualized macro time-series into a dense, daily-equivalent scalar series aligned with the trading calendar.
*   **Role & Equations:** Provides the $r_{\text{rf}}$ scalar required for the PM objective's cash return term ($-r_{\text{rf}} c$) and the shorting cost function $\phi^{\text{short}}(w) = r_{\text{rf}} \sum_{j=1}^N \max(0, -w_j)$.

### 12. `task_12_orchestrator` (Midpoint and Spread Construction)
*   **Inputs:** `universe_market_df`, `valid_decision_dates`, `universe_asset_ids`.
*   **Processes:** Computes the reference midpoint, calculates the fractional spread, and pivots to a wide-format matrix.
*   **Outputs:** The `kappa_spread_panel` DataFrame of shape $(T, N)$.
*   **Data Transformation:** Maps raw bid/ask prices into a unitless fractional spread matrix, explicitly reindexing columns to the canonical asset order and clamping negative values to zero.
*   **Role & Equations:** Implements the linear component of the transaction cost model. It computes $\kappa_{\text{spread}}$ for use in $\phi^{\text{tc}}(z) = (1/2)\kappa_{\text{spread}}^T|z| + \dots$

### 13. `task_13_orchestrator` (Volume and Returns Construction)
*   **Inputs:** `universe_market_df`, `universe_asset_ids`, `master_trading_calendar`, `manifest`.
*   **Processes:** Computes daily dollar volume and one-period realized returns, pivoting both to wide-format matrices.
*   **Outputs:** The `dollar_volume_panel`, `daily_returns_panel`, and augmented `manifest`.
*   **Data Transformation:** Maps prices and share volumes into dense $(T, N)$ matrices. Crucially, it unstacks prices *before* applying the temporal `.shift(1)` to prevent cross-asset contamination.
*   **Role & Equations:** Computes $\omega_{t,j} = \text{price}_{t,j} \times \text{volume}_{t,j}$ for the impact coefficient, and $r_{t,j} = (p_{t,j}/p_{t-1,j}) - 1$ as the primitive for the risk model and alpha generation.

### 14. `task_14_orchestrator` (Forward Returns Construction)
*   **Inputs:** `daily_returns_panel`, `valid_decision_dates`, `master_trading_calendar`, `manifest`.
*   **Processes:** Extracts 42-day forward windows using zero-copy NumPy slicing and computes the compound forward return vector.
*   **Outputs:** Dictionaries mapping dates to `(42, N)` window matrices and `(N,)` return vectors, plus the `manifest`.
*   **Data Transformation:** Slices a 2D matrix into a dictionary of 2D matrices, then reduces the temporal axis via `np.prod` to yield 1D vectors.
*   **Role & Equations:** Implements the forward-looking target for the synthetic alpha: $R_{t,j} = \prod_{s=1}^{42}(1+r_{t+s,j}) - 1$.

### 15. `task_15_orchestrator` (Volatility and Standardization)
*   **Inputs:** Forward return windows, `study_config`, `manifest`.
*   **Processes:** Computes sample standard deviation (`ddof=1`), standardizes returns, and winsorizes at $\pm 4.2$.
*   **Outputs:** Dictionaries of volatility vectors $\sigma_t$ and winsorized matrices $\tilde{\mathbf{R}}_t^{\text{win}}$.
*   **Data Transformation:** Applies column-wise reduction for volatility, broadcasting for standardization, and element-wise clipping for winsorization.
*   **Role & Equations:** Implements the risk model prerequisites: $\sigma_{t,j} = \text{std}(\dots)$ and $\tilde{r}_{t+s,j}^{\text{win}} = \max(-4.2, \min(4.2, r_{t+s,j}/\sigma_{t,j}))$.

### 16. `task_16_orchestrator` (Factor Structure Extraction)
*   **Inputs:** Winsorized windows, volatility vectors, `study_config`, `manifest`.
*   **Processes:** Computes correlation matrices, extracts top $J=15$ eigenpairs, constructs factor loadings, and computes idiosyncratic variances.
*   **Outputs:** Dictionaries for $F_t^{\text{annual}}$, $D_t^{\text{idio, annual}}$, and $\nu_t^{\text{daily}}$.
*   **Data Transformation:** Maps $(42, N)$ matrices to $(N, N)$ correlation matrices, then to $(N, J)$ eigenvectors via symmetric eigendecomposition, scaling the results by annualization factors.
*   **Role & Equations:** Implements the low-rank factor model: $\Sigma_t = F_t F_t^T + D_t$, where $F_t = \text{diag}(\sigma_t)Q_t \Lambda_t^{1/2}$.

### 17. `task_17_orchestrator` (PM Static Characteristics)
*   **Inputs:** `study_config`, `universe_asset_ids`, random generators, `manifest`.
*   **Processes:** Draws log-uniform NAVs, uniform risk targets, and exact-cardinality tradable masks.
*   **Outputs:** A dictionary of frozen `PMCharacteristics` dataclasses and the `manifest`.
*   **Data Transformation:** Maps random uniform draws into structured, immutable PM parameters.
*   **Role & Equations:** Initializes the heterogeneous PM environment: $V_0^{(i)} \sim \text{log-uniform}(10^{6.5}, 10^{7.5})$ and $\sigma_{\text{target}}^{(i)} \sim \mathcal{U}[0.06, 0.15]$.

### 18. `task_18_orchestrator` (PM Initial State)
*   **Inputs:** `pm_characteristics_dict`, `manifest`.
*   **Processes:** Initializes all-cash portfolios ($w=\mathbf{0}, c=1.0$) and computes relative NAV weights.
*   **Outputs:** The mutable `FirmState` object and the `manifest`.
*   **Data Transformation:** Aggregates individual NAVs to compute the global scalar $V_0$ and fractional weights $\lambda_0^{(i)}$.
*   **Role & Equations:** Establishes the coupling weights for the firm net trade: $\lambda_i = V_i / V$.

### 19. `task_19_orchestrator` (Alpha Calibration)
*   **Inputs:** `study_config`, random generators, `manifest`.
*   **Processes:** Draws target ICs, computes alpha/noise scaling factors, draws autocorrelations, and freezes the objects.
*   **Outputs:** A dictionary of frozen `AlphaCalibration` dataclasses and the `manifest`.
*   **Data Transformation:** Maps uniform draws into deterministic scaling scalars.
*   **Role & Equations:** Calibrates the signal-to-noise ratio: $c^{(i)} = (\text{IC}^{(i)})^2$ and $v^{(i)} = \sqrt{1/(\text{IC}^{(i)})^2 - 1}$.

### 20. `task_20_orchestrator` (VAR(1) and Lyapunov Solution)
*   **Inputs:** `alpha_calibration_dict`, `daily_returns_panel`, `study_config`, `manifest`.
*   **Processes:** Constructs the diagonal $\Phi$, the stationary covariance $S_E$, and solves the Lyapunov equation for $S_U$.
*   **Outputs:** The VAR(1) structural components (`phi_diag`, $S_E$, $S_U$, $\Sigma_{\text{Asset}}$).
*   **Data Transformation:** Exploits Kronecker properties to solve a $1736 \times 1736$ Lyapunov equation element-wise on a $4 \times 4$ matrix, avoiding massive dense matrix inversion.
*   **Role & Equations:** Implements the discrete-time Lyapunov equation: $\Sigma_E = \Phi \Sigma_E \Phi^T + \Sigma_U$.

### 21. `task_21_orchestrator` (Noise Path and Alpha Construction)
*   **Inputs:** VAR(1) components, forward returns, calibration dict, calendar, generators, `manifest`.
*   **Processes:** Simulates the noise path via Kronecker Cholesky sampling, constructs synthetic alphas, and validates empirical ICs.
*   **Outputs:** The `alpha_dictionary` mapping $(t, i)$ to $\alpha_t^{(i)}$.
*   **Data Transformation:** Transforms standard normal draws into autocorrelated, cross-correlated noise vectors, synthesizing them with realized returns.
*   **Role & Equations:** Implements the VAR(1) process $E_t = \Phi E_{t-1} + U_t$ and the synthetic alpha $\hat{R}_t^{(i)} = c^{(i)}(R_t + E_t^{(i)})$.

### 22. `task_22_orchestrator` (Transaction Cost Parameters)
*   **Inputs:** $\nu_t$, $\omega_t$, $\kappa_{\text{spread},t}$, $r_{\text{rf},t}$, endogenous $V_t$, `study_config`.
*   **Processes:** Computes the market impact coefficient and builds symbolic/numerical closures for TC and shorting costs.
*   **Outputs:** $\kappa_{\text{impact},t}$ and four functional closures.
*   **Data Transformation:** Maps raw liquidity vectors into evaluable CVXPY expressions, explicitly enforcing DCP compliance via `cp.abs(z)`.
*   **Role & Equations:** Implements the 3/2-power model: $(\kappa_{\text{impact}})_j = b_j \nu_j / \sqrt{\omega_j/V}$ and $\phi_{\text{tc}}(z) = \frac{1}{2}\kappa_{\text{spread}}^T|z| + \kappa_{\text{impact}}^T|z|^{3/2}$.

### 23. `task_23_orchestrator` (ADMM Scaling Matrix)
*   **Inputs:** $\kappa_{\text{impact},t}$.
*   **Processes:** Computes the square root heuristic and enforces a strict positivity floor.
*   **Outputs:** The 1D diagonal scaling vector $D_t^{\text{scale}}$.
*   **Data Transformation:** Maps the impact coefficient to the ADMM scaling space via element-wise square root.
*   **Role & Equations:** Implements the Hessian-based diagonal scaling heuristic: $D_{jj} = \sqrt{2(\kappa_{\text{impact}})_j}$.

### 24. `solve_pm_local_problem` (PM Local Optimization)
*   **Inputs:** Alpha, risk model, current state, constraints, TC/shorting closures.
*   **Processes:** Constructs the CVXPY problem, applies the factored risk constraint, solves, and extracts optimal vectors.
*   **Outputs:** The `PMOptimizationResult` NamedTuple.
*   **Data Transformation:** Maps numerical parameters into a symbolic convex graph, solves it, and flattens the result back to NumPy arrays.
*   **Role & Equations:** Implements the independent PM objective: $\min -\alpha^T w - r_{\text{rf}}c + \gamma_{\text{risk}}s_{\text{risk}} + \gamma_{\text{turn}}s_{\text{turn}} + \gamma_{\text{tc}}\phi_{\text{tc}}(z) + \gamma_{\text{short}}\phi_{\text{short}}(w)$.

### 25. `solve_admm_pm_update` (ADMM PM Update)
*   **Inputs:** Same as Task 24, plus $\ell^k$, $x^{i,k}$, $D_t^{\text{scale}}$, $\rho$, $\lambda^i$.
*   **Processes:** Modifies the local objective by scaling it by $\lambda^i$ and adding the linear signal and quadratic proximal terms.
*   **Outputs:** The `PMOptimizationResult` containing $x^{i,k+1}$.
*   **Data Transformation:** Modifies the symbolic CVXPY graph to include ADMM penalties, utilizing `warm_start=True` for rapid convergence.
*   **Role & Equations:** Implements Algorithm 1, Step 1: $x^{i,k+1} = \arg\min_x \left( \lambda^i f^i(x) + \lambda^i (\ell^k)^T Dx + \frac{\rho}{2} \|\lambda^i D(x - x^{i,k})\|_2^2 \right)$.

### 26. `task_26_orchestrator` (Independent Protocol Runner)
*   **Inputs:** `FirmState`, date $t$, exogenous data, `study_config`.
*   **Processes:** Computes endogenous TC parameters, executes independent PM solves, aggregates the firm net trade, and evaluates costs.
*   **Outputs:** The standardized `ProtocolRunOutput` dataclass.
*   **Data Transformation:** Aggregates individual optimal trades into a NAV-weighted firm net trade vector.
*   **Role & Equations:** Implements the baseline protocol where PMs ignore firm-level netting: $z_t^{\text{firm}} = \sum_{i=1}^M \lambda_t^{(i)} x_t^{(i)}$.

### 27. `task_27_orchestrator` (Full Cooperative Protocol Runner)
*   **Inputs:** `FirmState`, date $t$, exogenous data, `study_config`.
*   **Processes:** Constructs a massive joint CVXPY problem, replacing local TC/shorting costs with firm-level equivalents on the net trade, solves, and extracts.
*   **Outputs:** The standardized `ProtocolRunOutput` dataclass.
*   **Data Transformation:** Couples $M$ independent variable sets into a single optimization graph via the consensus constraint.
*   **Role & Equations:** Implements the theoretical joint optimum (Equation 3): $\min \sum_{i=1}^M \lambda_i f_i(x_i) + \gamma_{\text{tc}}\phi_{\text{tc}}(z)$ subject to $z = \sum \lambda_i x_i$.

### 28. `task_28_orchestrator` (ADMM Initialization)
*   **Inputs:** `FirmState`, date $t$, exogenous data, `study_config`.
*   **Processes:** Computes standalone initial trades $x^{i,0}$, initializes $u^0=\mathbf{0}$ and $z_{\text{sum}}^0$, and packages the state.
*   **Outputs:** The `ADMMState` dataclass and required closures.
*   **Data Transformation:** Maps independent solves into the starting state for the iterative ADMM loop.
*   **Role & Equations:** Implements Algorithm 1 Initialization: $x^{i,0} = \arg\min f^i(x)$ and $z_{\text{sum}}^0 = \sum \lambda_i x^{i,0}$.

### 29. `task_29_orchestrator` (ADMM Iteration Loop)
*   **Inputs:** `ADMMState`, `FirmState`, date $t$, exogenous data, closures, $K$.
*   **Processes:** Loops $K$ times, computing $\ell^k$, executing PM updates, solving the planner subproblem for $z_{\text{sum}}^{k+1}$, and updating $u^{k+1}$.
*   **Outputs:** The `ProtocolRunOutput` containing the final executed trades and the diagnostic trace.
*   **Data Transformation:** Iteratively refines trade vectors and dual shadow prices, converging toward the joint optimum.
*   **Role & Equations:** Implements the core of Algorithm 1, including the planner update: $z_{\text{sum}}^{k+1} = \arg\min_z (g(z) - (u^k)^T Dz + \frac{\rho}{2M} \|Dz - D\sum \lambda_i x^{i,k+1}\|_2^2)$.

### 30. `task_30_orchestrator` (State Transition Layer)
*   **Inputs:** `ProtocolRunOutput`, `FirmState`, date $t$, date $t+1$, returns, RF rate.
*   **Processes:** Verifies trade identities, computes portfolio returns, updates NAVs, drifts weights, and recomputes $\lambda$.
*   **Outputs:** The mutated `FirmState` object for $t+1$.
*   **Data Transformation:** Maps post-trade weights at $t$ to pre-trade weights at $t+1$ using realized asset returns.
*   **Role & Equations:** Implements the walk-forward accounting mechanics: $V_{t+1}^{(i)} = V_t^{(i)}(1 + r_{\text{portfolio},t}^{(i)})$ and $\lambda_{t+1}^{(i)} = V_{t+1}^{(i)} / \sum V_{t+1}^{(j)}$.

### 31. `task_31_orchestrator` (Input & Estimation Layer Orchestrator)
*   **Inputs:** The five raw input objects.
*   **Processes:** Chains Tasks 1 through 21 (Validation, Pre-estimation, Estimation).
*   **Outputs:** The fully populated `artifact_registry`.
*   **Data Transformation:** Orchestrates the transformation of raw vendor data into the complete set of exogenous parameters required for simulation.
*   **Role & Equations:** Serves as the master controller for the data engineering and econometric modeling phases of the research pipeline.

### 32. `task_32_orchestrator` (Protocol & Evaluation Layer Orchestrator)
*   **Inputs:** The `artifact_registry`.
*   **Processes:** Executes the walk-forward loop for all 4 protocols, computes performance metrics, and packages the artifacts.
*   **Outputs:** The `reproduction_package` dictionary.
*   **Data Transformation:** Orchestrates the transformation of static parameters into dynamic historical trajectories and summary statistics.
*   **Role & Equations:** Serves as the master controller for the simulation and evaluation phases, computing the final Sharpe ratios and cumulative paths.

### 33. `task_33_orchestrator` (Execution & Diagnostics Persistence)
*   **Inputs:** The five raw input objects and an output directory.
*   **Processes:** Invokes Tasks 31 and 32, extracts ADMM traces, computes residual summaries, and persists traces to `.npz` files.
*   **Outputs:** The finalized `reproduction_package`.
*   **Data Transformation:** Serializes massive in-memory trace lists into compressed disk archives.
*   **Role & Equations:** Ensures that the convergence diagnostics (primal and dual residuals) of the ADMM algorithm are permanently recorded for analysis.

### 34. `task_34_orchestrator` (Performance Metrics & Fidelity Classification)
*   **Inputs:** The `reproduction_package`.
*   **Processes:** Compares firm metrics against manuscript values, verifies PM qualitative claims, and assigns a fidelity classification.
*   **Outputs:** The augmented `reproduction_package`.
*   **Data Transformation:** Reduces performance tables into boolean verification flags and a discrete classification string.
*   **Role & Equations:** Implements the scientific validation of the reproduction, explicitly testing the claims made in Appendix B regarding indeterminate PM performance.

### 35. `task_35_orchestrator` (Archival & Fingerprinting)
*   **Inputs:** The `reproduction_package`, raw inputs, and output directory.
*   **Processes:** Serializes all artifacts to Parquet/JSON/NPZ, writes the manifest, computes SHA-256 hashes, and creates an immutable `.tar.gz` tarball.
*   **Outputs:** The file path to the tarball and the master fingerprint string.
*   **Data Transformation:** Transforms the entire Python memory state into a single, cryptographically verifiable binary file.
*   **Role & Equations:** Guarantees the absolute reproducibility and immutability of the research findings.

### 36. `run_distributed_cooperative_optimization_pipeline` (Top-Level Orchestrator)
*   **Inputs:** The five raw input objects and a base output directory.
*   **Processes:** Chains Tasks 33, 34, and 35 in a strict, non-interruptible sequence, handling early-exit validation failures gracefully.
*   **Outputs:** A summary dictionary containing the archive path, fingerprint, and fidelity classification.
*   **Data Transformation:** The ultimate function that maps raw market data into a finalized, peer-reviewable research archive.
*   **Role & Equations:** The definitive entry point for the entire "Distributed Method for Cooperative Transaction Cost Mitigation" codebase.

<br><br>

## **Usage Example**

In this exposition, we will operationalize the theoretical constructs of the "Distributed Method for Cooperative Transaction Cost Mitigation" framework. This granular, step-by-step guide demonstrates how to synthetically generate the requisite high-dimensional market data, ingest the YAML configuration, and execute the top-level research pipeline.

*Note: In accordance with the operational parameters of this environment, we assume that all callables reside within a single, contiguous Jupyter notebook namespace. Consequently, no external `.py` module imports are required for the pipeline functions. We further assume that the `config.yaml` file is saved in the current working directory.*

### **Step 1: Synthetic Data Generation (`raw_market_df`, `risk_free_rate_series`, `master_trading_calendar`, `asset_identifier_map`)**

To rigorously test the pipeline without violating proprietary data licenses (e.g., LSEG/Refinitiv or CRSP), we must construct a synthetic market environment. This environment must strictly adhere to the structural schemas enforced by Task 1 and exhibit mathematically plausible financial dynamics to prevent the pipeline's anomaly detection heuristics (e.g., negative prices, extreme returns) from halting execution.

**Methodology:**
1.  **Temporal Backbone:** We generate a strictly increasing business-day calendar from July 2000 to April 2025.
2.  **Asset Universe:** We instantiate $N=434$ unique assets, mapping internal integer IDs to synthetic vendor tickers.
3.  **Price Simulation:** We simulate asset trajectories using a vectorized, discretized Geometric Brownian Motion (GBM). This ensures prices remain strictly positive and returns exhibit realistic volatility ($\sigma \approx 2\%$ daily).
4.  **Microstructure Simulation:** Bid and ask prices are derived by applying a synthetic fractional spread to the GBM prices. Volumes are drawn from a log-normal distribution.
5.  **Schema Enforcement:** All columns are explicitly cast to the exact `numpy` dtypes required by the pipeline's structural validation layer.

```python
import pandas as pd
import numpy as np
import yaml
from typing import Dict, Any, Tuple

# Initialize isolated random generator for deterministic synthetic data generation
rng = np.random.default_rng(seed=42)

def generate_synthetic_market_environment(
    n_assets: int = 434,
    start_date: str = "2000-07-03",
    end_date: str = "2025-04-01"
) -> Tuple[pd.DataFrame, pd.Series, pd.DatetimeIndex, pd.DataFrame]:
    """
    Generates a high-fidelity, mathematically plausible synthetic market environment.

    Purpose:
        To construct the four primary data structures required by the cooperative
        transaction cost mitigation pipeline, strictly adhering to the schema
        and numerical invariants expected by the validation layer.

    Inputs:
        n_assets (int): The cross-sectional dimension of the universe (N).
        start_date (str): The start date of the simulation in 'YYYY-MM-DD' format.
        end_date (str): The end date of the simulation in 'YYYY-MM-DD' format.

    Processes:
        1. Calendar Generation: Creates a business-day DatetimeIndex.
        2. Identifier Mapping: Generates a DataFrame linking asset_id to vendor_id.
        3. Macro Series: Simulates a mean-reverting risk-free rate series.
        4. Panel Simulation: Uses Geometric Brownian Motion to simulate prices,
           applies synthetic spreads, and generates log-normal volumes.
        5. Schema Enforcement: Casts all arrays to strict dtypes and constructs
           the MultiIndex DataFrame.

    Outputs:
        Tuple containing:
            - raw_market_df (pd.DataFrame): The MultiIndex market panel.
            - risk_free_rate_series (pd.Series): The macro-financial scalar series.
            - master_trading_calendar (pd.DatetimeIndex): The temporal backbone.
            - asset_identifier_map (pd.DataFrame): The identity-resolution layer.
    """
    print("Initializing synthetic market environment generation...")

    # 1. Generate Master Trading Calendar (Business Days)
    master_trading_calendar = pd.date_range(start=start_date, end=end_date, freq='B')
    t_days = len(master_trading_calendar)

    # 2. Generate Asset Identifier Map
    asset_ids = np.arange(1, n_assets + 1, dtype=np.int64)
    tickers = [f"SYM{i:04d}" for i in asset_ids]
    vendor_ids = [f"VND{i:08d}" for i in asset_ids]
    
    asset_identifier_map = pd.DataFrame({
        "asset_id": asset_ids,
        "ticker": tickers,
        "vendor_id": vendor_ids
    })

    # 3. Generate Risk-Free Rate Series (Ornstein-Uhlenbeck process approximation)
    # Mean ~ 2.0% annualized, bounded to prevent negative rates
    rf_noise = rng.normal(loc=0.0, scale=0.001, size=t_days)
    rf_rates = np.clip(0.02 + np.cumsum(rf_noise), a_min=0.0001, a_max=0.10)
    risk_free_rate_series = pd.Series(
        data=rf_rates,
        index=master_trading_calendar,
        name="risk_free_rate"
    ).astype(np.float64)

    # 4. Simulate Market Panel via Geometric Brownian Motion (GBM)
    print(f"Simulating GBM trajectories for {n_assets} assets over {t_days} days...")
    
    # Daily drift ~ 0.02%, Daily Volatility ~ 2.0%
    mu, sigma = 0.0002, 0.02
    
    # Generate return shocks: shape (T, N)
    daily_shocks = rng.normal(loc=mu, scale=sigma, size=(t_days, n_assets))
    
    # Compute cumulative returns and exponentiate to get price paths
    cumulative_returns = np.cumsum(daily_shocks, axis=0)
    initial_prices = rng.uniform(10.0, 150.0, size=n_assets)
    trade_prices = initial_prices * np.exp(cumulative_returns)
    
    # Simulate fractional bid-ask spreads (e.g., 5 to 20 basis points)
    spread_fractions = rng.uniform(0.0005, 0.0020, size=(t_days, n_assets))
    half_spreads = trade_prices * (spread_fractions / 2.0)
    
    bid_prices = trade_prices - half_spreads
    ask_prices = trade_prices + half_spreads
    
    # Simulate share volume (Log-normal distribution)
    share_volumes = np.floor(rng.lognormal(mean=14.0, sigma=1.0, size=(t_days, n_assets)))
    
    # Simulate market capitalization (Price * Shares Outstanding)
    # Assume shares outstanding is roughly 100x daily volume
    shares_outstanding = share_volumes * rng.uniform(50, 150, size=(t_days, n_assets))
    market_caps = trade_prices * shares_outstanding

    # 5. Construct the MultiIndex DataFrame
    print("Constructing MultiIndex panel and enforcing strict schemas...")
    
    # Create the Cartesian product of dates and asset_ids
    multi_index = pd.MultiIndex.from_product(
        [master_trading_calendar, asset_ids],
        names=["date", "asset_id"]
    )
    
    # Flatten the 2D arrays to 1D for DataFrame construction (Fortran order to match MultiIndex)
    raw_market_df = pd.DataFrame({
        "ticker": np.tile(tickers, t_days),
        "adjusted_trade_price": trade_prices.flatten(),
        "adjusted_bid_price": bid_prices.flatten(),
        "adjusted_ask_price": ask_prices.flatten(),
        "share_volume": share_volumes.flatten().astype(np.int64),
        "market_cap_usd": market_caps.flatten(),
        "is_historical_sp500_constituent": np.ones(t_days * n_assets, dtype=bool),
        "is_primary_observation": np.ones(t_days * n_assets, dtype=bool),
        "currency": np.full(t_days * n_assets, "USD", dtype=object),
        "data_quality_flag": np.full(t_days * n_assets, "OK", dtype=object)
    }, index=multi_index)

    # Enforce strict dtypes
    raw_market_df = raw_market_df.astype({
        "ticker": "string",
        "adjusted_trade_price": "float64",
        "adjusted_bid_price": "float64",
        "adjusted_ask_price": "float64",
        "share_volume": "int64",
        "market_cap_usd": "float64",
        "is_historical_sp500_constituent": "bool",
        "is_primary_observation": "bool",
        "currency": "string",
        "data_quality_flag": "string"
    })

    print("Synthetic market environment generated successfully.")
    return raw_market_df, risk_free_rate_series, master_trading_calendar, asset_identifier_map

# Instantiate the synthetic data structures
(
    raw_market_df,
    risk_free_rate_series,
    master_trading_calendar,
    asset_identifier_map
) = generate_synthetic_market_environment()
```

### **Step 2: Loading the Configuration (`config.yaml`)**

The pipeline's execution is strictly governed by the deterministic parameters defined in the `config.yaml` file. We must read this file from the disk and parse it into a Python dictionary, ensuring that the hierarchical structure and data types are preserved.

**Methodology:**
1.  **File I/O:** Open `config.yaml` in read mode.
2.  **Parsing:** Utilize `yaml.safe_load` to securely deserialize the YAML document into a nested Python dictionary.
3.  **Validation:** Implement a `try-except` block to catch `FileNotFoundError` or `yaml.YAMLError`, ensuring the pipeline fails gracefully if the blueprint is missing or malformed.

```python
def load_study_configuration(filepath: str = "config.yaml") -> Dict[str, Any]:
    """
    Loads the study configuration parameters from a YAML file into a Python dictionary.

    Purpose:
        To ingest the deterministic hyperparameters, institutional constraints, and
        econometric specifications defined in the external configuration file.

    Inputs:
        filepath (str): The relative or absolute path to the YAML configuration file.
                        Default is "config.yaml".

    Processes:
        1. File Access: Attempts to open the specified file in read mode.
        2. Parsing: Uses PyYAML's safe_load to parse the YAML structure.
        3. Validation: Catches file existence errors and parsing errors.

    Outputs:
        Dict[str, Any]: A nested dictionary containing the study configuration.

    Raises:
        FileNotFoundError: If the config.yaml file is not found in the working directory.
        yaml.YAMLError: If the file contains invalid YAML syntax.
    """
    print(f"Attempting to load configuration from {filepath}...")
    
    try:
        # Open the file stream and parse the YAML content safely
        with open(filepath, "r") as file:
            config = yaml.safe_load(file)
            
        print("Configuration loaded successfully.")
        return config
        
    except FileNotFoundError:
        error_msg = f"Critical Error: {filepath} not found. The pipeline cannot proceed without the configuration blueprint."
        print(error_msg)
        raise
    except yaml.YAMLError as e:
        error_msg = f"Critical Error parsing YAML file {filepath}: {e}"
        print(error_msg)
        raise

# Load the configuration into memory
study_config = load_study_configuration("config.yaml")
```

### **Step 3: Executing the Pipeline (`run_distributed_cooperative_optimization_pipeline`)**

With the raw data structures and the configuration dictionary instantiated in memory, we invoke the top-level orchestrator. This function manages the entire lifecycle: structural validation, data cleansing, risk model estimation, VAR(1) alpha simulation, ADMM protocol execution, performance evaluation, and immutable archival.

**Methodology:**
1.  **Function Call:** Pass the five primary objects to `run_distributed_cooperative_optimization_pipeline`.
2.  **Output Handling:** The function returns a summary dictionary containing the final `reproduction_package`, the path to the `.tar.gz` archive, and the cryptographic fingerprint.
3.  **Inspection:** We extract and print the fidelity classification and the firm-level performance table to verify the success of the cooperative mitigation protocol.

```python
# ==============================================================================
# Execution of the End-to-End Study Pipeline
# ==============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("INITIATING DISTRIBUTED COOPERATIVE OPTIMIZATION PIPELINE")
    print("="*80)
    
    try:
        # Execute the top-level orchestrator
        # This will trigger the cascade of Tasks 1 through 35
        pipeline_summary = run_distributed_cooperative_optimization_pipeline(
            raw_market_df=raw_market_df,
            risk_free_rate_series=risk_free_rate_series,
            master_trading_calendar=master_trading_calendar,
            asset_identifier_map=asset_identifier_map,
            study_config=study_config,
            output_base_dir="./institutional_research_archive"
        )
        
        # ==============================================================================
        # Inspecting the Outputs
        # ==============================================================================
        
        # Check if the pipeline halted early due to validation failures
        if "validation_report" in pipeline_summary:
            print("\nPIPELINE HALTED DURING VALIDATION.")
            print("Please review the validation report for structural or semantic errors.")
        else:
            print("\n" + "="*80)
            print("PIPELINE EXECUTION COMPLETE")
            print("="*80)
            
            # 1. Extract Archival Metadata
            print(f"\n[Archival Information]")
            print(f"Archive Path: {pipeline_summary['archive_path']}")
            print(f"Master Fingerprint (SHA-256): {pipeline_summary['master_fingerprint']}")
            print(f"Total Execution Time: {pipeline_summary['total_execution_time']:.2f} seconds")
            
            # 2. Extract Fidelity Classification
            print(f"\n[Reproduction Fidelity]")
            print(f"Classification: {pipeline_summary['fidelity_classification']}")
            
            # 3. Extract Firm-Level Performance Metrics
            # This table demonstrates the transaction cost savings achieved via ADMM
            reproduction_package = pipeline_summary["reproduction_package"]
            firm_table = reproduction_package["performance_metrics"]["firm_table"]
            
            print(f"\n[Firm-Level Performance Metrics]")
            # Format the DataFrame for professional console output
            formatted_table = firm_table.copy()
            formatted_table["Return"] = (formatted_table["Return"] * 100).map("{:.2f}%".format)
            formatted_table["Volatility"] = (formatted_table["Volatility"] * 100).map("{:.2f}%".format)
            formatted_table["Sharpe"] = formatted_table["Sharpe"].map("{:.2f}".format)
            
            print(formatted_table.to_string())
            
    except Exception as e:
        print(f"\nPipeline execution failed with an unhandled exception: {e}")
```


<br><br>

## **Implemented Callables**

In [ ]:
# Task 1. Validate the structural integrity of all five input objects

class InputSchemaError(Exception):
    """Custom exception raised when input data structures violate the required schema."""
    pass

# ==============================================================================
# Task 1: Validate the structural integrity of all five input objects
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 1, Step 1: Validate the raw market DataFrame
# -------------------------------------------------------------------------------------------------------------------------------
def validate_raw_market_df(raw_market_df: pd.DataFrame) -> pd.DataFrame:
    """
    Validates the structural integrity of the raw market panel.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.

    Returns
    -------
    pd.DataFrame
        The validated and potentially normalized (e.g., sorted, tz-stripped) market DataFrame.

    Raises
    ------
    InputSchemaError
        If the DataFrame violates the required MultiIndex structure, level types, or column schema.
    """
    # Verify that the input is strictly a pandas DataFrame
    if not isinstance(raw_market_df, pd.DataFrame):
        raise InputSchemaError(f"raw_market_df must be a pd.DataFrame, got {type(raw_market_df)}")

    # Verify that the index is a MultiIndex
    if not isinstance(raw_market_df.index, pd.MultiIndex):
        raise InputSchemaError("raw_market_df must have a pd.MultiIndex.")

    # Verify that the MultiIndex has exactly two levels
    if raw_market_df.index.nlevels != 2:
        raise InputSchemaError(f"raw_market_df index must have exactly 2 levels, got {raw_market_df.index.nlevels}.")

    # Verify the exact names and order of the MultiIndex levels
    if list(raw_market_df.index.names) != ["date", "asset_id"]:
        raise InputSchemaError(f"raw_market_df index names must be ['date', 'asset_id'], got {list(raw_market_df.index.names)}.")

    # Extract the date level for type inspection
    date_level = raw_market_df.index.get_level_values("date")

    # Check if the date level is a datetime type
    if not pd.api.types.is_datetime64_any_dtype(date_level):
        raise InputSchemaError(f"The 'date' level must be datetime64, got {date_level.dtype}.")

    # Normalize timezone-aware datetimes to timezone-naive UTC to prevent downstream join failures
    if pd.api.types.is_datetime64tz_dtype(date_level):
        logger.info("Normalizing tz-aware 'date' level to tz-naive UTC.")
        # Reset index, strip timezone, and set index back to maintain structural integrity
        raw_market_df = raw_market_df.reset_index()
        raw_market_df["date"] = raw_market_df["date"].dt.tz_convert("UTC").dt.tz_localize(None)
        raw_market_df = raw_market_df.set_index(["date", "asset_id"])

    # Extract the asset_id level for type inspection
    asset_id_level = raw_market_df.index.get_level_values("asset_id")

    # Verify that the asset_id level contains no null values
    if asset_id_level.isna().any():
        raise InputSchemaError("The 'asset_id' level contains null values, which is strictly prohibited.")

    # Define the exact required column schema
    required_columns = [
        "ticker", "adjusted_trade_price", "adjusted_bid_price", "adjusted_ask_price",
        "share_volume", "market_cap_usd", "is_historical_sp500_constituent",
        "is_primary_observation", "currency", "data_quality_flag"
    ]

    # Compute the set difference to find any missing columns
    missing_cols = set(required_columns) - set(raw_market_df.columns)
    if missing_cols:
        raise InputSchemaError(f"raw_market_df is missing required columns: {missing_cols}")

    # Verify that the index is lexicographically sorted for efficient downstream slicing
    if not raw_market_df.index.is_monotonic_increasing:
        logger.info("raw_market_df MultiIndex is not sorted. Sorting lexicographically.")
        raw_market_df = raw_market_df.sort_index()

    return raw_market_df

# -------------------------------------------------------------------------------------------------------------------------------
# Task 1, Step 2: Validate the risk-free rate series and master trading calendar
# -------------------------------------------------------------------------------------------------------------------------------
def validate_macro_and_calendar(
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex
) -> Tuple[pd.Series, pd.DatetimeIndex]:
    """
    Validates the structural integrity of the macro-financial series and the trading calendar.

    Parameters
    ----------
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    Tuple[pd.Series, pd.DatetimeIndex]
        The validated risk-free rate series and master trading calendar.

    Raises
    ------
    InputSchemaError
        If either object violates its required structural schema.
    """
    # Verify that the risk-free rate input is strictly a pandas Series
    if not isinstance(risk_free_rate_series, pd.Series):
        raise InputSchemaError(f"risk_free_rate_series must be a pd.Series, got {type(risk_free_rate_series)}")

    # Attempt to cast the series index to a DatetimeIndex if it is not already one
    if not isinstance(risk_free_rate_series.index, pd.DatetimeIndex):
        logger.info("Converting risk_free_rate_series index to DatetimeIndex.")
        try:
            risk_free_rate_series.index = pd.DatetimeIndex(risk_free_rate_series.index)
        except Exception as e:
            raise InputSchemaError(f"Failed to convert risk_free_rate_series index to DatetimeIndex: {e}")

    # Normalize timezone-aware datetimes in the risk-free rate series to tz-naive UTC
    if pd.api.types.is_datetime64tz_dtype(risk_free_rate_series.index):
        logger.info("Normalizing tz-aware risk_free_rate_series index to tz-naive UTC.")
        risk_free_rate_series.index = risk_free_rate_series.index.tz_convert("UTC").tz_localize(None)

    # Enforce the index name convention for the risk-free rate series
    if risk_free_rate_series.index.name != "date":
        logger.warning(f"risk_free_rate_series index name is '{risk_free_rate_series.index.name}', setting to 'date'.")
        risk_free_rate_series.index.name = "date"

    # Enforce the series name convention
    if risk_free_rate_series.name != "risk_free_rate":
        raise InputSchemaError(f"risk_free_rate_series name must be 'risk_free_rate', got '{risk_free_rate_series.name}'.")

    # Verify that the risk-free rate values are numeric
    if not pd.api.types.is_numeric_dtype(risk_free_rate_series.dtype):
        raise InputSchemaError(f"risk_free_rate_series must have a numeric dtype, got {risk_free_rate_series.dtype}.")

    # Verify that the master trading calendar is strictly a pandas DatetimeIndex
    if not isinstance(master_trading_calendar, pd.DatetimeIndex):
        raise InputSchemaError(f"master_trading_calendar must be a pd.DatetimeIndex, got {type(master_trading_calendar)}")

    # Normalize timezone-aware datetimes in the calendar to tz-naive UTC
    if pd.api.types.is_datetime64tz_dtype(master_trading_calendar):
        logger.info("Normalizing tz-aware master_trading_calendar to tz-naive UTC.")
        master_trading_calendar = master_trading_calendar.tz_convert("UTC").tz_localize(None)

    # Verify strict monotonic increase of the calendar
    if not master_trading_calendar.is_monotonic_increasing:
        logger.info("master_trading_calendar is not sorted. Sorting now.")
        master_trading_calendar = master_trading_calendar.sort_values()

    # Verify that the calendar contains no duplicate dates
    if master_trading_calendar.duplicated().any():
        raise InputSchemaError("master_trading_calendar contains duplicate dates, which is strictly prohibited.")

    return risk_free_rate_series, master_trading_calendar

# -------------------------------------------------------------------------------------------------------------------------------
# Task 1, Step 3: Validate the identifier map and the study configuration dictionary
# -------------------------------------------------------------------------------------------------------------------------------
def validate_identifier_map_and_config(
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Validates the structural integrity of the asset identifier map and the study configuration dictionary.

    Parameters
    ----------
    asset_identifier_map : pd.DataFrame
        The permanent identifier map linking internal asset IDs to vendor identifiers.
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Returns
    -------
    Tuple[pd.DataFrame, Dict[str, Any]]
        The validated identifier map and configuration dictionary.

    Raises
    ------
    InputSchemaError
        If either object violates its required structural schema.
    """
    # Verify that the identifier map is strictly a pandas DataFrame
    if not isinstance(asset_identifier_map, pd.DataFrame):
        raise InputSchemaError(f"asset_identifier_map must be a pd.DataFrame, got {type(asset_identifier_map)}")

    # Rescue 'asset_id' if it was accidentally set as the DataFrame index
    if asset_identifier_map.index.name == "asset_id" and "asset_id" not in asset_identifier_map.columns:
        logger.info("asset_id found in index of asset_identifier_map. Resetting index.")
        asset_identifier_map = asset_identifier_map.reset_index()

    # Define the exact required columns for the identifier map
    required_map_cols = {"asset_id", "ticker", "vendor_id"}

    # Verify that the identifier map contains exactly the required columns
    if set(asset_identifier_map.columns) != required_map_cols:
        raise InputSchemaError(f"asset_identifier_map columns must be exactly {required_map_cols}, got {set(asset_identifier_map.columns)}")

    # Reorder columns to the canonical sequence for downstream consistency
    asset_identifier_map = asset_identifier_map[["asset_id", "ticker", "vendor_id"]]

    # Verify that the asset_id column contains no null values
    if asset_identifier_map["asset_id"].isna().any():
        raise InputSchemaError("asset_identifier_map contains null values in the 'asset_id' column.")

    # Verify that every asset_id in the map is strictly unique
    if not asset_identifier_map["asset_id"].is_unique:
        raise InputSchemaError("asset_identifier_map contains duplicate 'asset_id' values.")

    # Verify that the study configuration is strictly a Python dictionary
    if not isinstance(study_config, dict):
        raise InputSchemaError(f"study_config must be a dictionary, got {type(study_config)}")

    # Define the exact required top-level keys for the configuration dictionary
    required_config_keys = {
        "REQUIRED_EXTERNAL_INPUT_OBJECTS", "GLOBAL_FIRM_CONSTANTS", "ASSET_UNIVERSE_CONSTRUCTION",
        "RANDOM_SEED_REGISTRY", "PM_SLEEVE_CONFIGURATIONS", "INSTITUTIONAL_CONSTRAINTS",
        "TC_MODEL_PARAMETERS", "ADMM_HYPERPARAMETERS", "SIMULATION_ECONOMETRICS",
        "RISK_MODEL_SPECIFICATION", "DERIVED_DATA_OBJECTS", "RAW_DATA_SCHEMA",
        "PROTOCOL_REGISTRY", "BACKTEST_STATE_LOGIC", "SOLVER_AND_NUMERICAL_SETTINGS",
        "METHODOLOGY", "VISUALIZATION_PARAMETERS", "PROMPT_TEMPLATES", "LLM_SETTINGS"
    }

    # Compute the set difference to find any missing configuration keys
    missing_config_keys = required_config_keys - set(study_config.keys())
    if missing_config_keys:
        raise InputSchemaError(f"study_config is missing required top-level keys: {missing_config_keys}")

    return asset_identifier_map, study_config

# -------------------------------------------------------------------------------------------------------------------------------
# Task 1, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_1_orchestrator(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[pd.DataFrame, pd.Series, pd.DatetimeIndex, pd.DataFrame, Dict[str, Any]]:
    """
    Orchestrates the structural validation of all five primary input objects.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map linking internal asset IDs to vendor identifiers.
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Returns
    -------
    Tuple[pd.DataFrame, pd.Series, pd.DatetimeIndex, pd.DataFrame, Dict[str, Any]]
        The structurally validated and normalized input objects.

    Raises
    ------
    InputSchemaError
        If any of the input objects violate their required structural schema.
    """
    logger.info("Commencing Task 1: Structural validation of input objects.")

    # Execute Step 1: Validate the raw market panel
    val_raw_market_df = validate_raw_market_df(raw_market_df)
    logger.info("Successfully validated raw_market_df.")

    # Execute Step 2: Validate the macro series and trading calendar
    val_rf_series, val_calendar = validate_macro_and_calendar(
        risk_free_rate_series,
        master_trading_calendar
    )
    logger.info("Successfully validated risk_free_rate_series and master_trading_calendar.")

    # Execute Step 3: Validate the identifier map and configuration dictionary
    val_id_map, val_config = validate_identifier_map_and_config(
        asset_identifier_map,
        study_config
    )
    logger.info("Successfully validated asset_identifier_map and study_config.")

    logger.info("Task 1 complete. All input objects are structurally sound.")

    # Return the validated objects for downstream consumption
    return val_raw_market_df, val_rf_series, val_calendar, val_id_map, val_config


In [ ]:
# Task 2: Validate semantic correctness of all study parameters

class SemanticValidationError(Exception):
    """Custom exception raised when study parameters violate the required semantic values."""
    pass

# ==============================================================================
# Task 2: Validate semantic correctness of all study parameters
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 2, Step 1: Validate firm and institutional constants
# -------------------------------------------------------------------------------------------------------------------------------
def validate_firm_and_institutional_constants(study_config: Dict[str, Any]) -> None:
    """
    Extracts and verifies exact scalar values for global firm constants and institutional constraints.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Raises
    ------
    SemanticValidationError
        If any parameter is missing, of the wrong type, or deviates from the manuscript's exact values.
    """
    # Define the expected values for GLOBAL_FIRM_CONSTANTS
    expected_firm_constants = {
        "M_portfolio_managers": 4,
        "N_assets": 434,
        "gamma_tc": 0.15,
        "gamma_risk": 20.0,
        "gamma_turnover": 1.0,
        "gamma_shorting": 1.0
    }

    # Extract the firm constants section
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS")
    if firm_consts is None:
        raise SemanticValidationError("Missing 'GLOBAL_FIRM_CONSTANTS' section in study_config.")

    # Validate each firm constant
    for key, expected_val in expected_firm_constants.items():
        if key not in firm_consts:
            raise SemanticValidationError(f"Missing key '{key}' in GLOBAL_FIRM_CONSTANTS.")

        actual_val = firm_consts[key]

        # Handle integer validation with safe casting for floats that are exactly integers
        if isinstance(expected_val, int):
            if isinstance(actual_val, float) and actual_val.is_integer():
                logger.info(f"Casting float {actual_val} to int for parameter '{key}'.")
                actual_val = int(actual_val)
            if not isinstance(actual_val, int) or actual_val != expected_val:
                raise SemanticValidationError(
                    f"Parameter '{key}' must be exactly integer {expected_val}, got {actual_val} of type {type(actual_val)}."
                )
        # Handle float validation using strict tolerance
        elif isinstance(expected_val, float):
            if not isinstance(actual_val, (float, int)):
                raise SemanticValidationError(f"Parameter '{key}' must be numeric, got {type(actual_val)}.")
            if not math.isclose(actual_val, expected_val, rel_tol=0.0, abs_tol=1e-12):
                raise SemanticValidationError(
                    f"Parameter '{key}' must be exactly {expected_val}, got {actual_val}."
                )

    # Define the expected values for INSTITUTIONAL_CONSTRAINTS
    expected_inst_constraints = {
        "leverage_limit_L": 1.5,
        "concentration_limit_C": 0.2,
        "shorting_limit_S": 0.5,
        "turnover_limit_T": 0.2
    }

    # Extract the institutional constraints section
    inst_consts = study_config.get("INSTITUTIONAL_CONSTRAINTS")
    if inst_consts is None:
        raise SemanticValidationError("Missing 'INSTITUTIONAL_CONSTRAINTS' section in study_config.")

    # Validate each institutional constraint
    for key, expected_val in expected_inst_constraints.items():
        if key not in inst_consts:
            raise SemanticValidationError(f"Missing key '{key}' in INSTITUTIONAL_CONSTRAINTS.")

        actual_val = inst_consts[key]

        if not isinstance(actual_val, (float, int)):
            raise SemanticValidationError(f"Parameter '{key}' must be numeric, got {type(actual_val)}.")
        if not math.isclose(actual_val, expected_val, rel_tol=0.0, abs_tol=1e-12):
            raise SemanticValidationError(
                f"Parameter '{key}' must be exactly {expected_val}, got {actual_val}."
            )

# -------------------------------------------------------------------------------------------------------------------------------
# Task 2, Step 2: Validate ADMM, risk model, and simulation parameters
# -------------------------------------------------------------------------------------------------------------------------------
def validate_admm_risk_and_simulation_parameters(study_config: Dict[str, Any]) -> None:
    """
    Extracts and verifies exact scalar values for ADMM hyperparameters, risk model specs, and simulation econometrics.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Raises
    ------
    SemanticValidationError
        If any parameter deviates from the manuscript's exact values or structural requirements.
    """
    # Validate ADMM_HYPERPARAMETERS
    admm_params = study_config.get("ADMM_HYPERPARAMETERS")
    if admm_params is None:
        raise SemanticValidationError("Missing 'ADMM_HYPERPARAMETERS' section in study_config.")

    # Check rho penalty
    rho = admm_params.get("rho_penalty")
    if not isinstance(rho, (float, int)) or not math.isclose(rho, 10.0, abs_tol=1e-12):
        raise SemanticValidationError(f"'rho_penalty' must be exactly 10.0, got {rho}.")

    # Check dual extrapolation parameter
    varphi = admm_params.get("varphi_dual_extrapolation")
    if not isinstance(varphi, (float, int)) or not math.isclose(varphi, 1.0, abs_tol=1e-12):
        raise SemanticValidationError(f"'varphi_dual_extrapolation' must be exactly 1.0, got {varphi}.")

    # Check iteration counts compared
    k_counts = admm_params.get("iteration_counts_compared")
    if not isinstance(k_counts, list) or k_counts != [2, 5]:
        raise SemanticValidationError(f"'iteration_counts_compared' must be exactly the list [2, 5], got {k_counts}.")

    # Validate RISK_MODEL_SPECIFICATION
    risk_params = study_config.get("RISK_MODEL_SPECIFICATION")
    if risk_params is None:
        raise SemanticValidationError("Missing 'RISK_MODEL_SPECIFICATION' section in study_config.")

    # Check J factors
    j_factors = risk_params.get("J_factors")
    if not isinstance(j_factors, int) or j_factors != 15:
        raise SemanticValidationError(f"'J_factors' must be exactly integer 15, got {j_factors}.")

    # Check forward window days
    risk_window = risk_params.get("forward_window_days")
    if not isinstance(risk_window, int) or risk_window != 42:
        raise SemanticValidationError(f"'forward_window_days' in risk model must be exactly 42, got {risk_window}.")

    # Check winsorization standard deviations
    winsor_std = risk_params.get("winsorization_std")
    if not isinstance(winsor_std, (float, int)) or not math.isclose(winsor_std, 4.2, abs_tol=1e-12):
        raise SemanticValidationError(f"'winsorization_std' must be exactly 4.2, got {winsor_std}.")

    # Validate SIMULATION_ECONOMETRICS
    sim_params = study_config.get("SIMULATION_ECONOMETRICS")
    if sim_params is None:
        raise SemanticValidationError("Missing 'SIMULATION_ECONOMETRICS' section in study_config.")

    # Check cross-strategy noise correlation
    cross_corr = sim_params.get("cross_strategy_noise_correlation")
    if not isinstance(cross_corr, (float, int)) or not math.isclose(cross_corr, 0.3, abs_tol=1e-12):
        raise SemanticValidationError(f"'cross_strategy_noise_correlation' must be exactly 0.3, got {cross_corr}.")

    # Check forward return window days
    sim_window = sim_params.get("forward_return_window_days")
    if not isinstance(sim_window, int) or sim_window != 42:
        raise SemanticValidationError(f"'forward_return_window_days' in simulation must be exactly 42, got {sim_window}.")

    # Cross-validate the two window parameters to ensure temporal consistency
    if risk_window != sim_window:
        raise SemanticValidationError(
            f"Window mismatch: risk model uses {risk_window} days, simulation uses {sim_window} days. Both must be 42."
        )

# -------------------------------------------------------------------------------------------------------------------------------
# Task 2, Step 3: Validate protocol registry and LLM settings
# -------------------------------------------------------------------------------------------------------------------------------
def validate_protocol_registry_and_llm_settings(study_config: Dict[str, Any]) -> None:
    """
    Confirms the exact presence of the four evaluated protocols and the strict absence of LLM components.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Raises
    ------
    SemanticValidationError
        If the protocol registry is malformed or if LLM settings are not strictly None.
    """
    # Validate PROTOCOL_REGISTRY
    protocol_registry = study_config.get("PROTOCOL_REGISTRY")
    if protocol_registry is None or not isinstance(protocol_registry, dict):
        raise SemanticValidationError("Missing or invalid 'PROTOCOL_REGISTRY' section in study_config.")

    # Define the exact required protocol keys
    expected_protocols = {"independent", "full_cooperative", "admm_2_iter", "admm_5_iter"}
    actual_protocols = set(protocol_registry.keys())

    # Enforce strict set equality for protocol keys
    if actual_protocols != expected_protocols:
        raise SemanticValidationError(
            f"'PROTOCOL_REGISTRY' keys must be exactly {expected_protocols}, got {actual_protocols}."
        )

    # Validate LLM and Prompt settings
    if "PROMPT_TEMPLATES" not in study_config:
        raise SemanticValidationError("Missing 'PROMPT_TEMPLATES' key in study_config.")
    if study_config["PROMPT_TEMPLATES"] is not None:
        raise SemanticValidationError(
            f"'PROMPT_TEMPLATES' must be exactly None, got {type(study_config['PROMPT_TEMPLATES'])}. "
            "This study does not use LLM components."
        )

    if "LLM_SETTINGS" not in study_config:
        raise SemanticValidationError("Missing 'LLM_SETTINGS' key in study_config.")
    if study_config["LLM_SETTINGS"] is not None:
        raise SemanticValidationError(
            f"'LLM_SETTINGS' must be exactly None, got {type(study_config['LLM_SETTINGS'])}. "
            "This study does not use LLM components."
        )

# -------------------------------------------------------------------------------------------------------------------------------
# Task 2, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_2_orchestrator(study_config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Orchestrates the semantic validation of all study parameters.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing all study parameters.

    Returns
    -------
    Dict[str, Any]
        The semantically validated configuration dictionary.

    Raises
    ------
    SemanticValidationError
        If any parameter violates the manuscript's exact semantic requirements.
    """
    logger.info("Commencing Task 2: Semantic validation of study parameters.")

    # Execute Step 1: Validate firm and institutional constants
    validate_firm_and_institutional_constants(study_config)
    logger.info("Successfully validated GLOBAL_FIRM_CONSTANTS and INSTITUTIONAL_CONSTRAINTS.")

    # Execute Step 2: Validate ADMM, risk model, and simulation parameters
    validate_admm_risk_and_simulation_parameters(study_config)
    logger.info("Successfully validated ADMM_HYPERPARAMETERS, RISK_MODEL_SPECIFICATION, and SIMULATION_ECONOMETRICS.")

    # Execute Step 3: Validate protocol registry and LLM settings
    validate_protocol_registry_and_llm_settings(study_config)
    logger.info("Successfully validated PROTOCOL_REGISTRY and confirmed absence of LLM settings.")

    logger.info("Task 2 complete. All study parameters are semantically correct.")

    # Return the validated configuration dictionary
    return study_config


In [ ]:
# Task 3: Validate numerical ranges and cross-input consistency

class CrossInputConsistencyError(Exception):
    """Custom exception raised when referential integrity between input objects fails."""
    pass

class CalendarLengthError(Exception):
    """Custom exception raised when the trading calendar cannot support the forward window."""
    pass

class ValidationStatus(Enum):
    """Enumeration of possible validation statuses for input objects."""
    PASSED = "PASSED"
    FAILED = "FAILED"
    PASSED_WITH_PLACEHOLDER_RISK = "PASSED_WITH_PLACEHOLDER_RISK"

@dataclass(frozen=True)
class ValidationReport:
    """
    A strictly typed, immutable record of the Task 3 validation outcomes.

    Attributes
    ----------
    raw_market_df_status : ValidationStatus
        The validation status of the raw market panel.
    risk_free_rate_series_status : ValidationStatus
        The validation status of the macro-financial series.
    master_trading_calendar_status : ValidationStatus
        The validation status of the trading calendar.
    asset_identifier_map_status : ValidationStatus
        The validation status of the identifier map.
    study_config_status : ValidationStatus
        The validation status of the configuration dictionary.
    numerical_violations : Dict[str, int]
        A mapping of column names to their respective numerical violation counts.
    cross_consistency_flags : Dict[str, Any]
        A mapping of referential integrity checks to their outcomes.
    global_halt_required : bool
        A boolean flag indicating if the pipeline must be terminated.
    """
    raw_market_df_status: ValidationStatus
    risk_free_rate_series_status: ValidationStatus
    master_trading_calendar_status: ValidationStatus
    asset_identifier_map_status: ValidationStatus
    study_config_status: ValidationStatus
    numerical_violations: Dict[str, int]
    cross_consistency_flags: Dict[str, Any]
    global_halt_required: bool

# ==============================================================================
# Task 3: Validate numerical ranges and cross-input consistency
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 3, Step 1: Validate numerical ranges within the raw market panel
# -------------------------------------------------------------------------------------------------------------------------------
def check_numerical_ranges(raw_market_df: pd.DataFrame) -> Dict[str, int]:
    """
    Verifies that price fields are strictly positive and volume/market cap fields are non-negative.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.

    Returns
    -------
    Dict[str, int]
        A dictionary mapping each evaluated column to the count of rows violating the numerical constraints.
    """
    logger.info("Executing Task 3, Step 1: Checking numerical ranges in raw_market_df.")

    # Initialize the dictionary to store violation counts
    violation_counts: Dict[str, int] = {}

    # Define the columns that must be strictly greater than zero
    strictly_positive_cols = [
        "adjusted_trade_price",
        "adjusted_bid_price",
        "adjusted_ask_price"
    ]

    # Evaluate strictly positive constraints
    for col in strictly_positive_cols:
        # Extract the column series for vectorized evaluation
        series = raw_market_df[col]
        # A violation occurs if the value is <= 0, is NaN, or is infinite
        # We use bitwise OR (|) to combine these vectorized boolean masks
        violation_mask = (series <= 0.0) | series.isna() | np.isinf(series)
        # Sum the boolean mask to get the total count of violations
        violation_counts[col] = int(violation_mask.sum())

        if violation_counts[col] > 0:
            logger.warning(f"Found {violation_counts[col]} numerical violations in strictly positive column '{col}'.")

    # Define the columns that must be greater than or equal to zero
    non_negative_cols = [
        "share_volume",
        "market_cap_usd"
    ]

    # Evaluate non-negative constraints
    for col in non_negative_cols:
        # Extract the column series for vectorized evaluation
        series = raw_market_df[col]
        # A violation occurs if the value is < 0, is NaN, or is infinite
        violation_mask = (series < 0.0) | series.isna() | np.isinf(series)
        # Sum the boolean mask to get the total count of violations
        violation_counts[col] = int(violation_mask.sum())

        if violation_counts[col] > 0:
            logger.warning(f"Found {violation_counts[col]} numerical violations in non-negative column '{col}'.")

    # Specifically flag zero-volume entries as they cause singularities in the impact model
    zero_volume_mask = (raw_market_df["share_volume"] == 0.0)
    zero_volume_count = int(zero_volume_mask.sum())
    violation_counts["share_volume_exactly_zero"] = zero_volume_count

    if zero_volume_count > 0:
        logger.info(f"Found {zero_volume_count} rows with exactly zero share_volume. These will require clamping downstream.")

    return violation_counts

# -------------------------------------------------------------------------------------------------------------------------------
# Task 3, Step 2: Verify cross-input identifier and date consistency
# -------------------------------------------------------------------------------------------------------------------------------
def check_cross_input_consistency(
    raw_market_df: pd.DataFrame,
    asset_identifier_map: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    risk_free_rate_series: pd.Series
) -> Dict[str, Any]:
    """
    Verifies referential integrity across the market panel, identifier map, calendar, and macro series.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.

    Returns
    -------
    Dict[str, Any]
        A dictionary containing the results of the cross-consistency checks.

    Raises
    ------
    CrossInputConsistencyError
        If unmapped assets exist in the market panel or if the macro series fails to cover the calendar.
    """
    logger.info("Executing Task 3, Step 2: Verifying cross-input consistency.")

    # Initialize the dictionary to store consistency flags and metrics
    consistency_results: Dict[str, Any] = {}

    # Extract unique asset IDs from the market panel and cast to string to prevent type mismatch
    market_asset_ids: Set[str] = set(raw_market_df.index.get_level_values("asset_id").astype(str).unique())

    # Extract unique asset IDs from the identifier map and cast to string
    map_asset_ids: Set[str] = set(asset_identifier_map["asset_id"].astype(str).unique())

    # Compute assets present in the market data but missing from the identifier map
    orphan_in_market = market_asset_ids - map_asset_ids
    consistency_results["orphan_assets_in_market_count"] = len(orphan_in_market)

    # Halt execution if any market asset lacks an identifier mapping
    if orphan_in_market:
        error_msg = f"Critical referential integrity failure: {len(orphan_in_market)} assets in raw_market_df lack mappings in asset_identifier_map. Sample: {list(orphan_in_market)[:5]}"
        logger.error(error_msg)
        raise CrossInputConsistencyError(error_msg)

    # Compute assets present in the map but missing from the market data
    orphan_in_map = map_asset_ids - market_asset_ids
    consistency_results["orphan_assets_in_map_count"] = len(orphan_in_map)

    # Log a warning for map orphans, but do not halt as the map may legitimately be a superset
    if orphan_in_map:
        logger.warning(f"Found {len(orphan_in_map)} assets in asset_identifier_map with no corresponding market data. This is acceptable.")

    # Extract unique dates from the market panel
    # We normalize to tz-naive UTC to ensure safe comparison with the calendar
    market_dates_raw = raw_market_df.index.get_level_values("date")
    if pd.api.types.is_datetime64tz_dtype(market_dates_raw):
        market_dates_raw = market_dates_raw.tz_convert("UTC").tz_localize(None)
    market_dates: Set[pd.Timestamp] = set(market_dates_raw.unique())

    # Extract dates from the master trading calendar
    calendar_dates: Set[pd.Timestamp] = set(master_trading_calendar)

    # Compute dates present in the market data but missing from the calendar
    dates_not_in_calendar = market_dates - calendar_dates
    consistency_results["market_dates_excluded_from_calendar_count"] = len(dates_not_in_calendar)

    # Log excluded dates; these will be naturally filtered out during downstream reindexing
    if dates_not_in_calendar:
        logger.info(f"{len(dates_not_in_calendar)} dates in raw_market_df are not in master_trading_calendar and will be excluded.")

    # Verify that the risk-free rate series covers the entire master trading calendar
    cal_min_date = master_trading_calendar.min()
    cal_max_date = master_trading_calendar.max()

    # Normalize risk-free rate index to tz-naive UTC for safe comparison
    rf_index = risk_free_rate_series.index
    if pd.api.types.is_datetime64tz_dtype(rf_index):
        rf_index = rf_index.tz_convert("UTC").tz_localize(None)

    rf_min_date = rf_index.min()
    rf_max_date = rf_index.max()

    # The macro series must start on or before the calendar start to allow forward-filling
    if rf_min_date > cal_min_date:
        error_msg = f"Macro series coverage gap: risk_free_rate_series starts on {rf_min_date}, but calendar starts on {cal_min_date}."
        logger.error(error_msg)
        raise CrossInputConsistencyError(error_msg)

    # The macro series must end on or after the calendar end
    if rf_max_date < cal_max_date:
        error_msg = f"Macro series coverage gap: risk_free_rate_series ends on {rf_max_date}, but calendar ends on {cal_max_date}."
        logger.error(error_msg)
        raise CrossInputConsistencyError(error_msg)

    consistency_results["macro_series_coverage_valid"] = True
    logger.info("Cross-input consistency verified successfully.")

    return consistency_results

# -------------------------------------------------------------------------------------------------------------------------------
# Task 3, Step 3: Verify calendar length and compile the validation report
# -------------------------------------------------------------------------------------------------------------------------------
def compile_validation_report(
    master_trading_calendar: pd.DatetimeIndex,
    numerical_violations: Dict[str, int],
    cross_consistency_flags: Dict[str, Any]
) -> ValidationReport:
    """
    Verifies the calendar length requirement and compiles the final validation report.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    numerical_violations : Dict[str, int]
        The dictionary of numerical violation counts from Step 1.
    cross_consistency_flags : Dict[str, Any]
        The dictionary of referential integrity outcomes from Step 2.

    Returns
    -------
    ValidationReport
        A strictly typed dataclass containing the final validation statuses.

    Raises
    ------
    CalendarLengthError
        If the calendar contains fewer than 43 entries.
    """
    logger.info("Executing Task 3, Step 3: Verifying calendar length and compiling report.")

    # Verify that the calendar is long enough to support a 42-day forward window
    # A minimum of 43 dates is required: 1 decision date + 42 forward dates
    calendar_length = len(master_trading_calendar)
    if calendar_length < 43:
        error_msg = f"master_trading_calendar has only {calendar_length} entries. A minimum of 43 is required for the 42-day forward window."
        logger.error(error_msg)
        raise CalendarLengthError(error_msg)

    # Initialize the global halt flag to False
    global_halt = False

    # Determine the status for raw_market_df
    # If there are any numerical violations (excluding the acceptable exactly_zero volume), assign PASSED_WITH_PLACEHOLDER_RISK
    # because these rows will be filtered out in Task 7.
    total_strict_violations = sum(
        count for key, count in numerical_violations.items()
        if key != "share_volume_exactly_zero"
    )

    if total_strict_violations > 0:
        raw_market_status = ValidationStatus.PASSED_WITH_PLACEHOLDER_RISK
        logger.warning("raw_market_df assigned PASSED_WITH_PLACEHOLDER_RISK due to numerical violations.")
    else:
        raw_market_status = ValidationStatus.PASSED

    # Determine the status for risk_free_rate_series
    # Since coverage was verified in Step 2 (otherwise an exception would have been raised), it passes.
    # We assign PASSED_WITH_PLACEHOLDER_RISK to acknowledge the manuscript-unspecified daily conversion convention.
    rf_series_status = ValidationStatus.PASSED_WITH_PLACEHOLDER_RISK

    # Determine the status for master_trading_calendar
    # Length is verified above. We assign PASSED_WITH_PLACEHOLDER_RISK to acknowledge the unspecified exchange source.
    calendar_status = ValidationStatus.PASSED_WITH_PLACEHOLDER_RISK

    # Determine the status for asset_identifier_map
    # Referential integrity was verified in Step 2.
    id_map_status = ValidationStatus.PASSED

    # Determine the status for study_config
    # Assuming Task 2 passed, we assign PASSED_WITH_PLACEHOLDER_RISK to acknowledge the presence of placeholder values.
    config_status = ValidationStatus.PASSED_WITH_PLACEHOLDER_RISK

    # Construct the final immutable report
    report = ValidationReport(
        raw_market_df_status=raw_market_status,
        risk_free_rate_series_status=rf_series_status,
        master_trading_calendar_status=calendar_status,
        asset_identifier_map_status=id_map_status,
        study_config_status=config_status,
        numerical_violations=numerical_violations,
        cross_consistency_flags=cross_consistency_flags,
        global_halt_required=global_halt
    )

    logger.info("Validation report compiled successfully.")
    return report

# -------------------------------------------------------------------------------------------------------------------------------
# Task 3, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_3_orchestrator(
    raw_market_df: pd.DataFrame,
    asset_identifier_map: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    risk_free_rate_series: pd.Series
) -> ValidationReport:
    """
    Orchestrates the validation of numerical ranges and cross-input consistency.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.

    Returns
    -------
    ValidationReport
        A strictly typed dataclass containing the final validation statuses.

    Raises
    ------
    CrossInputConsistencyError
        If referential integrity checks fail.
    CalendarLengthError
        If the calendar is too short to support the required forward window.
    """
    logger.info("Commencing Task 3: Validation of numerical ranges and cross-input consistency.")

    # Execute Step 1: Check numerical ranges in the market panel
    numerical_violations = check_numerical_ranges(raw_market_df)

    # Execute Step 2: Verify cross-input referential integrity
    # This step will raise an exception and halt if critical orphans or coverage gaps are found
    cross_consistency_flags = check_cross_input_consistency(
        raw_market_df=raw_market_df,
        asset_identifier_map=asset_identifier_map,
        master_trading_calendar=master_trading_calendar,
        risk_free_rate_series=risk_free_rate_series
    )

    # Execute Step 3: Verify calendar length and compile the final report
    # This step will raise an exception if the calendar is too short
    validation_report = compile_validation_report(
        master_trading_calendar=master_trading_calendar,
        numerical_violations=numerical_violations,
        cross_consistency_flags=cross_consistency_flags
    )

    # Evaluate the global halt policy
    if validation_report.global_halt_required:
        logger.critical("Validation report indicates a global halt is required. Terminating pipeline.")
        raise RuntimeError("Pipeline halted due to critical validation failures.")

    logger.info("Task 3 complete. Numerical ranges and cross-input consistency validated.")

    # Return the immutable validation report
    return validation_report


In [ ]:
# Task 4: Freeze the execution environment and hash all inputs

class EnvironmentFreezeError(Exception):
    """Custom exception raised when the execution environment cannot be deterministically frozen."""
    pass

# ==============================================================================
# Task 4: Freeze the execution environment and hash all inputs
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 4, Step 1: Create the provenance manifest and record the execution environment
# -------------------------------------------------------------------------------------------------------------------------------
def record_execution_environment(study_config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Creates the foundational provenance manifest by recording OS, Python, package, and solver versions.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing solver and repository settings.

    Returns
    -------
    Dict[str, Any]
        The initialized provenance manifest containing the 'environment' sub-dictionary.

    Raises
    ------
    EnvironmentFreezeError
        If critical packages are missing or if the solver version cannot be determined.
    """
    logger.info("Executing Task 4, Step 1: Recording execution environment.")

    manifest: Dict[str, Any] = {"environment": {}}

    # Record Operating System and Python version
    manifest["environment"]["os_platform"] = platform.platform()
    manifest["environment"]["python_version"] = sys.version.replace('\n', ' ')

    # Record critical numerical package versions
    critical_packages = ["pandas", "numpy", "scipy", "cvxpy"]
    for pkg in critical_packages:
        try:
            manifest["environment"][f"{pkg}_version"] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            error_msg = f"Critical package '{pkg}' is not installed."
            logger.error(error_msg)
            raise EnvironmentFreezeError(error_msg)

    # Record Solver information
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})
    solver_name = solver_settings.get("solver_name", "UNKNOWN")
    manifest["environment"]["configured_solver"] = solver_name

    # Attempt to dynamically query the solver version
    try:
        if solver_name.upper() == "MOSEK":
            import mosek
            # MOSEK version is typically available via the Env object
            env = mosek.Env()
            major, minor, revision = env.getversion()
            manifest["environment"]["solver_version"] = f"{major}.{minor}.{revision}"
        elif solver_name.upper() == "SCS":
            import scs
            manifest["environment"]["solver_version"] = scs.__version__
        elif solver_name.upper() == "ECOS":
            import ecos
            manifest["environment"]["solver_version"] = ecos.__version__
        else:
            manifest["environment"]["solver_version"] = "version_query_not_implemented"
    except ImportError:
        error_msg = f"Configured solver '{solver_name}' is not installed or accessible."
        logger.error(error_msg)
        raise EnvironmentFreezeError(error_msg)
    except Exception as e:
        logger.warning(f"Could not determine version for solver '{solver_name}': {e}")
        manifest["environment"]["solver_version"] = "unknown"

    # Record Repository URL and Git Commit Hash
    methodology = study_config.get("METHODOLOGY", {})
    manifest["environment"]["repository_url"] = methodology.get("paper_repository", "not_specified")

    try:
        # Attempt to extract the current git HEAD commit hash
        commit_hash = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.STDOUT,
            text=True
        ).strip()
        manifest["environment"]["git_commit_hash"] = commit_hash
    except (subprocess.CalledProcessError, FileNotFoundError):
        logger.warning("Git repository not found or git command unavailable. Commit hash set to 'not_available'.")
        manifest["environment"]["git_commit_hash"] = "not_available"

    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 4, Step 2: Freeze random seeds and instantiate isolated generators
# -------------------------------------------------------------------------------------------------------------------------------
def freeze_random_generators(study_config: Dict[str, Any], manifest: Dict[str, Any]) -> Tuple[Dict[str, np.random.Generator], Dict[str, Any]]:
    """
    Validates random seeds and instantiates isolated numpy PCG64 generators for deterministic execution.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary containing the RANDOM_SEED_REGISTRY.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented with the seed records.

    Returns
    -------
    Tuple[Dict[str, np.random.Generator], Dict[str, Any]]
        A dictionary mapping seed domains to their respective Generator instances, and the augmented manifest.

    Raises
    ------
    EnvironmentFreezeError
        If any seed is missing, negative, or not an integer.
    """
    logger.info("Executing Task 4, Step 2: Freezing random seeds and instantiating generators.")

    seed_registry = study_config.get("RANDOM_SEED_REGISTRY")
    if not seed_registry:
        raise EnvironmentFreezeError("Missing 'RANDOM_SEED_REGISTRY' in study_config.")

    manifest["random_seeds"] = {}
    generators: Dict[str, np.random.Generator] = {}

    for domain, seed_val in seed_registry.items():
        # Validate that the seed is a non-negative integer
        if not isinstance(seed_val, int) or isinstance(seed_val, bool):
            raise EnvironmentFreezeError(f"Seed for '{domain}' must be an integer, got {type(seed_val)}.")
        if seed_val < 0:
            raise EnvironmentFreezeError(f"Seed for '{domain}' must be non-negative, got {seed_val}.")

        # Record the exact seed value in the manifest
        manifest["random_seeds"][domain] = seed_val

        # Instantiate an isolated PCG64 generator via default_rng
        generators[domain] = np.random.default_rng(seed_val)

    logger.info(f"Successfully instantiated {len(generators)} isolated random generators.")
    return generators, manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 4, Step 3: Compute deterministic cryptographic hashes of all input objects
# -------------------------------------------------------------------------------------------------------------------------------
def hash_input_objects(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Computes deterministic SHA-256 hashes of the five primary input objects to detect numerical drift.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented with the hashes.

    Returns
    -------
    Dict[str, Any]
        The fully augmented provenance manifest containing the 'input_hashes' sub-dictionary.
    """
    logger.info("Executing Task 4, Step 3: Computing cryptographic hashes of input objects.")

    manifest["input_hashes"] = {}

    def compute_sha256(byte_stream: bytes) -> str:
        """Helper to compute SHA-256 hex digest from bytes."""
        return hashlib.sha256(byte_stream).hexdigest()

    # 1. Hash raw_market_df
    # We use pd.util.hash_pandas_object which is deterministic and avoids pyarrow version dependencies.
    # The index is already sorted from Task 1. We sort columns alphabetically to ensure canonical order.
    sorted_cols_df = raw_market_df[sorted(raw_market_df.columns)]
    df_bytes = pd.util.hash_pandas_object(sorted_cols_df, index=True).values.tobytes()
    manifest["input_hashes"]["raw_market_df"] = compute_sha256(df_bytes)

    # 2. Hash risk_free_rate_series
    # Sort index to ensure canonical order
    sorted_rf = risk_free_rate_series.sort_index()
    rf_bytes = pd.util.hash_pandas_object(sorted_rf, index=True).values.tobytes()
    manifest["input_hashes"]["risk_free_rate_series"] = compute_sha256(rf_bytes)

    # 3. Hash master_trading_calendar
    # Convert the DatetimeIndex to an array of int64 nanosecond timestamps
    cal_bytes = master_trading_calendar.asi8.tobytes()
    manifest["input_hashes"]["master_trading_calendar"] = compute_sha256(cal_bytes)

    # 4. Hash asset_identifier_map
    # Sort by asset_id to ensure canonical row order, then sort columns
    sorted_map = asset_identifier_map.sort_values("asset_id")[sorted(asset_identifier_map.columns)]
    map_bytes = pd.util.hash_pandas_object(sorted_map, index=True).values.tobytes()
    manifest["input_hashes"]["asset_identifier_map"] = compute_sha256(map_bytes)

    # 5. Hash study_config
    # Serialize to JSON with sorted keys. default=str handles any non-serializable types safely.
    config_json = json.dumps(study_config, sort_keys=True, default=str)
    config_bytes = config_json.encode("utf-8")
    manifest["input_hashes"]["study_config"] = compute_sha256(config_bytes)

    logger.info("Successfully computed SHA-256 hashes for all 5 input objects.")
    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 4, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_4_orchestrator(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[Dict[str, Any], Dict[str, np.random.Generator]]:
    """
    Orchestrates the freezing of the execution environment, seed instantiation, and input hashing.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[Dict[str, Any], Dict[str, np.random.Generator]]
        A tuple containing the fully populated provenance manifest and the dictionary of isolated random generators.

    Raises
    ------
    EnvironmentFreezeError
        If the environment cannot be deterministically frozen or if seeds are invalid.
    """
    logger.info("Commencing Task 4: Freezing execution environment and hashing inputs.")

    # Execute Step 1: Record OS, Python, packages, and solver versions
    manifest = record_execution_environment(study_config)

    # Execute Step 2: Freeze seeds and instantiate isolated generators
    generators, manifest = freeze_random_generators(study_config, manifest)

    # Execute Step 3: Compute cryptographic hashes of all inputs
    manifest = hash_input_objects(
        raw_market_df=raw_market_df,
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar,
        asset_identifier_map=asset_identifier_map,
        study_config=study_config,
        manifest=manifest
    )

    logger.info("Task 4 complete. Environment frozen, generators instantiated, and inputs hashed.")

    # Return the immutable manifest and the active generators
    return manifest, generators


In [ ]:
# Task 5: Validate identifier integrity across input objects

class IdentifierIntegrityError(Exception):
    """Custom exception raised when identifier referential integrity fails."""
    pass

# ==============================================================================
# Task 5: Validate identifier integrity across input objects
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 5, Step 1: Verify identifier map uniqueness and vendor_id completeness
# -------------------------------------------------------------------------------------------------------------------------------
def verify_identifier_map_integrity(asset_identifier_map: pd.DataFrame) -> None:
    """
    Confirms that every asset_id in the map is strictly unique and checks for missing vendor IDs.

    Parameters
    ----------
    asset_identifier_map : pd.DataFrame
        The permanent identifier map linking internal asset IDs to vendor identifiers.

    Raises
    ------
    IdentifierIntegrityError
        If duplicate asset_id values are detected in the map.
    """
    logger.info("Executing Task 5, Step 1: Verifying identifier map uniqueness and completeness.")

    # Defensive invariant check: asset_id must be strictly unique
    if not asset_identifier_map["asset_id"].is_unique:
        # Extract the duplicated IDs for the error message
        duplicates = asset_identifier_map[asset_identifier_map.duplicated("asset_id", keep=False)]
        error_msg = f"Critical failure: asset_identifier_map contains duplicate asset_id values. Sample:\n{duplicates.head()}"
        logger.error(error_msg)
        raise IdentifierIntegrityError(error_msg)

    # Check for missing vendor_id values (handling true NaNs and empty/whitespace strings)
    vendor_id_series = asset_identifier_map["vendor_id"]
    missing_vendor_mask = vendor_id_series.isna() | (vendor_id_series.astype(str).str.strip() == "")
    missing_vendor_count = int(missing_vendor_mask.sum())

    if missing_vendor_count > 0:
        # We log a warning rather than halting, as some mapped assets may not enter the final universe
        logger.warning(
            f"Found {missing_vendor_count} assets in asset_identifier_map with missing or empty vendor_id. "
            "This is acceptable at this stage but may cause issues if these assets enter the final universe."
        )
    else:
        logger.info("All assets in asset_identifier_map possess a valid vendor_id.")

# -------------------------------------------------------------------------------------------------------------------------------
# Task 5, Step 2: Verify referential integrity between market panel and identifier map
# -------------------------------------------------------------------------------------------------------------------------------
def verify_market_panel_referential_integrity(
    raw_market_df: pd.DataFrame,
    asset_identifier_map: pd.DataFrame
) -> None:
    """
    Confirms that every unique asset_id in the market panel has a corresponding entry in the identifier map.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.

    Raises
    ------
    IdentifierIntegrityError
        If any asset_id in the market panel lacks a mapping in the identifier map.
    """
    logger.info("Executing Task 5, Step 2: Verifying referential integrity between market panel and identifier map.")

    # Extract unique asset IDs from the market panel and cast to string to prevent type mismatch
    market_asset_ids: Set[str] = set(raw_market_df.index.get_level_values("asset_id").astype(str).unique())

    # Extract unique asset IDs from the identifier map and cast to string
    map_asset_ids: Set[str] = set(asset_identifier_map["asset_id"].astype(str).unique())

    # Compute assets present in the market data but missing from the identifier map
    orphan_in_market = market_asset_ids - map_asset_ids

    # Halt execution if any market asset lacks an identifier mapping
    if orphan_in_market:
        error_msg = (
            f"Critical referential integrity failure: {len(orphan_in_market)} assets in raw_market_df "
            f"lack mappings in asset_identifier_map. Sample: {list(orphan_in_market)[:5]}"
        )
        logger.error(error_msg)
        raise IdentifierIntegrityError(error_msg)

    # Compute assets present in the map but missing from the market data
    orphan_in_map = map_asset_ids - market_asset_ids

    # Log a warning for map orphans, but do not halt as the map may legitimately be a superset
    if orphan_in_map:
        logger.info(
            f"Found {len(orphan_in_map)} assets in asset_identifier_map with no corresponding market data. "
            "This is acceptable as the map may be a superset."
        )

    logger.info("Referential integrity between market panel and identifier map verified successfully.")

# -------------------------------------------------------------------------------------------------------------------------------
# Task 5, Step 3: Validate calendar endpoints against study configuration
# -------------------------------------------------------------------------------------------------------------------------------
def validate_and_record_calendar_endpoints(
    master_trading_calendar: pd.DatetimeIndex,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Validates the calendar date range against the config placeholders and records exact endpoints in the manifest.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    study_config : Dict[str, Any]
        The master configuration dictionary containing the intended sample boundaries.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented with the exact calendar endpoints.

    Returns
    -------
    Dict[str, Any]
        The augmented provenance manifest containing the 'calendar_endpoints' sub-dictionary.
    """
    logger.info("Executing Task 5, Step 3: Validating calendar endpoints and augmenting manifest.")

    # Extract the absolute minimum and maximum dates from the calendar
    calendar_start = master_trading_calendar.min()
    calendar_end = master_trading_calendar.max()

    # Extract the intended sample boundaries from the configuration
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    intended_start_str = firm_consts.get("sample_start_date", "")
    intended_end_str = firm_consts.get("sample_end_date", "")

    # Soft validation: Check if the calendar start aligns with the intended "July 2000"
    if "July_2000" in intended_start_str:
        if calendar_start.year != 2000 or calendar_start.month != 7:
            logger.warning(
                f"Calendar start date ({calendar_start.strftime('%Y-%m-%d')}) deviates from the intended "
                f"placeholder '{intended_start_str}'. This may be due to business-day conventions."
            )

    # Soft validation: Check if the calendar end aligns with the intended "April 2025"
    if "April_2025" in intended_end_str:
        if calendar_end.year != 2025 or calendar_end.month != 4:
            logger.warning(
                f"Calendar end date ({calendar_end.strftime('%Y-%m-%d')}) deviates from the intended "
                f"placeholder '{intended_end_str}'. This is acceptable if extending for the forward window."
            )

    # Record the exact datetime endpoints as ISO 8601 strings in the provenance manifest
    manifest["calendar_endpoints"] = {
        "exact_start_date": calendar_start.isoformat(),
        "exact_end_date": calendar_end.isoformat(),
        "total_trading_days": len(master_trading_calendar)
    }

    logger.info(f"Calendar endpoints recorded: {calendar_start.strftime('%Y-%m-%d')} to {calendar_end.strftime('%Y-%m-%d')}.")
    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 5, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_5_orchestrator(
    raw_market_df: pd.DataFrame,
    asset_identifier_map: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Orchestrates the validation of identifier integrity and temporal boundaries.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Dict[str, Any]
        The fully augmented provenance manifest.

    Raises
    ------
    IdentifierIntegrityError
        If critical referential integrity checks fail.
    """
    logger.info("Commencing Task 5: Validation of identifier integrity across input objects.")

    # Execute Step 1: Verify identifier map uniqueness and completeness
    verify_identifier_map_integrity(asset_identifier_map)

    # Execute Step 2: Verify referential integrity between market panel and identifier map
    verify_market_panel_referential_integrity(
        raw_market_df=raw_market_df,
        asset_identifier_map=asset_identifier_map
    )

    # Execute Step 3: Validate calendar endpoints and augment the manifest
    manifest = validate_and_record_calendar_endpoints(
        master_trading_calendar=master_trading_calendar,
        study_config=study_config,
        manifest=manifest
    )

    logger.info("Task 5 complete. Identifier integrity verified and temporal boundaries recorded.")

    # Return the augmented manifest
    return manifest


In [ ]:
# Task 6: Validate and freeze the master trading calendar

class CalendarValidationError(Exception):
    """Custom exception raised when the master trading calendar violates structural or temporal requirements."""
    pass

# ==============================================================================
# Task 6: Validate and freeze the master trading calendar
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 6, Step 1: Validate calendar monotonicity, uniqueness, and business-day consistency
# -------------------------------------------------------------------------------------------------------------------------------
def validate_calendar_structure(master_trading_calendar: pd.DatetimeIndex) -> str:
    """
    Confirms the calendar is strictly increasing, unique, and contains only valid business days.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    str
        A descriptive string documenting the calendar validation convention applied.

    Raises
    ------
    CalendarValidationError
        If the calendar is not strictly increasing, contains duplicates, or includes weekends.
    """
    logger.info("Executing Task 6, Step 1: Validating calendar structure and business-day consistency.")

    # 1. Verify strict monotonic increase
    if not master_trading_calendar.is_monotonic_increasing:
        error_msg = "master_trading_calendar is not monotonically increasing."
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    # 2. Verify absence of duplicates
    if master_trading_calendar.has_duplicates:
        error_msg = "master_trading_calendar contains duplicate dates."
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    # 3. Verify absence of weekends (dayofweek: Monday=0, ..., Friday=4, Saturday=5, Sunday=6)
    weekend_mask = master_trading_calendar.dayofweek >= 5
    if weekend_mask.any():
        weekend_dates = master_trading_calendar[weekend_mask]
        error_msg = f"master_trading_calendar contains {len(weekend_dates)} weekend dates. Sample: {weekend_dates[:5].tolist()}"
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    # 4. Attempt optional holiday validation against a reference exchange calendar
    convention_used = "Strict weekend-only check applied. Exact exchange holiday calendar is manuscript-unspecified."
    try:
        import pandas_market_calendars as mcal
        nyse = mcal.get_calendar('NYSE')
        # Generate reference schedule covering the calendar's range
        ref_schedule = nyse.schedule(start_date=master_trading_calendar.min(), end_date=master_trading_calendar.max())
        ref_dates = pd.DatetimeIndex(ref_schedule.index).tz_localize(None)

        # Compute symmetric difference
        sym_diff = master_trading_calendar.symmetric_difference(ref_dates)
        if not sym_diff.empty:
            logger.warning(
                f"Calendar differs from standard NYSE schedule by {len(sym_diff)} dates. "
                f"This is acceptable as the exact source is unspecified. Sample diff: {sym_diff[:5].tolist()}"
            )
        convention_used = "Validated against pandas_market_calendars NYSE schedule with minor acceptable deviations."
    except ImportError:
        logger.info("pandas_market_calendars not installed. Proceeding with strict weekend-only validation.")

    logger.info("Calendar structure validated successfully.")
    return convention_used

# -------------------------------------------------------------------------------------------------------------------------------
# Task 6, Step 2: Compute and freeze the valid decision dates subset
# -------------------------------------------------------------------------------------------------------------------------------
def compute_valid_decision_dates(master_trading_calendar: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """
    Computes the subset of calendar dates that possess a complete 42-trading-day forward window.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The strictly increasing, validated trading calendar.

    Returns
    -------
    pd.DatetimeIndex
        The frozen subset of valid decision dates (T_valid).

    Raises
    ------
    CalendarValidationError
        If the calendar is too short to yield any valid decision dates.
    """
    logger.info("Executing Task 6, Step 2: Computing valid decision dates (T_valid).")

    total_length = len(master_trading_calendar)
    forward_window_days = 42

    # The calendar must have at least 1 decision date + 42 forward dates
    if total_length <= forward_window_days:
        error_msg = f"Calendar length ({total_length}) is insufficient to support a {forward_window_days}-day forward window."
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    # Slice the calendar to exclude the last 42 dates
    # If length is L, valid indices are 0 through L - 43 inclusive.
    # Slice notation [:-42] correctly drops the last 42 elements.
    valid_decision_dates = master_trading_calendar[:-forward_window_days]

    # Rigorous invariant assertion:
    # For the LAST date in the valid set, there must be EXACTLY 42 dates strictly after it in the master calendar.
    last_valid_date = valid_decision_dates[-1]
    dates_strictly_after = (master_trading_calendar > last_valid_date).sum()

    if dates_strictly_after != forward_window_days:
        error_msg = f"Invariant failure: Expected exactly {forward_window_days} dates after {last_valid_date}, found {dates_strictly_after}."
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    logger.info(f"Computed {len(valid_decision_dates)} valid decision dates. Last valid date: {last_valid_date.strftime('%Y-%m-%d')}.")

    # Return the DatetimeIndex. By convention, this object should be treated as immutable downstream.
    return valid_decision_dates

# -------------------------------------------------------------------------------------------------------------------------------
# Task 6, Step 3: Verify macro series coverage for forward-filling
# -------------------------------------------------------------------------------------------------------------------------------
def verify_macro_series_coverage(
    master_trading_calendar: pd.DatetimeIndex,
    risk_free_rate_series: pd.Series
) -> str:
    """
    Verifies that the risk-free rate series starts on or before the calendar to permit safe forward-filling.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.

    Returns
    -------
    str
        A descriptive string documenting the alignment method.

    Raises
    ------
    CalendarValidationError
        If the macro series starts after the calendar, preventing forward-fill.
    """
    logger.info("Executing Task 6, Step 3: Verifying macro series coverage for forward-filling.")

    cal_min = master_trading_calendar.min()

    # Ensure the risk-free rate index is tz-naive for safe comparison
    rf_index = risk_free_rate_series.index
    if pd.api.types.is_datetime64tz_dtype(rf_index):
        rf_index = rf_index.tz_convert("UTC").tz_localize(None)

    rf_min = rf_index.min()

    # If the macro series starts after the calendar, the initial calendar dates will be NaN after forward-fill
    if rf_min > cal_min:
        error_msg = (
            f"Macro series coverage gap: risk_free_rate_series starts on {rf_min.strftime('%Y-%m-%d')}, "
            f"but calendar starts on {cal_min.strftime('%Y-%m-%d')}. Forward-filling is impossible for leading dates."
        )
        logger.error(error_msg)
        raise CalendarValidationError(error_msg)

    alignment_method = "forward_fill_from_most_recent_FRED_observation"
    logger.info("Macro series coverage verified. Safe for forward-filling.")

    return alignment_method

# -------------------------------------------------------------------------------------------------------------------------------
# Task 6, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_6_orchestrator(
    master_trading_calendar: pd.DatetimeIndex,
    risk_free_rate_series: pd.Series,
    manifest: Dict[str, Any]
) -> Tuple[pd.DatetimeIndex, Dict[str, Any]]:
    """
    Orchestrates the validation and freezing of the master trading calendar and valid decision dates.

    Parameters
    ----------
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[pd.DatetimeIndex, Dict[str, Any]]
        A tuple containing the frozen valid decision dates (T_valid) and the augmented manifest.

    Raises
    ------
    CalendarValidationError
        If any structural or temporal calendar requirements are violated.
    """
    logger.info("Commencing Task 6: Validation and freezing of the master trading calendar.")

    # Execute Step 1: Validate calendar structure
    calendar_convention = validate_calendar_structure(master_trading_calendar)

    # Execute Step 2: Compute valid decision dates
    valid_decision_dates = compute_valid_decision_dates(master_trading_calendar)

    # Execute Step 3: Verify macro series coverage
    alignment_method = verify_macro_series_coverage(
        master_trading_calendar=master_trading_calendar,
        risk_free_rate_series=risk_free_rate_series
    )

    # Augment the provenance manifest with the documented conventions
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "exchange_calendar_source",
        "manuscript_status": "unspecified",
        "value_used": calendar_convention,
        "source": "rational placeholder"
    })

    manifest["placeholder_assumptions"].append({
        "convention_name": "macro_series_alignment",
        "manuscript_status": "unspecified",
        "value_used": alignment_method,
        "source": "rational placeholder"
    })

    logger.info("Task 6 complete. Calendar frozen and valid decision dates computed.")

    # Return the frozen valid decision dates and the augmented manifest
    return valid_decision_dates, manifest


In [ ]:
# Task 7: Filter raw market data for structural validity

class DataQualityError(Exception):
    """Custom exception raised when the filtered market panel still contains structural violations."""
    pass

# ==============================================================================
# Task 7: Filter raw market data for structural validity
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 7, Step 1 & 3: Apply filtering conditions and compile audit log
# -------------------------------------------------------------------------------------------------------------------------------
def apply_structural_filters(
    raw_market_df: pd.DataFrame,
    acceptable_quality_flags: List[str]
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Applies eight structural filtering conditions to the raw market panel and compiles an audit log.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    acceptable_quality_flags : List[str]
        A list of acceptable strings for the data_quality_flag column.

    Returns
    -------
    Tuple[pd.DataFrame, Dict[str, int]]
        The filtered DataFrame and a dictionary containing the audit log of removed rows.
    """
    logger.info("Executing Task 7, Step 1 & 3: Applying structural filters and compiling audit log.")

    n_total = len(raw_market_df)
    audit_log: Dict[str, int] = {"total_input_rows": n_total}

    # 1. Primary observation filter
    # Handle potential NaNs in nullable boolean columns by filling with False
    mask_primary = raw_market_df["is_primary_observation"].fillna(False).astype(bool)
    audit_log["rows_failing_primary_observation"] = int((~mask_primary).sum())

    # 2. Currency filter (case-insensitive)
    # Handle potential NaNs by filling with empty string before upper()
    mask_currency = raw_market_df["currency"].fillna("").astype(str).str.upper() == "USD"
    audit_log["rows_failing_currency_usd"] = int((~mask_currency).sum())

    # 3. Data quality flag filter
    if acceptable_quality_flags:
        mask_quality = raw_market_df["data_quality_flag"].isin(acceptable_quality_flags)
    else:
        # If no specific flags are provided, accept all non-null values as a placeholder convention
        mask_quality = raw_market_df["data_quality_flag"].notna()
    audit_log["rows_failing_quality_flag"] = int((~mask_quality).sum())

    # 4. Strictly positive adjusted trade price
    mask_trade_price = (raw_market_df["adjusted_trade_price"] > 0.0) & ~raw_market_df["adjusted_trade_price"].isna()
    audit_log["rows_failing_positive_trade_price"] = int((~mask_trade_price).sum())

    # 5. Strictly positive adjusted bid price
    mask_bid_price = (raw_market_df["adjusted_bid_price"] > 0.0) & ~raw_market_df["adjusted_bid_price"].isna()
    audit_log["rows_failing_positive_bid_price"] = int((~mask_bid_price).sum())

    # 6. Strictly positive adjusted ask price
    mask_ask_price = (raw_market_df["adjusted_ask_price"] > 0.0) & ~raw_market_df["adjusted_ask_price"].isna()
    audit_log["rows_failing_positive_ask_price"] = int((~mask_ask_price).sum())

    # 7. Non-negative share volume
    mask_volume = (raw_market_df["share_volume"] >= 0.0) & ~raw_market_df["share_volume"].isna()
    audit_log["rows_failing_nonneg_volume"] = int((~mask_volume).sum())

    # 8. Non-negative market capitalization
    mask_mcap = (raw_market_df["market_cap_usd"] >= 0.0) & ~raw_market_df["market_cap_usd"].isna()
    audit_log["rows_failing_nonneg_market_cap"] = int((~mask_mcap).sum())

    # Combine all masks using element-wise AND
    combined_mask = (
        mask_primary &
        mask_currency &
        mask_quality &
        mask_trade_price &
        mask_bid_price &
        mask_ask_price &
        mask_volume &
        mask_mcap
    )

    # Apply the combined mask to filter the DataFrame
    filtered_df = raw_market_df.loc[combined_mask].copy()

    n_retained = len(filtered_df)
    audit_log["total_rows_removed"] = n_total - n_retained
    audit_log["total_rows_retained"] = n_retained

    logger.info(f"Filtering complete. Retained {n_retained} out of {n_total} rows.")

    return filtered_df, audit_log

# -------------------------------------------------------------------------------------------------------------------------------
# Task 7, Step 2: Verify absence of duplicate index pairs
# -------------------------------------------------------------------------------------------------------------------------------
def verify_absence_of_duplicates(filtered_df: pd.DataFrame) -> None:
    """
    Verifies that the filtered panel contains no duplicate (date, asset_id) pairs.

    Parameters
    ----------
    filtered_df : pd.DataFrame
        The market panel after applying structural filters.

    Raises
    ------
    DataQualityError
        If duplicate index pairs remain in the filtered DataFrame.
    """
    logger.info("Executing Task 7, Step 2: Verifying absence of duplicate index pairs.")

    # Check for duplicated indices
    if filtered_df.index.duplicated().any():
        # Extract all rows involved in the duplication for diagnostic purposes
        dup_mask = filtered_df.index.duplicated(keep=False)
        dup_rows = filtered_df.loc[dup_mask]
        dup_count = len(dup_rows)

        error_msg = (
            f"Critical data quality failure: {dup_count} duplicate (date, asset_id) rows remain "
            f"after applying the 'is_primary_observation' filter. The data is malformed.\n"
            f"Sample of duplicates:\n{dup_rows.head(10)}"
        )
        logger.error(error_msg)
        raise DataQualityError(error_msg)

    logger.info("Duplicate verification passed. No duplicate (date, asset_id) pairs exist.")

# -------------------------------------------------------------------------------------------------------------------------------
# Task 7, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_7_orchestrator(
    raw_market_df: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    """
    Orchestrates the structural filtering of the raw market data and compiles the audit log.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[pd.DataFrame, Dict[str, int]]
        A tuple containing the structurally filtered DataFrame and the audit log dictionary.

    Raises
    ------
    DataQualityError
        If the filtered data still contains duplicate index pairs.
    """
    logger.info("Commencing Task 7: Filtering raw market data for structural validity.")

    # Extract acceptable quality flags from the configuration, if specified
    # The manuscript does not specify exact flags, so we default to an empty list which triggers the notna() fallback
    methodology_config = study_config.get("METHODOLOGY", {})
    acceptable_flags = methodology_config.get("acceptable_data_quality_flags", [])

    # Execute Step 1 & 3: Apply filters and generate audit log
    filtered_df, audit_log = apply_structural_filters(
        raw_market_df=raw_market_df,
        acceptable_quality_flags=acceptable_flags
    )

    # Execute Step 2: Verify uniqueness of the resulting MultiIndex
    verify_absence_of_duplicates(filtered_df)

    logger.info("Task 7 complete. Market data structurally validated and filtered.")

    # Return the filtered DataFrame and the audit log
    return filtered_df, audit_log


In [ ]:
# Task 8: Apply missing-data cleaning conventions and emit clean panel

class DataImputationError(Exception):
    """Custom exception raised when missing-data cleaning fails or leaves residual NaNs."""
    pass

# ==============================================================================
# Task 8: Apply missing-data cleaning conventions and emit clean panel
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 8, Step 1: Forward-fill bid and ask prices per asset
# -------------------------------------------------------------------------------------------------------------------------------
def forward_fill_bid_ask(
    filtered_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex
) -> Tuple[pd.DataFrame, int]:
    """
    Forward-fills missing bid and ask prices independently for each asset along the master trading calendar.

    Parameters
    ----------
    filtered_df : pd.DataFrame
        The structurally validated market panel from Task 7.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    Tuple[pd.DataFrame, int]
        The DataFrame with forward-filled bid/ask prices, and the total count of filled values.
    """
    logger.info("Executing Task 8, Step 1: Forward-filling bid and ask prices per asset.")

    # Define the columns to be forward-filled as per the manuscript
    fill_cols = ["adjusted_bid_price", "adjusted_ask_price"]

    # Count NaNs before filling to calculate the exact number of filled values
    initial_nans = filtered_df[fill_cols].isna().sum().sum()

    # To perform a highly efficient, vectorized forward-fill per asset, we unstack the asset_id level.
    # This creates a wide matrix where rows are dates and columns are assets.
    # We only unstack the specific columns needing imputation to save memory.
    wide_prices = filtered_df[fill_cols].unstack(level="asset_id")

    # Reindex the wide matrix to the master trading calendar.
    # This ensures that if an asset has a gap of several calendar days, those days are explicitly
    # represented as NaNs and subsequently forward-filled.
    wide_prices = wide_prices.reindex(master_trading_calendar)

    # Apply vectorized forward-fill along the date axis (axis=0).
    # This strictly prevents look-ahead bias (no backfilling) and prevents cross-asset contamination.
    wide_prices_filled = wide_prices.ffill(axis=0)

    # Restack the wide matrix back to the long-format MultiIndex series
    long_prices_filled = wide_prices_filled.stack(level="asset_id", dropna=False)

    # Realign the filled data with the original filtered_df index.
    # We use an inner join (via .loc) to ensure we only keep (date, asset_id) pairs that actually
    # exist in the filtered_df, discarding the combinatorial explosion of dates/assets created by unstacking.
    filtered_df = filtered_df.copy()

    # Update the specific columns with the filled data
    for col in fill_cols:
        # Extract the specific column from the restacked DataFrame
        filled_series = long_prices_filled[col]
        # Align and assign back to the main DataFrame
        filtered_df[col] = filled_series.reindex(filtered_df.index)

    # Count NaNs after filling
    final_nans = filtered_df[fill_cols].isna().sum().sum()

    # The number of successfully filled values
    filled_count = int(initial_nans - final_nans)

    logger.info(f"Successfully forward-filled {filled_count} missing bid/ask values.")

    return filtered_df, filled_count

# -------------------------------------------------------------------------------------------------------------------------------
# Task 8, Step 2: Apply missing-data strategy for trade price and volume
# -------------------------------------------------------------------------------------------------------------------------------
def handle_missing_trade_and_volume(
    df_filled: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[pd.DataFrame, int, int]:
    """
    Handles missing values in trade price and volume based on the configuration strategy.

    Parameters
    ----------
    df_filled : pd.DataFrame
        The market panel with forward-filled bid/ask prices.
    study_config : Dict[str, Any]
        The master configuration dictionary containing the missing data strategies.

    Returns
    -------
    Tuple[pd.DataFrame, int, int]
        The cleaned DataFrame, the count of rows dropped due to missing price, and missing volume.

    Raises
    ------
    DataImputationError
        If residual NaNs remain in critical numeric columns after applying the strategy.
    """
    logger.info("Executing Task 8, Step 2: Handling missing trade prices and volumes.")

    methodology = study_config.get("METHODOLOGY", {})
    price_strategy = methodology.get("missing_trade_price_strategy", "drop")
    volume_strategy = methodology.get("missing_volume_strategy", "drop")

    initial_len = len(df_filled)

    # Handle missing adjusted_trade_price
    if "drop" in price_strategy.lower():
        logger.info("Applying placeholder convention: dropping rows with missing adjusted_trade_price.")
        df_cleaned = df_filled.dropna(subset=["adjusted_trade_price"]).copy()
    else:
        # Fallback to drop if the strategy is unrecognized, to ensure numerical safety downstream
        logger.warning(f"Unrecognized price strategy '{price_strategy}'. Defaulting to 'drop'.")
        df_cleaned = df_filled.dropna(subset=["adjusted_trade_price"]).copy()

    len_after_price_drop = len(df_cleaned)
    dropped_price_count = initial_len - len_after_price_drop

    # Handle missing share_volume
    if "drop" in volume_strategy.lower():
        logger.info("Applying placeholder convention: dropping rows with missing share_volume.")
        df_cleaned = df_cleaned.dropna(subset=["share_volume"]).copy()
    else:
        logger.warning(f"Unrecognized volume strategy '{volume_strategy}'. Defaulting to 'drop'.")
        df_cleaned = df_cleaned.dropna(subset=["share_volume"]).copy()

    len_after_volume_drop = len(df_cleaned)
    dropped_volume_count = len_after_price_drop - len_after_volume_drop

    # Final invariant check: Ensure no NaNs remain in the four critical numeric columns
    critical_cols = ["adjusted_trade_price", "adjusted_bid_price", "adjusted_ask_price", "share_volume"]
    residual_nans = df_cleaned[critical_cols].isna().sum()

    if residual_nans.sum() > 0:
        error_msg = f"Critical failure: Residual NaNs remain in critical columns after cleaning phase:\n{residual_nans}"
        logger.error(error_msg)
        raise DataImputationError(error_msg)

    return df_cleaned, dropped_price_count, dropped_volume_count

# -------------------------------------------------------------------------------------------------------------------------------
# Task 8, Step 3: Emit clean panel and compile row-count summary
# -------------------------------------------------------------------------------------------------------------------------------
def finalize_clean_panel(
    df_cleaned: pd.DataFrame,
    audit_log: Dict[str, int],
    filled_bid_ask_count: int,
    dropped_price_count: int,
    dropped_volume_count: int
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Ensures the final DataFrame is lexicographically sorted and compiles the comprehensive audit summary.

    Parameters
    ----------
    df_cleaned : pd.DataFrame
        The fully cleaned market panel.
    audit_log : Dict[str, int]
        The audit log initiated in Task 7.
    filled_bid_ask_count : int
        Count of forward-filled bid/ask values.
    dropped_price_count : int
        Count of rows dropped due to missing trade price.
    dropped_volume_count : int
        Count of rows dropped due to missing volume.

    Returns
    -------
    Tuple[pd.DataFrame, Dict[str, Any]]
        The finalized clean_market_df and the augmented audit summary.
    """
    logger.info("Executing Task 8, Step 3: Finalizing clean panel and compiling summary.")

    # Ensure the MultiIndex is lexicographically sorted for downstream slicing efficiency
    if not df_cleaned.index.is_monotonic_increasing:
        df_cleaned = df_cleaned.sort_index()

    # Extract unique counts for the summary
    unique_assets = df_cleaned.index.get_level_values("asset_id").nunique()
    unique_dates = df_cleaned.index.get_level_values("date").nunique()
    final_rows = len(df_cleaned)

    # Augment the audit log with Task 8 metrics
    audit_summary: Dict[str, Any] = {
        "raw_input_rows": audit_log.get("total_input_rows", 0),
        "after_structural_filter": audit_log.get("total_rows_retained", 0),
        "forward_filled_bid_ask_count": filled_bid_ask_count,
        "dropped_missing_trade_price": dropped_price_count,
        "dropped_missing_volume": dropped_volume_count,
        "final_clean_rows": final_rows,
        "unique_assets_in_clean_panel": unique_assets,
        "unique_dates_in_clean_panel": unique_dates,
        "task_7_detailed_log": audit_log  # Nest the detailed Task 7 log for completeness
    }

    logger.info(f"Clean panel finalized. Contains {final_rows} rows, {unique_assets} assets, over {unique_dates} dates.")

    return df_cleaned, audit_summary

# -------------------------------------------------------------------------------------------------------------------------------
# Task 8, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_8_orchestrator(
    filtered_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    study_config: Dict[str, Any],
    task_7_audit_log: Dict[str, int]
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Orchestrates the missing-data imputation and cleaning conventions to produce the final clean market panel.

    Parameters
    ----------
    filtered_df : pd.DataFrame
        The structurally validated market panel from Task 7.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    task_7_audit_log : Dict[str, int]
        The audit log generated during Task 7.

    Returns
    -------
    Tuple[pd.DataFrame, Dict[str, Any]]
        A tuple containing the clean_market_df and the comprehensive row-count summary.

    Raises
    ------
    DataImputationError
        If the cleaning process fails to resolve all NaNs in critical columns.
    """
    logger.info("Commencing Task 8: Applying missing-data cleaning conventions.")

    # Execute Step 1: Forward-fill bid and ask prices
    df_filled, filled_count = forward_fill_bid_ask(
        filtered_df=filtered_df,
        master_trading_calendar=master_trading_calendar
    )

    # Execute Step 2: Handle missing trade prices and volumes
    df_cleaned, dropped_price, dropped_volume = handle_missing_trade_and_volume(
        df_filled=df_filled,
        study_config=study_config
    )

    # Execute Step 3: Finalize the panel and compile the summary
    clean_market_df, audit_summary = finalize_clean_panel(
        df_cleaned=df_cleaned,
        audit_log=task_7_audit_log,
        filled_bid_ask_count=filled_count,
        dropped_price_count=dropped_price,
        dropped_volume_count=dropped_volume
    )

    logger.info("Task 8 complete. Clean market panel emitted.")

    # Return the clean DataFrame and the summary dictionary
    return clean_market_df, audit_summary


In [ ]:
# Task 9: Construct the asset universe candidate set

class UniverseConstructionError(Exception):
    """Custom exception raised when the asset universe candidate pool is insufficient."""
    pass

# ==============================================================================
# Task 9: Construct the asset universe candidate set
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 9, Step 1: Identify historical S&P 500 constituents
# -------------------------------------------------------------------------------------------------------------------------------
def identify_sp500_candidates(clean_market_df: pd.DataFrame) -> Set[str]:
    """
    Identifies all assets that were historical S&P 500 constituents on at least one date.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel from Task 8.

    Returns
    -------
    Set[str]
        A set of asset_id strings that meet the historical constituent criteria.

    Raises
    ------
    UniverseConstructionError
        If no assets meet the criteria, indicating a severe upstream data error.
    """
    logger.info("Executing Task 9, Step 1: Identifying historical S&P 500 constituents.")

    # Group by asset_id and check if the constituent flag is True on ANY date
    # This is highly optimized in pandas
    ever_sp500_series = clean_market_df.groupby(level="asset_id")["is_historical_sp500_constituent"].any()

    # Extract the index values (asset_ids) where the condition is True
    # Cast to string to ensure consistent type handling downstream
    sp500_candidates = set(ever_sp500_series[ever_sp500_series].index.astype(str).tolist())

    candidate_count = len(sp500_candidates)
    logger.info(f"{candidate_count} assets were historical S&P 500 constituents on at least one date.")

    if candidate_count == 0:
        error_msg = "Zero assets identified as historical S&P 500 constituents. Check input data."
        logger.error(error_msg)
        raise UniverseConstructionError(error_msg)

    return sp500_candidates

# -------------------------------------------------------------------------------------------------------------------------------
# Task 9, Step 2: Apply sufficient-history filter
# -------------------------------------------------------------------------------------------------------------------------------
def apply_sufficient_history_filter(
    clean_market_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    coverage_threshold: float = 0.80
) -> Set[str]:
    """
    Filters assets based on their data history coverage relative to the master trading calendar.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    coverage_threshold : float, optional
        The minimum fraction of calendar dates an asset must have data for (default is 0.80).

    Returns
    -------
    Set[str]
        A set of asset_id strings that meet the sufficient history criteria.
    """
    logger.info(f"Executing Task 9, Step 2: Applying sufficient-history filter (threshold: {coverage_threshold:.1%}).")

    total_calendar_dates = len(master_trading_calendar)

    # Count the number of valid observations per asset
    # Since clean_market_df has no NaNs in adjusted_trade_price (enforced in Task 8),
    # the size of the group is exactly the number of valid observations.
    observation_counts = clean_market_df.groupby(level="asset_id").size()

    # Compute the coverage fraction relative to the FULL calendar, not just the asset's lifespan
    coverage_fractions = observation_counts / total_calendar_dates

    # Identify assets meeting the threshold
    sufficient_history_series = coverage_fractions >= coverage_threshold

    # Extract the asset_ids and cast to string
    history_candidates = set(sufficient_history_series[sufficient_history_series].index.astype(str).tolist())

    logger.info(f"{len(history_candidates)} assets met the {coverage_threshold:.1%} history coverage threshold.")

    return history_candidates

# -------------------------------------------------------------------------------------------------------------------------------
# Task 9, Step 3: Intersect candidates, verify counts, and emit funnel log
# -------------------------------------------------------------------------------------------------------------------------------
def finalize_candidate_set(
    sp500_candidates: Set[str],
    history_candidates: Set[str],
    total_clean_assets: int,
    required_universe_size: int = 434
) -> Tuple[List[str], Dict[str, int]]:
    """
    Intersects the candidate sets, verifies the minimum required count, and compiles the funnel log.

    Parameters
    ----------
    sp500_candidates : Set[str]
        Assets passing the S&P 500 filter.
    history_candidates : Set[str]
        Assets passing the history filter.
    total_clean_assets : int
        Total unique assets in the clean panel before filtering.
    required_universe_size : int, optional
        The target size of the final universe (default is 434).

    Returns
    -------
    Tuple[List[str], Dict[str, int]]
        The final list of candidate asset_ids and the filter funnel log.

    Raises
    ------
    UniverseConstructionError
        If the final candidate count is less than the required universe size.
    """
    logger.info("Executing Task 9, Step 3: Intersecting candidates and verifying counts.")

    # The final candidates must pass BOTH filters
    final_candidates_set = sp500_candidates.intersection(history_candidates)
    final_candidate_count = len(final_candidates_set)

    # Compile the funnel log
    funnel_log: Dict[str, int] = {
        "total_assets_in_clean_panel": total_clean_assets,
        "after_sp500_membership_filter": len(sp500_candidates),
        "after_sufficient_history_filter": len(history_candidates),
        "final_candidates_forwarded_to_ranking": final_candidate_count
    }

    logger.info(f"Filter funnel: {funnel_log}")

    # Verify that we have enough candidates to form the N=434 universe
    if final_candidate_count < required_universe_size:
        error_msg = (
            f"Insufficient candidates ({final_candidate_count} < {required_universe_size}) "
            f"to form the study universe. Relax the history filter or check input data."
        )
        logger.error(error_msg)
        raise UniverseConstructionError(error_msg)

    # Convert to a sorted list for deterministic downstream processing
    final_candidates_list = sorted(list(final_candidates_set))

    return final_candidates_list, funnel_log

# -------------------------------------------------------------------------------------------------------------------------------
# Task 9, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_9_orchestrator(
    clean_market_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[List[str], Dict[str, Any], Dict[str, int]]:
    """
    Orchestrates the construction of the asset universe candidate set.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[List[str], Dict[str, Any], Dict[str, int]]
        The list of candidate asset_ids, the augmented manifest, and the funnel log.

    Raises
    ------
    UniverseConstructionError
        If the candidate pool is too small to form the required universe.
    """
    logger.info("Commencing Task 9: Constructing the asset universe candidate set.")

    # Extract configuration parameters
    universe_config = study_config.get("ASSET_UNIVERSE_CONSTRUCTION", {})
    required_size = universe_config.get("final_universe_size", 434)

    # Determine the history threshold. The manuscript is unspecified, so we use a rational placeholder.
    # We default to 80% coverage.
    coverage_threshold = 0.80

    # Execute Step 1: Identify S&P 500 candidates
    sp500_candidates = identify_sp500_candidates(clean_market_df)

    # Execute Step 2: Apply history filter
    history_candidates = apply_sufficient_history_filter(
        clean_market_df=clean_market_df,
        master_trading_calendar=master_trading_calendar,
        coverage_threshold=coverage_threshold
    )

    # Execute Step 3: Intersect and verify
    total_clean_assets = clean_market_df.index.get_level_values("asset_id").nunique()
    candidate_asset_ids, funnel_log = finalize_candidate_set(
        sp500_candidates=sp500_candidates,
        history_candidates=history_candidates,
        total_clean_assets=total_clean_assets,
        required_universe_size=required_size
    )

    # Augment the provenance manifest with the placeholder convention
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "sufficient_history_filter",
        "manuscript_status": "unspecified",
        "value_used": f"{coverage_threshold * 100}% calendar coverage threshold",
        "source": "rational placeholder"
    })

    logger.info("Task 9 complete. Candidate set constructed and verified.")

    # Return the candidates, the updated manifest, and the funnel log
    return candidate_asset_ids, manifest, funnel_log


In [ ]:
# Task 10: Apply end-of-sample ranking and freeze the universe

class UniverseRankingError(Exception):
    """Custom exception raised when universe ranking or restriction fails."""
    pass

# ==============================================================================
# Task 10: Apply end-of-sample ranking and freeze the universe
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 10, Step 1: Extract market capitalization on the ranking date
# -------------------------------------------------------------------------------------------------------------------------------
def extract_ranking_market_caps(
    clean_market_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    candidate_asset_ids: List[str],
    study_config: Dict[str, Any]
) -> Tuple[pd.Series, pd.Timestamp]:
    """
    Determines the exact ranking date and extracts the most recent market cap for each candidate.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    candidate_asset_ids : List[str]
        The list of asset_ids that passed the Task 9 filters.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[pd.Series, pd.Timestamp]
        A Series mapping candidate asset_id to market_cap_usd, and the exact ranking date used.

    Raises
    ------
    UniverseRankingError
        If the ranking date cannot be determined or if no market cap data is available.
    """
    logger.info("Executing Task 10, Step 1: Extracting market caps on the ranking date.")

    # Determine the ranking date based on the config placeholder
    universe_config = study_config.get("ASSET_UNIVERSE_CONSTRUCTION", {})
    ranking_date_str = universe_config.get("market_cap_ranking_date", "")

    if "April_2025" in ranking_date_str:
        # Find the last trading day in April 2025
        april_2025_dates = master_trading_calendar[
            (master_trading_calendar.year == 2025) & (master_trading_calendar.month == 4)
        ]
        if not april_2025_dates.empty:
            ranking_date = april_2025_dates.max()
        else:
            # Fallback to the absolute maximum date in the calendar if April 2025 is not present
            ranking_date = master_trading_calendar.max()
            logger.warning(f"April 2025 not found in calendar. Falling back to max calendar date: {ranking_date.strftime('%Y-%m-%d')}")
    else:
        ranking_date = master_trading_calendar.max()

    logger.info(f"Exact ranking date determined as: {ranking_date.strftime('%Y-%m-%d')}")

    # Filter the panel to only include candidate assets and dates up to the ranking date
    # We use a 30-calendar-day lookback window to prevent using severely stale market caps for delisted assets
    lookback_date = ranking_date - pd.Timedelta(days=30)

    # Slice the MultiIndex efficiently
    idx = pd.IndexSlice
    try:
        # Extract the relevant slice
        slice_df = clean_market_df.loc[idx[lookback_date:ranking_date, candidate_asset_ids], ["market_cap_usd"]]
    except KeyError:
        raise UniverseRankingError("Failed to slice clean_market_df. Ensure index is sorted and candidates exist.")

    if slice_df.empty:
        raise UniverseRankingError(f"No market cap data found for any candidates between {lookback_date.date()} and {ranking_date.date()}.")

    # Group by asset_id and take the last available observation within the window
    # .last() returns the last non-null observation due to pandas groupby behavior
    latest_mcap_df = slice_df.groupby(level="asset_id").last()

    # Reindex to ensure all candidates are present. Missing ones get NaN (which we fill with 0.0)
    mcap_series = latest_mcap_df["market_cap_usd"].reindex(candidate_asset_ids).fillna(0.0)

    missing_count = (mcap_series == 0.0).sum()
    if missing_count > 0:
        logger.info(f"{missing_count} candidates had no market cap data within 30 days of the ranking date. Assigned 0.0.")

    return mcap_series, ranking_date

# -------------------------------------------------------------------------------------------------------------------------------
# Task 10, Step 2: Sort candidates and retain the top N assets
# -------------------------------------------------------------------------------------------------------------------------------
def sort_and_truncate_universe(
    mcap_series: pd.Series,
    required_universe_size: int = 434
) -> Tuple[str, ...]:
    """
    Sorts candidates descending by market cap (with deterministic tie-breaking) and retains the top N.

    Parameters
    ----------
    mcap_series : pd.Series
        Series mapping candidate asset_id to market_cap_usd.
    required_universe_size : int, optional
        The target size of the final universe (default is 434).

    Returns
    -------
    Tuple[str, ...]
        A frozen tuple of exactly 434 asset_ids in canonical order.

    Raises
    ------
    UniverseRankingError
        If there are fewer candidates than the required universe size.
    """
    logger.info(f"Executing Task 10, Step 2: Sorting candidates and retaining top {required_universe_size}.")

    if len(mcap_series) < required_universe_size:
        raise UniverseRankingError(f"Cannot select top {required_universe_size} from {len(mcap_series)} candidates.")

    # Convert Series to DataFrame to allow multi-column sorting for deterministic tie-breaking
    sort_df = mcap_series.reset_index()
    sort_df.columns = ["asset_id", "market_cap_usd"]

    # Sort by market_cap_usd DESCENDING, then by asset_id ASCENDING (tie-breaker)
    # na_position="last" ensures any residual NaNs fall to the bottom
    sorted_df = sort_df.sort_values(
        by=["market_cap_usd", "asset_id"],
        ascending=[False, True],
        na_position="last"
    )

    # Truncate to the required universe size
    top_n_df = sorted_df.head(required_universe_size)

    # Extract the asset_ids as a tuple to enforce immutability downstream
    universe_asset_ids = tuple(top_n_df["asset_id"].tolist())

    # Log the market cap range for sanity checking
    max_cap = top_n_df["market_cap_usd"].max()
    min_cap = top_n_df["market_cap_usd"].min()
    logger.info(f"Universe frozen. Market cap range on ranking date: ${min_cap:,.2f} to ${max_cap:,.2f}.")

    # Rigorous invariant assertion
    assert len(universe_asset_ids) == required_universe_size, "Truncated universe size mismatch."

    return universe_asset_ids

# -------------------------------------------------------------------------------------------------------------------------------
# Task 10, Step 3: Restrict the market panel to the frozen universe
# -------------------------------------------------------------------------------------------------------------------------------
def restrict_panel_to_universe(
    clean_market_df: pd.DataFrame,
    universe_asset_ids: Tuple[str, ...]
) -> pd.DataFrame:
    """
    Filters the clean market panel to include only the assets in the frozen universe.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of 434 selected asset_ids.

    Returns
    -------
    pd.DataFrame
        The restricted market panel containing only the universe assets.

    Raises
    ------
    UniverseRankingError
        If the restriction results in an incorrect number of unique assets.
    """
    logger.info("Executing Task 10, Step 3: Restricting market panel to the frozen universe.")

    # Convert tuple to list for pandas indexing
    universe_list = list(universe_asset_ids)

    # Slice the MultiIndex to retain only the universe assets
    idx = pd.IndexSlice
    universe_market_df = clean_market_df.loc[idx[:, universe_list], :].copy()

    # Ensure the resulting DataFrame is lexicographically sorted
    if not universe_market_df.index.is_monotonic_increasing:
        universe_market_df = universe_market_df.sort_index()

    # Verify the restriction was successful
    unique_assets_retained = universe_market_df.index.get_level_values("asset_id").nunique()
    if unique_assets_retained != len(universe_asset_ids):
        error_msg = f"Panel restriction failed. Expected {len(universe_asset_ids)} assets, got {unique_assets_retained}."
        logger.error(error_msg)
        raise UniverseRankingError(error_msg)

    logger.info(f"Panel successfully restricted. Final shape: {universe_market_df.shape}.")

    return universe_market_df

# -------------------------------------------------------------------------------------------------------------------------------
# Task 10, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_10_orchestrator(
    clean_market_df: pd.DataFrame,
    master_trading_calendar: pd.DatetimeIndex,
    candidate_asset_ids: List[str],
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[Tuple[str, ...], pd.DataFrame, Dict[str, Any]]:
    """
    Orchestrates the end-of-sample ranking, universe freezing, and panel restriction.

    Parameters
    ----------
    clean_market_df : pd.DataFrame
        The fully cleaned market panel.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    candidate_asset_ids : List[str]
        The list of asset_ids that passed the Task 9 filters.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Tuple[str, ...], pd.DataFrame, Dict[str, Any]]
        The frozen universe tuple, the restricted market panel, and the augmented manifest.

    Raises
    ------
    UniverseRankingError
        If ranking, sorting, or restriction fails.
    """
    logger.info("Commencing Task 10: Applying end-of-sample ranking and freezing the universe.")

    required_size = study_config.get("ASSET_UNIVERSE_CONSTRUCTION", {}).get("final_universe_size", 434)

    # Execute Step 1: Extract market caps on the ranking date
    mcap_series, exact_ranking_date = extract_ranking_market_caps(
        clean_market_df=clean_market_df,
        master_trading_calendar=master_trading_calendar,
        candidate_asset_ids=candidate_asset_ids,
        study_config=study_config
    )

    # Execute Step 2: Sort and truncate to exactly N=434
    universe_asset_ids = sort_and_truncate_universe(
        mcap_series=mcap_series,
        required_universe_size=required_size
    )

    # Execute Step 3: Restrict the panel
    universe_market_df = restrict_panel_to_universe(
        clean_market_df=clean_market_df,
        universe_asset_ids=universe_asset_ids
    )

    # Augment the provenance manifest
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["universe_ranking_date"] = exact_ranking_date.isoformat()

    manifest["placeholder_assumptions"].append({
        "convention_name": "market_cap_tie_breaker",
        "manuscript_status": "unspecified",
        "value_used": "ascending asset_id",
        "source": "rational placeholder"
    })

    logger.info("Task 10 complete. Universe frozen and panel restricted.")

    # Return the frozen tuple, the restricted panel, and the updated manifest
    return universe_asset_ids, universe_market_df, manifest


In [ ]:
# Task 11: Align the risk-free rate series to the master trading calendar

class MacroDataAlignmentError(Exception):
    """Custom exception raised when the macro-financial series cannot be properly aligned or contains missing values."""
    pass

# ==============================================================================
# Task 11: Align the risk-free rate series to the master trading calendar
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 11, Step 1: Reindex and forward-fill the risk-free rate series
# -------------------------------------------------------------------------------------------------------------------------------
def align_and_normalize_rf_series(
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex
) -> pd.Series:
    """
    Aligns the risk-free rate series to the trading calendar via forward-fill and normalizes percentages to decimals.

    Parameters
    ----------
    risk_free_rate_series : pd.Series
        The raw 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    pd.Series
        The calendar-aligned, decimal-normalized annual risk-free rate series.
    """
    logger.info("Executing Task 11, Step 1: Aligning and normalizing the risk-free rate series.")

    # Ensure the input series index is tz-naive UTC to match the calendar
    if pd.api.types.is_datetime64tz_dtype(risk_free_rate_series.index):
        risk_free_rate_series.index = risk_free_rate_series.index.tz_convert("UTC").tz_localize(None)

    # Reindex the series onto the master trading calendar.
    # We use method='ffill' to propagate the most recent FRED observation forward.
    # This strictly prevents look-ahead bias.
    aligned_rf_annual = risk_free_rate_series.reindex(master_trading_calendar, method='ffill')

    # Check if the rates are provided as percentages (e.g., 4.5) rather than decimals (0.045).
    # If the maximum value exceeds 1.0, it is highly probable they are percentages.
    max_rate = aligned_rf_annual.max()
    if max_rate > 1.0:
        logger.info(f"Maximum risk-free rate is {max_rate:.2f}. Normalizing percentages to decimals (dividing by 100).")
        aligned_rf_annual = aligned_rf_annual / 100.0
    else:
        logger.info(f"Maximum risk-free rate is {max_rate:.4f}. Assuming data is already in decimal format.")

    return aligned_rf_annual

# -------------------------------------------------------------------------------------------------------------------------------
# Task 11, Step 2: Convert annualized rate to daily equivalent
# -------------------------------------------------------------------------------------------------------------------------------
def convert_annual_to_daily_rate(
    aligned_rf_annual: pd.Series,
    study_config: Dict[str, Any]
) -> pd.Series:
    """
    Converts the annualized risk-free rate to a daily equivalent using the configured convention.

    Parameters
    ----------
    aligned_rf_annual : pd.Series
        The calendar-aligned, decimal-normalized annual risk-free rate series.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    pd.Series
        The daily equivalent risk-free rate series.
    """
    logger.info("Executing Task 11, Step 2: Converting annualized risk-free rate to daily equivalent.")

    # Extract the annualization basis from the configuration
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    annualization_basis = firm_consts.get("annualization_basis_trading_days", 252)

    # Apply the linear scaling convention: r_daily = r_annual / 252
    # This is a standard institutional placeholder when exact compounding is unspecified.
    rf_daily_series = aligned_rf_annual / float(annualization_basis)

    # Enforce the canonical name for downstream identification
    rf_daily_series.name = "rf_daily"

    logger.info(f"Applied linear daily conversion: divided annual rate by {annualization_basis}.")

    return rf_daily_series

# -------------------------------------------------------------------------------------------------------------------------------
# Task 11, Step 3: Verify completeness over valid decision dates
# -------------------------------------------------------------------------------------------------------------------------------
def verify_rf_completeness(
    rf_daily_series: pd.Series,
    valid_decision_dates: pd.DatetimeIndex
) -> None:
    """
    Verifies that the daily risk-free rate series contains no missing values on any valid decision date.

    Parameters
    ----------
    rf_daily_series : pd.Series
        The daily equivalent risk-free rate series.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.

    Raises
    ------
    MacroDataAlignmentError
        If any NaN values are detected within the valid decision dates.
    """
    logger.info("Executing Task 11, Step 3: Verifying risk-free rate completeness over valid decision dates.")

    try:
        # Extract the subset of the series corresponding to the valid decision dates
        rf_on_valid_dates = rf_daily_series.loc[valid_decision_dates]
    except KeyError as e:
        error_msg = f"Failed to slice rf_daily_series with valid_decision_dates. Index mismatch: {e}"
        logger.error(error_msg)
        raise MacroDataAlignmentError(error_msg)

    # Check for any missing values
    missing_mask = rf_on_valid_dates.isna()
    if missing_mask.any():
        missing_dates = rf_on_valid_dates[missing_mask].index
        error_msg = (
            f"Critical failure: rf_daily_series contains {len(missing_dates)} missing values "
            f"within the valid decision dates. Sample dates: {missing_dates[:5].tolist()}"
        )
        logger.error(error_msg)
        raise MacroDataAlignmentError(error_msg)

    # Verify the length matches exactly
    if len(rf_on_valid_dates) != len(valid_decision_dates):
        error_msg = (
            f"Length mismatch: rf_on_valid_dates has {len(rf_on_valid_dates)} entries, "
            f"but valid_decision_dates has {len(valid_decision_dates)}."
        )
        logger.error(error_msg)
        raise MacroDataAlignmentError(error_msg)

    logger.info("Risk-free rate series is fully populated for all valid decision dates.")

# -------------------------------------------------------------------------------------------------------------------------------
# Task 11, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_11_orchestrator(
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    valid_decision_dates: pd.DatetimeIndex,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[pd.Series, Dict[str, Any]]:
    """
    Orchestrates the alignment, normalization, and daily conversion of the risk-free rate series.

    Parameters
    ----------
    risk_free_rate_series : pd.Series
        The raw 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[pd.Series, Dict[str, Any]]
        The finalized daily risk-free rate series and the augmented manifest.

    Raises
    ------
    MacroDataAlignmentError
        If alignment fails or missing values persist.
    """
    logger.info("Commencing Task 11: Aligning the risk-free rate series to the master trading calendar.")

    # Execute Step 1: Align to calendar and normalize percentages
    aligned_rf_annual = align_and_normalize_rf_series(
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar
    )

    # Execute Step 2: Convert to daily rate
    rf_daily_series = convert_annual_to_daily_rate(
        aligned_rf_annual=aligned_rf_annual,
        study_config=study_config
    )

    # Execute Step 3: Verify completeness
    verify_rf_completeness(
        rf_daily_series=rf_daily_series,
        valid_decision_dates=valid_decision_dates
    )

    # Augment the provenance manifest with the placeholder convention
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "rf_daily_conversion_logic",
        "manuscript_status": "unspecified",
        "value_used": "linear division: r_annual / 252",
        "source": "rational placeholder"
    })

    logger.info("Task 11 complete. Risk-free rate series aligned and converted.")

    # Return the finalized series and the updated manifest
    return rf_daily_series, manifest


In [ ]:
# Task 12: Construct the reference midpoint and fractional spread

class SpreadCalculationError(Exception):
    """Custom exception raised when the bid-ask spread calculation encounters numerical singularities or structural failures."""
    pass

# ==============================================================================
# Task 12: Construct the reference midpoint and fractional spread
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 12, Step 1: Compute the reference midpoint price
# -------------------------------------------------------------------------------------------------------------------------------
def compute_reference_midpoint(universe_market_df: pd.DataFrame) -> pd.Series:
    """
    Computes the reference midpoint price for each (date, asset) pair in the universe.

    Equation: p_{ref,t,j} = (adjusted_bid_price_{t,j} + adjusted_ask_price_{t,j}) / 2

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel with a ["date", "asset_id"] MultiIndex.

    Returns
    -------
    pd.Series
        A Series containing the reference midpoint prices, indexed identically to the input DataFrame.

    Raises
    ------
    SpreadCalculationError
        If any computed midpoint price is less than or equal to zero, or if NaNs are generated.
    """
    logger.info("Executing Task 12, Step 1: Computing reference midpoint prices.")

    # Extract bid and ask series for vectorized computation
    bid_prices = universe_market_df["adjusted_bid_price"]
    ask_prices = universe_market_df["adjusted_ask_price"]

    # Compute the midpoint price: p_ref = (bid + ask) / 2
    reference_mid_price = (bid_prices + ask_prices) / 2.0
    reference_mid_price.name = "reference_mid_price"

    # Diagnostic: Check for crossed markets (bid > ask)
    crossed_market_mask = bid_prices > ask_prices
    crossed_count = int(crossed_market_mask.sum())
    if crossed_count > 0:
        logger.warning(f"Detected {crossed_count} instances of crossed markets (bid > ask). Midpoints computed normally.")

    # Rigorous invariant assertion: Midpoint prices must be strictly positive
    non_positive_mask = reference_mid_price <= 0.0
    if non_positive_mask.any():
        error_msg = f"Critical failure: {non_positive_mask.sum()} midpoint prices are <= 0.0. This violates price positivity invariants."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    # Verify absence of NaNs
    if reference_mid_price.isna().any():
        error_msg = "Critical failure: NaN values generated during midpoint computation."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    logger.info("Reference midpoint prices computed successfully.")
    return reference_mid_price

# -------------------------------------------------------------------------------------------------------------------------------
# Task 12, Step 2: Compute the fractional bid-ask spread
# -------------------------------------------------------------------------------------------------------------------------------
def compute_fractional_spread(
    universe_market_df: pd.DataFrame,
    reference_mid_price: pd.Series
) -> pd.Series:
    """
    Computes the unitless fractional bid-ask spread for each (date, asset) pair.

    Equation: kappa_{spread,t,j} = (adjusted_ask_price_{t,j} - adjusted_bid_price_{t,j}) / p_{ref,t,j}

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel.
    reference_mid_price : pd.Series
        The strictly positive reference midpoint prices computed in Step 1.

    Returns
    -------
    pd.Series
        A Series containing the fractional bid-ask spreads.

    Raises
    ------
    SpreadCalculationError
        If the computation results in infinite or NaN values.
    """
    logger.info("Executing Task 12, Step 2: Computing fractional bid-ask spreads.")

    # Extract bid and ask series
    bid_prices = universe_market_df["adjusted_bid_price"]
    ask_prices = universe_market_df["adjusted_ask_price"]

    # Compute the fractional spread
    # The denominator is guaranteed > 0 by Step 1, preventing ZeroDivisionError
    fractional_spread = (ask_prices - bid_prices) / reference_mid_price
    fractional_spread.name = "fractional_spread"

    # Verify absence of infinite values (which could occur from extreme floating-point underflow)
    if np.isinf(fractional_spread).any():
        error_msg = "Critical failure: Infinite values generated during fractional spread computation."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    # Verify absence of NaNs
    if fractional_spread.isna().any():
        error_msg = "Critical failure: NaN values generated during fractional spread computation."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    logger.info("Fractional bid-ask spreads computed successfully.")
    return fractional_spread

# -------------------------------------------------------------------------------------------------------------------------------
# Task 12, Step 3: Construct the wide-format daily spread panel
# -------------------------------------------------------------------------------------------------------------------------------
def construct_wide_spread_panel(
    fractional_spread: pd.Series,
    valid_decision_dates: pd.DatetimeIndex,
    universe_asset_ids: Tuple[str, ...]
) -> pd.DataFrame:
    """
    Transforms the long-format spread series into a canonical wide-format matrix, enforcing non-negativity.

    Parameters
    ----------
    fractional_spread : pd.Series
        The long-format fractional bid-ask spreads with a ["date", "asset_id"] MultiIndex.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.

    Returns
    -------
    pd.DataFrame
        A wide-format DataFrame of shape (len(valid_decision_dates), 434) containing non-negative spreads.

    Raises
    ------
    SpreadCalculationError
        If the final panel shape does not exactly match the required dimensions.
    """
    logger.info("Executing Task 12, Step 3: Constructing the wide-format daily spread panel.")

    # Unstack the asset_id level to pivot the data from long to wide format
    # Rows become dates, columns become asset_ids
    spread_wide = fractional_spread.unstack(level="asset_id")

    # Reindex both axes to enforce the canonical temporal and cross-sectional dimensions.
    # Missing entries (e.g., assets not trading on a specific date) are filled with 0.0.
    # A spread of 0.0 is numerically safe because untradeable assets will have their trade weights
    # constrained to zero in the PM optimization, nullifying the cost term.
    canonical_columns = list(universe_asset_ids)
    spread_wide = spread_wide.reindex(
        index=valid_decision_dates,
        columns=canonical_columns,
        fill_value=0.0
    )

    # Enforce non-negativity: clamp any negative spreads (from crossed markets) to 0.0
    negative_mask = spread_wide < 0.0
    negative_count = int(negative_mask.sum().sum())
    if negative_count > 0:
        logger.warning(f"Clamping {negative_count} negative spread entries to 0.0 to satisfy cost model requirements.")
        spread_wide = spread_wide.clip(lower=0.0)

    # Rigorous invariant assertion: Verify exact shape
    expected_shape = (len(valid_decision_dates), len(universe_asset_ids))
    if spread_wide.shape != expected_shape:
        error_msg = f"Shape mismatch: Expected {expected_shape}, got {spread_wide.shape}."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    # Rigorous invariant assertion: Verify strict non-negativity
    if (spread_wide < 0.0).any().any():
        error_msg = "Critical failure: Negative spreads persist after clamping."
        logger.error(error_msg)
        raise SpreadCalculationError(error_msg)

    logger.info(f"Wide-format spread panel constructed successfully. Shape: {spread_wide.shape}.")
    return spread_wide

# -------------------------------------------------------------------------------------------------------------------------------
# Task 12, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_12_orchestrator(
    universe_market_df: pd.DataFrame,
    valid_decision_dates: pd.DatetimeIndex,
    universe_asset_ids: Tuple[str, ...]
) -> pd.DataFrame:
    """
    Orchestrates the computation of reference midpoints, fractional spreads, and the wide-format spread panel.

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.

    Returns
    -------
    pd.DataFrame
        The canonical wide-format daily spread panel (kappa_spread_t).

    Raises
    ------
    SpreadCalculationError
        If any numerical singularities or structural failures occur during computation.
    """
    logger.info("Commencing Task 12: Constructing the reference midpoint and fractional spread.")

    # Execute Step 1: Compute reference midpoint prices
    reference_mid_price = compute_reference_midpoint(
        universe_market_df=universe_market_df
    )

    # Execute Step 2: Compute fractional bid-ask spreads
    fractional_spread = compute_fractional_spread(
        universe_market_df=universe_market_df,
        reference_mid_price=reference_mid_price
    )

    # Execute Step 3: Construct the canonical wide-format panel
    kappa_spread_panel = construct_wide_spread_panel(
        fractional_spread=fractional_spread,
        valid_decision_dates=valid_decision_dates,
        universe_asset_ids=universe_asset_ids
    )

    logger.info("Task 12 complete. Canonical spread panel emitted.")

    # Return the finalized wide-format spread panel
    return kappa_spread_panel


In [ ]:
# Task 13: Construct dollar volume and daily asset returns

class MarketFeatureError(Exception):
    """Custom exception raised when market feature computation (volume or returns) fails structural invariants."""
    pass

# ==============================================================================
# Task 13: Construct dollar volume and daily asset returns
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 13, Step 1: Compute daily dollar volume
# -------------------------------------------------------------------------------------------------------------------------------
def compute_dollar_volume_panel(
    universe_market_df: pd.DataFrame,
    universe_asset_ids: Tuple[str, ...],
    master_trading_calendar: pd.DatetimeIndex
) -> pd.DataFrame:
    """
    Computes the daily dollar volume for each asset and constructs a canonical wide-format panel.

    Equation: omega_{t,j} = adjusted_trade_price_{t,j} * share_volume_{t,j}

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    pd.DataFrame
        A wide-format DataFrame of shape (len(master_trading_calendar), 434) containing non-negative dollar volumes.

    Raises
    ------
    MarketFeatureError
        If negative dollar volumes are generated or if the final shape is incorrect.
    """
    logger.info("Executing Task 13, Step 1: Computing daily dollar volume panel.")

    # Extract price and volume series for vectorized computation
    trade_prices = universe_market_df["adjusted_trade_price"]
    share_volumes = universe_market_df["share_volume"]

    # Compute element-wise dollar volume: omega_{t,j} = p_{t,j} * v_{t,j}
    dollar_volume = trade_prices * share_volumes
    dollar_volume.name = "dollar_volume"

    # Unstack the asset_id level to pivot the data from long to wide format
    # Rows become dates, columns become asset_ids
    dollar_volume_wide = dollar_volume.unstack(level="asset_id")

    # Reindex both axes to enforce the canonical temporal and cross-sectional dimensions.
    # Missing entries (e.g., assets not trading on a specific date or not yet listed) are filled with 0.0.
    canonical_columns = list(universe_asset_ids)
    dollar_volume_wide = dollar_volume_wide.reindex(
        index=master_trading_calendar,
        columns=canonical_columns,
        fill_value=0.0
    )

    # Diagnostic: Count exactly zero entries
    zero_volume_mask = dollar_volume_wide == 0.0
    zero_count = int(zero_volume_mask.sum().sum())
    if zero_count > 0:
        logger.info(f"Detected {zero_count} zero-volume entries in the wide panel. These will be handled downstream.")

    # Rigorous invariant assertion: Verify strict non-negativity
    if (dollar_volume_wide < 0.0).any().any():
        error_msg = "Critical failure: Negative dollar volumes computed. Check input price and volume non-negativity."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    # Rigorous invariant assertion: Verify exact shape
    expected_shape = (len(master_trading_calendar), len(universe_asset_ids))
    if dollar_volume_wide.shape != expected_shape:
        error_msg = f"Shape mismatch in dollar volume panel: Expected {expected_shape}, got {dollar_volume_wide.shape}."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    logger.info(f"Dollar volume panel constructed successfully. Shape: {dollar_volume_wide.shape}.")
    return dollar_volume_wide

# -------------------------------------------------------------------------------------------------------------------------------
# Task 13, Step 2: Compute one-period realized returns
# -------------------------------------------------------------------------------------------------------------------------------
def compute_daily_returns_panel(
    universe_market_df: pd.DataFrame,
    universe_asset_ids: Tuple[str, ...],
    master_trading_calendar: pd.DatetimeIndex
) -> pd.DataFrame:
    """
    Computes the one-period realized returns for each asset and constructs a canonical wide-format panel.

    Equation: r_{t,j} = (p_{t,j}^{adj} / p_{t-1,j}^{adj}) - 1

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    pd.DataFrame
        A wide-format DataFrame of shape (len(master_trading_calendar), 434) containing daily returns.

    Raises
    ------
    MarketFeatureError
        If the temporal shift operation fails or if the final shape is incorrect.
    """
    logger.info("Executing Task 13, Step 2: Computing daily realized returns panel.")

    # Extract the adjusted trade price series
    trade_prices = universe_market_df["adjusted_trade_price"]

    # Unstack the asset_id level FIRST to ensure the shift operation does not cross-contaminate assets
    price_wide = trade_prices.unstack(level="asset_id")

    # Reindex to the master trading calendar to ensure all dates are present in chronological order
    # We do NOT fill NaNs here; missing prices should result in missing returns
    canonical_columns = list(universe_asset_ids)
    price_wide = price_wide.reindex(
        index=master_trading_calendar,
        columns=canonical_columns
    )

    # Compute one-period returns: r_{t,j} = (p_{t,j} / p_{t-1,j}) - 1
    # The .shift(1) operates along axis=0 (dates) because the index is the calendar
    returns_wide = (price_wide / price_wide.shift(1)) - 1.0

    # Diagnostic: Identify and log extreme returns (> 50% or < -50%)
    extreme_returns_mask = (returns_wide > 0.50) | (returns_wide < -0.50)
    extreme_count = int(extreme_returns_mask.sum().sum())
    if extreme_count > 0:
        logger.warning(f"Detected {extreme_count} extreme daily returns (> ±50%). These are retained but flagged for audit.")

    # Rigorous invariant assertion: The first row must be entirely NaN because there is no prior price
    if not returns_wide.iloc[0].isna().all():
        error_msg = "Critical failure: The first row of the returns panel is not entirely NaN. Shift logic failed."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    # Rigorous invariant assertion: Verify exact shape
    expected_shape = (len(master_trading_calendar), len(universe_asset_ids))
    if returns_wide.shape != expected_shape:
        error_msg = f"Shape mismatch in returns panel: Expected {expected_shape}, got {returns_wide.shape}."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    logger.info(f"Daily returns panel constructed successfully. Shape: {returns_wide.shape}.")
    return returns_wide

# -------------------------------------------------------------------------------------------------------------------------------
# Task 13, Step 3: Verify alignment and augment the provenance manifest
# -------------------------------------------------------------------------------------------------------------------------------
def verify_and_augment_manifest(
    dollar_volume_panel: pd.DataFrame,
    daily_returns_panel: pd.DataFrame,
    universe_asset_ids: Tuple[str, ...],
    master_trading_calendar: pd.DatetimeIndex,
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Verifies the structural alignment of both panels and records their metadata in the provenance manifest.

    Parameters
    ----------
    dollar_volume_panel : pd.DataFrame
        The wide-format dollar volume panel.
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Dict[str, Any]
        The augmented provenance manifest containing panel metadata.

    Raises
    ------
    MarketFeatureError
        If the panels do not perfectly align with the canonical universe or calendar.
    """
    logger.info("Executing Task 13, Step 3: Verifying panel alignment and augmenting manifest.")

    # 1. Verify identical column indices matching the canonical universe
    canonical_columns = pd.Index(list(universe_asset_ids))

    if not dollar_volume_panel.columns.equals(canonical_columns):
        error_msg = "Column mismatch: dollar_volume_panel columns do not match universe_asset_ids."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    if not daily_returns_panel.columns.equals(canonical_columns):
        error_msg = "Column mismatch: daily_returns_panel columns do not match universe_asset_ids."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    # 2. Verify row indices exactly match the master trading calendar
    if not dollar_volume_panel.index.equals(master_trading_calendar):
        error_msg = "Row mismatch: dollar_volume_panel index does not match master_trading_calendar."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    if not daily_returns_panel.index.equals(master_trading_calendar):
        error_msg = "Row mismatch: daily_returns_panel index does not match master_trading_calendar."
        logger.error(error_msg)
        raise MarketFeatureError(error_msg)

    # 3. Extract metadata for the manifest
    vol_nans = int(dollar_volume_panel.isna().sum().sum())
    vol_zeros = int((dollar_volume_panel == 0.0).sum().sum())
    ret_nans = int(daily_returns_panel.isna().sum().sum())
    ret_extremes = int(((daily_returns_panel > 0.50) | (daily_returns_panel < -0.50)).sum().sum())

    # 4. Augment the manifest
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["dollar_volume_panel"] = {
        "shape": dollar_volume_panel.shape,
        "nan_count": vol_nans,
        "zero_count": vol_zeros,
        "computation_convention": "adjusted_trade_price * share_volume (manuscript-unspecified placeholder)"
    }

    manifest["derived_artifacts_metadata"]["daily_returns_panel"] = {
        "shape": daily_returns_panel.shape,
        "nan_count": ret_nans,
        "extreme_returns_count": ret_extremes,
        "computation_convention": "standard one-period total return"
    }

    # Record the placeholder assumption
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "dollar_volume_price_field",
        "manuscript_status": "unspecified",
        "value_used": "adjusted_trade_price",
        "source": "rational placeholder"
    })

    logger.info("Panel alignment verified. Manifest augmented with artifact metadata.")
    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 13, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_13_orchestrator(
    universe_market_df: pd.DataFrame,
    universe_asset_ids: Tuple[str, ...],
    master_trading_calendar: pd.DatetimeIndex,
    manifest: Dict[str, Any]
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Orchestrates the computation of daily dollar volume and realized returns panels.

    Parameters
    ----------
    universe_market_df : pd.DataFrame
        The fully cleaned, universe-restricted market panel.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids in canonical order.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]
        The dollar volume panel, the daily returns panel, and the augmented manifest.

    Raises
    ------
    MarketFeatureError
        If any structural invariants or mathematical constraints are violated during computation.
    """
    logger.info("Commencing Task 13: Constructing dollar volume and daily asset returns.")

    # Execute Step 1: Compute dollar volume panel
    dollar_volume_panel = compute_dollar_volume_panel(
        universe_market_df=universe_market_df,
        universe_asset_ids=universe_asset_ids,
        master_trading_calendar=master_trading_calendar
    )

    # Execute Step 2: Compute daily returns panel
    daily_returns_panel = compute_daily_returns_panel(
        universe_market_df=universe_market_df,
        universe_asset_ids=universe_asset_ids,
        master_trading_calendar=master_trading_calendar
    )

    # Execute Step 3: Verify alignment and augment manifest
    manifest = verify_and_augment_manifest(
        dollar_volume_panel=dollar_volume_panel,
        daily_returns_panel=daily_returns_panel,
        universe_asset_ids=universe_asset_ids,
        master_trading_calendar=master_trading_calendar,
        manifest=manifest
    )

    logger.info("Task 13 complete. Dollar volume and daily returns panels emitted.")

    # Return the finalized wide-format panels and the updated manifest
    return dollar_volume_panel, daily_returns_panel, manifest


In [ ]:
# Task 14: Construct forward realized return objects

class ForwardReturnError(Exception):
    """Custom exception raised when forward return window extraction or computation fails."""
    pass

# ==============================================================================
# Task 14: Construct forward realized return objects
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 14, Step 1: Extract the matrix of daily returns over the next 42 trading days
# -------------------------------------------------------------------------------------------------------------------------------
def extract_forward_return_windows(
    daily_returns_panel: pd.DataFrame,
    valid_decision_dates: pd.DatetimeIndex,
    master_trading_calendar: pd.DatetimeIndex
) -> Dict[pd.Timestamp, np.ndarray]:
    """
    Extracts a 42-day forward-looking window of daily returns for each valid decision date.

    Parameters
    ----------
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel of shape (T, N).
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.

    Returns
    -------
    Dict[pd.Timestamp, np.ndarray]
        A dictionary mapping each valid decision date to a NumPy array of shape (42, 434).

    Raises
    ------
    ForwardReturnError
        If the extracted windows do not possess the exact required shape of (42, N).
    """
    logger.info("Executing Task 14, Step 1: Extracting 42-day forward return windows.")

    forward_window_days = 42
    n_assets = daily_returns_panel.shape[1]

    # Extract the underlying NumPy array for high-performance, zero-copy slicing
    returns_array = daily_returns_panel.to_numpy()

    forward_return_windows: Dict[pd.Timestamp, np.ndarray] = {}

    # By definition (Task 6), valid_decision_dates are exactly the first L-42 dates of the calendar.
    # Therefore, the i-th valid date corresponds to the i-th row in the returns_array.
    # The 42 forward days strictly after date i are at indices [i+1 : i+1+42].
    for i, current_date in enumerate(valid_decision_dates):
        # Slice the array: rows i+1 through i+42 inclusive
        start_idx = i + 1
        end_idx = start_idx + forward_window_days

        window_matrix = returns_array[start_idx:end_idx, :]

        # Rigorous invariant assertion: Verify exact shape
        if window_matrix.shape != (forward_window_days, n_assets):
            error_msg = (
                f"Shape mismatch for date {current_date.strftime('%Y-%m-%d')}: "
                f"Expected {(forward_window_days, n_assets)}, got {window_matrix.shape}."
            )
            logger.error(error_msg)
            raise ForwardReturnError(error_msg)

        forward_return_windows[current_date] = window_matrix

    logger.info(f"Successfully extracted {len(forward_return_windows)} forward return windows.")
    return forward_return_windows

# -------------------------------------------------------------------------------------------------------------------------------
# Task 14, Step 2: Compute the scalar forward realized return vector
# -------------------------------------------------------------------------------------------------------------------------------
def compute_compound_forward_returns(
    forward_return_windows: Dict[pd.Timestamp, np.ndarray]
) -> Dict[pd.Timestamp, np.ndarray]:
    """
    Computes the scalar forward realized return vector R_t for each decision date.

    Equation: R_{t,j} = \prod_{s=1}^{42}(1 + r_{t+s,j}) - 1

    Parameters
    ----------
    forward_return_windows : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to their (42, N) forward return matrices.

    Returns
    -------
    Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to their (N,) compound forward return vectors.
    """
    logger.info("Executing Task 14, Step 2: Computing compound forward realized returns.")

    forward_return_vectors: Dict[pd.Timestamp, np.ndarray] = {}

    for current_date, window_matrix in forward_return_windows.items():
        # Compute the compound return along the temporal axis (axis=0)
        # np.prod propagates NaNs, which is mathematically correct: if any daily return
        # in the 42-day window is missing, the 42-day compound return is undefined.
        compound_return_vector = np.prod(1.0 + window_matrix, axis=0) - 1.0

        forward_return_vectors[current_date] = compound_return_vector

    logger.info(f"Successfully computed {len(forward_return_vectors)} compound forward return vectors.")
    return forward_return_vectors

# -------------------------------------------------------------------------------------------------------------------------------
# Task 14, Step 3: Verify completeness and augment the provenance manifest
# -------------------------------------------------------------------------------------------------------------------------------
def verify_and_persist_forward_returns(
    forward_return_windows: Dict[pd.Timestamp, np.ndarray],
    forward_return_vectors: Dict[pd.Timestamp, np.ndarray],
    valid_decision_dates: pd.DatetimeIndex,
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Verifies the structural completeness of the forward return objects and augments the manifest.

    Parameters
    ----------
    forward_return_windows : Dict[pd.Timestamp, np.ndarray]
        Dictionary of forward return matrices.
    forward_return_vectors : Dict[pd.Timestamp, np.ndarray]
        Dictionary of compound forward return vectors.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Dict[str, Any]
        The augmented provenance manifest.

    Raises
    ------
    ForwardReturnError
        If the dictionaries are missing dates or contain malformed arrays.
    """
    logger.info("Executing Task 14, Step 3: Verifying completeness and augmenting manifest.")

    expected_dates = set(valid_decision_dates)
    window_dates = set(forward_return_windows.keys())
    vector_dates = set(forward_return_vectors.keys())

    # 1. Verify key sets match exactly
    if window_dates != expected_dates:
        error_msg = "Mismatch between valid_decision_dates and forward_return_windows keys."
        logger.error(error_msg)
        raise ForwardReturnError(error_msg)

    if vector_dates != expected_dates:
        error_msg = "Mismatch between valid_decision_dates and forward_return_vectors keys."
        logger.error(error_msg)
        raise ForwardReturnError(error_msg)

    # 2. Verify shapes and count NaNs
    total_vector_nans = 0
    n_assets = len(next(iter(forward_return_vectors.values())))

    for dt in valid_decision_dates:
        if forward_return_windows[dt].shape != (42, n_assets):
            raise ForwardReturnError(f"Invalid window shape at {dt}.")
        if forward_return_vectors[dt].shape != (n_assets,):
            raise ForwardReturnError(f"Invalid vector shape at {dt}.")

        total_vector_nans += int(np.isnan(forward_return_vectors[dt]).sum())

    # 3. Estimate memory footprint (8 bytes per float64)
    n_dates = len(valid_decision_dates)
    mem_windows_mb = (n_dates * 42 * n_assets * 8) / (1024 * 1024)
    logger.info(f"Estimated memory footprint of forward_return_windows: {mem_windows_mb:.2f} MB.")

    # 4. Augment the manifest
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["forward_return_objects"] = {
        "total_valid_dates": n_dates,
        "window_shape_per_date": (42, n_assets),
        "vector_shape_per_date": (n_assets,),
        "total_nan_entries_in_vectors": total_vector_nans
    }

    # Record the placeholder assumption for the cumulative return convention
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "forward_return_cumulative_convention",
        "manuscript_status": "unspecified",
        "value_used": "compound return: prod(1+r) - 1",
        "source": "rational placeholder"
    })

    logger.info("Forward return objects verified. Manifest augmented.")
    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 14, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_14_orchestrator(
    daily_returns_panel: pd.DataFrame,
    valid_decision_dates: pd.DatetimeIndex,
    master_trading_calendar: pd.DatetimeIndex,
    manifest: Dict[str, Any]
) -> Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]:
    """
    Orchestrates the extraction of forward return windows and the computation of compound forward returns.

    Parameters
    ----------
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]
        The dictionary of forward return windows, the dictionary of forward return vectors, and the augmented manifest.

    Raises
    ------
    ForwardReturnError
        If any structural invariants or mathematical constraints are violated during computation.
    """
    logger.info("Commencing Task 14: Constructing forward realized return objects.")

    # Execute Step 1: Extract forward return windows
    forward_return_windows = extract_forward_return_windows(
        daily_returns_panel=daily_returns_panel,
        valid_decision_dates=valid_decision_dates,
        master_trading_calendar=master_trading_calendar
    )

    # Execute Step 2: Compute compound forward returns
    forward_return_vectors = compute_compound_forward_returns(
        forward_return_windows=forward_return_windows
    )

    # Execute Step 3: Verify completeness and augment manifest
    manifest = verify_and_persist_forward_returns(
        forward_return_windows=forward_return_windows,
        forward_return_vectors=forward_return_vectors,
        valid_decision_dates=valid_decision_dates,
        manifest=manifest
    )

    logger.info("Task 14 complete. Forward return objects emitted.")

    # Return the finalized dictionaries and the updated manifest
    return forward_return_windows, forward_return_vectors, manifest


In [ ]:
# Task 15: Estimate per-date asset volatilities and standardized returns

class VolatilityEstimationError(Exception):
    """Custom exception raised when volatility estimation or standardization fails structural invariants."""
    pass

# ==============================================================================
# Task 15: Estimate per-date asset volatilities and standardized returns
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 15, Step 1: Compute per-asset volatilities over the forward window
# -------------------------------------------------------------------------------------------------------------------------------
def estimate_per_date_volatilities(
    window_matrix: np.ndarray,
    min_periods: int = 10,
    vol_floor: float = 1e-10
) -> np.ndarray:
    """
    Computes the sample standard deviation for each asset over the 42-day forward window.

    Equation: sigma_{t,j} = std({r_{t+1,j}, ..., r_{t+42,j}})

    Parameters
    ----------
    window_matrix : np.ndarray
        A 2D NumPy array of shape (42, N) containing forward daily returns.
    min_periods : int, optional
        The minimum number of valid (non-NaN) observations required to compute volatility (default is 10).
    vol_floor : float, optional
        The minimum allowable volatility to prevent division-by-zero singularities (default is 1e-10).

    Returns
    -------
    np.ndarray
        A 1D NumPy array of shape (N,) containing the daily volatilities.

    Raises
    ------
    VolatilityEstimationError
        If the input matrix is not 2D or if the output shape is incorrect.
    """
    # Verify the input is a 2D matrix
    if window_matrix.ndim != 2:
        raise VolatilityEstimationError(f"Expected 2D window_matrix, got {window_matrix.ndim}D.")

    # Extract the number of assets (columns)
    n_assets = window_matrix.shape[1]

    # Count the number of valid (non-NaN) observations per asset along the temporal axis (axis=0)
    valid_obs_count = np.sum(~np.isnan(window_matrix), axis=0)

    # Compute the sample standard deviation (ddof=1) ignoring NaNs
    # Suppress RuntimeWarnings for all-NaN slices, as we handle them explicitly below
    with np.errstate(invalid='ignore'):
        sigma_t = np.nanstd(window_matrix, axis=0, ddof=1)

    # Enforce the minimum observation threshold: set volatility to NaN if insufficient data exists
    sigma_t = np.where(valid_obs_count >= min_periods, sigma_t, np.nan)

    # Enforce the strict positive floor to prevent division-by-zero during standardization
    # We only apply the floor to non-NaN entries
    sigma_t = np.where(~np.isnan(sigma_t) & (sigma_t < vol_floor), vol_floor, sigma_t)

    # Rigorous invariant assertion: Verify exact output shape
    if sigma_t.shape != (n_assets,):
        raise VolatilityEstimationError(f"Shape mismatch: Expected {(n_assets,)}, got {sigma_t.shape}.")

    return sigma_t

# -------------------------------------------------------------------------------------------------------------------------------
# Task 15, Step 2: Standardize the forward returns
# -------------------------------------------------------------------------------------------------------------------------------
def standardize_forward_returns(
    window_matrix: np.ndarray,
    sigma_t: np.ndarray
) -> np.ndarray:
    """
    Standardizes the forward returns by dividing each asset's daily returns by its estimated volatility.

    Equation: r_tilde_{t+s,j} = r_{t+s,j} / sigma_{t,j}

    Parameters
    ----------
    window_matrix : np.ndarray
        A 2D NumPy array of shape (42, N) containing raw forward daily returns.
    sigma_t : np.ndarray
        A 1D NumPy array of shape (N,) containing the daily volatilities.

    Returns
    -------
    np.ndarray
        A 2D NumPy array of shape (42, N) containing the standardized returns.

    Raises
    ------
    VolatilityEstimationError
        If the shapes of the inputs are incompatible for broadcasting.
    """
    # Verify dimensional compatibility
    if window_matrix.shape[1] != sigma_t.shape[0]:
        raise VolatilityEstimationError(
            f"Dimension mismatch: window_matrix has {window_matrix.shape[1]} columns, "
            f"but sigma_t has length {sigma_t.shape[0]}."
        )

    # Perform element-wise division using NumPy broadcasting
    # sigma_t[np.newaxis, :] reshapes (N,) to (1, N), allowing it to broadcast across the 42 rows
    standardized_matrix = window_matrix / sigma_t[np.newaxis, :]

    # Rigorous invariant assertion: Verify the output shape matches the input window shape
    if standardized_matrix.shape != window_matrix.shape:
        raise VolatilityEstimationError(
            f"Shape mismatch after standardization: Expected {window_matrix.shape}, got {standardized_matrix.shape}."
        )

    return standardized_matrix

# -------------------------------------------------------------------------------------------------------------------------------
# Task 15, Step 3: Winsorize the standardized returns
# -------------------------------------------------------------------------------------------------------------------------------
def winsorize_standardized_returns(
    standardized_matrix: np.ndarray,
    clip_std: float = 4.2
) -> Tuple[np.ndarray, int]:
    """
    Winsorizes each standardized observation at the specified standard deviation threshold.

    Equation: r_tilde_win_{t+s,j} = max(-4.2, min(4.2, r_tilde_{t+s,j}))

    Parameters
    ----------
    standardized_matrix : np.ndarray
        A 2D NumPy array of shape (42, N) containing standardized returns.
    clip_std : float, optional
        The standard deviation threshold for winsorization (default is 4.2).

    Returns
    -------
    Tuple[np.ndarray, int]
        The winsorized 2D NumPy array, and the total count of clipped entries.
    """
    # Identify entries that exceed the clipping threshold (ignoring NaNs)
    # This boolean mask is used purely for diagnostic counting
    with np.errstate(invalid='ignore'):
        exceeds_threshold_mask = np.abs(standardized_matrix) > clip_std

    # Sum the boolean mask to get the total number of clipped entries
    clipped_count = int(np.sum(exceeds_threshold_mask))

    # Apply the highly optimized C-level clipping function
    # NaNs are unaffected by np.clip and remain NaNs
    winsorized_matrix = np.clip(standardized_matrix, -clip_std, clip_std)

    return winsorized_matrix, clipped_count

# -------------------------------------------------------------------------------------------------------------------------------
# Task 15, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_15_orchestrator(
    forward_return_windows: Dict[pd.Timestamp, np.ndarray],
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]:
    """
    Orchestrates the estimation of per-date volatilities, standardization, and winsorization.

    Parameters
    ----------
    forward_return_windows : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to their (42, N) forward return matrices.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]
        The dictionary of volatility vectors, the dictionary of winsorized windows, and the augmented manifest.

    Raises
    ------
    VolatilityEstimationError
        If any structural invariants or mathematical constraints are violated during computation.
    """
    logger.info("Commencing Task 15: Estimating per-date asset volatilities and standardized returns.")

    # Extract the winsorization threshold from the configuration
    risk_config = study_config.get("RISK_MODEL_SPECIFICATION", {})
    winsorization_std = risk_config.get("winsorization_std", 4.2)

    # Initialize the output dictionaries
    volatility_vectors: Dict[pd.Timestamp, np.ndarray] = {}
    winsorized_windows: Dict[pd.Timestamp, np.ndarray] = {}

    total_clipped_entries = 0
    total_nan_vols = 0

    # Iterate chronologically over every valid decision date
    for current_date, window_matrix in forward_return_windows.items():

        # Execute Step 1: Estimate per-asset volatilities
        sigma_t = estimate_per_date_volatilities(
            window_matrix=window_matrix,
            min_periods=10,
            vol_floor=1e-10
        )

        # Track the number of assets with NaN volatility for diagnostics
        total_nan_vols += int(np.isnan(sigma_t).sum())

        # Execute Step 2: Standardize the forward returns
        standardized_matrix = standardize_forward_returns(
            window_matrix=window_matrix,
            sigma_t=sigma_t
        )

        # Execute Step 3: Winsorize the standardized returns
        winsorized_matrix, clipped_count = winsorize_standardized_returns(
            standardized_matrix=standardized_matrix,
            clip_std=winsorization_std
        )

        # Accumulate diagnostic statistics
        total_clipped_entries += clipped_count

        # Store the computed artifacts
        volatility_vectors[current_date] = sigma_t
        winsorized_windows[current_date] = winsorized_matrix

    logger.info(f"Processed {len(forward_return_windows)} dates. Total clipped entries: {total_clipped_entries}.")

    # Augment the provenance manifest with metadata and placeholder conventions
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["volatility_and_standardization"] = {
        "total_nan_volatilities_across_all_dates": total_nan_vols,
        "total_winsorized_entries_across_all_dates": total_clipped_entries,
        "winsorization_threshold": winsorization_std
    }

    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "volatility_estimation_ddof",
        "manuscript_status": "unspecified",
        "value_used": "ddof=1 (sample standard deviation)",
        "source": "rational placeholder"
    })

    manifest["placeholder_assumptions"].append({
        "convention_name": "volatility_zero_floor",
        "manuscript_status": "unspecified",
        "value_used": "1e-10",
        "source": "numerical stability guardrail"
    })

    logger.info("Task 15 complete. Volatility vectors and winsorized windows emitted.")

    # Return the finalized dictionaries and the updated manifest
    return volatility_vectors, winsorized_windows, manifest


In [ ]:
# Task 16: Extract factor structure and construct the covariance matrix

class RiskModelError(Exception):
    """Custom exception raised when factor extraction or covariance construction fails structural invariants."""
    pass

# ==============================================================================
# Task 16: Extract factor structure and construct the covariance matrix
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 16, Step 1: Compute correlation matrix and extract top J eigenpairs
# -------------------------------------------------------------------------------------------------------------------------------
def extract_top_eigenpairs(
    winsorized_matrix: np.ndarray,
    j_factors: int = 15
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Computes the sample correlation matrix and extracts the top J eigenvectors and eigenvalues.

    Parameters
    ----------
    winsorized_matrix : np.ndarray
        A 2D NumPy array of shape (42, N) containing winsorized standardized returns.
    j_factors : int, optional
        The number of top factors to extract (default is 15).

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        Q_t: The eigenvector matrix of shape (N, J).
        Lambda_t: The eigenvalue vector of shape (J,).

    Raises
    ------
    RiskModelError
        If the eigendecomposition fails or returns invalid shapes.
    """
    n_obs, n_assets = winsorized_matrix.shape

    # Identify assets with valid data (not all NaNs)
    # An asset is valid if it has at least 2 non-NaN observations to compute correlation
    valid_mask = np.sum(~np.isnan(winsorized_matrix), axis=0) > 1
    valid_indices = np.where(valid_mask)[0]
    n_valid = len(valid_indices)

    if n_valid < j_factors:
        raise RiskModelError(f"Only {n_valid} valid assets available, cannot extract {j_factors} factors.")

    # Extract the sub-matrix of valid assets
    valid_matrix = winsorized_matrix[:, valid_mask]

    # Explicitly demean the columns. Winsorization may shift the mean slightly away from zero.
    # We use nanmean to ignore any residual NaNs within the valid columns.
    with np.errstate(invalid='ignore'):
        col_means = np.nanmean(valid_matrix, axis=0)
        demeaned_matrix = valid_matrix - col_means[np.newaxis, :]

    # Compute the correlation matrix using pandas for robust pairwise NaN handling
    # We convert to DataFrame temporarily to leverage .corr() which handles missing data gracefully
    valid_df = pd.DataFrame(demeaned_matrix)
    corr_matrix_valid = valid_df.corr().to_numpy()

    # Fill any residual NaNs in the correlation matrix (e.g., from zero variance after demeaning) with 0.0,
    # and ensure the diagonal is exactly 1.0
    np.fill_diagonal(corr_matrix_valid, 1.0)
    corr_matrix_valid = np.nan_to_num(corr_matrix_valid, nan=0.0)

    # Perform symmetric eigendecomposition
    # eigh returns eigenvalues in ascending order
    try:
        eigenvalues_valid, eigenvectors_valid = scipy.linalg.eigh(corr_matrix_valid)
    except Exception as e:
        raise RiskModelError(f"Eigendecomposition failed: {e}")

    # Extract the top J eigenpairs (the last J elements, reversed to be descending)
    top_evals_valid = eigenvalues_valid[-j_factors:][::-1]
    top_evecs_valid = eigenvectors_valid[:, -j_factors:][:, ::-1]

    # Enforce deterministic eigenvector sign convention:
    # For each eigenvector, the element with the largest absolute value must be positive.
    for k in range(j_factors):
        max_idx = np.argmax(np.abs(top_evecs_valid[:, k]))
        if top_evecs_valid[max_idx, k] < 0:
            top_evecs_valid[:, k] *= -1.0

    # Reconstruct the full N x J eigenvector matrix, inserting zeros for invalid assets
    Q_t = np.zeros((n_assets, j_factors), dtype=np.float64)
    Q_t[valid_indices, :] = top_evecs_valid

    # The eigenvalues remain the same (shape J,)
    Lambda_t = top_evals_valid

    # Rigorous invariant assertion
    if Q_t.shape != (n_assets, j_factors):
        raise RiskModelError(f"Q_t shape mismatch: Expected {(n_assets, j_factors)}, got {Q_t.shape}.")
    if Lambda_t.shape != (j_factors,):
        raise RiskModelError(f"Lambda_t shape mismatch: Expected {(j_factors,)}, got {Lambda_t.shape}.")

    return Q_t, Lambda_t

# -------------------------------------------------------------------------------------------------------------------------------
# Task 16, Step 2: Construct factor loadings and idiosyncratic variance
# -------------------------------------------------------------------------------------------------------------------------------
def construct_factor_components(
    Q_t: np.ndarray,
    Lambda_t: np.ndarray,
    sigma_t: np.ndarray,
    idio_floor: float = 1e-8
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Constructs the factor loading matrix and the idiosyncratic variance diagonal.

    Equations:
    F_t = diag(sigma_t) * Q_t * Lambda_t^(1/2)
    (D_t^idio)_{jj} = sigma_{t,j}^2 - (F_t F_t^T)_{jj}

    Parameters
    ----------
    Q_t : np.ndarray
        The eigenvector matrix of shape (N, J).
    Lambda_t : np.ndarray
        The eigenvalue vector of shape (J,).
    sigma_t : np.ndarray
        The daily volatility vector of shape (N,).
    idio_floor : float, optional
        The minimum allowable idiosyncratic variance (default is 1e-8).

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        F_t: The factor loading matrix of shape (N, J).
        D_t_idio: The idiosyncratic variance vector of shape (N,).

    Raises
    ------
    RiskModelError
        If dimensional mismatches occur during broadcasting.
    """
    n_assets, j_factors = Q_t.shape

    # Guard against numerical noise producing tiny negative eigenvalues
    safe_lambda = np.maximum(Lambda_t, 0.0)
    sqrt_lambda = np.sqrt(safe_lambda)

    # Handle NaNs in sigma_t by temporarily replacing them with 0.0 for the matrix multiplication
    # The resulting F_t rows will be zero, which is correct for assets with no data
    safe_sigma = np.nan_to_num(sigma_t, nan=0.0)

    # Construct F_t using vectorized broadcasting
    # sigma_t[:, np.newaxis] scales the rows; sqrt_lambda[np.newaxis, :] scales the columns
    F_t = safe_sigma[:, np.newaxis] * Q_t * sqrt_lambda[np.newaxis, :]

    # Compute the diagonal of F_t F_t^T efficiently without forming the full N x N matrix
    # The diagonal elements are simply the sum of squares of the rows of F_t
    factor_variance = np.sum(F_t**2, axis=1)

    # Compute idiosyncratic variance: Total Variance - Factor Variance
    D_t_idio = (safe_sigma**2) - factor_variance

    # Clamp negative idiosyncratic variances to the positive floor
    # Negative values can occur if the factor model over-explains the variance due to in-sample fitting
    D_t_idio = np.maximum(D_t_idio, idio_floor)

    # Restore NaNs for assets that originally had NaN volatility
    nan_mask = np.isnan(sigma_t)
    F_t[nan_mask, :] = np.nan
    D_t_idio[nan_mask] = np.nan

    # Rigorous invariant assertion
    if F_t.shape != (n_assets, j_factors):
        raise RiskModelError(f"F_t shape mismatch: Expected {(n_assets, j_factors)}, got {F_t.shape}.")
    if D_t_idio.shape != (n_assets,):
        raise RiskModelError(f"D_t_idio shape mismatch: Expected {(n_assets,)}, got {D_t_idio.shape}.")

    return F_t, D_t_idio

# -------------------------------------------------------------------------------------------------------------------------------
# Task 16, Step 3: Form the covariance matrix and define volatility inputs
# -------------------------------------------------------------------------------------------------------------------------------
def synthesize_risk_model_artifacts(
    F_t: np.ndarray,
    D_t_idio: np.ndarray,
    annualization_factor: float = 252.0
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Synthesizes the daily volatility input for the TC model and the annualized factored covariance for the PM solver.

    Equations:
    nu_{t,j} = sqrt((F_t F_t^T)_{jj} + (D_t^idio)_{jj})
    F_t^annual = sqrt(252) * F_t
    D_t^idio,annual = 252 * D_t^idio

    Parameters
    ----------
    F_t : np.ndarray
        The daily factor loading matrix of shape (N, J).
    D_t_idio : np.ndarray
        The daily idiosyncratic variance vector of shape (N,).
    annualization_factor : float, optional
        The scalar used to annualize the covariance matrix (default is 252.0).

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        nu_t_daily: The daily volatility vector of shape (N,).
        F_t_annual: The annualized factor loading matrix of shape (N, J).
        D_t_idio_annual: The annualized idiosyncratic variance vector of shape (N,).
    """
    # Compute the daily volatility vector nu_t directly from the factored components
    # This avoids forming the full N x N covariance matrix
    factor_variance = np.sum(F_t**2, axis=1)

    # Suppress warnings for NaNs
    with np.errstate(invalid='ignore'):
        nu_t_daily = np.sqrt(factor_variance + D_t_idio)

    # Scale the components to annual units for the PM risk constraint
    # If Sigma_annual = 252 * Sigma_daily, then:
    # 252 * (F F^T + D) = (sqrt(252)*F)(sqrt(252)*F)^T + 252*D
    F_t_annual = np.sqrt(annualization_factor) * F_t
    D_t_idio_annual = annualization_factor * D_t_idio

    return nu_t_daily, F_t_annual, D_t_idio_annual

# -------------------------------------------------------------------------------------------------------------------------------
# Task 16, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_16_orchestrator(
    winsorized_windows: Dict[pd.Timestamp, np.ndarray],
    volatility_vectors: Dict[pd.Timestamp, np.ndarray],
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]:
    """
    Orchestrates the extraction of the factor structure and the construction of the risk model artifacts.

    Parameters
    ----------
    winsorized_windows : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to their (42, N) winsorized return matrices.
    volatility_vectors : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to their (N,) daily volatility vectors.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[pd.Timestamp, np.ndarray], Dict[str, Any]]
        Dictionaries for F_t_annual, D_t_idio_annual, nu_t_daily, and the augmented manifest.

    Raises
    ------
    RiskModelError
        If any structural invariants or mathematical constraints are violated during computation.
    """
    logger.info("Commencing Task 16: Extracting factor structure and constructing the covariance matrix.")

    # Extract configuration parameters
    risk_config = study_config.get("RISK_MODEL_SPECIFICATION", {})
    j_factors = risk_config.get("J_factors", 15)

    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    annualization_factor = float(firm_consts.get("annualization_basis_trading_days", 252.0))

    # Initialize the output dictionaries
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray] = {}
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray] = {}
    nu_t_daily_dict: Dict[pd.Timestamp, np.ndarray] = {}

    # Iterate chronologically over every valid decision date
    for current_date, winsorized_matrix in winsorized_windows.items():
        sigma_t = volatility_vectors[current_date]

        # Execute Step 1: Extract top J eigenpairs
        Q_t, Lambda_t = extract_top_eigenpairs(
            winsorized_matrix=winsorized_matrix,
            j_factors=j_factors
        )

        # Execute Step 2: Construct factor loadings and idiosyncratic variance
        F_t_daily, D_t_idio_daily = construct_factor_components(
            Q_t=Q_t,
            Lambda_t=Lambda_t,
            sigma_t=sigma_t,
            idio_floor=1e-8
        )

        # Execute Step 3: Synthesize final artifacts and apply annualization
        nu_t_daily, F_t_annual, D_t_idio_annual = synthesize_risk_model_artifacts(
            F_t=F_t_daily,
            D_t_idio=D_t_idio_daily,
            annualization_factor=annualization_factor
        )

        # Store the computed artifacts
        F_t_annual_dict[current_date] = F_t_annual
        D_t_idio_annual_dict[current_date] = D_t_idio_annual
        nu_t_daily_dict[current_date] = nu_t_daily

    logger.info(f"Processed {len(winsorized_windows)} dates. Risk model artifacts generated.")

    # Augment the provenance manifest with metadata and placeholder conventions
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["risk_model_artifacts"] = {
        "j_factors_extracted": j_factors,
        "annualization_factor_applied": annualization_factor,
        "F_t_annual_shape": (434, j_factors),
        "D_t_idio_annual_shape": (434,),
        "nu_t_daily_shape": (434,)
    }

    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "idiosyncratic_variance_residual_formula",
        "manuscript_status": "unspecified",
        "value_used": "sigma^2 - diag(F F^T), clamped at 1e-8",
        "source": "rational placeholder"
    })

    manifest["placeholder_assumptions"].append({
        "convention_name": "covariance_annualization_logic",
        "manuscript_status": "unspecified",
        "value_used": f"Multiply daily covariance by {annualization_factor}",
        "source": "rational placeholder"
    })

    logger.info("Task 16 complete. Factored covariance matrices and daily volatilities emitted.")

    # Return the finalized dictionaries and the updated manifest
    return F_t_annual_dict, D_t_idio_annual_dict, nu_t_daily_dict, manifest


In [ ]:
# Task 17: Draw PM static characteristics

class PMInitializationError(Exception):
    """Custom exception raised when PM static characteristics generation fails structural or mathematical invariants."""
    pass

@dataclass(frozen=True)
class PMCharacteristics:
    """
    A strictly typed, immutable record of a Portfolio Manager's static characteristics.

    Attributes
    ----------
    pm_id : int
        The unique integer identifier for the PM (1 to M).
    initial_nav_usd : float
        The initial Net Asset Value V_0^(i) in USD.
    risk_target_annualized : float
        The annualized volatility target sigma_target^(i).
    tradable_mask : np.ndarray
        A boolean array of length N indicating the PM's tradable universe.
    n_tradable_assets : int
        The exact count of assets the PM is permitted to trade.
    """
    pm_id: int
    initial_nav_usd: float
    risk_target_annualized: float
    tradable_mask: np.ndarray
    n_tradable_assets: int

    def __post_init__(self):
        """Defensive invariant assertions post-initialization."""
        if self.initial_nav_usd <= 0:
            raise PMInitializationError(f"PM {self.pm_id} initial NAV must be strictly positive.")
        if self.risk_target_annualized <= 0:
            raise PMInitializationError(f"PM {self.pm_id} risk target must be strictly positive.")
        if self.tradable_mask.dtype != bool:
            raise PMInitializationError(f"PM {self.pm_id} tradable mask must be boolean.")
        if int(self.tradable_mask.sum()) != self.n_tradable_assets:
            raise PMInitializationError(f"PM {self.pm_id} mask sum does not match n_tradable_assets.")

# ==============================================================================
# Task 17: Draw PM static characteristics
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 17, Step 1: Draw initial NAVs and risk targets
# -------------------------------------------------------------------------------------------------------------------------------
def draw_navs_and_risk_targets(
    m_pms: int,
    rng_nav: np.random.Generator,
    rng_risk: np.random.Generator
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Draws the initial NAVs and annualized risk targets for all PMs.

    Equations:
    V_0^(i) ~ log-uniform(10^6.5, 10^7.5)
    sigma_target^(i) ~ U[0.06, 0.15]

    Parameters
    ----------
    m_pms : int
        The number of portfolio managers (M).
    rng_nav : np.random.Generator
        The isolated random generator for NAV draws.
    rng_risk : np.random.Generator
        The isolated random generator for risk target draws.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        initial_navs: 1D array of length M containing V_0^(i).
        risk_targets: 1D array of length M containing sigma_target^(i).

    Raises
    ------
    PMInitializationError
        If the drawn values fall outside the mathematically defined bounds.
    """
    logger.info("Executing Task 17, Step 1: Drawing initial NAVs and risk targets.")

    # Draw log-uniform NAVs: 10^U where U ~ U[6.5, 7.5]
    log_navs = rng_nav.uniform(low=6.5, high=7.5, size=m_pms)
    initial_navs = 10.0 ** log_navs

    # Draw uniform risk targets: sigma_target ~ U[0.06, 0.15]
    risk_targets = rng_risk.uniform(low=0.06, high=0.15, size=m_pms)

    # Rigorous invariant assertions
    if not np.all((initial_navs >= 10**6.5) & (initial_navs <= 10**7.5)):
        raise PMInitializationError("Drawn NAVs fall outside the [10^6.5, 10^7.5] bound.")

    if not np.all((risk_targets >= 0.06) & (risk_targets <= 0.15)):
        raise PMInitializationError("Drawn risk targets fall outside the [0.06, 0.15] bound.")

    logger.info(f"Successfully drawn characteristics for {m_pms} PMs.")
    return initial_navs, risk_targets

# -------------------------------------------------------------------------------------------------------------------------------
# Task 17, Step 2: Draw random binary tradable-universe masks
# -------------------------------------------------------------------------------------------------------------------------------
def draw_tradable_universe_masks(
    m_pms: int,
    n_assets: int,
    fraction: float,
    rng_mask: np.random.Generator
) -> List[np.ndarray]:
    """
    Draws a random binary tradable-universe mask for each PM using exact-cardinality sampling.

    Parameters
    ----------
    m_pms : int
        The number of portfolio managers (M).
    n_assets : int
        The total number of assets in the canonical universe (N).
    fraction : float
        The fraction of the universe each PM is permitted to trade (e.g., 0.75).
    rng_mask : np.random.Generator
        The isolated random generator for mask draws.

    Returns
    -------
    List[np.ndarray]
        A list of M boolean arrays, each of length N.

    Raises
    ------
    PMInitializationError
        If the exact cardinality constraint is violated.
    """
    logger.info(f"Executing Task 17, Step 2: Drawing tradable-universe masks (fraction: {fraction}).")

    # Calculate exact cardinality: floor(0.75 * 434) = 325
    exact_cardinality = int(np.floor(fraction * n_assets))

    masks: List[np.ndarray] = []

    for i in range(m_pms):
        # Select exactly 'exact_cardinality' indices uniformly at random without replacement
        selected_indices = rng_mask.choice(n_assets, size=exact_cardinality, replace=False)

        # Construct the boolean mask
        mask_i = np.zeros(n_assets, dtype=bool)
        mask_i[selected_indices] = True

        # Rigorous invariant assertion
        if mask_i.sum() != exact_cardinality:
            raise PMInitializationError(f"Mask for PM {i+1} has {mask_i.sum()} True values, expected {exact_cardinality}.")

        masks.append(mask_i)

    logger.info(f"Successfully drawn {m_pms} masks, each with exactly {exact_cardinality} tradable assets.")
    return masks

# -------------------------------------------------------------------------------------------------------------------------------
# Task 17, Step 3: Emit frozen PM characteristics object
# -------------------------------------------------------------------------------------------------------------------------------
def freeze_pm_characteristics(
    initial_navs: np.ndarray,
    risk_targets: np.ndarray,
    masks: List[np.ndarray]
) -> Dict[int, PMCharacteristics]:
    """
    Aggregates the drawn attributes into immutable PMCharacteristics dataclasses.

    Parameters
    ----------
    initial_navs : np.ndarray
        1D array of length M containing V_0^(i).
    risk_targets : np.ndarray
        1D array of length M containing sigma_target^(i).
    masks : List[np.ndarray]
        List of M boolean arrays representing the tradable universes.

    Returns
    -------
    Dict[int, PMCharacteristics]
        A dictionary mapping pm_id (1 to M) to its frozen PMCharacteristics object.
    """
    logger.info("Executing Task 17, Step 3: Freezing PM characteristics into immutable objects.")

    m_pms = len(initial_navs)
    pm_characteristics_dict: Dict[int, PMCharacteristics] = {}

    for i in range(m_pms):
        pm_id = i + 1
        n_tradable = int(masks[i].sum())

        # Instantiate the frozen dataclass
        # The __post_init__ method will automatically enforce invariants
        char_obj = PMCharacteristics(
            pm_id=pm_id,
            initial_nav_usd=float(initial_navs[i]),
            risk_target_annualized=float(risk_targets[i]),
            tradable_mask=masks[i],
            n_tradable_assets=n_tradable
        )

        pm_characteristics_dict[pm_id] = char_obj

    logger.info("PM characteristics successfully frozen.")
    return pm_characteristics_dict

# -------------------------------------------------------------------------------------------------------------------------------
# Task 17, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_17_orchestrator(
    study_config: Dict[str, Any],
    universe_asset_ids: Tuple[str, ...],
    generators: Dict[str, np.random.Generator],
    manifest: Dict[str, Any]
) -> Tuple[Dict[int, PMCharacteristics], Dict[str, Any]]:
    """
    Orchestrates the drawing and freezing of static PM characteristics.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary.
    universe_asset_ids : Tuple[str, ...]
        The frozen tuple of exactly 434 asset_ids.
    generators : Dict[str, np.random.Generator]
        The dictionary of isolated random generators instantiated in Task 4.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[int, PMCharacteristics], Dict[str, Any]]
        The dictionary of frozen PM characteristics and the augmented manifest.

    Raises
    ------
    PMInitializationError
        If any structural invariants or mathematical constraints are violated during generation.
    """
    logger.info("Commencing Task 17: Drawing PM static characteristics.")

    # Extract configuration parameters
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    m_pms = firm_consts.get("M_portfolio_managers", 4)
    n_assets = len(universe_asset_ids)

    pm_config = study_config.get("PM_SLEEVE_CONFIGURATIONS", {}).get("manager_initialization_logic", {})
    fraction = pm_config.get("asset_universe_fraction_per_pm", 0.75)

    # Extract specific generators
    try:
        rng_nav = generators["pm_nav_seed"]
        rng_risk = generators["pm_risk_target_seed"]
        rng_mask = generators["pm_universe_mask_seed"]
    except KeyError as e:
        raise PMInitializationError(f"Required random generator missing: {e}")

    # Execute Step 1: Draw continuous parameters
    initial_navs, risk_targets = draw_navs_and_risk_targets(
        m_pms=m_pms,
        rng_nav=rng_nav,
        rng_risk=rng_risk
    )

    # Execute Step 2: Draw discrete masks
    masks = draw_tradable_universe_masks(
        m_pms=m_pms,
        n_assets=n_assets,
        fraction=fraction,
        rng_mask=rng_mask
    )

    # Execute Step 3: Freeze characteristics
    pm_characteristics_dict = freeze_pm_characteristics(
        initial_navs=initial_navs,
        risk_targets=risk_targets,
        masks=masks
    )

    # Augment the provenance manifest with the exact drawn values for auditability
    if "pm_static_characteristics" not in manifest:
        manifest["pm_static_characteristics"] = {}

    for pm_id, char_obj in pm_characteristics_dict.items():
        manifest["pm_static_characteristics"][f"PM_{pm_id}"] = {
            "initial_nav_usd": char_obj.initial_nav_usd,
            "risk_target_annualized": char_obj.risk_target_annualized,
            "n_tradable_assets": char_obj.n_tradable_assets
        }

    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "tradable_subset_method",
        "manuscript_status": "unspecified",
        "value_used": "exact cardinality sampling (replace=False)",
        "source": "rational placeholder"
    })

    logger.info("Task 17 complete. PM static characteristics emitted.")

    # Return the frozen dictionary and the updated manifest
    return pm_characteristics_dict, manifest


In [ ]:
# Task 18: Set PM initial portfolio state and compute firm NAV weights

class StateInitializationError(Exception):
    """Custom exception raised when PM or Firm state initialization fails structural or mathematical invariants."""
    pass

@dataclass
class PMDynamicState:
    """
    A strictly typed, mutable record of a Portfolio Manager's state at a specific point in time.
    This object will be mutated during the walk-forward simulation.

    Attributes
    ----------
    pm_id : int
        The unique integer identifier for the PM.
    w_curr : np.ndarray
        The current portfolio weight vector of length N.
    c_curr : float
        The current cash position as a fraction of NAV.
    nav_usd : float
        The current Net Asset Value in USD.
    nav_weight : float
        The current relative weight of the PM's NAV to the firm's total NAV (lambda).
    risk_target_annualized : float
        The static annualized volatility target.
    tradable_mask : np.ndarray
        The static boolean array indicating the PM's tradable universe.
    """
    pm_id: int
    w_curr: np.ndarray
    c_curr: float
    nav_usd: float
    nav_weight: float
    risk_target_annualized: float
    tradable_mask: np.ndarray

@dataclass
class FirmState:
    """
    A strictly typed, mutable record of the Firm's aggregate state.
    This object serves as the seed for the walk-forward simulation.

    Attributes
    ----------
    firm_nav_usd : float
        The aggregate Net Asset Value of the firm in USD.
    pm_states : Dict[int, PMDynamicState]
        A dictionary mapping pm_id to the respective PM's dynamic state.
    current_date : pd.Timestamp or None
        The current date of the simulation state (None at initialization).
    """
    firm_nav_usd: float
    pm_states: Dict[int, PMDynamicState]
    current_date: Any = None

# ==============================================================================
# Task 18: Set PM initial portfolio state and compute firm NAV weights
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 18, Step 1: Set initial portfolio and cash state for each PM
# -------------------------------------------------------------------------------------------------------------------------------
def initialize_pm_portfolios(
    pm_characteristics_dict: Dict[int, Any]
) -> Dict[int, Tuple[np.ndarray, float]]:
    """
    Initializes the portfolio weights and cash position for each PM to an all-cash state.

    Equations:
    w_{curr,0}^{(i)} = 0_N
    c_{curr,0}^{(i)} = 1.0

    Parameters
    ----------
    pm_characteristics_dict : Dict[int, PMCharacteristics]
        Dictionary mapping pm_id to its frozen static characteristics.

    Returns
    -------
    Dict[int, Tuple[np.ndarray, float]]
        A dictionary mapping pm_id to a tuple of (initial_weights, initial_cash).

    Raises
    ------
    StateInitializationError
        If the cash identity constraint is violated during initialization.
    """
    logger.info("Executing Task 18, Step 1: Initializing PM portfolios to all-cash state.")

    initial_portfolios: Dict[int, Tuple[np.ndarray, float]] = {}

    for pm_id, char_obj in pm_characteristics_dict.items():
        # Extract N from the length of the tradable mask to ensure dimensional consistency
        n_assets = len(char_obj.tradable_mask)

        # Initialize weights to exactly zero
        w_curr_0 = np.zeros(n_assets, dtype=np.float64)

        # Initialize cash to 1.0 (100% of NAV)
        c_curr_0 = 1.0

        # Rigorous invariant assertion: Verify the cash identity 1 - 1^T w = c
        cash_identity_check = 1.0 - np.sum(w_curr_0)
        if not np.isclose(cash_identity_check, c_curr_0, atol=1e-14):
            error_msg = f"Cash identity violation for PM {pm_id}: 1 - sum(w) = {cash_identity_check}, but c = {c_curr_0}."
            logger.error(error_msg)
            raise StateInitializationError(error_msg)

        initial_portfolios[pm_id] = (w_curr_0, c_curr_0)

    logger.info(f"Successfully initialized {len(initial_portfolios)} PM portfolios.")
    return initial_portfolios

# -------------------------------------------------------------------------------------------------------------------------------
# Task 18, Step 2: Compute initial firm NAV and PM NAV weights
# -------------------------------------------------------------------------------------------------------------------------------
def compute_initial_nav_weights(
    pm_characteristics_dict: Dict[int, Any]
) -> Tuple[float, Dict[int, float]]:
    """
    Computes the initial aggregate firm NAV and the relative NAV weight for each PM.

    Equations:
    V_0 = \sum_{i=1}^{M} V_0^{(i)}
    \lambda_0^{(i)} = V_0^{(i)} / V_0

    Parameters
    ----------
    pm_characteristics_dict : Dict[int, PMCharacteristics]
        Dictionary mapping pm_id to its frozen static characteristics.

    Returns
    -------
    Tuple[float, Dict[int, float]]
        The total firm NAV (V_0) and a dictionary mapping pm_id to its relative weight (\lambda_0^{(i)}).

    Raises
    ------
    StateInitializationError
        If the firm NAV is non-positive or if the weights do not sum to exactly 1.0.
    """
    logger.info("Executing Task 18, Step 2: Computing initial firm NAV and relative PM weights.")

    # Compute total firm NAV
    v_0_firm = sum(char_obj.initial_nav_usd for char_obj in pm_characteristics_dict.values())

    if v_0_firm <= 0:
        raise StateInitializationError(f"Firm NAV must be strictly positive, got {v_0_firm}.")

    lambda_0_dict: Dict[int, float] = {}

    # Compute relative weights
    for pm_id, char_obj in pm_characteristics_dict.items():
        lambda_0_dict[pm_id] = char_obj.initial_nav_usd / v_0_firm

    # Rigorous invariant assertion: Weights must sum to 1.0
    total_weight = sum(lambda_0_dict.values())
    if not np.isclose(total_weight, 1.0, atol=1e-14):
        error_msg = f"NAV weights do not sum to 1.0. Sum = {total_weight}."
        logger.error(error_msg)
        raise StateInitializationError(error_msg)

    logger.info(f"Initial Firm NAV: ${v_0_firm:,.2f}. Weights computed successfully.")
    return v_0_firm, lambda_0_dict

# -------------------------------------------------------------------------------------------------------------------------------
# Task 18, Step 3: Emit the initial state object
# -------------------------------------------------------------------------------------------------------------------------------
def construct_master_state_object(
    pm_characteristics_dict: Dict[int, Any],
    initial_portfolios: Dict[int, Tuple[np.ndarray, float]],
    v_0_firm: float,
    lambda_0_dict: Dict[int, float]
) -> FirmState:
    """
    Aggregates all initial parameters into a comprehensive, deep-copyable FirmState object.

    Parameters
    ----------
    pm_characteristics_dict : Dict[int, PMCharacteristics]
        Dictionary mapping pm_id to its frozen static characteristics.
    initial_portfolios : Dict[int, Tuple[np.ndarray, float]]
        Dictionary mapping pm_id to (w_curr_0, c_curr_0).
    v_0_firm : float
        The total initial firm NAV.
    lambda_0_dict : Dict[int, float]
        Dictionary mapping pm_id to its initial relative weight.

    Returns
    -------
    FirmState
        The fully populated initial state object, ready to seed the walk-forward simulation.

    Raises
    ------
    StateInitializationError
        If any numerical field is NaN or infinite.
    """
    logger.info("Executing Task 18, Step 3: Constructing the master initial state object.")

    pm_states: Dict[int, PMDynamicState] = {}

    for pm_id, char_obj in pm_characteristics_dict.items():
        w_curr, c_curr = initial_portfolios[pm_id]
        nav_usd = char_obj.initial_nav_usd
        nav_weight = lambda_0_dict[pm_id]

        # Verify finiteness of scalar fields
        if not np.isfinite([c_curr, nav_usd, nav_weight, char_obj.risk_target_annualized]).all():
            raise StateInitializationError(f"Non-finite scalar detected in PM {pm_id} state initialization.")

        # Verify finiteness of vector fields
        if not np.isfinite(w_curr).all():
            raise StateInitializationError(f"Non-finite weight detected in PM {pm_id} state initialization.")

        # Construct the dynamic state object for this PM
        pm_state = PMDynamicState(
            pm_id=pm_id,
            w_curr=w_curr,
            c_curr=c_curr,
            nav_usd=nav_usd,
            nav_weight=nav_weight,
            risk_target_annualized=char_obj.risk_target_annualized,
            tradable_mask=char_obj.tradable_mask
        )

        pm_states[pm_id] = pm_state

    # Construct the global firm state
    master_state = FirmState(
        firm_nav_usd=v_0_firm,
        pm_states=pm_states,
        current_date=None  # Will be set at the start of the walk-forward loop
    )

    logger.info("Master initial state object constructed successfully.")
    return master_state

# -------------------------------------------------------------------------------------------------------------------------------
# Task 18, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_18_orchestrator(
    pm_characteristics_dict: Dict[int, Any],
    manifest: Dict[str, Any]
) -> Tuple[FirmState, Dict[str, Any]]:
    """
    Orchestrates the initialization of PM portfolios and the computation of firm NAV weights.

    Parameters
    ----------
    pm_characteristics_dict : Dict[int, PMCharacteristics]
        Dictionary mapping pm_id to its frozen static characteristics.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[FirmState, Dict[str, Any]]
        The master initial state object and the augmented manifest.

    Raises
    ------
    StateInitializationError
        If any structural invariants or mathematical constraints are violated during initialization.
    """
    logger.info("Commencing Task 18: Setting PM initial portfolio state and computing firm NAV weights.")

    # Execute Step 1: Initialize portfolios to all-cash
    initial_portfolios = initialize_pm_portfolios(
        pm_characteristics_dict=pm_characteristics_dict
    )

    # Execute Step 2: Compute firm NAV and relative weights
    v_0_firm, lambda_0_dict = compute_initial_nav_weights(
        pm_characteristics_dict=pm_characteristics_dict
    )

    # Execute Step 3: Construct the master state object
    initial_firm_state = construct_master_state_object(
        pm_characteristics_dict=pm_characteristics_dict,
        initial_portfolios=initial_portfolios,
        v_0_firm=v_0_firm,
        lambda_0_dict=lambda_0_dict
    )

    # Augment the provenance manifest with the placeholder convention
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "initial_pm_portfolio_state",
        "manuscript_status": "unspecified",
        "value_used": "all-cash (w=0, c=1.0)",
        "source": "rational placeholder"
    })

    logger.info("Task 18 complete. Initial FirmState object emitted.")

    # Return the master state object and the updated manifest
    return initial_firm_state, manifest


In [ ]:
# Task 19: Draw PM signal calibration parameters

class SignalCalibrationError(Exception):
    """Custom exception raised when PM signal calibration fails structural or mathematical invariants."""
    pass

@dataclass(frozen=True)
class AlphaCalibration:
    """
    A strictly typed, immutable record of a Portfolio Manager's synthetic alpha calibration parameters.

    Attributes
    ----------
    pm_id : int
        The unique integer identifier for the PM (1 to M).
    ic_target : float
        The target Information Coefficient (IC^(i)).
    c_alpha : float
        The alpha scaling factor (c^(i) = (IC^(i))^2).
    v_noise : float
        The noise scaling factor (v^(i) = sqrt(1/(IC^(i))^2 - 1)).
    autocorr_phi : float
        The temporal autocorrelation coefficient for the VAR(1) process (phi^(i)).
    """
    pm_id: int
    ic_target: float
    c_alpha: float
    v_noise: float
    autocorr_phi: float

    def __post_init__(self):
        """Defensive invariant assertions post-initialization."""
        if not (0.0 < self.ic_target < 1.0):
            raise SignalCalibrationError(f"PM {self.pm_id} IC target must be in (0, 1).")
        if self.c_alpha <= 0.0:
            raise SignalCalibrationError(f"PM {self.pm_id} alpha scaling factor must be strictly positive.")
        if self.v_noise <= 0.0:
            raise SignalCalibrationError(f"PM {self.pm_id} noise scaling factor must be strictly positive.")
        if not (-1.0 < self.autocorr_phi < 1.0):
            raise SignalCalibrationError(f"PM {self.pm_id} autocorrelation must be in (-1, 1) for VAR(1) stationarity.")

# ==============================================================================
# Task 19: Draw PM signal calibration parameters
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 19, Step 1: Draw target ICs and compute alpha scaling factors
# -------------------------------------------------------------------------------------------------------------------------------
def draw_ic_and_compute_alpha_scale(
    m_pms: int,
    rng_ic: np.random.Generator
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Draws the target Information Coefficient for each PM and computes the alpha scaling factor.

    Equations:
    IC^(i) ~ U[0.06, 0.10]
    c^(i) = (IC^(i))^2

    Parameters
    ----------
    m_pms : int
        The number of portfolio managers (M).
    rng_ic : np.random.Generator
        The isolated random generator for IC draws.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        ic_targets: 1D array of length M containing IC^(i).
        c_alpha: 1D array of length M containing c^(i).

    Raises
    ------
    SignalCalibrationError
        If the drawn ICs fall outside the mathematically defined bounds.
    """
    logger.info("Executing Task 19, Step 1: Drawing target ICs and computing alpha scaling factors.")

    # Draw uniform IC targets: IC ~ U[0.06, 0.10]
    ic_targets = rng_ic.uniform(low=0.06, high=0.10, size=m_pms)

    # Rigorous invariant assertion: IC must be strictly positive to prevent division by zero later
    if not np.all((ic_targets >= 0.06) & (ic_targets <= 0.10)):
        raise SignalCalibrationError("Drawn IC targets fall outside the [0.06, 0.10] bound.")

    # Compute alpha scaling factor: c = IC^2
    c_alpha = ic_targets ** 2

    logger.info(f"Successfully drawn IC targets and computed alpha scales for {m_pms} PMs.")
    return ic_targets, c_alpha

# -------------------------------------------------------------------------------------------------------------------------------
# Task 19, Step 2: Compute the noise scaling factor
# -------------------------------------------------------------------------------------------------------------------------------
def compute_noise_scaling_factor(ic_targets: np.ndarray) -> np.ndarray:
    """
    Computes the noise scaling factor for each PM based on their target IC.

    Equation: v^(i) = sqrt(1 / (IC^(i))^2 - 1)

    Parameters
    ----------
    ic_targets : np.ndarray
        1D array of length M containing the target Information Coefficients.

    Returns
    -------
    np.ndarray
        1D array of length M containing the noise scaling factors v^(i).

    Raises
    ------
    SignalCalibrationError
        If the IC targets are invalid, leading to mathematical singularities.
    """
    logger.info("Executing Task 19, Step 2: Computing noise scaling factors.")

    # Defensive invariant assertion: IC must be in (0, 1)
    if not np.all((ic_targets > 0.0) & (ic_targets < 1.0)):
        raise SignalCalibrationError("IC targets must be strictly between 0 and 1 to compute noise scaling.")

    # Compute noise scaling factor: v = sqrt(1 / IC^2 - 1)
    v_noise = np.sqrt((1.0 / (ic_targets ** 2)) - 1.0)

    # Verify strict positivity
    if not np.all(v_noise > 0.0):
        raise SignalCalibrationError("Computed noise scaling factors must be strictly positive.")

    logger.info("Successfully computed noise scaling factors.")
    return v_noise

# -------------------------------------------------------------------------------------------------------------------------------
# Task 19, Step 3: Draw temporal autocorrelation and emit frozen calibration object
# -------------------------------------------------------------------------------------------------------------------------------
def draw_autocorr_and_freeze_calibration(
    ic_targets: np.ndarray,
    c_alpha: np.ndarray,
    v_noise: np.ndarray,
    rng_phi: np.random.Generator
) -> Dict[int, AlphaCalibration]:
    """
    Draws the VAR(1) temporal autocorrelation coefficient and aggregates all parameters into immutable objects.

    Equation: phi^(i) ~ U[0.75, 0.85]

    Parameters
    ----------
    ic_targets : np.ndarray
        1D array of length M containing IC^(i).
    c_alpha : np.ndarray
        1D array of length M containing c^(i).
    v_noise : np.ndarray
        1D array of length M containing v^(i).
    rng_phi : np.random.Generator
        The isolated random generator for autocorrelation draws.

    Returns
    -------
    Dict[int, AlphaCalibration]
        A dictionary mapping pm_id (1 to M) to its frozen AlphaCalibration object.

    Raises
    ------
    SignalCalibrationError
        If the drawn autocorrelations violate stationarity bounds.
    """
    logger.info("Executing Task 19, Step 3: Drawing temporal autocorrelation and freezing calibration objects.")

    m_pms = len(ic_targets)

    # Draw uniform temporal autocorrelation: phi ~ U[0.75, 0.85]
    autocorr_phis = rng_phi.uniform(low=0.75, high=0.85, size=m_pms)

    # Rigorous invariant assertion: phi must be in (-1, 1) for VAR(1) stationarity
    if not np.all((autocorr_phis >= 0.75) & (autocorr_phis <= 0.85)):
        raise SignalCalibrationError("Drawn autocorrelation coefficients fall outside the [0.75, 0.85] bound.")

    alpha_calibration_dict: Dict[int, AlphaCalibration] = {}

    for i in range(m_pms):
        pm_id = i + 1

        # Instantiate the frozen dataclass
        # The __post_init__ method will automatically enforce invariants
        calib_obj = AlphaCalibration(
            pm_id=pm_id,
            ic_target=float(ic_targets[i]),
            c_alpha=float(c_alpha[i]),
            v_noise=float(v_noise[i]),
            autocorr_phi=float(autocorr_phis[i])
        )

        alpha_calibration_dict[pm_id] = calib_obj

    logger.info("Alpha calibration parameters successfully frozen.")
    return alpha_calibration_dict

# -------------------------------------------------------------------------------------------------------------------------------
# Task 19, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_19_orchestrator(
    study_config: Dict[str, Any],
    generators: Dict[str, np.random.Generator],
    manifest: Dict[str, Any]
) -> Tuple[Dict[int, AlphaCalibration], Dict[str, Any]]:
    """
    Orchestrates the drawing and freezing of PM signal calibration parameters.

    Parameters
    ----------
    study_config : Dict[str, Any]
        The master configuration dictionary.
    generators : Dict[str, np.random.Generator]
        The dictionary of isolated random generators instantiated in Task 4.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[int, AlphaCalibration], Dict[str, Any]]
        The dictionary of frozen AlphaCalibration objects and the augmented manifest.

    Raises
    ------
    SignalCalibrationError
        If any structural invariants or mathematical constraints are violated during generation.
    """
    logger.info("Commencing Task 19: Drawing PM signal calibration parameters.")

    # Extract configuration parameters
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    m_pms = firm_consts.get("M_portfolio_managers", 4)

    # Extract specific generators
    try:
        rng_ic = generators["pm_ic_target_seed"]
        rng_phi = generators["pm_var_phi_seed"]
    except KeyError as e:
        raise SignalCalibrationError(f"Required random generator missing: {e}")

    # Execute Step 1: Draw ICs and compute alpha scaling
    ic_targets, c_alpha = draw_ic_and_compute_alpha_scale(
        m_pms=m_pms,
        rng_ic=rng_ic
    )

    # Execute Step 2: Compute noise scaling factors
    v_noise = compute_noise_scaling_factor(
        ic_targets=ic_targets
    )

    # Execute Step 3: Draw autocorrelation and freeze objects
    alpha_calibration_dict = draw_autocorr_and_freeze_calibration(
        ic_targets=ic_targets,
        c_alpha=c_alpha,
        v_noise=v_noise,
        rng_phi=rng_phi
    )

    # Augment the provenance manifest with the exact drawn values for auditability
    if "pm_alpha_calibration" not in manifest:
        manifest["pm_alpha_calibration"] = {}

    for pm_id, calib_obj in alpha_calibration_dict.items():
        manifest["pm_alpha_calibration"][f"PM_{pm_id}"] = {
            "ic_target": calib_obj.ic_target,
            "c_alpha": calib_obj.c_alpha,
            "v_noise": calib_obj.v_noise,
            "autocorr_phi": calib_obj.autocorr_phi
        }

    logger.info("Task 19 complete. PM signal calibration parameters emitted.")

    # Return the frozen dictionary and the updated manifest
    return alpha_calibration_dict, manifest


In [ ]:
# Task 20: Construct the VAR(1) transition structure and solve the Lyapunov equation

class VAREquationError(Exception):
    """Custom exception raised when VAR(1) construction or Lyapunov solution fails mathematical invariants."""
    pass

# ===================================================================================
# Task 20: Construct the VAR(1) transition structure and solve the Lyapunov equation
# ===================================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 20, Step 1: Construct the block-diagonal VAR(1) transition matrix
# -------------------------------------------------------------------------------------------------------------------------------
def construct_var_transition_matrix(
    alpha_calibration_dict: Dict[int, Any],
    n_assets: int
) -> np.ndarray:
    """
    Constructs the diagonal representation of the VAR(1) transition matrix Phi.

    Equation: Phi = blkdiag(phi^(1) I_N, phi^(2) I_N, phi^(3) I_N, phi^(4) I_N)

    Parameters
    ----------
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    n_assets : int
        The total number of assets in the canonical universe (N).

    Returns
    -------
    np.ndarray
        A 1D NumPy array of length M*N representing the diagonal of Phi.

    Raises
    ------
    VAREquationError
        If the resulting diagonal vector has an incorrect shape.
    """
    logger.info("Executing Task 20, Step 1: Constructing the VAR(1) transition matrix diagonal.")

    m_pms = len(alpha_calibration_dict)

    # Extract the temporal autocorrelation coefficients in strict PM order (1 to M)
    autocorr_phis = np.array([
        alpha_calibration_dict[pm_id].autocorr_phi
        for pm_id in sorted(alpha_calibration_dict.keys())
    ])

    # Since each block is phi^(i) * I_N, the entire matrix Phi is strictly diagonal.
    # We construct only the 1D diagonal vector to save memory (O(MN) vs O((MN)^2)).
    # np.repeat duplicates each phi exactly N times.
    phi_diag = np.repeat(autocorr_phis, n_assets)

    # Rigorous invariant assertion: Verify exact shape
    expected_length = m_pms * n_assets
    if phi_diag.shape != (expected_length,):
        error_msg = f"Shape mismatch for phi_diag: Expected {(expected_length,)}, got {phi_diag.shape}."
        logger.error(error_msg)
        raise VAREquationError(error_msg)

    logger.info(f"Successfully constructed VAR(1) transition diagonal of length {expected_length}.")
    return phi_diag

# -------------------------------------------------------------------------------------------------------------------------------
# Task 20, Step 2: Construct cross-strategy noise correlation and empirical asset covariance
# -------------------------------------------------------------------------------------------------------------------------------
def construct_stationary_covariance_components(
    alpha_calibration_dict: Dict[int, Any],
    daily_returns_panel: pd.DataFrame,
    cross_strategy_corr: float = 0.3
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Constructs the cross-strategy noise matrix S_E and the empirical asset covariance Sigma_Asset.

    Equations:
    (S_E)_{ij} = (v^{(i)})^2 if i == j, else 0.3 * v^{(i)} * v^{(j)}
    Sigma_E = S_E \otimes Sigma_Asset

    Parameters
    ----------
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel of shape (T, N).
    cross_strategy_corr : float, optional
        The correlation scalar for off-diagonal elements (default is 0.3).

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        S_E: The cross-strategy noise matrix of shape (M, M).
        Sigma_Asset: The empirical asset covariance matrix of shape (N, N).

    Raises
    ------
    VAREquationError
        If the computed matrices are not symmetric positive semi-definite (PSD).
    """
    logger.info("Executing Task 20, Step 2: Constructing stationary covariance components.")

    m_pms = len(alpha_calibration_dict)
    n_assets = daily_returns_panel.shape[1]

    # 1. Construct S_E
    v_noise = np.array([
        alpha_calibration_dict[pm_id].v_noise
        for pm_id in sorted(alpha_calibration_dict.keys())
    ])

    # Compute the outer product scaled by the cross-strategy correlation
    S_E = cross_strategy_corr * np.outer(v_noise, v_noise)

    # Overwrite the diagonal with the exact variance (v^(i))^2
    np.fill_diagonal(S_E, v_noise ** 2)

    # Verify S_E is PSD (eigenvalues >= 0)
    evals_SE = np.linalg.eigvalsh(S_E)
    if np.any(evals_SE < -1e-10):
        raise VAREquationError(f"S_E is not positive semi-definite. Min eigenvalue: {np.min(evals_SE)}")

    # 2. Construct Sigma_Asset
    # The manuscript does not specify if this is rolling or full-sample.
    # We use the full-sample empirical covariance, dropping dates with any NaNs to ensure PSD.
    clean_returns = daily_returns_panel.dropna(how="any")
    n_clean_dates = len(clean_returns)
    logger.info(f"Computing full-sample Sigma_Asset using {n_clean_dates} complete observation dates.")

    Sigma_Asset = clean_returns.cov().to_numpy()

    # Verify Sigma_Asset is PSD
    evals_Sigma = np.linalg.eigvalsh(Sigma_Asset)
    if np.any(evals_Sigma < -1e-10):
        # If slightly negative due to numerical noise, clamp to zero
        logger.warning(f"Sigma_Asset has slightly negative eigenvalues (min: {np.min(evals_Sigma)}). Clamping to 0.")
        # Reconstruct PSD matrix
        evals_Sigma = np.maximum(evals_Sigma, 0.0)
        evecs_Sigma = np.linalg.eigh(Sigma_Asset)[1]
        Sigma_Asset = evecs_Sigma @ np.diag(evals_Sigma) @ evecs_Sigma.T

    # Rigorous invariant assertions
    if S_E.shape != (m_pms, m_pms):
        raise VAREquationError(f"S_E shape mismatch: Expected {(m_pms, m_pms)}, got {S_E.shape}.")
    if Sigma_Asset.shape != (n_assets, n_assets):
        raise VAREquationError(f"Sigma_Asset shape mismatch: Expected {(n_assets, n_assets)}, got {Sigma_Asset.shape}.")

    return S_E, Sigma_Asset

# -------------------------------------------------------------------------------------------------------------------------------
# Task 20, Step 3: Solve the discrete-time Lyapunov equation
# -------------------------------------------------------------------------------------------------------------------------------
def solve_lyapunov_for_innovation_covariance(
    S_E: np.ndarray,
    alpha_calibration_dict: Dict[int, Any]
) -> np.ndarray:
    """
    Solves the Lyapunov equation for the innovation covariance component S_U.

    Equation: Sigma_U = Sigma_E - Phi Sigma_E Phi^T
    Exploiting Kronecker structure: (S_U)_{ij} = (S_E)_{ij} * (1 - phi^(i) * phi^(j))

    Parameters
    ----------
    S_E : np.ndarray
        The cross-strategy noise matrix of shape (M, M).
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.

    Returns
    -------
    np.ndarray
        S_U: The innovation cross-strategy matrix of shape (M, M).

    Raises
    ------
    VAREquationError
        If the resulting S_U matrix is not positive semi-definite.
    """
    logger.info("Executing Task 20, Step 3: Solving the discrete-time Lyapunov equation.")

    m_pms = S_E.shape[0]

    autocorr_phis = np.array([
        alpha_calibration_dict[pm_id].autocorr_phi
        for pm_id in sorted(alpha_calibration_dict.keys())
    ])

    # Compute the scaling matrix: 1 - phi^(i) * phi^(j)
    phi_outer = np.outer(autocorr_phis, autocorr_phis)
    scale_matrix = 1.0 - phi_outer

    # Compute S_U via element-wise multiplication
    S_U = S_E * scale_matrix

    # Verify S_U is PSD (eigenvalues >= 0)
    # This is a critical mathematical invariant for the VAR(1) process to be valid
    evals_SU = np.linalg.eigvalsh(S_U)
    if np.any(evals_SU < -1e-10):
        error_msg = f"Critical failure: S_U is not positive semi-definite. Min eigenvalue: {np.min(evals_SU)}. Check phi and IC parameters."
        logger.error(error_msg)
        raise VAREquationError(error_msg)

    # Clamp any tiny negative eigenvalues from numerical noise
    if np.any(evals_SU < 0.0):
        evals_SU = np.maximum(evals_SU, 0.0)
        evecs_SU = np.linalg.eigh(S_U)[1]
        S_U = evecs_SU @ np.diag(evals_SU) @ evecs_SU.T

    logger.info("Lyapunov equation solved successfully. S_U is positive semi-definite.")
    return S_U

# -------------------------------------------------------------------------------------------------------------------------------
# Task 20, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_20_orchestrator(
    alpha_calibration_dict: Dict[int, Any],
    daily_returns_panel: pd.DataFrame,
    study_config: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]:
    """
    Orchestrates the construction of the VAR(1) transition structure and solves the Lyapunov equation.

    Parameters
    ----------
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, Any]]
        phi_diag: The 1D diagonal of the transition matrix.
        S_E: The stationary cross-strategy covariance matrix.
        S_U: The innovation cross-strategy covariance matrix.
        Sigma_Asset: The empirical asset covariance matrix.
        manifest: The augmented provenance manifest.

    Raises
    ------
    VAREquationError
        If any structural invariants or mathematical constraints are violated during computation.
    """
    logger.info("Commencing Task 20: Constructing VAR(1) structure and solving Lyapunov equation.")

    n_assets = daily_returns_panel.shape[1]
    sim_config = study_config.get("SIMULATION_ECONOMETRICS", {})
    cross_corr = sim_config.get("cross_strategy_noise_correlation", 0.3)

    # Execute Step 1: Construct transition matrix diagonal
    phi_diag = construct_var_transition_matrix(
        alpha_calibration_dict=alpha_calibration_dict,
        n_assets=n_assets
    )

    # Execute Step 2: Construct stationary covariance components
    S_E, Sigma_Asset = construct_stationary_covariance_components(
        alpha_calibration_dict=alpha_calibration_dict,
        daily_returns_panel=daily_returns_panel,
        cross_strategy_corr=cross_corr
    )

    # Execute Step 3: Solve Lyapunov equation for innovation covariance
    S_U = solve_lyapunov_for_innovation_covariance(
        S_E=S_E,
        alpha_calibration_dict=alpha_calibration_dict
    )

    # Augment the provenance manifest with metadata and placeholder conventions
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["var_process_components"] = {
        "phi_diag_shape": phi_diag.shape,
        "S_E_shape": S_E.shape,
        "S_U_shape": S_U.shape,
        "Sigma_Asset_shape": Sigma_Asset.shape,
        "kronecker_structure_utilized": True
    }

    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "Sigma_Asset_for_VAR_construction",
        "manuscript_status": "unspecified",
        "value_used": "full-sample empirical covariance (dropping NaN dates)",
        "source": "rational placeholder"
    })

    logger.info("Task 20 complete. VAR(1) structural components emitted.")

    # Return the factored components and the updated manifest
    return phi_diag, S_E, S_U, Sigma_Asset, manifest


In [ ]:
# Task 21: Simulate the noise path and construct PM alpha vectors

class AlphaSimulationError(Exception):
    """Custom exception raised when VAR(1) simulation or alpha construction fails mathematical invariants."""
    pass

# ==============================================================================
# Task 21: Simulate the noise path and construct PM alpha vectors
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 21, Step 1: Simulate the VAR(1) noise path
# -------------------------------------------------------------------------------------------------------------------------------
def simulate_var_noise_path(
    phi_diag: np.ndarray,
    S_E: np.ndarray,
    S_U: np.ndarray,
    Sigma_Asset: np.ndarray,
    master_trading_calendar: pd.DatetimeIndex,
    rng_var: np.random.Generator
) -> Dict[pd.Timestamp, np.ndarray]:
    """
    Simulates the VAR(1) noise process E_t = Phi E_{t-1} + U_t for all calendar dates.

    Parameters
    ----------
    phi_diag : np.ndarray
        The 1D diagonal of the transition matrix (length MN).
    S_E : np.ndarray
        The stationary cross-strategy covariance matrix (M x M).
    S_U : np.ndarray
        The innovation cross-strategy covariance matrix (M x M).
    Sigma_Asset : np.ndarray
        The empirical asset covariance matrix (N x N).
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    rng_var : np.random.Generator
        The isolated random generator for VAR innovations.

    Returns
    -------
    Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping each calendar date to the noise state vector E_t of length MN.

    Raises
    ------
    AlphaSimulationError
        If Cholesky decomposition fails despite regularization.
    """
    logger.info("Executing Task 21, Step 1: Simulating the VAR(1) noise path.")

    m_pms = S_E.shape[0]
    n_assets = Sigma_Asset.shape[0]
    mn_length = m_pms * n_assets

    # Regularize Sigma_Asset to ensure strict positive definiteness for Cholesky
    ridge = 1e-8 * np.eye(n_assets)
    try:
        L_Sigma = np.linalg.cholesky(Sigma_Asset + ridge)
        L_SE = np.linalg.cholesky(S_E)
        L_SU = np.linalg.cholesky(S_U)
    except np.linalg.LinAlgError as e:
        raise AlphaSimulationError(f"Cholesky decomposition failed: {e}")

    noise_state_dict: Dict[pd.Timestamp, np.ndarray] = {}

    # Initialize E_0 from the stationary distribution N(0, S_E \otimes Sigma_Asset)
    # Using Kronecker sampling: E_0_matrix = L_Sigma @ Z @ L_SE^T
    Z_0 = rng_var.standard_normal((n_assets, m_pms))
    E_0_matrix = L_Sigma @ Z_0 @ L_SE.T

    # Flatten using Fortran (column-major) order so that the first N elements belong to PM 1
    E_prev = E_0_matrix.flatten(order='F')

    # Simulate the path chronologically
    for current_date in master_trading_calendar:
        # Draw innovation U_t ~ N(0, S_U \otimes Sigma_Asset)
        Z_t = rng_var.standard_normal((n_assets, m_pms))
        U_t_matrix = L_Sigma @ Z_t @ L_SU.T
        U_t = U_t_matrix.flatten(order='F')

        # Evolve VAR(1): E_t = Phi E_{t-1} + U_t
        # Since Phi is diagonal, this is an element-wise multiplication
        E_t = phi_diag * E_prev + U_t

        # Store and update state
        noise_state_dict[current_date] = E_t
        E_prev = E_t

    # Rigorous invariant assertion
    if len(noise_state_dict) != len(master_trading_calendar):
        raise AlphaSimulationError("Noise path length does not match calendar length.")

    logger.info(f"Successfully simulated noise path for {len(master_trading_calendar)} dates.")
    return noise_state_dict

# -------------------------------------------------------------------------------------------------------------------------------
# Task 21, Step 2: Construct PM synthetic alpha vectors
# -------------------------------------------------------------------------------------------------------------------------------
def construct_synthetic_alphas(
    noise_state_dict: Dict[pd.Timestamp, np.ndarray],
    forward_return_vectors: Dict[pd.Timestamp, np.ndarray],
    alpha_calibration_dict: Dict[int, Any],
    valid_decision_dates: pd.DatetimeIndex
) -> Dict[Tuple[pd.Timestamp, int], np.ndarray]:
    """
    Constructs the synthetic alpha vector for each PM at each valid decision date.

    Equation: alpha_t^{(i)} = c^{(i)}(R_t + E_t^{(i)})

    Parameters
    ----------
    noise_state_dict : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to the full noise state vector E_t.
    forward_return_vectors : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to the forward return vector R_t.
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.

    Returns
    -------
    Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Dictionary mapping (date, pm_id) to the alpha vector of length N.
    """
    logger.info("Executing Task 21, Step 2: Constructing PM synthetic alpha vectors.")

    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray] = {}

    # Extract N from the first available forward return vector
    n_assets = len(next(iter(forward_return_vectors.values())))

    for current_date in valid_decision_dates:
        R_t = forward_return_vectors[current_date]
        E_t_full = noise_state_dict[current_date]

        for pm_id, calib_obj in alpha_calibration_dict.items():
            c_alpha = calib_obj.c_alpha

            # Extract the PM-specific block from the flattened noise vector
            # PM 1 is at index 0 to N-1, PM 2 is at N to 2N-1, etc.
            start_idx = (pm_id - 1) * n_assets
            end_idx = pm_id * n_assets
            E_t_i = E_t_full[start_idx:end_idx]

            # Compute alpha: c * (R + E)
            alpha_t_i = c_alpha * (R_t + E_t_i)

            # Handle NaNs: If R_t has NaNs, alpha will have NaNs.
            # We impute NaNs with 0.0 (zero expected excess return) to prevent solver failure.
            nan_mask = np.isnan(alpha_t_i)
            if nan_mask.any():
                alpha_t_i[nan_mask] = 0.0

            alpha_dictionary[(current_date, pm_id)] = alpha_t_i

    logger.info(f"Successfully constructed {len(alpha_dictionary)} alpha vectors.")
    return alpha_dictionary

# -------------------------------------------------------------------------------------------------------------------------------
# Task 21, Step 3: Validate empirical Information Coefficients
# -------------------------------------------------------------------------------------------------------------------------------
def validate_empirical_ics(
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    forward_return_vectors: Dict[pd.Timestamp, np.ndarray],
    alpha_calibration_dict: Dict[int, Any],
    valid_decision_dates: pd.DatetimeIndex
) -> Dict[int, float]:
    """
    Computes the empirical cross-sectional IC for each PM and compares it to the target.

    Parameters
    ----------
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Dictionary mapping (date, pm_id) to the alpha vector.
    forward_return_vectors : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to the forward return vector R_t.
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates.

    Returns
    -------
    Dict[int, float]
        Dictionary mapping pm_id to its mean empirical IC.
    """
    logger.info("Executing Task 21, Step 3: Validating empirical Information Coefficients.")

    empirical_ics: Dict[int, float] = {}

    for pm_id, calib_obj in alpha_calibration_dict.items():
        target_ic = calib_obj.ic_target
        daily_ics = []

        for current_date in valid_decision_dates:
            alpha_t = alpha_dictionary[(current_date, pm_id)]
            R_t = forward_return_vectors[current_date]

            # Compute correlation only on valid (non-NaN) intersection
            valid_mask = ~np.isnan(R_t)

            # Need at least 2 points to compute correlation
            if valid_mask.sum() > 1:
                # np.corrcoef returns a 2x2 matrix, we want the off-diagonal element
                corr = np.corrcoef(alpha_t[valid_mask], R_t[valid_mask])[0, 1]
                if not np.isnan(corr):
                    daily_ics.append(corr)

        mean_empirical_ic = float(np.mean(daily_ics))
        empirical_ics[pm_id] = mean_empirical_ic

        # Diagnostic check: warn if deviation exceeds 0.03
        deviation = abs(mean_empirical_ic - target_ic)
        if deviation > 0.03:
            logger.warning(
                f"PM {pm_id} empirical IC ({mean_empirical_ic:.4f}) deviates from target "
                f"({target_ic:.4f}) by {deviation:.4f}. This is a diagnostic warning, not a failure."
            )
        else:
            logger.info(f"PM {pm_id} empirical IC ({mean_empirical_ic:.4f}) aligns with target ({target_ic:.4f}).")

    return empirical_ics

# -------------------------------------------------------------------------------------------------------------------------------
# Task 21, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_21_orchestrator(
    phi_diag: np.ndarray,
    S_E: np.ndarray,
    S_U: np.ndarray,
    Sigma_Asset: np.ndarray,
    forward_return_vectors: Dict[pd.Timestamp, np.ndarray],
    alpha_calibration_dict: Dict[int, Any],
    master_trading_calendar: pd.DatetimeIndex,
    valid_decision_dates: pd.DatetimeIndex,
    generators: Dict[str, np.random.Generator],
    manifest: Dict[str, Any]
) -> Tuple[Dict[Tuple[pd.Timestamp, int], np.ndarray], Dict[str, Any]]:
    """
    Orchestrates the simulation of the noise path and the construction of PM alpha vectors.

    Parameters
    ----------
    phi_diag : np.ndarray
        The 1D diagonal of the transition matrix.
    S_E : np.ndarray
        The stationary cross-strategy covariance matrix.
    S_U : np.ndarray
        The innovation cross-strategy covariance matrix.
    Sigma_Asset : np.ndarray
        The empirical asset covariance matrix.
    forward_return_vectors : Dict[pd.Timestamp, np.ndarray]
        Dictionary mapping dates to the forward return vector R_t.
    alpha_calibration_dict : Dict[int, AlphaCalibration]
        Dictionary mapping pm_id to its frozen AlphaCalibration object.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    valid_decision_dates : pd.DatetimeIndex
        The frozen subset of dates on which PM optimizations will occur.
    generators : Dict[str, np.random.Generator]
        The dictionary of isolated random generators.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Tuple[Dict[Tuple[pd.Timestamp, int], np.ndarray], Dict[str, Any]]
        The dictionary of synthetic alpha vectors and the augmented manifest.

    Raises
    ------
    AlphaSimulationError
        If any structural invariants or mathematical constraints are violated during simulation.
    """
    logger.info("Commencing Task 21: Simulating noise path and constructing PM alpha vectors.")

    try:
        rng_var = generators["var_innovation_seed"]
    except KeyError as e:
        raise AlphaSimulationError(f"Required random generator missing: {e}")

    # Execute Step 1: Simulate VAR(1) noise path
    noise_state_dict = simulate_var_noise_path(
        phi_diag=phi_diag,
        S_E=S_E,
        S_U=S_U,
        Sigma_Asset=Sigma_Asset,
        master_trading_calendar=master_trading_calendar,
        rng_var=rng_var
    )

    # Execute Step 2: Construct synthetic alphas
    alpha_dictionary = construct_synthetic_alphas(
        noise_state_dict=noise_state_dict,
        forward_return_vectors=forward_return_vectors,
        alpha_calibration_dict=alpha_calibration_dict,
        valid_decision_dates=valid_decision_dates
    )

    # Execute Step 3: Validate empirical ICs
    empirical_ics = validate_empirical_ics(
        alpha_dictionary=alpha_dictionary,
        forward_return_vectors=forward_return_vectors,
        alpha_calibration_dict=alpha_calibration_dict,
        valid_decision_dates=valid_decision_dates
    )

    # Augment the provenance manifest with metadata and placeholder conventions
    if "derived_artifacts_metadata" not in manifest:
        manifest["derived_artifacts_metadata"] = {}

    manifest["derived_artifacts_metadata"]["empirical_information_coefficients"] = empirical_ics

    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "var_initial_state_E0",
        "manuscript_status": "unspecified",
        "value_used": "drawn from stationary distribution N(0, Sigma_E)",
        "source": "rational placeholder"
    })

    manifest["placeholder_assumptions"].append({
        "convention_name": "alpha_nan_imputation",
        "manuscript_status": "unspecified",
        "value_used": "NaNs in alpha replaced with 0.0 to prevent solver failure",
        "source": "numerical stability guardrail"
    })

    logger.info("Task 21 complete. Synthetic alpha vectors emitted.")

    # Return the alpha dictionary and the updated manifest
    # Note: noise_state_dict is intentionally NOT returned to allow garbage collection and save memory
    return alpha_dictionary, manifest


In [ ]:
# Task 22: Construct daily transaction-cost parameters

class TransactionCostParameterError(Exception):
    """Custom exception raised when transaction cost parameterization fails mathematical invariants."""
    pass

# ==============================================================================
# Task 22: Construct daily transaction-cost parameters
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 22, Step 1: Compute the market-impact coefficient vector
# -------------------------------------------------------------------------------------------------------------------------------
def compute_market_impact_coefficient(
    nu_t: np.ndarray,
    omega_t: np.ndarray,
    v_firm_t: float,
    impact_coefficient_b: float = 1.0,
    clamp_value: float = 1e6
) -> np.ndarray:
    """
    Computes the market-impact coefficient vector for a specific date and firm NAV.

    Equation: (kappa_impact)_{t,j} = b_j * nu_{t,j} / sqrt(omega_{t,j} / V_t)

    Parameters
    ----------
    nu_t : np.ndarray
        The daily volatility vector of shape (N,).
    omega_t : np.ndarray
        The daily dollar volume vector of shape (N,).
    v_firm_t : float
        The endogenous, protocol-specific firm NAV at date t.
    impact_coefficient_b : float, optional
        The uniform impact coefficient (default is 1.0).
    clamp_value : float, optional
        The prohibitively large penalty assigned to assets with zero/NaN volume (default is 1e6).

    Returns
    -------
    np.ndarray
        The market-impact coefficient vector kappa_impact_t of shape (N,).

    Raises
    ------
    TransactionCostParameterError
        If the firm NAV is non-positive or if the output contains NaNs.
    """
    if v_firm_t <= 0.0:
        raise TransactionCostParameterError(f"Firm NAV must be strictly positive, got {v_firm_t}.")

    # Identify assets with zero, negative, or NaN dollar volume
    # These assets have no liquidity; trading them should incur infinite cost.
    with np.errstate(invalid='ignore'):
        invalid_volume_mask = (omega_t <= 0.0) | np.isnan(omega_t)

    # Compute the weight-equivalent market volume
    # We use np.where to avoid division by zero warnings; invalid entries are temporarily set to 1.0
    safe_omega = np.where(invalid_volume_mask, 1.0, omega_t)
    weight_equivalent_volume = safe_omega / v_firm_t

    # Compute kappa_impact
    kappa_impact_t = (impact_coefficient_b * nu_t) / np.sqrt(weight_equivalent_volume)

    # Apply the clamp value to assets with invalid volume or NaN volatility
    invalid_mask_total = invalid_volume_mask | np.isnan(nu_t)
    kappa_impact_t = np.where(invalid_mask_total, clamp_value, kappa_impact_t)

    # Rigorous invariant assertion
    if np.isnan(kappa_impact_t).any() or np.isinf(kappa_impact_t).any():
        raise TransactionCostParameterError("kappa_impact_t contains NaN or infinite values after clamping.")

    return kappa_impact_t

# -------------------------------------------------------------------------------------------------------------------------------
# Task 22, Step 2: Emit the complete transaction-cost function closures
# -------------------------------------------------------------------------------------------------------------------------------
def build_transaction_cost_closures(
    kappa_spread_t: np.ndarray,
    kappa_impact_t: np.ndarray,
    gamma_tc: float = 0.15
) -> Tuple[Callable[[cp.Variable], cp.Expression], Callable[[np.ndarray], float]]:
    """
    Constructs symbolic (CVXPY) and numerical (NumPy) evaluators for the 3/2-power transaction cost model.

    Equation: phi_tc(z) = 0.5 * kappa_spread^T |z| + kappa_impact^T |z|^(3/2)

    Parameters
    ----------
    kappa_spread_t : np.ndarray
        The fractional bid-ask spread vector of shape (N,).
    kappa_impact_t : np.ndarray
        The market-impact coefficient vector of shape (N,).
    gamma_tc : float, optional
        The scaling parameter for the transaction cost term (default is 0.15).

    Returns
    -------
    Tuple[Callable[[cp.Variable], cp.Expression], Callable[[np.ndarray], float]]
        cvxpy_tc_evaluator: A closure taking a CVXPY variable z and returning a convex Expression.
        numpy_tc_evaluator: A closure taking a NumPy array z and returning a scalar float.
    """

    def cvxpy_tc_evaluator(z: cp.Variable) -> cp.Expression:
        """Symbolic CVXPY evaluator for the transaction cost."""
        # cp.abs(z) is convex and non-negative, satisfying the DCP rules for cp.power(..., 1.5)
        half_spread_cost = 0.5 * (kappa_spread_t @ cp.abs(z))
        impact_cost = kappa_impact_t @ cp.power(cp.abs(z), 1.5)
        return gamma_tc * (half_spread_cost + impact_cost)

    def numpy_tc_evaluator(z: np.ndarray) -> float:
        """Numerical NumPy evaluator for the realized transaction cost."""
        abs_z = np.abs(z)
        half_spread_cost = 0.5 * np.dot(kappa_spread_t, abs_z)
        impact_cost = np.dot(kappa_impact_t, abs_z ** 1.5)
        return float(gamma_tc * (half_spread_cost + impact_cost))

    return cvxpy_tc_evaluator, numpy_tc_evaluator

# -------------------------------------------------------------------------------------------------------------------------------
# Task 22, Step 3: Define the shorting-cost function closures
# -------------------------------------------------------------------------------------------------------------------------------
def build_shorting_cost_closures(
    rf_daily_t: float,
    gamma_short: float = 1.0
) -> Tuple[Callable[[cp.Variable], cp.Expression], Callable[[cp.Variable, np.ndarray], cp.Expression]]:
    """
    Constructs symbolic (CVXPY) evaluators for the PM-local and Firm-level shorting costs.

    Equation: phi_short(w) = r_rf * sum(max(0, -w_j))

    Parameters
    ----------
    rf_daily_t : float
        The daily risk-free rate scalar.
    gamma_short : float, optional
        The scaling parameter for the shorting cost term (default is 1.0).

    Returns
    -------
    Tuple[Callable[[cp.Variable], cp.Expression], Callable[[cp.Variable, np.ndarray], cp.Expression]]
        cvxpy_short_local: Closure taking PM weights w and returning the local shorting cost.
        cvxpy_short_firm: Closure taking firm net trade z and pre-trade firm weights w_curr_firm.
    """

    def cvxpy_short_local(w: cp.Variable) -> cp.Expression:
        """
        Symbolic evaluator for a single PM's local shorting cost.
        Applies the borrow cost to the PM's gross short positions.
        """
        # cp.pos(-w) is equivalent to max(0, -w) and is DCP convex
        return gamma_short * rf_daily_t * cp.sum(cp.pos(-w))

    def cvxpy_short_firm(z_firm: cp.Variable, w_curr_firm: np.ndarray) -> cp.Expression:
        """
        Symbolic evaluator for the firm's net shorting cost.
        Applies the borrow cost to the firm's net post-trade positions.
        w_post_firm = w_curr_firm + z_firm
        """
        w_post_firm = w_curr_firm + z_firm
        return gamma_short * rf_daily_t * cp.sum(cp.pos(-w_post_firm))

    return cvxpy_short_local, cvxpy_short_firm

# -------------------------------------------------------------------------------------------------------------------------------
# Task 22, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_22_orchestrator(
    nu_t: np.ndarray,
    omega_t: np.ndarray,
    kappa_spread_t: np.ndarray,
    rf_daily_t: float,
    v_firm_t: float,
    study_config: Dict[str, Any]
) -> Tuple[np.ndarray, Callable, Callable, Callable, Callable]:
    """
    Orchestrates the construction of daily transaction-cost and shorting-cost parameters.
    Designed to be called inside the highly-performant walk-forward loop.

    Parameters
    ----------
    nu_t : np.ndarray
        The daily volatility vector.
    omega_t : np.ndarray
        The daily dollar volume vector.
    kappa_spread_t : np.ndarray
        The fractional bid-ask spread vector.
    rf_daily_t : float
        The daily risk-free rate.
    v_firm_t : float
        The endogenous, protocol-specific firm NAV at date t.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[np.ndarray, Callable, Callable, Callable, Callable]
        kappa_impact_t: The computed market-impact coefficient vector.
        cvxpy_tc_evaluator: Symbolic TC closure.
        numpy_tc_evaluator: Numerical TC closure.
        cvxpy_short_local: Symbolic local shorting cost closure.
        cvxpy_short_firm: Symbolic firm net shorting cost closure.
    """
    # Extract configuration parameters
    firm_consts = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    gamma_tc = firm_consts.get("gamma_tc", 0.15)
    gamma_short = firm_consts.get("gamma_shorting", 1.0)

    tc_params = study_config.get("TC_MODEL_PARAMETERS", {})
    impact_b = tc_params.get("impact_coefficient_b", 1.0)

    # Execute Step 1: Compute market-impact coefficient
    kappa_impact_t = compute_market_impact_coefficient(
        nu_t=nu_t,
        omega_t=omega_t,
        v_firm_t=v_firm_t,
        impact_coefficient_b=impact_b,
        clamp_value=1e6
    )

    # Execute Step 2: Build TC closures
    cvxpy_tc_evaluator, numpy_tc_evaluator = build_transaction_cost_closures(
        kappa_spread_t=kappa_spread_t,
        kappa_impact_t=kappa_impact_t,
        gamma_tc=gamma_tc
    )

    # Execute Step 3: Build shorting cost closures
    cvxpy_short_local, cvxpy_short_firm = build_shorting_cost_closures(
        rf_daily_t=rf_daily_t,
        gamma_short=gamma_short
    )

    # Return the lightweight tuple of parameters and closures for the optimization engine
    return kappa_impact_t, cvxpy_tc_evaluator, numpy_tc_evaluator, cvxpy_short_local, cvxpy_short_firm


In [ ]:
# Task 23: Construct the ADMM diagonal scaling matrix

class ADMMScalingError(Exception):
    """Custom exception raised when the ADMM scaling matrix fails strict positivity invariants."""
    pass

# ==============================================================================
# Task 23: Construct the ADMM diagonal scaling matrix
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 23, Step 1: Compute the diagonal scaling entries
# -------------------------------------------------------------------------------------------------------------------------------
def compute_admm_scaling_diagonal(
    kappa_impact_t: np.ndarray,
    positivity_floor: float = 1e-6
) -> np.ndarray:
    """
    Computes the diagonal entries of the ADMM scaling matrix D_t^{scale}.

    Equation: D_{t,jj}^{scale} = sqrt(2 * (kappa_impact)_{t,j})

    Parameters
    ----------
    kappa_impact_t : np.ndarray
        The market-impact coefficient vector of shape (N,), computed dynamically using the current firm NAV.
    positivity_floor : float, optional
        The minimum allowable value for the diagonal entries to ensure strict positivity (default is 1e-6).

    Returns
    -------
    np.ndarray
        A 1D NumPy array of shape (N,) representing the diagonal of D_t^{scale}.

    Raises
    ------
    ADMMScalingError
        If the input vector contains NaNs or infinite values.
    """
    # Verify input integrity
    if np.isnan(kappa_impact_t).any() or np.isinf(kappa_impact_t).any():
        raise ADMMScalingError("Input kappa_impact_t contains NaN or infinite values.")

    # Compute the heuristic scaling diagonal: sqrt(2 * kappa_impact)
    # We use np.maximum to guard against any theoretical negative values before the sqrt,
    # though Task 22 should guarantee non-negativity.
    safe_kappa = np.maximum(kappa_impact_t, 0.0)
    d_scale_diag_t = np.sqrt(2.0 * safe_kappa)

    # Apply the strict positivity floor
    # The ADMM algorithm requires D \in R_{++}^{N \times N}. A zero entry would destroy the strong convexity
    # of the proximal term, leading to solver failure or algorithmic divergence.
    d_scale_diag_t = np.maximum(d_scale_diag_t, positivity_floor)

    return d_scale_diag_t

# -------------------------------------------------------------------------------------------------------------------------------
# Task 23, Step 2 & 3: Verify strict positivity and emit the diagonal vector
# -------------------------------------------------------------------------------------------------------------------------------
def verify_and_emit_scaling_matrix(
    d_scale_diag_t: np.ndarray
) -> np.ndarray:
    """
    Verifies the strict positivity of the scaling diagonal and emits it for downstream element-wise operations.

    Note: This function intentionally does NOT form the dense N x N matrix to conserve memory and compute.
    All downstream ADMM operations must use element-wise multiplication.

    Parameters
    ----------
    d_scale_diag_t : np.ndarray
        The computed diagonal scaling vector of shape (N,).

    Returns
    -------
    np.ndarray
        The verified diagonal scaling vector.

    Raises
    ------
    ADMMScalingError
        If any entry in the vector is less than or equal to zero.
    """
    # Rigorous invariant assertion: Strict positivity
    if not np.all(d_scale_diag_t > 0.0):
        error_msg = "Critical failure: ADMM scaling matrix diagonal contains non-positive entries."
        logger.error(error_msg)
        raise ADMMScalingError(error_msg)

    # Return the 1D vector.
    # Persistence (Step 3) is handled by the caller (the walk-forward loop),
    # as this vector is endogenous to the specific protocol's NAV trajectory.
    return d_scale_diag_t

# -------------------------------------------------------------------------------------------------------------------------------
# Task 23, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_23_orchestrator(
    kappa_impact_t: np.ndarray
) -> np.ndarray:
    """
    Orchestrates the construction and verification of the ADMM diagonal scaling matrix.
    Designed to be highly performant for execution inside the walk-forward loop.

    Parameters
    ----------
    kappa_impact_t : np.ndarray
        The market-impact coefficient vector of shape (N,), computed dynamically using the current firm NAV.

    Returns
    -------
    np.ndarray
        The verified diagonal scaling vector D_t^{scale} of shape (N,).

    Raises
    ------
    ADMMScalingError
        If mathematical invariants (like strict positivity) are violated.
    """
    # Execute Step 1: Compute the diagonal entries
    d_scale_diag_t = compute_admm_scaling_diagonal(
        kappa_impact_t=kappa_impact_t,
        positivity_floor=1e-6
    )

    # Execute Step 2 & 3: Verify positivity and emit
    verified_d_scale_diag_t = verify_and_emit_scaling_matrix(
        d_scale_diag_t=d_scale_diag_t
    )

    # Return the lightweight 1D vector for use in the ADMM proximal updates
    return verified_d_scale_diag_t


In [ ]:
# Task 24: Define the PM local optimization problem callable

class PMSolverError(Exception):
    """Custom exception raised when the PM local optimization problem fails to solve optimally."""
    pass

class PMOptimizationResult(NamedTuple):
    """
    A strictly typed, immutable record of the PM's local optimization output.

    Attributes
    ----------
    w_opt : np.ndarray
        The optimal post-trade portfolio weight vector of shape (N,).
    c_opt : float
        The optimal post-trade cash position.
    z_opt : np.ndarray
        The optimal trade weight vector of shape (N,).
    status : str
        The CVXPY solver status (e.g., 'optimal', 'infeasible').
    objective_value : float
        The final value of the objective function.
    """
    w_opt: np.ndarray
    c_opt: float
    z_opt: np.ndarray
    status: str
    objective_value: float

# ==============================================================================
# Task 24: Define the PM local optimization problem callable
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 24, Steps 1, 2 & 3: Construct, solve, and extract the PM local problem
# -------------------------------------------------------------------------------------------------------------------------------
def solve_pm_local_problem(
    alpha_t_i: np.ndarray,
    F_t_annual: np.ndarray,
    D_t_idio_annual: np.ndarray,
    w_curr_i: np.ndarray,
    c_curr_i: float,
    rf_daily_t: float,
    mask_i: np.ndarray,
    sigma_target_i: float,
    cvxpy_tc_evaluator: Callable[[cp.Variable], cp.Expression],
    cvxpy_short_local: Callable[[cp.Variable], cp.Expression],
    institutional_constraints: Dict[str, float],
    firm_global_params: Dict[str, float],
    solver_settings: Dict[str, Any]
) -> PMOptimizationResult:
    """
    Constructs and solves the PM's local convex optimization problem (Equation 6).

    Parameters
    ----------
    alpha_t_i : np.ndarray
        The synthetic alpha vector of shape (N,).
    F_t_annual : np.ndarray
        The annualized factor loading matrix of shape (N, J).
    D_t_idio_annual : np.ndarray
        The annualized idiosyncratic variance vector of shape (N,).
    w_curr_i : np.ndarray
        The current portfolio weight vector of shape (N,).
    c_curr_i : float
        The current cash position.
    rf_daily_t : float
        The daily risk-free rate.
    mask_i : np.ndarray
        The boolean tradable-universe mask of shape (N,).
    sigma_target_i : float
        The annualized volatility target.
    cvxpy_tc_evaluator : Callable[[cp.Variable], cp.Expression]
        Closure returning the symbolic transaction cost expression.
    cvxpy_short_local : Callable[[cp.Variable], cp.Expression]
        Closure returning the symbolic local shorting cost expression.
    institutional_constraints : Dict[str, float]
        Dictionary containing L, C, S, T limits.
    firm_global_params : Dict[str, float]
        Dictionary containing penalty weights (gamma_risk, gamma_turnover).
    solver_settings : Dict[str, Any]
        Dictionary containing solver configuration (name, tolerances).

    Returns
    -------
    PMOptimizationResult
        A NamedTuple containing the optimal w, c, z vectors and solver diagnostics.

    Raises
    ------
    PMSolverError
        If the problem is infeasible or unbounded, based on the configured failure policy.
    """
    n_assets = len(alpha_t_i)

    # Extract constraints and penalties
    L = institutional_constraints.get("leverage_limit_L", 1.5)
    C = institutional_constraints.get("concentration_limit_C", 0.2)
    S = institutional_constraints.get("shorting_limit_S", 0.5)
    T = institutional_constraints.get("turnover_limit_T", 0.2)

    gamma_risk = firm_global_params.get("gamma_risk", 20.0)
    gamma_turn = firm_global_params.get("gamma_turnover", 1.0)

    # -------------------------------------------------------------------------
    # 1. Define CVXPY Variables
    # -------------------------------------------------------------------------
    w = cp.Variable(n_assets, name="w")
    c = cp.Variable(name="c")
    z = cp.Variable(n_assets, name="z")
    s_risk = cp.Variable(nonneg=True, name="s_risk")
    s_turn = cp.Variable(nonneg=True, name="s_turn")

    # -------------------------------------------------------------------------
    # 2. Construct Objective Function
    # -------------------------------------------------------------------------
    # Alpha return (linear)
    alpha_term = -alpha_t_i @ w

    # Cash return (linear)
    cash_term = -rf_daily_t * c

    # Penalties (linear in slacks)
    risk_penalty = gamma_risk * s_risk
    turn_penalty = gamma_turn * s_turn

    # Transaction cost (convex, 3/2 power)
    tc_term = cvxpy_tc_evaluator(z)

    # Shorting cost (convex, piecewise linear)
    short_term = cvxpy_short_local(w)

    objective = cp.Minimize(
        alpha_term + cash_term + risk_penalty + turn_penalty + tc_term + short_term
    )

    # -------------------------------------------------------------------------
    # 3. Construct Constraints
    # -------------------------------------------------------------------------
    constraints = []

    # Trade identity: w - w_curr = z
    constraints.append(w - w_curr_i == z)

    # Cash identity: 1 - 1^T w = c
    constraints.append(1.0 - cp.sum(w) == c)

    # Leverage limit: ||w||_1 <= L
    constraints.append(cp.norm1(w) <= L)

    # Concentration limit: |w| <= C
    constraints.append(cp.abs(w) <= C)

    # Shorting limit: 1^T max(0, -w) <= S
    constraints.append(cp.sum(cp.pos(-w)) <= S)

    # Turnover limit with slack: ||w - w_curr||_2 + |c - c_curr| <= 2T + s_turn
    constraints.append(cp.norm2(z) + cp.abs(c - c_curr_i) <= 2.0 * T + s_turn)

    # Factored Risk constraint with slack: ||Sigma^(1/2) w||_2 <= sigma_target + s_risk
    # We construct the stacked vector y = [F^T w; diag(sqrt(D_idio)) w]
    # This avoids forming the dense N x N covariance matrix.
    # Note: F_t_annual and D_t_idio_annual must be used to match sigma_target_i.

    # Handle potential NaNs in F_t and D_t by replacing with 0.0 for the matrix ops.
    # The mask constraint below ensures w_j = 0 for these assets anyway.
    safe_F = np.nan_to_num(F_t_annual, nan=0.0)
    safe_D = np.nan_to_num(D_t_idio_annual, nan=0.0)

    factor_risk_component = safe_F.T @ w
    idio_risk_component = cp.multiply(np.sqrt(safe_D), w)

    stacked_risk_vector = cp.hstack([factor_risk_component, idio_risk_component])
    constraints.append(cp.norm2(stacked_risk_vector) <= sigma_target_i + s_risk)

    # Tradable-universe mask: w_j = 0 for assets not in the PM's universe
    # ~mask_i inverts the boolean array, selecting non-tradable assets
    non_tradable_mask = ~mask_i
    if non_tradable_mask.any():
        constraints.append(w[non_tradable_mask] == 0.0)

    # -------------------------------------------------------------------------
    # 4. Solve the Problem
    # -------------------------------------------------------------------------
    problem = cp.Problem(objective, constraints)

    solver_name = solver_settings.get("solver_name", "MOSEK")
    warm_start = solver_settings.get("solver_warm_start", True)

    # Map solver name string to CVXPY constant
    solver_map = {"MOSEK": cp.MOSEK, "SCS": cp.SCS, "ECOS": cp.ECOS}
    cvxpy_solver = solver_map.get(solver_name.upper(), cp.MOSEK)

    try:
        problem.solve(solver=cvxpy_solver, warm_start=warm_start)
    except cp.error.SolverError as e:
        raise PMSolverError(f"CVXPY SolverError encountered: {e}")

    # -------------------------------------------------------------------------
    # 5. Extract Results and Handle Failures
    # -------------------------------------------------------------------------
    status = problem.status

    if status in ["optimal", "optimal_inaccurate"]:
        if status == "optimal_inaccurate":
            logger.debug("Solver returned 'optimal_inaccurate'. Proceeding with approximate solution.")

        # Extract values and flatten to 1D NumPy arrays
        w_opt = np.array(w.value).flatten()
        c_opt = float(c.value)
        z_opt = np.array(z.value).flatten()
        obj_val = float(problem.value)

        # Rigorous invariant assertion: Verify trade identity numerically
        if not np.allclose(w_opt - w_curr_i, z_opt, atol=1e-5):
            logger.warning("Numerical precision issue: w_opt - w_curr != z_opt within 1e-5 tolerance.")

        return PMOptimizationResult(
            w_opt=w_opt,
            c_opt=c_opt,
            z_opt=z_opt,
            status=status,
            objective_value=obj_val
        )
    else:
        # Handle infeasible or unbounded status based on failure policy
        failure_policy = solver_settings.get("failure_policy", "raise_and_log_full_state_for_reproducibility")

        error_msg = f"PM local optimization failed with status: '{status}'."
        logger.error(error_msg)

        if failure_policy == "raise_and_log_full_state_for_reproducibility":
            raise PMSolverError(error_msg)
        else:
            # Fallback: return zero trade if policy allows (not recommended for strict fidelity)
            logger.warning("Failure policy bypassed. Returning zero trade.")
            return PMOptimizationResult(
                w_opt=w_curr_i,
                c_opt=c_curr_i,
                z_opt=np.zeros(n_assets),
                status=status,
                objective_value=0.0
            )

# -------------------------------------------------------------------------------------------------------------------------------
# Task 24, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
# Note: For Task 24, the orchestrator is synonymous with the `solve_pm_local_problem` callable itself,
# as the construction, solving, and extraction are inextricably linked within the CVXPY problem lifecycle.
# The function above serves as the complete, standalone implementation for Task 24.


In [ ]:
# Task 25: Define the ADMM-modified PM update callable

class ADMMPMSolverError(Exception):
    """Custom exception raised when the ADMM-modified PM local optimization problem fails."""
    pass

class PMOptimizationResult(NamedTuple):
    """
    Technical Description:
        A strictly typed, immutable record designed to encapsulate the complete output of a Portfolio Manager's
        local convex optimization problem (Equation 6). This structure ensures that the primal solutions
        (weights, cash, and trades) are coupled with solver metadata to maintain an audit trail for
        distributed coordination and walk-forward accounting.

    Parameters:
        w_opt (np.ndarray):
            The optimal post-trade portfolio weight vector w* of shape (N,).
            Represents the target allocation for the next period.
        c_opt (float):
            The optimal post-trade cash position c* as a fraction of NAV.
            Satisfies the budget constraint 1 - 1^T w = c.
        z_opt (np.ndarray):
            The optimal trade weight vector z* of shape (N,).
            Defined by the identity w - w_curr = z.
        status (str):
            The termination status reported by the CVXPY solver (e.g., 'optimal', 'infeasible').
        objective_value (float):
            The scalar value of the minimized objective function f(x) at the solution.

    Returns:
        PMOptimizationResult:
            An immutable instance containing the synchronized optimization state.

    Raises:
        ValueError:
            If the dimensions of w_opt and z_opt are inconsistent or if numerical fields are non-finite.
    """
    # Define the optimal portfolio weight vector w* resulting from the optimization
    w_opt: np.ndarray
    # Define the optimal cash position c* as a scalar fraction of the portfolio NAV
    c_opt: float
    # Define the optimal trade vector z* which represents the change in weights
    z_opt: np.ndarray
    # Define the solver status string to track convergence or failure modes
    status: str
    # Define the final scalar value of the objective function at the optimal point
    objective_value: float

    def validate(self) -> None:
        """
        Technical Description:
            Performs post-instantiation validation to ensure the numerical integrity and
            dimensional consistency of the optimization result.

        Processes:
            1. Verifies that weight and trade vectors possess identical shapes.
            2. Confirms that all numerical outputs are finite (non-NaN and non-infinite).
            3. Validates that the solver status is a non-empty string.

        Raises:
            ValueError: If any structural or numerical invariant is violated.
        """
        # Check if the shape of the weight vector matches the shape of the trade vector
        if self.w_opt.shape != self.z_opt.shape:
            # Raise error if dimensions are inconsistent, preventing downstream linear algebra failures
            raise ValueError(f"Dimension mismatch: w_opt {self.w_opt.shape} != z_opt {self.z_opt.shape}")

        # Verify that the cash position, objective value, and vectors contain only finite numbers
        if not (np.isfinite(self.c_opt) and np.isfinite(self.objective_value) and
                np.all(np.isfinite(self.w_opt)) and np.all(np.isfinite(self.z_opt))):
            # Raise error if NaNs or Infs are detected, as they invalidate the backtest state
            raise ValueError("Optimization result contains non-finite numerical values (NaN or Inf).")

        # Ensure the status field is populated to allow for proper failure handling in the orchestrator
        if not isinstance(self.status, str) or len(self.status) == 0:
            # Raise error if the status is missing or malformed
            raise ValueError("Invalid solver status: status must be a non-empty string.")

# ==============================================================================
# Task 25: Define the ADMM-modified PM update callable
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 25, Steps 1, 2 & 3: Construct, solve, and extract the ADMM-modified PM problem
# -------------------------------------------------------------------------------------------------------------------------------
def solve_admm_pm_update(
    alpha_t_i: np.ndarray,
    F_t_annual: np.ndarray,
    D_t_idio_annual: np.ndarray,
    w_curr_i: np.ndarray,
    c_curr_i: float,
    rf_daily_t: float,
    mask_i: np.ndarray,
    sigma_target_i: float,
    cvxpy_tc_evaluator: Callable[[cp.Variable], cp.Expression],
    cvxpy_short_local: Callable[[cp.Variable], cp.Expression],
    institutional_constraints: Dict[str, float],
    firm_global_params: Dict[str, float],
    solver_settings: Dict[str, Any],
    ell_k: np.ndarray,
    x_prev_i: np.ndarray,
    D_scale_diag_t: np.ndarray,
    rho_admm: float,
    lambda_i: float
) -> PMOptimizationResult:
    """
    Constructs and solves the ADMM-modified PM convex optimization problem (Algorithm 1, Step 1).

    Parameters
    ----------
    alpha_t_i : np.ndarray
        The synthetic alpha vector of shape (N,).
    F_t_annual : np.ndarray
        The annualized factor loading matrix of shape (N, J).
    D_t_idio_annual : np.ndarray
        The annualized idiosyncratic variance vector of shape (N,).
    w_curr_i : np.ndarray
        The current portfolio weight vector of shape (N,).
    c_curr_i : float
        The current cash position.
    rf_daily_t : float
        The daily risk-free rate.
    mask_i : np.ndarray
        The boolean tradable-universe mask of shape (N,).
    sigma_target_i : float
        The annualized volatility target.
    cvxpy_tc_evaluator : Callable[[cp.Variable], cp.Expression]
        Closure returning the symbolic transaction cost expression.
    cvxpy_short_local : Callable[[cp.Variable], cp.Expression]
        Closure returning the symbolic local shorting cost expression.
    institutional_constraints : Dict[str, float]
        Dictionary containing L, C, S, T limits.
    firm_global_params : Dict[str, float]
        Dictionary containing penalty weights (gamma_risk, gamma_turnover).
    solver_settings : Dict[str, Any]
        Dictionary containing solver configuration (name, tolerances).
    ell_k : np.ndarray
        The planner broadcast signal vector of shape (N,).
    x_prev_i : np.ndarray
        The PM's trade vector from the previous ADMM iteration (x^{i,k}) of shape (N,).
    D_scale_diag_t : np.ndarray
        The ADMM diagonal scaling vector of shape (N,).
    rho_admm : float
        The ADMM penalty parameter (rho).
    lambda_i : float
        The PM's relative NAV weight.

    Returns
    -------
    PMOptimizationResult
        A NamedTuple containing the optimal w, c, z (which is x^{i,k+1}) vectors and diagnostics.

    Raises
    ------
    ADMMPMSolverError
        If the problem is infeasible or unbounded, based on the configured failure policy.
    """
    n_assets = len(alpha_t_i)

    # Extract constraints and penalties
    L = institutional_constraints.get("leverage_limit_L", 1.5)
    C = institutional_constraints.get("concentration_limit_C", 0.2)
    S = institutional_constraints.get("shorting_limit_S", 0.5)
    T = institutional_constraints.get("turnover_limit_T", 0.2)

    gamma_risk = firm_global_params.get("gamma_risk", 20.0)
    gamma_turn = firm_global_params.get("gamma_turnover", 1.0)

    # -------------------------------------------------------------------------
    # 1. Define CVXPY Variables
    # -------------------------------------------------------------------------
    w = cp.Variable(n_assets, name="w")
    c = cp.Variable(name="c")
    z = cp.Variable(n_assets, name="z")  # z is mathematically equivalent to x^i in the ADMM formulation
    s_risk = cp.Variable(nonneg=True, name="s_risk")
    s_turn = cp.Variable(nonneg=True, name="s_turn")

    # -------------------------------------------------------------------------
    # 2. Construct the ADMM-Modified Objective Function
    # -------------------------------------------------------------------------
    # Original local objective terms
    alpha_term = -alpha_t_i @ w
    cash_term = -rf_daily_t * c
    risk_penalty = gamma_risk * s_risk
    turn_penalty = gamma_turn * s_turn
    tc_term = cvxpy_tc_evaluator(z)
    short_term = cvxpy_short_local(w)

    # Sum the original local objective
    local_obj = alpha_term + cash_term + risk_penalty + turn_penalty + tc_term + short_term

    # ADMM Linear Signal Term: lambda_i * (ell^k)^T * D * z
    # We pre-compute the element-wise product of ell_k and D_scale_diag_t
    linear_signal_vector = ell_k * D_scale_diag_t
    admm_linear_term = lambda_i * (linear_signal_vector @ z)

    # ADMM Quadratic Proximal Term: (rho / 2) * || lambda_i * D * (z - x^{i,k}) ||_2^2
    # cp.multiply performs element-wise multiplication, preserving DCP convexity
    scaling_vector = lambda_i * D_scale_diag_t
    admm_proximal_term = (rho_admm / 2.0) * cp.sum_squares(cp.multiply(scaling_vector, z - x_prev_i))

    # The final objective scales the local objective by lambda_i and adds the ADMM terms
    objective = cp.Minimize(
        (lambda_i * local_obj) + admm_linear_term + admm_proximal_term
    )

    # -------------------------------------------------------------------------
    # 3. Construct Constraints (Identical to Task 24)
    # -------------------------------------------------------------------------
    constraints = []

    # Trade identity: w - w_curr = z
    constraints.append(w - w_curr_i == z)

    # Cash identity: 1 - 1^T w = c
    constraints.append(1.0 - cp.sum(w) == c)

    # Leverage limit: ||w||_1 <= L
    constraints.append(cp.norm1(w) <= L)

    # Concentration limit: |w| <= C
    constraints.append(cp.abs(w) <= C)

    # Shorting limit: 1^T max(0, -w) <= S
    constraints.append(cp.sum(cp.pos(-w)) <= S)

    # Turnover limit with slack: ||w - w_curr||_2 + |c - c_curr| <= 2T + s_turn
    constraints.append(cp.norm2(z) + cp.abs(c - c_curr_i) <= 2.0 * T + s_turn)

    # Factored Risk constraint with slack: ||Sigma^(1/2) w||_2 <= sigma_target + s_risk
    safe_F = np.nan_to_num(F_t_annual, nan=0.0)
    safe_D = np.nan_to_num(D_t_idio_annual, nan=0.0)

    factor_risk_component = safe_F.T @ w
    idio_risk_component = cp.multiply(np.sqrt(safe_D), w)

    stacked_risk_vector = cp.hstack([factor_risk_component, idio_risk_component])
    constraints.append(cp.norm2(stacked_risk_vector) <= sigma_target_i + s_risk)

    # Tradable-universe mask: w_j = 0 for assets not in the PM's universe
    non_tradable_mask = ~mask_i
    if non_tradable_mask.any():
        constraints.append(w[non_tradable_mask] == 0.0)

    # -------------------------------------------------------------------------
    # 4. Solve the Problem
    # -------------------------------------------------------------------------
    problem = cp.Problem(objective, constraints)

    solver_name = solver_settings.get("solver_name", "MOSEK")
    # Warm starting is highly recommended for ADMM iterations
    warm_start = solver_settings.get("solver_warm_start", True)

    solver_map = {"MOSEK": cp.MOSEK, "SCS": cp.SCS, "ECOS": cp.ECOS}
    cvxpy_solver = solver_map.get(solver_name.upper(), cp.MOSEK)

    try:
        problem.solve(solver=cvxpy_solver, warm_start=warm_start)
    except cp.error.SolverError as e:
        raise ADMMPMSolverError(f"CVXPY SolverError encountered during ADMM update: {e}")

    # -------------------------------------------------------------------------
    # 5. Extract Results and Handle Failures
    # -------------------------------------------------------------------------
    status = problem.status

    if status in ["optimal", "optimal_inaccurate"]:
        if status == "optimal_inaccurate":
            logger.debug("Solver returned 'optimal_inaccurate' during ADMM update. Proceeding.")

        w_opt = np.array(w.value).flatten()
        c_opt = float(c.value)
        z_opt = np.array(z.value).flatten()  # This is x^{i,k+1}
        obj_val = float(problem.value)

        # Rigorous invariant assertion: Verify trade identity numerically
        if not np.allclose(w_opt - w_curr_i, z_opt, atol=1e-5):
            logger.warning("Numerical precision issue: w_opt - w_curr != z_opt within 1e-5 tolerance.")

        return PMOptimizationResult(
            w_opt=w_opt,
            c_opt=c_opt,
            z_opt=z_opt,
            status=status,
            objective_value=obj_val
        )
    else:
        failure_policy = solver_settings.get("failure_policy", "raise_and_log_full_state_for_reproducibility")

        error_msg = f"ADMM PM local optimization failed with status: '{status}'."
        logger.error(error_msg)

        if failure_policy == "raise_and_log_full_state_for_reproducibility":
            raise ADMMPMSolverError(error_msg)
        else:
            logger.warning("Failure policy bypassed. Returning previous iterate x^{i,k} to maintain ADMM stability.")
            return PMOptimizationResult(
                w_opt=w_curr_i + x_prev_i, # Approximate w_opt based on previous trade
                c_opt=c_curr_i,            # Approximate c_opt
                z_opt=x_prev_i,            # Return previous trade
                status=status,
                objective_value=0.0
            )

# -------------------------------------------------------------------------------------------------------------------------------
# Task 25, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
# Note: Similar to Task 24, the orchestrator is synonymous with the `solve_admm_pm_update` callable itself,
# as the construction, solving, and extraction are inextricably linked within the CVXPY problem lifecycle.
# The function above serves as the complete, standalone implementation for Task 25.


In [ ]:
# Task 26: Define the independent protocol runner

class ProtocolExecutionError(Exception):
    """Custom exception raised when a protocol runner fails to execute or aggregate trades."""
    pass

@dataclass
class ProtocolRunOutput:
    """
    A strictly typed, standardized output contract for all protocol runners at a single decision date.
    This structure is consumed by the state-transition accounting layer (Task 30).

    Attributes
    ----------
    protocol_name : str
        The identifier of the protocol (e.g., 'independent', 'full_cooperative').
    pm_trade_vectors : Dict[int, np.ndarray]
        Dictionary mapping pm_id to the optimal trade vector x_t^{(i)} of shape (N,).
    pm_portfolio_weights : Dict[int, np.ndarray]
        Dictionary mapping pm_id to the optimal post-trade weights w_t^{*(i)} of shape (N,).
    pm_cash_positions : Dict[int, float]
        Dictionary mapping pm_id to the optimal post-trade cash position c_t^{*(i)}.
    firm_net_trade : np.ndarray
        The NAV-weighted aggregate firm trade vector z_t^{firm} of shape (N,).
    firm_transaction_cost : float
        The realized transaction cost evaluated on the firm net trade.
    pm_transaction_costs : Dict[int, float]
        Dictionary mapping pm_id to the transaction cost evaluated on their individual trade.
    solver_statuses : Dict[int, str]
        Dictionary mapping pm_id to the CVXPY solver status.
    admm_trace : Optional[Any]
        The diagnostic trace of ADMM iterations (None for non-ADMM protocols).
    """
    protocol_name: str
    pm_trade_vectors: Dict[int, np.ndarray]
    pm_portfolio_weights: Dict[int, np.ndarray]
    pm_cash_positions: Dict[int, float]
    firm_net_trade: np.ndarray
    firm_transaction_cost: float
    pm_transaction_costs: Dict[int, float]
    solver_statuses: Dict[int, str]
    admm_trace: Optional[Any] = None

# ==============================================================================
# Task 26: Define the independent protocol runner
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 26, Step 1: Execute independent PM local optimizations
# -------------------------------------------------------------------------------------------------------------------------------
def execute_independent_pm_solves(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    rf_daily_series: pd.Series,
    cvxpy_tc_evaluator: Any,
    cvxpy_short_local: Any,
    study_config: Dict[str, Any]
) -> Dict[int, Any]: # Dict[int, PMOptimizationResult]
    """
    Executes the standalone local optimization for each PM independently.

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    cvxpy_short_local : Callable
        The symbolic local shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Dict[int, PMOptimizationResult]
        A dictionary mapping pm_id to its optimization result.

    Raises
    ------
    ProtocolExecutionError
        If any PM solve fails critically.
    """
    logger.debug(f"Executing Task 26, Step 1: Independent PM solves for date {current_date.date()}.")

    pm_results = {}

    # Extract exogenous data for date t
    F_t_annual = F_t_annual_dict[current_date]
    D_t_idio_annual = D_t_idio_annual_dict[current_date]
    rf_daily_t = float(rf_daily_series.loc[current_date])

    inst_constraints = study_config.get("INSTITUTIONAL_CONSTRAINTS", {})
    firm_params = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})

    # Iterate over PMs sequentially to ensure deterministic execution
    for pm_id, pm_state in current_state.pm_states.items():
        alpha_t_i = alpha_dictionary[(current_date, pm_id)]

        try:
            # Invoke the Task 24 callable
            # Note: solve_pm_local_problem is assumed to be imported/available in the environment
            result = solve_pm_local_problem(
                alpha_t_i=alpha_t_i,
                F_t_annual=F_t_annual,
                D_t_idio_annual=D_t_idio_annual,
                w_curr_i=pm_state.w_curr,
                c_curr_i=pm_state.c_curr,
                rf_daily_t=rf_daily_t,
                mask_i=pm_state.tradable_mask,
                sigma_target_i=pm_state.risk_target_annualized,
                cvxpy_tc_evaluator=cvxpy_tc_evaluator,
                cvxpy_short_local=cvxpy_short_local,
                institutional_constraints=inst_constraints,
                firm_global_params=firm_params,
                solver_settings=solver_settings
            )
            pm_results[pm_id] = result

        except Exception as e:
            error_msg = f"Independent solve failed for PM {pm_id} on {current_date.date()}: {e}"
            logger.error(error_msg)
            raise ProtocolExecutionError(error_msg)

    return pm_results

# -------------------------------------------------------------------------------------------------------------------------------
# Task 26, Step 2: Compute firm net trade and evaluate transaction costs
# -------------------------------------------------------------------------------------------------------------------------------
def compute_independent_aggregates(
    pm_results: Dict[int, Any], # Dict[int, PMOptimizationResult]
    current_state: Any, # FirmState
    numpy_tc_evaluator: Any
) -> Tuple[np.ndarray, float, Dict[int, float]]:
    """
    Computes the ex-post firm net trade and evaluates realized transaction costs.

    Equations:
    z_t^{firm} = \sum_{i=1}^M \lambda_t^{(i)} x_t^{(i)}
    TC_{firm} = \phi_t^{tc}(z_t^{firm})

    Parameters
    ----------
    pm_results : Dict[int, PMOptimizationResult]
        The optimization results from Step 1.
    current_state : FirmState
        The current dynamic state containing the NAV weights lambda_t.
    numpy_tc_evaluator : Callable
        The numerical transaction cost closure.

    Returns
    -------
    Tuple[np.ndarray, float, Dict[int, float]]
        The firm net trade vector, the firm TC, and a dictionary of individual PM TCs.
    """
    logger.debug("Executing Task 26, Step 2: Computing firm net trade and evaluating costs.")

    n_assets = len(next(iter(pm_results.values())).z_opt)
    z_firm_t = np.zeros(n_assets, dtype=np.float64)
    pm_tcs = {}

    # Aggregate trades using current NAV weights
    for pm_id, result in pm_results.items():
        lambda_i = current_state.pm_states[pm_id].nav_weight
        x_t_i = result.z_opt  # z_opt is the trade vector x_t^{(i)}

        z_firm_t += lambda_i * x_t_i

        # Evaluate individual PM TC (what they think they pay in isolation)
        pm_tcs[pm_id] = numpy_tc_evaluator(x_t_i)

    # Evaluate the actual firm TC on the netted trade
    tc_firm_t = numpy_tc_evaluator(z_firm_t)

    # Diagnostic: Calculate netting benefit
    independent_tc_sum = sum(current_state.pm_states[pm_id].nav_weight * tc for pm_id, tc in pm_tcs.items())
    netting_benefit = independent_tc_sum - tc_firm_t

    if netting_benefit < -1e-8:
        logger.warning(f"Negative netting benefit detected: {netting_benefit}. This implies trades are perfectly correlated and market impact is super-additive.")

    return z_firm_t, tc_firm_t, pm_tcs

# -------------------------------------------------------------------------------------------------------------------------------
# Task 26, Step 3: Package outputs into the standardized contract
# -------------------------------------------------------------------------------------------------------------------------------
def package_protocol_output(
    pm_results: Dict[int, Any],
    z_firm_t: np.ndarray,
    tc_firm_t: float,
    pm_tcs: Dict[int, float]
) -> ProtocolRunOutput:
    """
    Packages the results into the standardized ProtocolRunOutput dataclass.

    Parameters
    ----------
    pm_results : Dict[int, PMOptimizationResult]
        The optimization results from Step 1.
    z_firm_t : np.ndarray
        The firm net trade vector.
    tc_firm_t : float
        The firm transaction cost.
    pm_tcs : Dict[int, float]
        The individual PM transaction costs.

    Returns
    -------
    ProtocolRunOutput
        The standardized output object.
    """
    logger.debug("Executing Task 26, Step 3: Packaging outputs into standardized contract.")

    pm_trade_vectors = {}
    pm_portfolio_weights = {}
    pm_cash_positions = {}
    solver_statuses = {}

    for pm_id, result in pm_results.items():
        pm_trade_vectors[pm_id] = result.z_opt
        pm_portfolio_weights[pm_id] = result.w_opt
        pm_cash_positions[pm_id] = result.c_opt
        solver_statuses[pm_id] = result.status

    output = ProtocolRunOutput(
        protocol_name="independent",
        pm_trade_vectors=pm_trade_vectors,
        pm_portfolio_weights=pm_portfolio_weights,
        pm_cash_positions=pm_cash_positions,
        firm_net_trade=z_firm_t,
        firm_transaction_cost=tc_firm_t,
        pm_transaction_costs=pm_tcs,
        solver_statuses=solver_statuses,
        admm_trace=None
    )

    return output

# -------------------------------------------------------------------------------------------------------------------------------
# Task 26, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_26_orchestrator(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    nu_t_daily_dict: Dict[pd.Timestamp, np.ndarray],
    dollar_volume_panel: pd.DataFrame,
    kappa_spread_panel: pd.DataFrame,
    rf_daily_series: pd.Series,
    study_config: Dict[str, Any]
) -> ProtocolRunOutput:
    """
    Orchestrates the execution of the independent protocol for a single decision date.

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    nu_t_daily_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed daily volatilities.
    dollar_volume_panel : pd.DataFrame
        The wide-format dollar volume panel.
    kappa_spread_panel : pd.DataFrame
        The wide-format fractional spread panel.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    ProtocolRunOutput
        The standardized output object containing trades and costs.

    Raises
    ------
    ProtocolExecutionError
        If the protocol fails to execute or aggregate correctly.
    """
    # 1. Dynamically compute endogenous TC parameters using Task 22 orchestrator
    # Note: task_22_orchestrator is assumed to be imported/available
    nu_t = nu_t_daily_dict[current_date]
    omega_t = dollar_volume_panel.loc[current_date].to_numpy()
    kappa_spread_t = kappa_spread_panel.loc[current_date].to_numpy()
    rf_daily_t = float(rf_daily_series.loc[current_date])
    v_firm_t = current_state.firm_nav_usd

    _, cvxpy_tc_evaluator, numpy_tc_evaluator, cvxpy_short_local, _ = task_22_orchestrator(
        nu_t=nu_t,
        omega_t=omega_t,
        kappa_spread_t=kappa_spread_t,
        rf_daily_t=rf_daily_t,
        v_firm_t=v_firm_t,
        study_config=study_config
    )

    # 2. Execute Step 1: Independent PM solves
    pm_results = execute_independent_pm_solves(
        current_state=current_state,
        current_date=current_date,
        alpha_dictionary=alpha_dictionary,
        F_t_annual_dict=F_t_annual_dict,
        D_t_idio_annual_dict=D_t_idio_annual_dict,
        rf_daily_series=rf_daily_series,
        cvxpy_tc_evaluator=cvxpy_tc_evaluator,
        cvxpy_short_local=cvxpy_short_local,
        study_config=study_config
    )

    # 3. Execute Step 2: Compute aggregates
    z_firm_t, tc_firm_t, pm_tcs = compute_independent_aggregates(
        pm_results=pm_results,
        current_state=current_state,
        numpy_tc_evaluator=numpy_tc_evaluator
    )

    # 4. Execute Step 3: Package output
    output = package_protocol_output(
        pm_results=pm_results,
        z_firm_t=z_firm_t,
        tc_firm_t=tc_firm_t,
        pm_tcs=pm_tcs
    )

    return output


In [ ]:
# Task 27: Define the full cooperative protocol runner

# ==============================================================================
# Task 27: Define the full cooperative protocol runner
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 27, Steps 1 & 2: Construct, solve, and extract the centralized cooperative problem
# -------------------------------------------------------------------------------------------------------------------------------
def execute_cooperative_joint_solve(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    rf_daily_series: pd.Series,
    cvxpy_tc_evaluator: Any,
    cvxpy_short_firm: Any,
    study_config: Dict[str, Any]
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray], Dict[int, float], np.ndarray, str]:
    """
    Constructs and solves the centralized cooperative optimization problem (Equation 3).

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    cvxpy_short_firm : Callable
        The symbolic firm net shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray], Dict[int, float], np.ndarray, str]
        pm_trade_vectors: Dict mapping pm_id to optimal trade vector z_opt.
        pm_portfolio_weights: Dict mapping pm_id to optimal weight vector w_opt.
        pm_cash_positions: Dict mapping pm_id to optimal cash c_opt.
        firm_net_trade: The optimal firm net trade vector z_firm_opt.
        status: The CVXPY solver status.

    Raises
    ------
    ProtocolExecutionError
        If the joint problem is infeasible or unbounded.
    """
    logger.debug(f"Executing Task 27, Steps 1 & 2: Cooperative joint solve for date {current_date.date()}.")

    # Extract exogenous data for date t
    F_t_annual = F_t_annual_dict[current_date]
    D_t_idio_annual = D_t_idio_annual_dict[current_date]
    rf_daily_t = float(rf_daily_series.loc[current_date])

    inst_constraints = study_config.get("INSTITUTIONAL_CONSTRAINTS", {})
    firm_params = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})

    L = inst_constraints.get("leverage_limit_L", 1.5)
    C = inst_constraints.get("concentration_limit_C", 0.2)
    S = inst_constraints.get("shorting_limit_S", 0.5)
    T = inst_constraints.get("turnover_limit_T", 0.2)

    gamma_risk = firm_global_params.get("gamma_risk", 20.0)
    gamma_turn = firm_global_params.get("gamma_turnover", 1.0)

    n_assets = len(next(iter(current_state.pm_states.values())).w_curr)
    m_pms = len(current_state.pm_states)

    # -------------------------------------------------------------------------
    # 1. Define CVXPY Variables
    # -------------------------------------------------------------------------
    # We use dictionaries keyed by pm_id to store the variables for each PM
    w_vars = {pm_id: cp.Variable(n_assets, name=f"w_{pm_id}") for pm_id in current_state.pm_states}
    c_vars = {pm_id: cp.Variable(name=f"c_{pm_id}") for pm_id in current_state.pm_states}
    z_vars = {pm_id: cp.Variable(n_assets, name=f"z_{pm_id}") for pm_id in current_state.pm_states}
    s_risk_vars = {pm_id: cp.Variable(nonneg=True, name=f"s_risk_{pm_id}") for pm_id in current_state.pm_states}
    s_turn_vars = {pm_id: cp.Variable(nonneg=True, name=f"s_turn_{pm_id}") for pm_id in current_state.pm_states}

    # Firm net trade variable
    z_firm = cp.Variable(n_assets, name="z_firm")

    # -------------------------------------------------------------------------
    # 2. Construct Objective Function
    # -------------------------------------------------------------------------
    joint_objective_terms = []
    constraints = []

    # Pre-compute safe risk matrices to handle potential NaNs
    safe_F = np.nan_to_num(F_t_annual, nan=0.0)
    safe_D = np.nan_to_num(D_t_idio_annual, nan=0.0)

    # Iterate over PMs to build local objectives and constraints
    for pm_id, pm_state in current_state.pm_states.items():
        lambda_i = pm_state.nav_weight
        alpha_t_i = alpha_dictionary[(current_date, pm_id)]

        w = w_vars[pm_id]
        c = c_vars[pm_id]
        z = z_vars[pm_id]
        s_risk = s_risk_vars[pm_id]
        s_turn = s_turn_vars[pm_id]

        # Local objective terms (excluding TC and shorting cost)
        alpha_term = -alpha_t_i @ w
        cash_term = -rf_daily_t * c
        risk_penalty = gamma_risk * s_risk
        turn_penalty = gamma_turn * s_turn

        local_obj = alpha_term + cash_term + risk_penalty + turn_penalty

        # Scale local objective by NAV weight and add to joint objective
        joint_objective_terms.append(lambda_i * local_obj)

        # Local constraints
        constraints.append(w - pm_state.w_curr == z)
        constraints.append(1.0 - cp.sum(w) == c)
        constraints.append(cp.norm1(w) <= L)
        constraints.append(cp.abs(w) <= C)
        constraints.append(cp.sum(cp.pos(-w)) <= S)
        constraints.append(cp.norm2(z) + cp.abs(c - pm_state.c_curr) <= 2.0 * T + s_turn)

        # Factored Risk constraint
        factor_risk_component = safe_F.T @ w
        idio_risk_component = cp.multiply(np.sqrt(safe_D), w)
        stacked_risk_vector = cp.hstack([factor_risk_component, idio_risk_component])
        constraints.append(cp.norm2(stacked_risk_vector) <= pm_state.risk_target_annualized + s_risk)

        # Tradable-universe mask
        non_tradable_mask = ~pm_state.tradable_mask
        if non_tradable_mask.any():
            constraints.append(w[non_tradable_mask] == 0.0)

    # Firm-level coupling constraint: z_firm = sum(lambda_i * z_i)
    weighted_trades = [current_state.pm_states[pm_id].nav_weight * z_vars[pm_id] for pm_id in current_state.pm_states]
    constraints.append(z_firm == cp.sum(weighted_trades))

    # Firm-level transaction cost
    firm_tc_term = cvxpy_tc_evaluator(z_firm)
    joint_objective_terms.append(firm_tc_term)

    # Firm-level shorting cost
    # Requires pre-trade aggregate weights: w_curr_firm = sum(lambda_i * w_curr_i)
    w_curr_firm = sum(pm_state.nav_weight * pm_state.w_curr for pm_state in current_state.pm_states.values())
    firm_short_term = cvxpy_short_firm(z_firm, w_curr_firm)
    joint_objective_terms.append(firm_short_term)

    # Final Joint Objective
    objective = cp.Minimize(cp.sum(joint_objective_terms))

    # -------------------------------------------------------------------------
    # 3. Solve the Problem
    # -------------------------------------------------------------------------
    problem = cp.Problem(objective, constraints)

    solver_name = solver_settings.get("solver_name", "MOSEK")
    warm_start = solver_settings.get("solver_warm_start", True)

    solver_map = {"MOSEK": cp.MOSEK, "SCS": cp.SCS, "ECOS": cp.ECOS}
    cvxpy_solver = solver_map.get(solver_name.upper(), cp.MOSEK)

    try:
        problem.solve(solver=cvxpy_solver, warm_start=warm_start)
    except cp.error.SolverError as e:
        raise ProtocolExecutionError(f"CVXPY SolverError encountered during cooperative solve: {e}")

    # -------------------------------------------------------------------------
    # 4. Extract Results
    # -------------------------------------------------------------------------
    status = problem.status

    if status in ["optimal", "optimal_inaccurate"]:
        if status == "optimal_inaccurate":
            logger.debug("Solver returned 'optimal_inaccurate' during cooperative solve. Proceeding.")

        pm_trade_vectors = {}
        pm_portfolio_weights = {}
        pm_cash_positions = {}

        for pm_id in current_state.pm_states:
            pm_trade_vectors[pm_id] = np.array(z_vars[pm_id].value).flatten()
            pm_portfolio_weights[pm_id] = np.array(w_vars[pm_id].value).flatten()
            pm_cash_positions[pm_id] = float(c_vars[pm_id].value)

        z_firm_opt = np.array(z_firm.value).flatten()

        # Rigorous invariant assertion: Verify consensus constraint numerically
        computed_z_firm = sum(current_state.pm_states[pm_id].nav_weight * pm_trade_vectors[pm_id] for pm_id in current_state.pm_states)
        if not np.allclose(z_firm_opt, computed_z_firm, atol=1e-5):
            logger.warning("Numerical precision issue: z_firm != sum(lambda * z_i) within 1e-5 tolerance.")

        return pm_trade_vectors, pm_portfolio_weights, pm_cash_positions, z_firm_opt, status
    else:
        failure_policy = solver_settings.get("failure_policy", "raise_and_log_full_state_for_reproducibility")
        error_msg = f"Cooperative joint optimization failed with status: '{status}'."
        logger.error(error_msg)

        if failure_policy == "raise_and_log_full_state_for_reproducibility":
            raise ProtocolExecutionError(error_msg)
        else:
            # Fallback: return zero trades
            logger.warning("Failure policy bypassed. Returning zero trades for cooperative protocol.")
            pm_trade_vectors = {pm_id: np.zeros(n_assets) for pm_id in current_state.pm_states}
            pm_portfolio_weights = {pm_id: state.w_curr for pm_id, state in current_state.pm_states.items()}
            pm_cash_positions = {pm_id: state.c_curr for pm_id, state in current_state.pm_states.items()}
            return pm_trade_vectors, pm_portfolio_weights, pm_cash_positions, np.zeros(n_assets), status

# -------------------------------------------------------------------------------------------------------------------------------
# Task 27, Step 3: Evaluate costs and package outputs
# -------------------------------------------------------------------------------------------------------------------------------
def evaluate_and_package_cooperative_output(
    pm_trade_vectors: Dict[int, np.ndarray],
    pm_portfolio_weights: Dict[int, np.ndarray],
    pm_cash_positions: Dict[int, float],
    z_firm_opt: np.ndarray,
    status: str,
    numpy_tc_evaluator: Any
) -> Any: # ProtocolRunOutput
    """
    Evaluates the firm transaction cost and packages the results into the standardized contract.

    Parameters
    ----------
    pm_trade_vectors : Dict[int, np.ndarray]
        Dictionary mapping pm_id to optimal trade vector.
    pm_portfolio_weights : Dict[int, np.ndarray]
        Dictionary mapping pm_id to optimal weight vector.
    pm_cash_positions : Dict[int, float]
        Dictionary mapping pm_id to optimal cash.
    z_firm_opt : np.ndarray
        The optimal firm net trade vector.
    status : str
        The CVXPY solver status.
    numpy_tc_evaluator : Callable
        The numerical transaction cost closure.

    Returns
    -------
    ProtocolRunOutput
        The standardized output object.
    """
    logger.debug("Executing Task 27, Step 3: Evaluating costs and packaging cooperative output.")

    # Evaluate the actual firm TC on the netted trade
    tc_firm_t = numpy_tc_evaluator(z_firm_opt)

    # Evaluate individual PM TC for diagnostic comparison (what they would have paid in isolation)
    pm_tcs = {}
    for pm_id, x_t_i in pm_trade_vectors.items():
        pm_tcs[pm_id] = numpy_tc_evaluator(x_t_i)

    solver_statuses = {pm_id: status for pm_id in pm_trade_vectors.keys()}

    # Note: ProtocolRunOutput is assumed to be imported/available
    output = ProtocolRunOutput(
        protocol_name="full_cooperative",
        pm_trade_vectors=pm_trade_vectors,
        pm_portfolio_weights=pm_portfolio_weights,
        pm_cash_positions=pm_cash_positions,
        firm_net_trade=z_firm_opt,
        firm_transaction_cost=tc_firm_t,
        pm_transaction_costs=pm_tcs,
        solver_statuses=solver_statuses,
        admm_trace=None
    )

    return output

# -------------------------------------------------------------------------------------------------------------------------------
# Task 27, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_27_orchestrator(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    nu_t_daily_dict: Dict[pd.Timestamp, np.ndarray],
    dollar_volume_panel: pd.DataFrame,
    kappa_spread_panel: pd.DataFrame,
    rf_daily_series: pd.Series,
    study_config: Dict[str, Any]
) -> Any: # ProtocolRunOutput
    """
    Orchestrates the execution of the full cooperative protocol for a single decision date.

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    nu_t_daily_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed daily volatilities.
    dollar_volume_panel : pd.DataFrame
        The wide-format dollar volume panel.
    kappa_spread_panel : pd.DataFrame
        The wide-format fractional spread panel.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    ProtocolRunOutput
        The standardized output object containing trades and costs.

    Raises
    ------
    ProtocolExecutionError
        If the protocol fails to execute or aggregate correctly.
    """
    # 1. Dynamically compute endogenous TC parameters using Task 22 orchestrator
    nu_t = nu_t_daily_dict[current_date]
    omega_t = dollar_volume_panel.loc[current_date].to_numpy()
    kappa_spread_t = kappa_spread_panel.loc[current_date].to_numpy()
    rf_daily_t = float(rf_daily_series.loc[current_date])
    v_firm_t = current_state.firm_nav_usd

    _, cvxpy_tc_evaluator, numpy_tc_evaluator, _, cvxpy_short_firm = task_22_orchestrator(
        nu_t=nu_t,
        omega_t=omega_t,
        kappa_spread_t=kappa_spread_t,
        rf_daily_t=rf_daily_t,
        v_firm_t=v_firm_t,
        study_config=study_config
    )

    # 2. Execute Steps 1 & 2: Construct and solve the joint problem
    pm_trade_vectors, pm_portfolio_weights, pm_cash_positions, z_firm_opt, status = execute_cooperative_joint_solve(
        current_state=current_state,
        current_date=current_date,
        alpha_dictionary=alpha_dictionary,
        F_t_annual_dict=F_t_annual_dict,
        D_t_idio_annual_dict=D_t_idio_annual_dict,
        rf_daily_series=rf_daily_series,
        cvxpy_tc_evaluator=cvxpy_tc_evaluator,
        cvxpy_short_firm=cvxpy_short_firm,
        study_config=study_config
    )

    # 3. Execute Step 3: Evaluate costs and package output
    output = evaluate_and_package_cooperative_output(
        pm_trade_vectors=pm_trade_vectors,
        pm_portfolio_weights=pm_portfolio_weights,
        pm_cash_positions=pm_cash_positions,
        z_firm_opt=z_firm_opt,
        status=status,
        numpy_tc_evaluator=numpy_tc_evaluator
    )

    return output


In [ ]:
# Task 28: Define the ADMM protocol runner — initialization

class ADMMInitializationError(Exception):
    """Custom exception raised when the ADMM protocol initialization fails."""
    pass

@dataclass
class ADMMState:
    """
    A strictly typed record of the ADMM protocol state at a specific iteration k.
    For initialization, this represents k=0.

    Attributes
    ----------
    x_k : Dict[int, np.ndarray]
        Dictionary mapping pm_id to the trade vector x^{i,k} of shape (N,).
    z_sum_k : np.ndarray
        The planner's consensus aggregate variable z_{sum}^k of shape (N,).
    u_k : np.ndarray
        The planner's dual variable u^k of shape (N,).
    """
    x_k: Dict[int, np.ndarray]
    z_sum_k: np.ndarray
    u_k: np.ndarray

# ==============================================================================
# Task 28: Define the ADMM protocol runner — initialization
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 28, Step 1: Compute the standalone initialization trade for each PM
# -------------------------------------------------------------------------------------------------------------------------------
def compute_admm_initial_trades(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    rf_daily_series: pd.Series,
    cvxpy_tc_evaluator: Any,
    cvxpy_short_local: Any,
    study_config: Dict[str, Any]
) -> Dict[int, np.ndarray]:
    """
    Computes the standalone initialization trade x^{i,0} = argmin f^i(x) for each PM.

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs (specific to the ADMM protocol run).
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    cvxpy_short_local : Callable
        The symbolic local shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Dict[int, np.ndarray]
        A dictionary mapping pm_id to the initial trade vector x^{i,0}.

    Raises
    ------
    ADMMInitializationError
        If any PM solve fails critically during initialization.
    """
    logger.debug(f"Executing Task 28, Step 1: Computing ADMM initial trades for date {current_date.date()}.")

    x_init_dict: Dict[int, np.ndarray] = {}

    # Extract exogenous data for date t
    F_t_annual = F_t_annual_dict[current_date]
    D_t_idio_annual = D_t_idio_annual_dict[current_date]
    rf_daily_t = float(rf_daily_series.loc[current_date])

    inst_constraints = study_config.get("INSTITUTIONAL_CONSTRAINTS", {})
    firm_params = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})

    # Iterate over PMs sequentially
    for pm_id, pm_state in current_state.pm_states.items():
        alpha_t_i = alpha_dictionary[(current_date, pm_id)]

        try:
            # Invoke the Task 24 callable to solve the standalone problem
            # Note: solve_pm_local_problem is assumed to be imported/available
            result = solve_pm_local_problem(
                alpha_t_i=alpha_t_i,
                F_t_annual=F_t_annual,
                D_t_idio_annual=D_t_idio_annual,
                w_curr_i=pm_state.w_curr,
                c_curr_i=pm_state.c_curr,
                rf_daily_t=rf_daily_t,
                mask_i=pm_state.tradable_mask,
                sigma_target_i=pm_state.risk_target_annualized,
                cvxpy_tc_evaluator=cvxpy_tc_evaluator,
                cvxpy_short_local=cvxpy_short_local,
                institutional_constraints=inst_constraints,
                firm_global_params=firm_params,
                solver_settings=solver_settings
            )
            # The trade vector z_opt corresponds to x^{i,0} in the ADMM formulation
            x_init_dict[pm_id] = result.z_opt

        except Exception as e:
            error_msg = f"ADMM initialization solve failed for PM {pm_id} on {current_date.date()}: {e}"
            logger.error(error_msg)
            raise ADMMInitializationError(error_msg)

    return x_init_dict

# -------------------------------------------------------------------------------------------------------------------------------
# Task 28, Step 2: Initialize the planner's dual and aggregate variables
# -------------------------------------------------------------------------------------------------------------------------------
def initialize_planner_variables(
    x_init_dict: Dict[int, np.ndarray],
    current_state: Any # FirmState
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Initializes the dual variable u^0 and the consensus variable z_sum^0.

    Equations:
    u^0 = 0_N
    z_sum^0 = \sum_{i=1}^M \lambda_t^{(i)} x^{i,0}

    Parameters
    ----------
    x_init_dict : Dict[int, np.ndarray]
        Dictionary mapping pm_id to the initial trade vector x^{i,0}.
    current_state : FirmState
        The current dynamic state containing the NAV weights lambda_t.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        u_0: The initial dual variable vector of shape (N,).
        z_sum_0: The initial consensus variable vector of shape (N,).
    """
    logger.debug("Executing Task 28, Step 2: Initializing planner dual and aggregate variables.")

    # Extract N from the first available trade vector
    n_assets = len(next(iter(x_init_dict.values())))

    # Initialize dual variable to exactly zero
    u_0 = np.zeros(n_assets, dtype=np.float64)

    # Initialize consensus variable to the NAV-weighted sum of initial trades
    z_sum_0 = np.zeros(n_assets, dtype=np.float64)
    for pm_id, x_i_0 in x_init_dict.items():
        lambda_i = current_state.pm_states[pm_id].nav_weight
        z_sum_0 += lambda_i * x_i_0

    # Rigorous invariant assertion
    if u_0.shape != (n_assets,) or z_sum_0.shape != (n_assets,):
        raise ADMMInitializationError("Shape mismatch during planner variable initialization.")

    return u_0, z_sum_0

# -------------------------------------------------------------------------------------------------------------------------------
# Task 28, Step 3: Emit the initial ADMM state tuple
# -------------------------------------------------------------------------------------------------------------------------------
def package_admm_initial_state(
    x_init_dict: Dict[int, np.ndarray],
    z_sum_0: np.ndarray,
    u_0: np.ndarray
) -> ADMMState:
    """
    Packages the initialization artifacts into the strictly typed ADMMState dataclass.

    Parameters
    ----------
    x_init_dict : Dict[int, np.ndarray]
        Dictionary mapping pm_id to x^{i,0}.
    z_sum_0 : np.ndarray
        The initial consensus variable.
    u_0 : np.ndarray
        The initial dual variable.

    Returns
    -------
    ADMMState
        The packaged state object for iteration k=0.
    """
    logger.debug("Executing Task 28, Step 3: Packaging initial ADMM state.")

    # We pass the arrays directly. The iteration loop (Task 29) will treat them as immutable
    # for the current step k and generate new arrays for step k+1.
    initial_state = ADMMState(
        x_k=x_init_dict,
        z_sum_k=z_sum_0,
        u_k=u_0
    )

    return initial_state

# -------------------------------------------------------------------------------------------------------------------------------
# Task 28, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_28_orchestrator(
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    nu_t_daily_dict: Dict[pd.Timestamp, np.ndarray],
    dollar_volume_panel: pd.DataFrame,
    kappa_spread_panel: pd.DataFrame,
    rf_daily_series: pd.Series,
    study_config: Dict[str, Any]
) -> Tuple[ADMMState, np.ndarray, Callable, Callable, Callable]:
    """
    Orchestrates the initialization of the ADMM protocol for a single decision date.

    Parameters
    ----------
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    nu_t_daily_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed daily volatilities.
    dollar_volume_panel : pd.DataFrame
        The wide-format dollar volume panel.
    kappa_spread_panel : pd.DataFrame
        The wide-format fractional spread panel.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[ADMMState, np.ndarray, Callable, Callable, Callable]
        The initial ADMM state (k=0), the D_scale_diag_t vector, and the three TC/shorting closures.
        These artifacts are required by the subsequent ADMM iteration loop (Task 29).

    Raises
    ------
    ADMMInitializationError
        If the initialization fails to execute correctly.
    """
    # 1. Dynamically compute endogenous TC parameters using Task 22 orchestrator
    # Note: task_22_orchestrator is assumed to be imported/available
    nu_t = nu_t_daily_dict[current_date]
    omega_t = dollar_volume_panel.loc[current_date].to_numpy()
    kappa_spread_t = kappa_spread_panel.loc[current_date].to_numpy()
    rf_daily_t = float(rf_daily_series.loc[current_date])
    v_firm_t = current_state.firm_nav_usd

    kappa_impact_t, cvxpy_tc_evaluator, numpy_tc_evaluator, cvxpy_short_local, cvxpy_short_firm = task_22_orchestrator(
        nu_t=nu_t,
        omega_t=omega_t,
        kappa_spread_t=kappa_spread_t,
        rf_daily_t=rf_daily_t,
        v_firm_t=v_firm_t,
        study_config=study_config
    )

    # 2. Dynamically compute the ADMM scaling diagonal using Task 23 orchestrator
    # Note: task_23_orchestrator is assumed to be imported/available
    D_scale_diag_t = task_23_orchestrator(kappa_impact_t=kappa_impact_t)

    # 3. Execute Step 1: Compute initial standalone trades
    x_init_dict = compute_admm_initial_trades(
        current_state=current_state,
        current_date=current_date,
        alpha_dictionary=alpha_dictionary,
        F_t_annual_dict=F_t_annual_dict,
        D_t_idio_annual_dict=D_t_idio_annual_dict,
        rf_daily_series=rf_daily_series,
        cvxpy_tc_evaluator=cvxpy_tc_evaluator,
        cvxpy_short_local=cvxpy_short_local,
        study_config=study_config
    )

    # 4. Execute Step 2: Initialize planner variables
    u_0, z_sum_0 = initialize_planner_variables(
        x_init_dict=x_init_dict,
        current_state=current_state
    )

    # 5. Execute Step 3: Package initial state
    initial_admm_state = package_admm_initial_state(
        x_init_dict=x_init_dict,
        z_sum_0=z_sum_0,
        u_0=u_0
    )

    # Return the state and the closures/scaling vector needed for the iteration loop
    return initial_admm_state, D_scale_diag_t, cvxpy_tc_evaluator, numpy_tc_evaluator, cvxpy_short_firm


In [ ]:
# Task 29: Define the ADMM protocol runner — iteration loop

class ADMMIterationError(Exception):
    """Custom exception raised when the ADMM iteration loop fails to solve or converge."""
    pass

# ==============================================================================
# Task 29: Define the ADMM protocol runner — iteration loop
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 29, Step 1: Compute sharing signal and execute distributed PM updates
# -------------------------------------------------------------------------------------------------------------------------------
def execute_admm_distributed_update(
    admm_state: Any, # ADMMState
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual: np.ndarray,
    D_t_idio_annual: np.ndarray,
    rf_daily_t: float,
    D_scale_diag_t: np.ndarray,
    cvxpy_tc_evaluator: Any,
    cvxpy_short_local: Any,
    study_config: Dict[str, Any]
) -> Tuple[np.ndarray, Dict[int, Any]]: # Tuple[np.ndarray, Dict[int, PMOptimizationResult]]
    """
    Computes the sharing signal l^k and executes the ADMM-modified PM updates (Algorithm 1, Step 1).

    Parameters
    ----------
    admm_state : ADMMState
        The ADMM state at the start of iteration k.
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual : np.ndarray
        Pre-computed annualized factor loadings for date t.
    D_t_idio_annual : np.ndarray
        Pre-computed annualized idiosyncratic variances for date t.
    rf_daily_t : float
        The daily risk-free rate for date t.
    D_scale_diag_t : np.ndarray
        The ADMM diagonal scaling vector for date t.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    cvxpy_short_local : Callable
        The symbolic local shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[np.ndarray, Dict[int, PMOptimizationResult]]
        ell_k: The broadcast signal vector of shape (N,).
        pm_results: Dictionary mapping pm_id to its optimization result for iteration k+1.

    Raises
    ------
    ADMMIterationError
        If any PM solve fails critically.
    """
    m_pms = len(current_state.pm_states)
    n_assets = len(D_scale_diag_t)

    admm_params = study_config.get("ADMM_HYPERPARAMETERS", {})
    rho_admm = admm_params.get("rho_penalty", 10.0)

    inst_constraints = study_config.get("INSTITUTIONAL_CONSTRAINTS", {})
    firm_params = study_config.get("GLOBAL_FIRM_CONSTANTS", {})
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})

    # 1. Compute the NAV-weighted sum of current PM trades: sum(lambda_j * x^{j,k})
    nav_weighted_sum_k = np.zeros(n_assets, dtype=np.float64)
    for pm_id, x_j_k in admm_state.x_k.items():
        lambda_j = current_state.pm_states[pm_id].nav_weight
        nav_weighted_sum_k += lambda_j * x_j_k

    # 2. Compute the sharing signal: ell^k = u^k + (rho/M) * (-D * z_sum^k + D * sum(lambda_j * x^{j,k}))
    scaled_residual = D_scale_diag_t * (-admm_state.z_sum_k + nav_weighted_sum_k)
    ell_k = admm_state.u_k + (rho_admm / m_pms) * scaled_residual

    pm_results = {}

    # 3. Execute PM updates sequentially
    for pm_id, pm_state in current_state.pm_states.items():
        alpha_t_i = alpha_dictionary[(current_date, pm_id)]
        x_prev_i = admm_state.x_k[pm_id]
        lambda_i = pm_state.nav_weight

        try:
            # Invoke the Task 25 callable
            # Note: solve_admm_pm_update is assumed to be imported/available
            result = solve_admm_pm_update(
                alpha_t_i=alpha_t_i,
                F_t_annual=F_t_annual,
                D_t_idio_annual=D_t_idio_annual,
                w_curr_i=pm_state.w_curr,
                c_curr_i=pm_state.c_curr,
                rf_daily_t=rf_daily_t,
                mask_i=pm_state.tradable_mask,
                sigma_target_i=pm_state.risk_target_annualized,
                cvxpy_tc_evaluator=cvxpy_tc_evaluator,
                cvxpy_short_local=cvxpy_short_local,
                institutional_constraints=inst_constraints,
                firm_global_params=firm_params,
                solver_settings=solver_settings,
                ell_k=ell_k,
                x_prev_i=x_prev_i,
                D_scale_diag_t=D_scale_diag_t,
                rho_admm=rho_admm,
                lambda_i=lambda_i
            )
            pm_results[pm_id] = result

        except Exception as e:
            error_msg = f"ADMM PM update failed for PM {pm_id} on {current_date.date()}: {e}"
            logger.error(error_msg)
            raise ADMMIterationError(error_msg)

    return ell_k, pm_results

# -------------------------------------------------------------------------------------------------------------------------------
# Task 29, Step 2: Execute gathered central update and dual update
# -------------------------------------------------------------------------------------------------------------------------------
def execute_admm_gathered_update(
    pm_results: Dict[int, Any], # Dict[int, PMOptimizationResult]
    admm_state: Any, # ADMMState
    current_state: Any, # FirmState
    D_scale_diag_t: np.ndarray,
    cvxpy_tc_evaluator: Any,
    cvxpy_short_firm: Any,
    study_config: Dict[str, Any]
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    """
    Solves the planner's consensus subproblem and updates the dual variable (Algorithm 1, Step 2).

    Parameters
    ----------
    pm_results : Dict[int, PMOptimizationResult]
        The optimization results from Step 1 containing x^{i,k+1}.
    admm_state : ADMMState
        The ADMM state at the start of iteration k.
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    D_scale_diag_t : np.ndarray
        The ADMM diagonal scaling vector for date t.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    cvxpy_short_firm : Callable
        The symbolic firm net shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, float, float]
        z_sum_new: The updated consensus variable z_{sum}^{k+1}.
        u_new: The updated dual variable u^{k+1}.
        primal_residual: The L2 norm of the primal feasibility gap.
        dual_residual: The L2 norm of the dual feasibility gap.

    Raises
    ------
    ADMMIterationError
        If the planner subproblem is infeasible or unbounded.
    """
    m_pms = len(current_state.pm_states)
    n_assets = len(D_scale_diag_t)

    admm_params = study_config.get("ADMM_HYPERPARAMETERS", {})
    rho_admm = admm_params.get("rho_penalty", 10.0)
    varphi = admm_params.get("varphi_dual_extrapolation", 1.0)
    solver_settings = study_config.get("SOLVER_AND_NUMERICAL_SETTINGS", {})

    # 1. Compute the target: sum(lambda_i * D * x^{i,k+1})
    target_vector = np.zeros(n_assets, dtype=np.float64)
    nav_weighted_sum_new = np.zeros(n_assets, dtype=np.float64)

    for pm_id, result in pm_results.items():
        lambda_i = current_state.pm_states[pm_id].nav_weight
        x_new_i = result.z_opt
        nav_weighted_sum_new += lambda_i * x_new_i
        target_vector += lambda_i * D_scale_diag_t * x_new_i

    # 2. Define CVXPY Variable for the planner
    z = cp.Variable(n_assets, name="z_sum")

    # 3. Construct Planner Objective
    # Firm TC and Shorting Cost (g(z))
    w_curr_firm = sum(pm_state.nav_weight * pm_state.w_curr for pm_state in current_state.pm_states.values())
    g_z = cvxpy_tc_evaluator(z) + cvxpy_short_firm(z, w_curr_firm)

    # Linear dual term: -(u^k)^T * D * z
    dual_linear_term = (admm_state.u_k * D_scale_diag_t) @ z

    # Quadratic proximal term: (rho / 2M) * || D * z - target ||_2^2
    proximal_term = (rho_admm / (2.0 * m_pms)) * cp.sum_squares(cp.multiply(D_scale_diag_t, z) - target_vector)

    objective = cp.Minimize(g_z - dual_linear_term + proximal_term)

    # No additional constraints (Z = R^N)
    problem = cp.Problem(objective)

    # 4. Solve Planner Subproblem
    solver_name = solver_settings.get("solver_name", "MOSEK")
    warm_start = solver_settings.get("solver_warm_start", True)
    solver_map = {"MOSEK": cp.MOSEK, "SCS": cp.SCS, "ECOS": cp.ECOS}
    cvxpy_solver = solver_map.get(solver_name.upper(), cp.MOSEK)

    try:
        problem.solve(solver=cvxpy_solver, warm_start=warm_start)
    except cp.error.SolverError as e:
        raise ADMMIterationError(f"CVXPY SolverError encountered during planner update: {e}")

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ADMMIterationError(f"Planner subproblem failed with status: '{problem.status}'.")

    z_sum_new = np.array(z.value).flatten()

    # 5. Dual Update: u^{k+1} = u^k + (varphi * rho / M) * (D * sum(lambda_j * x^{j,k+1}) - D * z_sum^{k+1})
    scaled_z_sum_new = D_scale_diag_t * z_sum_new
    u_new = admm_state.u_k + (varphi * rho_admm / m_pms) * (target_vector - scaled_z_sum_new)

    # 6. Compute Convergence Diagnostics (Residuals)
    # Primal residual: || D * (sum(lambda_i * x^{i,k+1}) - z_sum^{k+1}) ||_2
    primal_residual = float(np.linalg.norm(target_vector - scaled_z_sum_new))

    # Dual residual: || (rho / M) * D * (z_sum^{k+1} - z_sum^k) ||_2
    dual_residual = float(np.linalg.norm((rho_admm / m_pms) * D_scale_diag_t * (z_sum_new - admm_state.z_sum_k)))

    return z_sum_new, u_new, primal_residual, dual_residual

# -------------------------------------------------------------------------------------------------------------------------------
# Task 29, Step 3: Orchestrate the K-iteration loop and package outputs
# -------------------------------------------------------------------------------------------------------------------------------
def task_29_orchestrator(
    initial_admm_state: Any, # ADMMState
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    alpha_dictionary: Dict[Tuple[pd.Timestamp, int], np.ndarray],
    F_t_annual_dict: Dict[pd.Timestamp, np.ndarray],
    D_t_idio_annual_dict: Dict[pd.Timestamp, np.ndarray],
    rf_daily_series: pd.Series,
    D_scale_diag_t: np.ndarray,
    cvxpy_tc_evaluator: Any,
    numpy_tc_evaluator: Any,
    cvxpy_short_local: Any,
    cvxpy_short_firm: Any,
    study_config: Dict[str, Any],
    iteration_count_K: int
) -> Any: # ProtocolRunOutput
    """
    Orchestrates the full ADMM iteration loop for a single decision date and packages the final executable state.

    Parameters
    ----------
    initial_admm_state : ADMMState
        The ADMM state at k=0 (from Task 28).
    current_state : FirmState
        The current dynamic state of the firm and all PMs.
    current_date : pd.Timestamp
        The decision date t.
    alpha_dictionary : Dict[Tuple[pd.Timestamp, int], np.ndarray]
        Pre-computed synthetic alpha vectors.
    F_t_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized factor loadings.
    D_t_idio_annual_dict : Dict[pd.Timestamp, np.ndarray]
        Pre-computed annualized idiosyncratic variances.
    rf_daily_series : pd.Series
        The daily risk-free rate series.
    D_scale_diag_t : np.ndarray
        The ADMM diagonal scaling vector.
    cvxpy_tc_evaluator : Callable
        The symbolic transaction cost closure.
    numpy_tc_evaluator : Callable
        The numerical transaction cost closure.
    cvxpy_short_local : Callable
        The symbolic local shorting cost closure.
    cvxpy_short_firm : Callable
        The symbolic firm net shorting cost closure.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    iteration_count_K : int
        The number of ADMM iterations to perform (e.g., 2 or 5).

    Returns
    -------
    ProtocolRunOutput
        The standardized output object containing the final executable trades and the ADMM trace.

    Raises
    ------
    ADMMIterationError
        If the iteration loop fails.
    """
    logger.debug(f"Commencing Task 29: ADMM iteration loop (K={iteration_count_K}) for date {current_date.date()}.")

    # Extract exogenous data for date t
    F_t_annual = F_t_annual_dict[current_date]
    D_t_idio_annual = D_t_idio_annual_dict[current_date]
    rf_daily_t = float(rf_daily_series.loc[current_date])

    # Initialize state and trace
    admm_state = initial_admm_state
    admm_trace: List[Dict[str, Any]] = []

    # The final PM results will be stored here
    final_pm_results = {}

    # Execute the K-iteration loop
    for k in range(iteration_count_K):

        # Step 1: Distributed PM Update
        ell_k, pm_results = execute_admm_distributed_update(
            admm_state=admm_state,
            current_state=current_state,
            current_date=current_date,
            alpha_dictionary=alpha_dictionary,
            F_t_annual=F_t_annual,
            D_t_idio_annual=D_t_idio_annual,
            rf_daily_t=rf_daily_t,
            D_scale_diag_t=D_scale_diag_t,
            cvxpy_tc_evaluator=cvxpy_tc_evaluator,
            cvxpy_short_local=cvxpy_short_local,
            study_config=study_config
        )

        # Step 2: Gathered Central Update
        z_sum_new, u_new, primal_res, dual_res = execute_admm_gathered_update(
            pm_results=pm_results,
            admm_state=admm_state,
            current_state=current_state,
            D_scale_diag_t=D_scale_diag_t,
            cvxpy_tc_evaluator=cvxpy_tc_evaluator,
            cvxpy_short_firm=cvxpy_short_firm,
            study_config=study_config
        )

        # Extract x^{i,k+1} for the trace and next iteration
        x_new_dict = {pm_id: res.z_opt for pm_id, res in pm_results.items()}

        # Record trace for iteration k
        trace_entry = {
            "iteration": k,
            "ell_k": ell_k,
            "z_sum_k": admm_state.z_sum_k,
            "u_k": admm_state.u_k,
            "x_k": admm_state.x_k,
            "primal_residual": primal_res,
            "dual_residual": dual_res
        }
        admm_trace.append(trace_entry)

        # Update state for next iteration
        # Note: ADMMState is redefined here to avoid mutating the previous trace entries
        # Note: ADMMState is assumed to be imported/available
        admm_state = ADMMState(
            x_k=x_new_dict,
            z_sum_k=z_sum_new,
            u_k=u_new
        )

        final_pm_results = pm_results

    # -------------------------------------------------------------------------
    # Final Execution and Packaging
    # -------------------------------------------------------------------------
    # The executed firm net trade is the NAV-weighted sum of the FINAL PM iterates x^{i,K},
    # NOT the planner variable z_sum^K, because ADMM has not fully converged.
    n_assets = len(D_scale_diag_t)
    z_firm_executed = np.zeros(n_assets, dtype=np.float64)
    pm_tcs = {}

    pm_trade_vectors = {}
    pm_portfolio_weights = {}
    pm_cash_positions = {}
    solver_statuses = {}

    for pm_id, result in final_pm_results.items():
        lambda_i = current_state.pm_states[pm_id].nav_weight
        x_final_i = result.z_opt

        z_firm_executed += lambda_i * x_final_i
        pm_tcs[pm_id] = numpy_tc_evaluator(x_final_i)

        pm_trade_vectors[pm_id] = x_final_i
        pm_portfolio_weights[pm_id] = result.w_opt
        pm_cash_positions[pm_id] = result.c_opt
        solver_statuses[pm_id] = result.status

    # Evaluate the actual firm TC on the executed netted trade
    tc_firm_executed = numpy_tc_evaluator(z_firm_executed)

    # Note: ProtocolRunOutput is assumed to be imported/available
    output = ProtocolRunOutput(
        protocol_name=f"admm_{iteration_count_K}_iter",
        pm_trade_vectors=pm_trade_vectors,
        pm_portfolio_weights=pm_portfolio_weights,
        pm_cash_positions=pm_cash_positions,
        firm_net_trade=z_firm_executed,
        firm_transaction_cost=tc_firm_executed,
        pm_transaction_costs=pm_tcs,
        solver_statuses=solver_statuses,
        admm_trace=admm_trace
    )

    return output


In [ ]:
# Task 30: Define the execution, accounting, and state-transition layer

class StateTransitionError(Exception):
    """Custom exception raised when the post-trade accounting or state transition fails."""
    pass

# ==============================================================================
# Task 30: Define the execution, accounting, and state-transition layer
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 30, Step 1: Verify trade identity and extract post-trade state
# -------------------------------------------------------------------------------------------------------------------------------
def verify_and_extract_post_trade_state(
    protocol_output: Any, # ProtocolRunOutput
    current_state: Any # FirmState
) -> Dict[int, Dict[str, Any]]:
    """
    Verifies the trade identity w_post - w_curr = z for each PM and extracts the post-trade state.

    Parameters
    ----------
    protocol_output : ProtocolRunOutput
        The standardized output from any protocol runner.
    current_state : FirmState
        The pre-trade dynamic state of the firm.

    Returns
    -------
    Dict[int, Dict[str, Any]]
        A dictionary mapping pm_id to their verified post-trade weights and cash.

    Raises
    ------
    StateTransitionError
        If the trade identity is violated for any PM.
    """
    logger.debug("Executing Task 30, Step 1: Verifying trade identity and extracting post-trade state.")

    verified_post_trade_states: Dict[int, Dict[str, Any]] = {}

    for pm_id, pm_state in current_state.pm_states.items():
        w_curr = pm_state.w_curr
        w_post = protocol_output.pm_portfolio_weights[pm_id]
        z_trade = protocol_output.pm_trade_vectors[pm_id]
        c_post = protocol_output.pm_cash_positions[pm_id]

        # Rigorous invariant assertion: Trade identity
        # We use a tolerance of 1e-5 to account for solver precision limits
        if not np.allclose(w_post - w_curr, z_trade, atol=1e-5):
            max_diff = np.max(np.abs((w_post - w_curr) - z_trade))
            error_msg = f"Trade identity violation for PM {pm_id}. Max diff: {max_diff}"
            logger.error(error_msg)
            raise StateTransitionError(error_msg)

        verified_post_trade_states[pm_id] = {
            "w_post": w_post,
            "c_post": c_post
        }

    return verified_post_trade_states

# -------------------------------------------------------------------------------------------------------------------------------
# Task 30, Step 2: Update cash, NAV, and drift weights based on realized returns
# -------------------------------------------------------------------------------------------------------------------------------
def compute_post_return_drift(
    verified_post_trade_states: Dict[int, Dict[str, Any]],
    current_state: Any, # FirmState
    r_t_plus_1: np.ndarray,
    rf_daily_t: float,
    tc_firm_t: float
) -> Dict[int, Dict[str, Any]]:
    """
    Computes the portfolio return, updates NAV, and drifts weights for each PM.

    Equations:
    r_portfolio = w_post^T * r_{t+1} + c_post * r_rf - lambda * TC_firm
    V_{t+1} = V_t * (1 + r_portfolio)
    w_{curr,t+1} = (w_post * (1 + r_{t+1})) / (1 + r_portfolio)
    c_{curr,t+1} = 1 - sum(w_{curr,t+1})

    Parameters
    ----------
    verified_post_trade_states : Dict[int, Dict[str, Any]]
        The verified post-trade weights and cash for each PM.
    current_state : FirmState
        The pre-trade dynamic state.
    r_t_plus_1 : np.ndarray
        The vector of realized asset returns from t to t+1.
    rf_daily_t : float
        The daily risk-free rate at time t.
    tc_firm_t : float
        The realized firm transaction cost.

    Returns
    -------
    Dict[int, Dict[str, Any]]
        A dictionary mapping pm_id to their new drifted state (V_{t+1}, w_{curr,t+1}, c_{curr,t+1}).

    Raises
    ------
    StateTransitionError
        If a PM's NAV drops below zero.
    """
    logger.debug("Executing Task 30, Step 2: Computing post-return drift and updating NAVs.")

    drifted_states: Dict[int, Dict[str, Any]] = {}

    # Handle NaNs in the return vector (e.g., assets that stopped trading)
    # We treat missing returns as 0.0 to allow the dot product to succeed.
    safe_r_t_plus_1 = np.nan_to_num(r_t_plus_1, nan=0.0)

    for pm_id, post_state in verified_post_trade_states.items():
        w_post = post_state["w_post"]
        c_post = post_state["c_post"]

        pm_current = current_state.pm_states[pm_id]
        v_t = pm_current.nav_usd
        lambda_t = pm_current.nav_weight

        # 1. Compute portfolio return
        # The TC attribution rule: PM pays their NAV-weighted share of the firm TC.
        # This is a manuscript-unspecified placeholder convention.
        asset_return = np.dot(w_post, safe_r_t_plus_1)
        cash_return = c_post * rf_daily_t
        tc_attribution = lambda_t * tc_firm_t

        r_portfolio = asset_return + cash_return - tc_attribution

        # 2. Update NAV
        v_t_plus_1 = v_t * (1.0 + r_portfolio)

        if v_t_plus_1 <= 0.0:
            error_msg = f"Critical failure: PM {pm_id} NAV dropped to {v_t_plus_1}. Portfolio bankrupt."
            logger.error(error_msg)
            raise StateTransitionError(error_msg)

        # 3. Drift weights
        # Each asset's weight grows by its individual return, normalized by the total portfolio return
        w_drifted = (w_post * (1.0 + safe_r_t_plus_1)) / (1.0 + r_portfolio)

        # 4. Recompute cash
        # Cash is the residual to ensure weights sum to 1.0
        c_drifted = 1.0 - np.sum(w_drifted)

        drifted_states[pm_id] = {
            "nav_usd": v_t_plus_1,
            "w_curr": w_drifted,
            "c_curr": c_drifted
        }

    return drifted_states

# -------------------------------------------------------------------------------------------------------------------------------
# Task 30, Step 3: Recompute NAV weights and finalize state transition
# -------------------------------------------------------------------------------------------------------------------------------
def finalize_state_transition(
    current_state: Any, # FirmState
    drifted_states: Dict[int, Dict[str, Any]],
    next_date: pd.Timestamp
) -> Any: # FirmState
    """
    Recomputes the relative NAV weights and updates the FirmState object in-place.

    Equation: lambda_{t+1}^{(i)} = V_{t+1}^{(i)} / sum(V_{t+1}^{(j)})

    Parameters
    ----------
    current_state : FirmState
        The FirmState object to be mutated.
    drifted_states : Dict[int, Dict[str, Any]]
        The new NAVs, weights, and cash for each PM.
    next_date : pd.Timestamp
        The date t+1 to stamp on the new state.

    Returns
    -------
    FirmState
        The mutated FirmState object, now representing the state at t+1.

    Raises
    ------
    StateTransitionError
        If the new NAV weights do not sum to 1.0.
    """
    logger.debug("Executing Task 30, Step 3: Recomputing NAV weights and finalizing state transition.")

    # 1. Compute new firm NAV
    v_firm_t_plus_1 = sum(state["nav_usd"] for state in drifted_states.values())
    current_state.firm_nav_usd = v_firm_t_plus_1

    # 2. Update PM states and compute new lambda weights
    total_weight = 0.0
    for pm_id, new_state in drifted_states.items():
        pm_obj = current_state.pm_states[pm_id]

        # Update dynamic fields
        pm_obj.nav_usd = new_state["nav_usd"]
        pm_obj.w_curr = new_state["w_curr"]
        pm_obj.c_curr = new_state["c_curr"]

        # Compute and update lambda
        new_lambda = new_state["nav_usd"] / v_firm_t_plus_1
        pm_obj.nav_weight = new_lambda

        total_weight += new_lambda

    # Rigorous invariant assertion
    if not np.isclose(total_weight, 1.0, atol=1e-14):
        error_msg = f"State transition failure: New NAV weights sum to {total_weight}, expected 1.0."
        logger.error(error_msg)
        raise StateTransitionError(error_msg)

    # 3. Stamp the new date
    current_state.current_date = next_date

    return current_state

# -------------------------------------------------------------------------------------------------------------------------------
# Task 30, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_30_orchestrator(
    protocol_output: Any, # ProtocolRunOutput
    current_state: Any, # FirmState
    current_date: pd.Timestamp,
    next_date: pd.Timestamp,
    daily_returns_panel: pd.DataFrame,
    rf_daily_series: pd.Series
) -> Any: # FirmState
    """
    Orchestrates the execution, accounting, and state-transition layer for a single time step.
    This function is protocol-agnostic and mutates the current_state in-place.

    Parameters
    ----------
    protocol_output : ProtocolRunOutput
        The standardized output from the protocol runner at date t.
    current_state : FirmState
        The dynamic state of the firm at date t.
    current_date : pd.Timestamp
        The decision date t.
    next_date : pd.Timestamp
        The next calendar date t+1.
    daily_returns_panel : pd.DataFrame
        The wide-format daily returns panel.
    rf_daily_series : pd.Series
        The daily risk-free rate series.

    Returns
    -------
    FirmState
        The mutated FirmState object, now representing the state at t+1.

    Raises
    ------
    StateTransitionError
        If any accounting invariants are violated.
    """
    # Extract exogenous data for the transition from t to t+1
    # We need the return realized ON next_date
    try:
        r_t_plus_1 = daily_returns_panel.loc[next_date].to_numpy()
    except KeyError:
        raise StateTransitionError(f"Return data for next_date {next_date.date()} not found in panel.")

    rf_daily_t = float(rf_daily_series.loc[current_date])
    tc_firm_t = protocol_output.firm_transaction_cost

    # Execute Step 1: Verify trades and extract post-trade state
    verified_post_trade_states = verify_and_extract_post_trade_state(
        protocol_output=protocol_output,
        current_state=current_state
    )

    # Execute Step 2: Compute drift and update NAVs
    drifted_states = compute_post_return_drift(
        verified_post_trade_states=verified_post_trade_states,
        current_state=current_state,
        r_t_plus_1=r_t_plus_1,
        rf_daily_t=rf_daily_t,
        tc_firm_t=tc_firm_t
    )

    # Execute Step 3: Finalize transition
    updated_state = finalize_state_transition(
        current_state=current_state,
        drifted_states=drifted_states,
        next_date=next_date
    )

    return updated_state


In [ ]:
# Task 31: Create the end-to-end orchestrator callable — input and estimation layer

class PipelineExecutionError(Exception):
    """Custom exception raised when the end-to-end pipeline fails during pre-computation."""
    pass

# ==================================================================================
# Task 31: Create the end-to-end orchestrator callable — input and estimation layer
# ==================================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 31, Step 1: Invoke validation callables (Tasks 1-3)
# -------------------------------------------------------------------------------------------------------------------------------
def execute_validation_layer(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Tuple[Dict[str, Any], Any]: # Tuple[Dict[str, Any], ValidationReport]
    """
    Executes Tasks 1-3 to validate the structural, semantic, and numerical integrity of all inputs.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Tuple[Dict[str, Any], ValidationReport]
        The initialized artifact registry containing validated inputs, and the validation report.

    Raises
    ------
    PipelineExecutionError
        If any validation task raises an exception, it is caught and re-raised to halt the pipeline.
    """
    logger.info("Executing Task 31, Step 1: Invoking validation layer (Tasks 1-3).")

    artifact_registry: Dict[str, Any] = {}

    try:
        # Task 1: Structural Validation
        val_raw_df, val_rf_series, val_calendar, val_id_map, val_config = task_1_orchestrator(
            raw_market_df, risk_free_rate_series, master_trading_calendar, asset_identifier_map, study_config
        )

        # Task 2: Semantic Validation
        val_config = task_2_orchestrator(val_config)

        # Task 3: Numerical and Cross-Input Consistency
        validation_report = task_3_orchestrator(
            raw_market_df=val_raw_df,
            asset_identifier_map=val_id_map,
            master_trading_calendar=val_calendar,
            risk_free_rate_series=val_rf_series
        )

    except Exception as e:
        error_msg = f"Critical validation failure: {e}"
        logger.error(error_msg)
        raise PipelineExecutionError(error_msg)

    # Populate the registry with validated inputs
    artifact_registry["raw_market_df"] = val_raw_df
    artifact_registry["risk_free_rate_series"] = val_rf_series
    artifact_registry["master_trading_calendar"] = val_calendar
    artifact_registry["asset_identifier_map"] = val_id_map
    artifact_registry["study_config"] = val_config

    return artifact_registry, validation_report

# -------------------------------------------------------------------------------------------------------------------------------
# Task 31, Step 2: Invoke pre-estimation callables (Tasks 4-14)
# -------------------------------------------------------------------------------------------------------------------------------
def execute_pre_estimation_layer(artifact_registry: Dict[str, Any]) -> Dict[str, Any]:
    """
    Executes Tasks 4-14 to cleanse data, construct the universe, and build market features.

    Parameters
    ----------
    artifact_registry : Dict[str, Any]
        The registry containing validated inputs.

    Returns
    -------
    Dict[str, Any]
        The updated artifact registry containing all pre-estimation artifacts.
    """
    logger.info("Executing Task 31, Step 2: Invoking pre-estimation layer (Tasks 4-14).")

    # Extract inputs
    raw_df = artifact_registry["raw_market_df"]
    rf_series = artifact_registry["risk_free_rate_series"]
    calendar = artifact_registry["master_trading_calendar"]
    id_map = artifact_registry["asset_identifier_map"]
    config = artifact_registry["study_config"]

    # Task 4: Freeze environment and hash inputs
    manifest, generators = task_4_orchestrator(raw_df, rf_series, calendar, id_map, config)
    artifact_registry["manifest"] = manifest
    artifact_registry["generators"] = generators

    # Task 5: Validate identifier integrity
    manifest = task_5_orchestrator(raw_df, id_map, calendar, config, manifest)

    # Task 6: Validate and freeze calendar
    valid_decision_dates, manifest = task_6_orchestrator(calendar, rf_series, manifest)
    artifact_registry["valid_decision_dates"] = valid_decision_dates

    # Tasks 7 & 8: Cleanse market data
    filtered_df, task_7_log = task_7_orchestrator(raw_df, config)
    clean_market_df, audit_summary = task_8_orchestrator(filtered_df, calendar, config, task_7_log)
    artifact_registry["clean_market_df"] = clean_market_df
    manifest["data_cleaning_audit"] = audit_summary

    # Tasks 9 & 10: Construct universe
    candidate_ids, manifest, funnel_log = task_9_orchestrator(clean_market_df, calendar, config, manifest)
    universe_asset_ids, universe_market_df, manifest = task_10_orchestrator(
        clean_market_df, calendar, candidate_ids, config, manifest
    )
    artifact_registry["universe_asset_ids"] = universe_asset_ids
    artifact_registry["universe_market_df"] = universe_market_df
    manifest["universe_construction_funnel"] = funnel_log

    # Task 11: Align risk-free rate
    rf_daily_series, manifest = task_11_orchestrator(rf_series, calendar, valid_decision_dates, config, manifest)
    artifact_registry["rf_daily_series"] = rf_daily_series

    # Tasks 12 & 13: Construct spreads, volume, and returns
    kappa_spread_panel = task_12_orchestrator(universe_market_df, valid_decision_dates, universe_asset_ids)
    dollar_volume_panel, daily_returns_panel, manifest = task_13_orchestrator(
        universe_market_df, universe_asset_ids, calendar, manifest
    )
    artifact_registry["kappa_spread_panel"] = kappa_spread_panel
    artifact_registry["dollar_volume_panel"] = dollar_volume_panel
    artifact_registry["daily_returns_panel"] = daily_returns_panel

    # Task 14: Construct forward returns
    fwd_windows, fwd_vectors, manifest = task_14_orchestrator(
        daily_returns_panel, valid_decision_dates, calendar, manifest
    )
    artifact_registry["forward_return_windows"] = fwd_windows
    artifact_registry["forward_return_vectors"] = fwd_vectors
    artifact_registry["manifest"] = manifest

    return artifact_registry

# -------------------------------------------------------------------------------------------------------------------------------
# Task 31, Step 3: Invoke estimation callables (Tasks 15-21)
# -------------------------------------------------------------------------------------------------------------------------------
def execute_estimation_layer(artifact_registry: Dict[str, Any]) -> Dict[str, Any]:
    """
    Executes Tasks 15-21 to estimate the risk model, initialize PMs, and generate synthetic alpha.

    Parameters
    ----------
    artifact_registry : Dict[str, Any]
        The registry containing pre-estimation artifacts.

    Returns
    -------
    Dict[str, Any]
        The fully populated artifact registry ready for protocol execution.
    """
    logger.info("Executing Task 31, Step 3: Invoking estimation layer (Tasks 15-21).")

    # Extract inputs
    fwd_windows = artifact_registry["forward_return_windows"]
    fwd_vectors = artifact_registry["forward_return_vectors"]
    daily_returns = artifact_registry["daily_returns_panel"]
    calendar = artifact_registry["master_trading_calendar"]
    valid_dates = artifact_registry["valid_decision_dates"]
    universe_ids = artifact_registry["universe_asset_ids"]
    config = artifact_registry["study_config"]
    manifest = artifact_registry["manifest"]
    generators = artifact_registry["generators"]

    # Tasks 15 & 16: Risk Model
    vol_vectors, win_windows, manifest = task_15_orchestrator(fwd_windows, config, manifest)
    F_t_ann, D_t_idio_ann, nu_t_daily, manifest = task_16_orchestrator(
        win_windows, vol_vectors, config, manifest
    )
    artifact_registry["F_t_annual_dict"] = F_t_ann
    artifact_registry["D_t_idio_annual_dict"] = D_t_idio_ann
    artifact_registry["nu_t_daily_dict"] = nu_t_daily

    # Tasks 17 & 18: PM Initialization
    pm_chars, manifest = task_17_orchestrator(config, universe_ids, generators, manifest)
    initial_firm_state, manifest = task_18_orchestrator(pm_chars, manifest)
    artifact_registry["pm_characteristics_dict"] = pm_chars
    artifact_registry["initial_firm_state"] = initial_firm_state

    # Tasks 19, 20 & 21: Synthetic Alpha
    alpha_calib, manifest = task_19_orchestrator(config, generators, manifest)
    phi_diag, S_E, S_U, Sigma_Asset, manifest = task_20_orchestrator(
        alpha_calib, daily_returns, config, manifest
    )
    alpha_dict, manifest = task_21_orchestrator(
        phi_diag, S_E, S_U, Sigma_Asset, fwd_vectors, alpha_calib, calendar, valid_dates, generators, manifest
    )
    artifact_registry["alpha_dictionary"] = alpha_dict
    artifact_registry["manifest"] = manifest

    return artifact_registry

# -------------------------------------------------------------------------------------------------------------------------------
# Task 31, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_31_orchestrator(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Union[Dict[str, Any], Any]: # Union[Dict[str, Any], ValidationReport]
    """
    Master orchestrator for the input and estimation layer (Tasks 1-21).
    Chains all pre-computation callables into a single end-to-end pipeline.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Union[Dict[str, Any], ValidationReport]
        The fully populated artifact registry if successful, or the ValidationReport if validation fails.
    """
    logger.info("Commencing Task 31: End-to-end orchestrator — input and estimation layer.")

    try:
        # Execute Step 1: Validation Layer
        artifact_registry, validation_report = execute_validation_layer(
            raw_market_df, risk_free_rate_series, master_trading_calendar, asset_identifier_map, study_config
        )

        # If validation requires a halt, it would have raised an exception in Step 1.
        # If we reach here, validation passed (or passed with placeholder risk).

        # Execute Step 2: Pre-estimation Layer
        artifact_registry = execute_pre_estimation_layer(artifact_registry)

        # Execute Step 3: Estimation Layer
        artifact_registry = execute_estimation_layer(artifact_registry)

        logger.info("Task 31 complete. Input and estimation layer executed successfully.")
        return artifact_registry

    except PipelineExecutionError as e:
        logger.critical(f"Pipeline halted during pre-computation: {e}")
        # If validation failed, return the report (if it was generated before the crash)
        # Otherwise, re-raise to ensure the failure is not swallowed silently.
        raise


In [ ]:
# Task 32: Create the end-to-end orchestrator callable — protocol and evaluation layer

class SimulationExecutionError(Exception):
    """Custom exception raised when the walk-forward simulation or evaluation fails."""
    pass

# =====================================================================================
# Task 32: Create the end-to-end orchestrator callable — protocol and evaluation layer
# =====================================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 32, Step 1: Execute the walk-forward daily loop for each protocol
# -------------------------------------------------------------------------------------------------------------------------------
def execute_walk_forward_simulation(
    artifact_registry: Dict[str, Any]
) -> Dict[str, Dict[str, Any]]:
    """
    Executes the walk-forward daily loop for all four protocols.

    Parameters
    ----------
    artifact_registry : Dict[str, Any]
        The fully populated registry containing all pre-computed exogenous data and initial states.

    Returns
    -------
    Dict[str, Dict[str, Any]]
        A dictionary mapping protocol names to their respective historical trajectories and traces.

    Raises
    ------
    SimulationExecutionError
        If any protocol fails during the walk-forward loop.
    """
    logger.info("Executing Task 32, Step 1: Running walk-forward simulations for all protocols.")

    # Extract required artifacts
    initial_firm_state = artifact_registry["initial_firm_state"]
    valid_dates = artifact_registry["valid_decision_dates"]
    calendar = artifact_registry["master_trading_calendar"]
    alpha_dict = artifact_registry["alpha_dictionary"]
    F_t_ann = artifact_registry["F_t_annual_dict"]
    D_t_idio_ann = artifact_registry["D_t_idio_annual_dict"]
    nu_t_daily = artifact_registry["nu_t_daily_dict"]
    dollar_vol = artifact_registry["dollar_volume_panel"]
    kappa_spread = artifact_registry["kappa_spread_panel"]
    rf_series = artifact_registry["rf_daily_series"]
    daily_returns = artifact_registry["daily_returns_panel"]
    config = artifact_registry["study_config"]

    protocols = ["independent", "full_cooperative", "admm_2_iter", "admm_5_iter"]
    protocol_results: Dict[str, Dict[str, Any]] = {}

    for protocol in protocols:
        logger.info(f"Starting walk-forward simulation for protocol: {protocol}")

        # Deep-copy the initial state to ensure strict isolation between protocols
        current_state = copy.deepcopy(initial_firm_state)

        # Initialize history accumulators
        history = {
            "firm_nav": [],
            "firm_tc": [],
            "pm_navs": {pm_id: [] for pm_id in current_state.pm_states},
            "pm_tcs": {pm_id: [] for pm_id in current_state.pm_states},
            "firm_net_trades": [],
            "admm_trace": []
        }

        # Record initial NAVs (t=0)
        history["firm_nav"].append(current_state.firm_nav_usd)
        for pm_id, pm_state in current_state.pm_states.items():
            history["pm_navs"][pm_id].append(pm_state.nav_usd)

        try:
            for i, current_date in enumerate(valid_dates):
                # Determine the next calendar date for the state transition
                # valid_dates are guaranteed to have at least 42 subsequent dates in the calendar
                current_idx = calendar.get_loc(current_date)
                next_date = calendar[current_idx + 1]

                # 1. Invoke the specific protocol runner
                if protocol == "independent":
                    output = task_26_orchestrator(
                        current_state, current_date, alpha_dict, F_t_ann, D_t_idio_ann,
                        nu_t_daily, dollar_vol, kappa_spread, rf_series, config
                    )
                elif protocol == "full_cooperative":
                    output = task_27_orchestrator(
                        current_state, current_date, alpha_dict, F_t_ann, D_t_idio_ann,
                        nu_t_daily, dollar_vol, kappa_spread, rf_series, config
                    )
                elif protocol.startswith("admm"):
                    k_iters = int(protocol.split("_")[1])
                    # Initialize ADMM
                    init_state, D_scale, cvxpy_tc, np_tc, cvxpy_short = task_28_orchestrator(
                        current_state, current_date, alpha_dict, F_t_ann, D_t_idio_ann,
                        nu_t_daily, dollar_vol, kappa_spread, rf_series, config
                    )
                    # Run ADMM loop
                    output = task_29_orchestrator(
                        init_state, current_state, current_date, alpha_dict, F_t_ann, D_t_idio_ann,
                        rf_series, D_scale, cvxpy_tc, np_tc, cvxpy_short, cvxpy_short, config, k_iters
                    )
                    if output.admm_trace:
                        history["admm_trace"].append({current_date: output.admm_trace})
                else:
                    raise ValueError(f"Unknown protocol: {protocol}")

                # 2. Record trades and costs for date t
                history["firm_net_trades"].append(output.firm_net_trade)
                history["firm_tc"].append(output.firm_transaction_cost)
                for pm_id, tc in output.pm_transaction_costs.items():
                    history["pm_tcs"][pm_id].append(tc)

                # 3. Invoke state transition to t+1
                current_state = task_30_orchestrator(
                    output, current_state, current_date, next_date, daily_returns, rf_series
                )

                # 4. Record post-transition NAVs for date t+1
                history["firm_nav"].append(current_state.firm_nav_usd)
                for pm_id, pm_state in current_state.pm_states.items():
                    history["pm_navs"][pm_id].append(pm_state.nav_usd)

                # Optional: Log progress every 1000 dates
                if (i + 1) % 1000 == 0:
                    logger.info(f"  {protocol}: Processed {i + 1} / {len(valid_dates)} dates.")

            protocol_results[protocol] = history
            logger.info(f"Completed walk-forward simulation for protocol: {protocol}")

        except Exception as e:
            error_msg = f"Simulation failed for protocol '{protocol}' at date {current_date.date()}: {e}"
            logger.error(error_msg)
            raise SimulationExecutionError(error_msg)

    return protocol_results

# -------------------------------------------------------------------------------------------------------------------------------
# Task 32, Step 2: Compute firm-level and PM-level performance metrics
# -------------------------------------------------------------------------------------------------------------------------------
def compute_performance_metrics(
    protocol_results: Dict[str, Dict[str, Any]],
    annualization_factor: float = 252.0
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Computes annualized return, volatility, Sharpe ratio, and cumulative paths.

    Parameters
    ----------
    protocol_results : Dict[str, Dict[str, Any]]
        The historical trajectories for each protocol.
    annualization_factor : float, optional
        The scalar used to annualize daily metrics (default is 252.0).

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]
        Firm performance table, PM performance table, and cumulative paths dictionary.
    """
    logger.info("Executing Task 32, Step 2: Computing performance metrics.")

    firm_metrics = []
    pm_metrics = []
    cumulative_paths = {}

    for protocol, history in protocol_results.items():
        # 1. Firm-level metrics
        firm_navs = np.array(history["firm_nav"])
        # Daily returns: V_{t+1} / V_t - 1
        firm_returns = (firm_navs[1:] / firm_navs[:-1]) - 1.0

        # Geometric annualized return
        total_days = len(firm_returns)
        ann_return = ((firm_navs[-1] / firm_navs[0]) ** (annualization_factor / total_days)) - 1.0

        # Annualized volatility (sample std dev)
        ann_vol = np.std(firm_returns, ddof=1) * np.sqrt(annualization_factor)

        # Sharpe Ratio (Total Return convention as per manuscript derivation)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

        firm_metrics.append({
            "Protocol": protocol,
            "Return": ann_return,
            "Volatility": ann_vol,
            "Sharpe": sharpe
        })

        # Cumulative paths
        cum_ret_path = np.cumprod(1.0 + firm_returns)
        cum_tc_path = np.cumsum(history["firm_tc"])

        cumulative_paths[protocol] = {
            "firm_cum_return": cum_ret_path,
            "firm_cum_tc": cum_tc_path,
            "pm_cum_returns": {},
            "pm_cum_tcs": {}
        }

        # 2. PM-level metrics
        for pm_id, navs in history["pm_navs"].items():
            pm_navs = np.array(navs)
            pm_returns = (pm_navs[1:] / pm_navs[:-1]) - 1.0

            pm_ann_ret = ((pm_navs[-1] / pm_navs[0]) ** (annualization_factor / total_days)) - 1.0
            pm_ann_vol = np.std(pm_returns, ddof=1) * np.sqrt(annualization_factor)
            pm_sharpe = pm_ann_ret / pm_ann_vol if pm_ann_vol > 0 else 0.0

            pm_metrics.append({
                "Protocol": protocol,
                "PM_ID": pm_id,
                "Return": pm_ann_ret,
                "Volatility": pm_ann_vol,
                "Sharpe": pm_sharpe
            })

            cumulative_paths[protocol]["pm_cum_returns"][pm_id] = np.cumprod(1.0 + pm_returns)
            cumulative_paths[protocol]["pm_cum_tcs"][pm_id] = np.cumsum(history["pm_tcs"][pm_id])

    # Format tables
    firm_df = pd.DataFrame(firm_metrics).set_index("Protocol")
    pm_df = pd.DataFrame(pm_metrics).set_index(["Protocol", "PM_ID"])

    logger.info("Performance metrics computed successfully.")
    return firm_df, pm_df, cumulative_paths

# -------------------------------------------------------------------------------------------------------------------------------
# Task 32, Step 3: Emit the structured reproduction package
# -------------------------------------------------------------------------------------------------------------------------------
def package_reproduction_artifacts(
    artifact_registry: Dict[str, Any],
    protocol_results: Dict[str, Dict[str, Any]],
    firm_df: pd.DataFrame,
    pm_df: pd.DataFrame,
    cumulative_paths: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Aggregates all artifacts into a single, serializable reproduction package.

    Parameters
    ----------
    artifact_registry : Dict[str, Any]
        The registry containing exogenous data and the manifest.
    protocol_results : Dict[str, Dict[str, Any]]
        The raw historical trajectories.
    firm_df : pd.DataFrame
        The firm performance table.
    pm_df : pd.DataFrame
        The PM performance table.
    cumulative_paths : Dict[str, Any]
        The cumulative return and cost paths.

    Returns
    -------
    Dict[str, Any]
        The complete structured reproduction package.
    """
    logger.info("Executing Task 32, Step 3: Packaging reproduction artifacts.")

    # Augment manifest with evaluation conventions
    manifest = artifact_registry["manifest"]
    if "placeholder_assumptions" not in manifest:
        manifest["placeholder_assumptions"] = []

    manifest["placeholder_assumptions"].append({
        "convention_name": "sharpe_ratio_calculation",
        "manuscript_status": "unspecified",
        "value_used": "Total Return / Volatility (no risk-free rate deduction)",
        "source": "derived from manuscript reported numbers"
    })

    reproduction_package = {
        "manifest": manifest,
        "exogenous_artifacts": {
            "clean_market_df": artifact_registry["clean_market_df"],
            "universe_asset_ids": artifact_registry["universe_asset_ids"],
            "rf_daily_series": artifact_registry["rf_daily_series"],
            "valid_decision_dates": artifact_registry["valid_decision_dates"],
            "pm_characteristics": artifact_registry["pm_characteristics_dict"]
            # Note: Massive tensors (like forward windows) are excluded from the final package
            # to save space, as they can be deterministically reconstructed from clean_market_df.
        },
        "protocol_results": protocol_results,
        "performance_metrics": {
            "firm_table": firm_df,
            "pm_table": pm_df,
            "cumulative_paths": cumulative_paths
        }
    }

    logger.info("Reproduction package assembled successfully.")
    return reproduction_package

# -------------------------------------------------------------------------------------------------------------------------------
# Task 32, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_32_orchestrator(
    artifact_registry: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Master orchestrator for the protocol execution and evaluation layer.

    Parameters
    ----------
    artifact_registry : Dict[str, Any]
        The fully populated registry from Task 31.

    Returns
    -------
    Dict[str, Any]
        The complete structured reproduction package.

    Raises
    ------
    SimulationExecutionError
        If the walk-forward loop or evaluation fails.
    """
    logger.info("Commencing Task 32: End-to-end orchestrator — protocol and evaluation layer.")

    # Execute Step 1: Walk-forward simulation
    protocol_results = execute_walk_forward_simulation(
        artifact_registry=artifact_registry
    )

    # Execute Step 2: Compute metrics
    config = artifact_registry["study_config"]
    ann_factor = float(config.get("GLOBAL_FIRM_CONSTANTS", {}).get("annualization_basis_trading_days", 252.0))

    firm_df, pm_df, cumulative_paths = compute_performance_metrics(
        protocol_results=protocol_results,
        annualization_factor=ann_factor
    )

    # Execute Step 3: Package artifacts
    reproduction_package = package_reproduction_artifacts(
        artifact_registry=artifact_registry,
        protocol_results=protocol_results,
        firm_df=firm_df,
        pm_df=pm_df,
        cumulative_paths=cumulative_paths
    )

    logger.info("Task 32 complete. Protocol execution and evaluation finished.")
    return reproduction_package


In [ ]:
# Task 33: Run the orchestrator for all four protocols and persist diagnostics

class DiagnosticsPersistenceError(Exception):
    """Custom exception raised when ADMM traces cannot be extracted or saved to disk."""
    pass

# ==============================================================================
# Task 33: Run the orchestrator for all four protocols and persist diagnostics
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 33, Step 1: Invoke the end-to-end orchestrator and capture results
# -------------------------------------------------------------------------------------------------------------------------------
def execute_full_pipeline(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any]
) -> Union[Dict[str, Any], Any]: # Union[Dict[str, Any], ValidationReport]
    """
    Invokes the master orchestrators to execute the entire pipeline from validation to evaluation.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.

    Returns
    -------
    Union[Dict[str, Any], ValidationReport]
        The complete reproduction package dictionary, or a ValidationReport if early halt occurred.
    """
    logger.info("Executing Task 33, Step 1: Invoking the end-to-end pipeline.")
    start_time = time.time()

    # 1. Execute the Input and Estimation Layer (Tasks 1-21)
    # Note: task_31_orchestrator is assumed to be imported/available
    registry_or_report = task_31_orchestrator(
        raw_market_df=raw_market_df,
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar,
        asset_identifier_map=asset_identifier_map,
        study_config=study_config
    )

    # Check if validation failed and returned a report instead of the registry
    # Assuming ValidationReport is a dataclass or specific type
    if not isinstance(registry_or_report, dict):
        logger.critical("Pipeline halted during validation. Returning ValidationReport.")
        return registry_or_report

    artifact_registry = registry_or_report

    # 2. Execute the Protocol and Evaluation Layer (Tasks 22-32)
    # Note: task_32_orchestrator is assumed to be imported/available
    reproduction_package = task_32_orchestrator(
        artifact_registry=artifact_registry
    )

    end_time = time.time()
    duration_seconds = end_time - start_time
    hours, rem = divmod(duration_seconds, 3600)
    minutes, seconds = divmod(rem, 60)

    logger.info(f"Full pipeline execution completed in {int(hours)}h {int(minutes)}m {seconds:.2f}s.")

    # Record execution time in the manifest
    reproduction_package["manifest"]["total_execution_time_seconds"] = duration_seconds

    return reproduction_package

# -------------------------------------------------------------------------------------------------------------------------------
# Task 33, Step 2: Extract, summarize, and persist ADMM iteration traces
# -------------------------------------------------------------------------------------------------------------------------------
def persist_admm_traces_and_summarize(
    reproduction_package: Dict[str, Any],
    output_dir: Path
) -> Dict[str, Any]:
    """
    Extracts ADMM traces, computes residual summary statistics, and saves traces to compressed NumPy archives.

    Parameters
    ----------
    reproduction_package : Dict[str, Any]
        The complete reproduction package containing protocol results.
    output_dir : Path
        The directory where the .npz trace files will be saved.

    Returns
    -------
    Dict[str, Any]
        The reproduction package with the manifest augmented by convergence diagnostics.

    Raises
    ------
    DiagnosticsPersistenceError
        If file I/O fails or trace structures are malformed.
    """
    logger.info("Executing Task 33, Step 2: Persisting ADMM traces and computing convergence summaries.")

    try:
        output_dir.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        raise DiagnosticsPersistenceError(f"Failed to create output directory {output_dir}: {e}")

    manifest = reproduction_package["manifest"]
    if "admm_convergence_diagnostics" not in manifest:
        manifest["admm_convergence_diagnostics"] = {}

    protocol_results = reproduction_package.get("protocol_results", {})
    admm_protocols = [p for p in protocol_results.keys() if p.startswith("admm")]

    for protocol in admm_protocols:
        history = protocol_results[protocol]
        raw_trace_list = history.get("admm_trace", [])

        if not raw_trace_list:
            logger.warning(f"No ADMM trace found for protocol {protocol}.")
            continue

        # raw_trace_list is a list of dicts: [{date: [iter0_dict, iter1_dict, ...]}, ...]
        # We need to aggregate residuals across all dates for each iteration k

        # Determine K from the first valid date's trace
        first_date_trace = list(raw_trace_list[0].values())[0]
        k_iters = len(first_date_trace)

        primal_residuals_by_k = {k: [] for k in range(k_iters)}
        dual_residuals_by_k = {k: [] for k in range(k_iters)}

        # Prepare flat arrays for persistence
        # To avoid massive memory spikes, we save the raw trace list directly using np.savez_compressed
        # after converting it to a more numpy-friendly format if necessary, or simply save the dict array.
        # For strict fidelity, we save the exact object.

        for date_entry in raw_trace_list:
            for date, iter_list in date_entry.items():
                for k_dict in iter_list:
                    k = k_dict["iteration"]
                    primal_residuals_by_k[k].append(k_dict["primal_residual"])
                    dual_residuals_by_k[k].append(k_dict["dual_residual"])

        # Compute summary statistics
        summary_stats = {}
        for k in range(k_iters):
            p_res = np.array(primal_residuals_by_k[k])
            d_res = np.array(dual_residuals_by_k[k])

            summary_stats[f"iteration_{k}"] = {
                "primal_residual_median": float(np.median(p_res)),
                "primal_residual_95th": float(np.percentile(p_res, 95)),
                "dual_residual_median": float(np.median(d_res)),
                "dual_residual_95th": float(np.percentile(d_res, 95))
            }

        manifest["admm_convergence_diagnostics"][protocol] = summary_stats

        # Persist to disk
        trace_file_path = output_dir / f"{protocol}_trace.npz"
        try:
            # np.savez_compressed can handle arrays of dictionaries (object arrays)
            np.savez_compressed(trace_file_path, trace_data=np.array(raw_trace_list, dtype=object))
            logger.info(f"Successfully saved {protocol} trace to {trace_file_path} (Size: {trace_file_path.stat().st_size / 1e6:.2f} MB).")
        except Exception as e:
            raise DiagnosticsPersistenceError(f"Failed to save trace for {protocol}: {e}")

    reproduction_package["manifest"] = manifest
    return reproduction_package

# -------------------------------------------------------------------------------------------------------------------------------
# Task 33, Step 3: Verify and enforce protocol ordering in the performance table
# -------------------------------------------------------------------------------------------------------------------------------
def verify_performance_table_ordering(
    reproduction_package: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Verifies that the output table's protocol ordering matches the paper's canonical sequence.

    Parameters
    ----------
    reproduction_package : Dict[str, Any]
        The complete reproduction package.

    Returns
    -------
    Dict[str, Any]
        The reproduction package with the correctly ordered performance table.
    """
    logger.info("Executing Task 33, Step 3: Verifying performance table ordering.")

    canonical_order = ["independent", "full_cooperative", "admm_2_iter", "admm_5_iter"]

    firm_table: pd.DataFrame = reproduction_package["performance_metrics"]["firm_table"]

    current_order = firm_table.index.tolist()

    if current_order != canonical_order:
        logger.warning(f"Table ordering mismatch. Expected {canonical_order}, got {current_order}. Reordering now.")
        # Reindex to enforce canonical order. Missing rows will become NaN, which is acceptable if a protocol failed.
        firm_table = firm_table.reindex(canonical_order)
        reproduction_package["performance_metrics"]["firm_table"] = firm_table
    else:
        logger.info("Performance table ordering matches the canonical sequence.")

    # Verify required columns exist
    required_cols = {"Return", "Volatility", "Sharpe"}
    actual_cols = set(firm_table.columns)
    if not required_cols.issubset(actual_cols):
        logger.error(f"Performance table is missing required columns. Expected {required_cols}, got {actual_cols}.")

    return reproduction_package

# -------------------------------------------------------------------------------------------------------------------------------
# Task 33, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_33_orchestrator(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any],
    output_directory: str = "./reproduction_artifacts"
) -> Union[Dict[str, Any], Any]:
    """
    Master orchestrator that runs the entire pipeline, persists diagnostics, and verifies outputs.

    Parameters
    ----------
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    output_directory : str, optional
        The path where diagnostic traces will be saved (default is "./reproduction_artifacts").

    Returns
    -------
    Union[Dict[str, Any], ValidationReport]
        The finalized reproduction package, or a ValidationReport if the pipeline halted early.
    """
    logger.info("Commencing Task 33: Run orchestrator and persist diagnostics.")

    # Execute Step 1: Run the full pipeline
    result = execute_full_pipeline(
        raw_market_df=raw_market_df,
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar,
        asset_identifier_map=asset_identifier_map,
        study_config=study_config
    )

    # If the result is not a dictionary, it means validation failed and returned a report.
    if not isinstance(result, dict):
        logger.warning("Pipeline returned a ValidationReport. Halting Task 33 execution.")
        return result

    reproduction_package = result
    out_dir_path = Path(output_directory)

    # Execute Step 2: Persist ADMM traces and compute summaries
    reproduction_package = persist_admm_traces_and_summarize(
        reproduction_package=reproduction_package,
        output_dir=out_dir_path
    )

    # Execute Step 3: Verify table ordering
    reproduction_package = verify_performance_table_ordering(
        reproduction_package=reproduction_package
    )

    logger.info("Task 33 complete. Pipeline executed, diagnostics persisted, and outputs verified.")
    return reproduction_package


In [ ]:
# Task 34: Compute firm-level and PM-level performance metrics

class PerformanceEvaluationError(Exception):
    """Custom exception raised when performance evaluation or fidelity classification fails."""
    pass

# ==============================================================================
# Task 34: Compute firm-level and PM-level performance metrics
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 34, Step 1: Compare firm-level metrics against reported values
# -------------------------------------------------------------------------------------------------------------------------------
def evaluate_firm_level_fidelity(
    firm_table: pd.DataFrame
) -> Tuple[pd.DataFrame, str]:
    """
    Compares the reproduced firm-level metrics against the manuscript's reported values and assigns a fidelity classification.

    Parameters
    ----------
    firm_table : pd.DataFrame
        The reproduced firm performance table with protocols as the index.

    Returns
    -------
    Tuple[pd.DataFrame, str]
        A comparison DataFrame showing reproduced vs. reported metrics and absolute differences,
        and the preliminary fidelity classification string.

    Raises
    ------
    PerformanceEvaluationError
        If the firm_table is missing required protocols or metrics.
    """
    logger.info("Executing Task 34, Step 1: Evaluating firm-level fidelity against reported values.")

    # Manuscript reported values (Table 3)
    # Format: {Protocol: {"Return": val, "Volatility": val, "Sharpe": val}}
    reported_values = {
        "independent": {"Return": 0.1563, "Volatility": 0.0977, "Sharpe": 1.60},
        "full_cooperative": {"Return": 0.1759, "Volatility": 0.0858, "Sharpe": 2.05},
        "admm_2_iter": {"Return": 0.1870, "Volatility": 0.0916, "Sharpe": 2.04},
        "admm_5_iter": {"Return": 0.1863, "Volatility": 0.0896, "Sharpe": 2.08}
    }

    reported_df = pd.DataFrame.from_dict(reported_values, orient="index")

    # Ensure the reproduced table has the same index and columns
    try:
        reproduced_df = firm_table.loc[reported_df.index, reported_df.columns]
    except KeyError as e:
        raise PerformanceEvaluationError(f"Reproduced firm_table is missing required protocols or columns: {e}")

    # Compute absolute differences
    diff_df = (reproduced_df - reported_df).abs()

    # Construct a multi-level comparison table for clear reporting
    comparison_df = pd.concat(
        [reproduced_df, reported_df, diff_df],
        axis=1,
        keys=["Reproduced", "Reported", "Absolute_Difference"]
    )

    # Determine fidelity classification based on tolerance bands
    # Exact-number consistent: Return/Vol diff < 0.0005 (0.05%), Sharpe diff < 0.02
    exact_return_vol = (diff_df[["Return", "Volatility"]] < 0.0005).all().all()
    exact_sharpe = (diff_df["Sharpe"] < 0.02).all()

    # High-fidelity method-consistent: Return/Vol diff < 0.01 (1.0%), Sharpe diff < 0.10
    high_fid_return_vol = (diff_df[["Return", "Volatility"]] < 0.01).all().all()
    high_fid_sharpe = (diff_df["Sharpe"] < 0.10).all()

    # Check qualitative ordering: Cooperative Sharpe > Independent Sharpe
    indep_sharpe = reproduced_df.loc["independent", "Sharpe"]
    coop_sharpes = reproduced_df.loc[["full_cooperative", "admm_2_iter", "admm_5_iter"], "Sharpe"]
    ordering_correct = (coop_sharpes > indep_sharpe).all()

    if exact_return_vol and exact_sharpe and ordering_correct:
        classification = "Exact-number consistent"
    elif high_fid_return_vol and high_fid_sharpe and ordering_correct:
        classification = "High-fidelity method-consistent"
    elif ordering_correct:
        classification = "Method-consistent (Qualitative ordering holds, but numerical drift exceeds high-fidelity bounds)"
    else:
        classification = "Method-inconsistent"

    logger.info(f"Firm-level fidelity classification: {classification}")
    return comparison_df, classification

# -------------------------------------------------------------------------------------------------------------------------------
# Task 34, Step 2: Verify PM-level qualitative findings
# -------------------------------------------------------------------------------------------------------------------------------
def verify_pm_qualitative_findings(
    pm_table: pd.DataFrame,
    cumulative_paths: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Verifies the manuscript's qualitative claims regarding individual PM outcomes under cooperation.

    Parameters
    ----------
    pm_table : pd.DataFrame
        The reproduced PM performance table with MultiIndex ["Protocol", "PM_ID"].
    cumulative_paths : Dict[str, Any]
        The dictionary containing cumulative transaction cost paths for each PM.

    Returns
    -------
    Dict[str, Any]
        A structured report detailing the outcome of the qualitative verifications.
    """
    logger.info("Executing Task 34, Step 2: Verifying PM-level qualitative findings.")

    verification_report = {}

    # Claim 1: Every PM experiences lower transaction costs under cooperation
    tc_reduction_holds = True
    pm_ids = pm_table.index.get_level_values("PM_ID").unique()

    indep_paths = cumulative_paths.get("independent", {}).get("pm_cum_tcs", {})

    for pm_id in pm_ids:
        if pm_id not in indep_paths:
            continue

        final_indep_tc = indep_paths[pm_id][-1]

        for coop_protocol in ["full_cooperative", "admm_2_iter", "admm_5_iter"]:
            coop_paths = cumulative_paths.get(coop_protocol, {}).get("pm_cum_tcs", {})
            if pm_id in coop_paths:
                final_coop_tc = coop_paths[pm_id][-1]
                if final_coop_tc >= final_indep_tc:
                    tc_reduction_holds = False
                    logger.warning(f"Claim 1 failed: PM {pm_id} TC under {coop_protocol} ({final_coop_tc}) >= independent ({final_indep_tc}).")

    verification_report["claim_1_all_pms_lower_tc"] = tc_reduction_holds

    # Claim 2: Cooperation does not guarantee higher returns or Sharpe for every PM
    # We check if AT LEAST ONE PM has a lower Sharpe under ANY cooperative protocol compared to independent
    indep_sharpes = pm_table.xs("independent", level="Protocol")["Sharpe"]

    sharpe_degradation_found = False
    degraded_pms = []

    for coop_protocol in ["full_cooperative", "admm_2_iter", "admm_5_iter"]:
        if coop_protocol in pm_table.index.get_level_values("Protocol"):
            coop_sharpes = pm_table.xs(coop_protocol, level="Protocol")["Sharpe"]

            # Compare element-wise
            degraded = coop_sharpes < indep_sharpes
            if degraded.any():
                sharpe_degradation_found = True
                degraded_pms.extend(degraded[degraded].index.tolist())

    verification_report["claim_2_indeterminate_pm_performance"] = sharpe_degradation_found
    verification_report["pms_with_degraded_sharpe"] = list(set(degraded_pms))

    if sharpe_degradation_found:
        logger.info(f"Claim 2 verified: PMs {list(set(degraded_pms))} experienced Sharpe degradation under cooperation.")
    else:
        logger.warning("Claim 2 failed: All PMs improved under cooperation. This contradicts the manuscript's indeterminate performance claim.")

    return verification_report

# -------------------------------------------------------------------------------------------------------------------------------
# Task 34, Step 3: Classify overall reproduction fidelity and document in manifest
# -------------------------------------------------------------------------------------------------------------------------------
def finalize_fidelity_classification(
    preliminary_classification: str,
    pm_verification_report: Dict[str, Any],
    manifest: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Synthesizes the firm and PM evaluations into a final fidelity classification and augments the manifest.

    Parameters
    ----------
    preliminary_classification : str
        The classification derived from firm-level numerical tolerances.
    pm_verification_report : Dict[str, Any]
        The outcomes of the PM-level qualitative checks.
    manifest : Dict[str, Any]
        The provenance manifest to be augmented.

    Returns
    -------
    Dict[str, Any]
        The finalized provenance manifest.
    """
    logger.info("Executing Task 34, Step 3: Finalizing fidelity classification.")

    # Strict logical hierarchy: Qualitative failures downgrade the classification
    final_classification = preliminary_classification

    if not pm_verification_report.get("claim_1_all_pms_lower_tc", False):
        logger.error("Critical qualitative failure: Not all PMs experienced lower TC.")
        final_classification = "Method-inconsistent"

    if not pm_verification_report.get("claim_2_indeterminate_pm_performance", False):
        logger.warning("Qualitative deviation: All PMs improved, contradicting the indeterminate claim.")
        # We don't strictly downgrade to Method-inconsistent here because this is highly dependent
        # on the specific random seed realization, but we note it.
        if final_classification == "Exact-number consistent":
            final_classification = "High-fidelity method-consistent (Seed-dependent PM deviation)"

    manifest["fidelity_classification"] = final_classification
    manifest["pm_qualitative_verification"] = pm_verification_report

    if "inconsistent" in final_classification.lower():
        logger.critical(f"Final Reproduction Fidelity: {final_classification}. Review placeholder assumptions.")
    else:
        logger.info(f"Final Reproduction Fidelity: {final_classification}.")

    return manifest

# -------------------------------------------------------------------------------------------------------------------------------
# Task 34, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_34_orchestrator(
    reproduction_package: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Orchestrates the computation of performance metrics, fidelity evaluation, and manifest finalization.

    Parameters
    ----------
    reproduction_package : Dict[str, Any]
        The complete reproduction package containing performance tables and the manifest.

    Returns
    -------
    Dict[str, Any]
        The reproduction package augmented with the evaluation summary and finalized manifest.

    Raises
    ------
    PerformanceEvaluationError
        If the evaluation process encounters missing data or structural failures.
    """
    logger.info("Commencing Task 34: Compute firm-level and PM-level performance metrics.")

    firm_table = reproduction_package["performance_metrics"]["firm_table"]
    pm_table = reproduction_package["performance_metrics"]["pm_table"]
    cumulative_paths = reproduction_package["performance_metrics"]["cumulative_paths"]
    manifest = reproduction_package["manifest"]

    # Execute Step 1: Evaluate firm-level fidelity
    comparison_df, preliminary_classification = evaluate_firm_level_fidelity(
        firm_table=firm_table
    )

    # Execute Step 2: Verify PM-level qualitative findings
    pm_verification_report = verify_pm_qualitative_findings(
        pm_table=pm_table,
        cumulative_paths=cumulative_paths
    )

    # Execute Step 3: Finalize classification and augment manifest
    manifest = finalize_fidelity_classification(
        preliminary_classification=preliminary_classification,
        pm_verification_report=pm_verification_report,
        manifest=manifest
    )

    # Inject the evaluation summary into the reproduction package
    reproduction_package["evaluation_summary"] = {
        "firm_comparison_table": comparison_df,
        "pm_qualitative_report": pm_verification_report,
        "final_classification": manifest["fidelity_classification"]
    }
    reproduction_package["manifest"] = manifest

    logger.info("Task 34 complete. Performance evaluated and fidelity classified.")
    return reproduction_package


In [ ]:
# Task 35: Archive the full reproduction package

class ArchivalError(Exception):
    """Custom exception raised when the archival process fails to write or fingerprint artifacts."""
    pass

# ==============================================================================
# Task 35: Archive the full reproduction package
# ==============================================================================

# -------------------------------------------------------------------------------------------------------------------------------
# Task 35, Step 1: Archive every input and derived artifact
# -------------------------------------------------------------------------------------------------------------------------------
def archive_data_artifacts(
    reproduction_package: Dict[str, Any],
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any],
    archive_dir: Path
) -> None:
    """
    Serializes all input and derived artifacts into their strictly preserved formats within a hierarchical directory tree.

    Parameters
    ----------
    reproduction_package : Dict[str, Any]
        The complete reproduction package containing derived artifacts and metrics.
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    archive_dir : Path
        The root directory for the archive.

    Raises
    ------
    ArchivalError
        If any file I/O operation fails.
    """
    logger.info("Executing Task 35, Step 1: Archiving input and derived data artifacts.")

    try:
        # Create directory hierarchy
        inputs_dir = archive_dir / "inputs"
        derived_dir = archive_dir / "derived"
        protocols_dir = archive_dir / "protocols"
        metrics_dir = archive_dir / "metrics"

        for d in [inputs_dir, derived_dir, protocols_dir, metrics_dir]:
            d.mkdir(parents=True, exist_ok=True)

        # --- Archive Inputs ---
        raw_market_df.to_parquet(inputs_dir / "raw_market_df.parquet")
        risk_free_rate_series.to_frame().to_parquet(inputs_dir / "risk_free_rate_series.parquet")
        asset_identifier_map.to_parquet(inputs_dir / "asset_identifier_map.parquet")

        with open(inputs_dir / "master_trading_calendar.json", "w") as f:
            json.dump([dt.isoformat() for dt in master_trading_calendar], f, indent=4)

        with open(inputs_dir / "study_config.json", "w") as f:
            json.dump(study_config, f, indent=4, sort_keys=True, default=str)

        # --- Archive Derived Artifacts ---
        exo_artifacts = reproduction_package.get("exogenous_artifacts", {})

        if "clean_market_df" in exo_artifacts:
            exo_artifacts["clean_market_df"].to_parquet(derived_dir / "clean_market_df.parquet")

        if "universe_asset_ids" in exo_artifacts:
            with open(derived_dir / "universe_asset_ids.json", "w") as f:
                json.dump(list(exo_artifacts["universe_asset_ids"]), f, indent=4)

        if "valid_decision_dates" in exo_artifacts:
            with open(derived_dir / "valid_decision_dates.json", "w") as f:
                json.dump([dt.isoformat() for dt in exo_artifacts["valid_decision_dates"]], f, indent=4)

        if "pm_characteristics" in exo_artifacts:
            # Convert dataclasses to dicts for JSON serialization
            pm_chars_dict = {
                pm_id: {
                    "pm_id": obj.pm_id,
                    "initial_nav_usd": obj.initial_nav_usd,
                    "risk_target_annualized": obj.risk_target_annualized,
                    "n_tradable_assets": obj.n_tradable_assets,
                    "tradable_mask": obj.tradable_mask.tolist()
                }
                for pm_id, obj in exo_artifacts["pm_characteristics"].items()
            }
            with open(derived_dir / "pm_characteristics.json", "w") as f:
                json.dump(pm_chars_dict, f, indent=4)

        # --- Archive Protocol Results ---
        protocol_results = reproduction_package.get("protocol_results", {})
        for protocol, history in protocol_results.items():
            p_dir = protocols_dir / protocol
            p_dir.mkdir(exist_ok=True)

            # Save trajectories as compressed npz
            np.savez_compressed(
                p_dir / "state_trajectories.npz",
                firm_nav=np.array(history.get("firm_nav", [])),
                firm_tc=np.array(history.get("firm_tc", [])),
                firm_net_trades=np.array(history.get("firm_net_trades", []))
            )

            # Note: ADMM traces were already persisted in Task 33, but we ensure they are in the archive
            # If they exist in the history object, we save them here as well for completeness
            if "admm_trace" in history and history["admm_trace"]:
                np.savez_compressed(
                    p_dir / "admm_trace.npz",
                    trace_data=np.array(history["admm_trace"], dtype=object)
                )

        # --- Archive Metrics ---
        metrics = reproduction_package.get("performance_metrics", {})
        if "firm_table" in metrics:
            metrics["firm_table"].to_csv(metrics_dir / "firm_performance_table.csv")
        if "pm_table" in metrics:
            metrics["pm_table"].to_csv(metrics_dir / "pm_performance_tables.csv")

        # Save cumulative paths
        if "cumulative_paths" in metrics:
            np.savez_compressed(
                metrics_dir / "cumulative_paths.npz",
                paths=np.array([metrics["cumulative_paths"]], dtype=object)
            )

        logger.info("All data artifacts successfully serialized to the archive directory.")

    except Exception as e:
        error_msg = f"Failed to archive data artifacts: {e}"
        logger.error(error_msg)
        raise ArchivalError(error_msg)

# -------------------------------------------------------------------------------------------------------------------------------
# Task 35, Step 2: Archive the provenance manifest
# -------------------------------------------------------------------------------------------------------------------------------
def archive_provenance_manifest(
    manifest: Dict[str, Any],
    archive_dir: Path
) -> None:
    """
    Serializes the provenance manifest to a human-readable JSON document in the archive root.

    Parameters
    ----------
    manifest : Dict[str, Any]
        The complete provenance manifest containing environment, hashes, and placeholder assumptions.
    archive_dir : Path
        The root directory for the archive.

    Raises
    ------
    ArchivalError
        If the manifest cannot be serialized.
    """
    logger.info("Executing Task 35, Step 2: Archiving the provenance manifest.")

    manifest_path = archive_dir / "manifest.json"

    # Custom JSON encoder to handle any residual NumPy types
    class NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, np.integer):
                return int(obj)
            if isinstance(obj, np.floating):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return super(NumpyEncoder, self).default(obj)

    try:
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=4, sort_keys=True, cls=NumpyEncoder)
        logger.info(f"Provenance manifest successfully written to {manifest_path}.")
    except Exception as e:
        error_msg = f"Failed to serialize provenance manifest: {e}"
        logger.error(error_msg)
        raise ArchivalError(error_msg)

# -------------------------------------------------------------------------------------------------------------------------------
# Task 35, Step 3: Verify self-containment, compute fingerprint, and freeze archive
# -------------------------------------------------------------------------------------------------------------------------------
def freeze_and_fingerprint_archive(
    archive_dir: Path,
    tarball_path: Path
) -> str:
    """
    Computes the SHA-256 hash of every file, creates a master fingerprint, and compresses the archive into an immutable tarball.

    Parameters
    ----------
    archive_dir : Path
        The root directory containing all archived files.
    tarball_path : Path
        The destination path for the final .tar.gz file.

    Returns
    -------
    str
        The master SHA-256 fingerprint of the entire archive.

    Raises
    ------
    ArchivalError
        If hashing or compression fails.
    """
    logger.info("Executing Task 35, Step 3: Computing fingerprint and freezing archive.")

    file_hashes: Dict[str, str] = {}

    try:
        # 1. Compute hash of every file in the archive directory
        for root, _, files in os.walk(archive_dir):
            for file in files:
                file_path = Path(root) / file
                # Skip the fingerprint file itself if it exists
                if file == "fingerprint.json":
                    continue

                with open(file_path, "rb") as f:
                    file_hash = hashlib.sha256(f.read()).hexdigest()

                # Store relative path as key
                rel_path = str(file_path.relative_to(archive_dir))
                file_hashes[rel_path] = file_hash

        # 2. Compute master fingerprint
        # Sort the relative paths to ensure deterministic ordering
        sorted_paths = sorted(file_hashes.keys())
        concatenated_hashes = "".join([file_hashes[p] for p in sorted_paths])
        master_fingerprint = hashlib.sha256(concatenated_hashes.encode("utf-8")).hexdigest()

        # Write the fingerprint file into the archive directory
        fingerprint_data = {
            "master_fingerprint": master_fingerprint,
            "file_hashes": file_hashes
        }
        with open(archive_dir / "fingerprint.json", "w") as f:
            json.dump(fingerprint_data, f, indent=4, sort_keys=True)

        # 3. Compress into an immutable .tar.gz tarball
        with tarfile.open(tarball_path, "w:gz") as tar:
            tar.add(archive_dir, arcname=archive_dir.name)

        logger.info(f"Archive successfully frozen into {tarball_path}.")
        logger.info(f"Master Archive Fingerprint (SHA-256): {master_fingerprint}")

        return master_fingerprint

    except Exception as e:
        error_msg = f"Failed to freeze and fingerprint archive: {e}"
        logger.error(error_msg)
        raise ArchivalError(error_msg)

# -------------------------------------------------------------------------------------------------------------------------------
# Task 35, Orchestrator Function
# -------------------------------------------------------------------------------------------------------------------------------
def task_35_orchestrator(
    reproduction_package: Dict[str, Any],
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any],
    base_output_dir: str = "./reproduction_archive"
) -> Tuple[Path, str]:
    """
    Master orchestrator that serializes all artifacts, writes the manifest, and freezes the final immutable archive.

    Parameters
    ----------
    reproduction_package : Dict[str, Any]
        The complete reproduction package containing derived artifacts and metrics.
    raw_market_df : pd.DataFrame
        The primary raw market panel input.
    risk_free_rate_series : pd.Series
        The 3-month U.S. Treasury Bill rate series.
    master_trading_calendar : pd.DatetimeIndex
        The canonical sequence of valid trading dates.
    asset_identifier_map : pd.DataFrame
        The permanent identifier map.
    study_config : Dict[str, Any]
        The master configuration dictionary.
    base_output_dir : str, optional
        The base directory where the archive will be constructed (default is "./reproduction_archive").

    Returns
    -------
    Tuple[Path, str]
        The path to the final .tar.gz tarball and the master SHA-256 fingerprint.

    Raises
    ------
    ArchivalError
        If the archival process fails at any step.
    """
    logger.info("Commencing Task 35: Archive the full reproduction package.")

    archive_dir = Path(base_output_dir)
    tarball_path = Path(f"{base_output_dir}.tar.gz")

    # Execute Step 1: Archive data artifacts
    archive_data_artifacts(
        reproduction_package=reproduction_package,
        raw_market_df=raw_market_df,
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar,
        asset_identifier_map=asset_identifier_map,
        study_config=study_config,
        archive_dir=archive_dir
    )

    # Execute Step 2: Archive provenance manifest
    manifest = reproduction_package.get("manifest", {})
    archive_provenance_manifest(
        manifest=manifest,
        archive_dir=archive_dir
    )

    # Execute Step 3: Freeze and fingerprint
    master_fingerprint = freeze_and_fingerprint_archive(
        archive_dir=archive_dir,
        tarball_path=tarball_path
    )

    logger.info("Task 35 complete. Full reproduction package archived and frozen.")

    return tarball_path, master_fingerprint


In [ ]:
# Top-Level Orchestrator

class FinalPipelineError(Exception):
    """Custom exception raised when the top-level research pipeline fails critically."""
    pass

# ==============================================================================
# FINAL TASK: Top-Level Research Pipeline Orchestrator
# ==============================================================================

def run_distributed_cooperative_optimization_pipeline(
    raw_market_df: pd.DataFrame,
    risk_free_rate_series: pd.Series,
    master_trading_calendar: pd.DatetimeIndex,
    asset_identifier_map: pd.DataFrame,
    study_config: Dict[str, Any],
    output_base_dir: str = "./research_outputs"
) -> Dict[str, Any]:
    """
    Technical Description:
        The definitive entry point for the "Distributed Method for Cooperative Transaction Cost Mitigation"
        research pipeline. This orchestrator executes the end-to-end workflow: validating inputs,
        simulating the four coordination protocols (Independent, Full Cooperative, ADMM-2, ADMM-5),
        evaluating firm and PM-level performance, classifying reproduction fidelity, and
        generating an immutable, fingerprinted research archive.

    Parameters:
        raw_market_df (pd.DataFrame):
            The primary raw market panel with MultiIndex ["date", "asset_id"].
        risk_free_rate_series (pd.Series):
            The 3-month U.S. Treasury Bill rate series from FRED.
        master_trading_calendar (pd.DatetimeIndex):
            The canonical sequence of valid trading dates for the 2000-2025 sample.
        asset_identifier_map (pd.DataFrame):
            The identity-resolution layer linking internal IDs to vendor symbols.
        study_config (Dict[str, Any]):
            The master configuration dictionary (the "blueprint").
        output_base_dir (str):
            The filesystem path where all diagnostic traces and the final archive will be saved.

    Returns:
        Dict[str, Any]:
            A summary dictionary containing the final 'reproduction_package', the 'tarball_path',
            and the 'master_fingerprint'. If validation fails, returns the 'ValidationReport'.

    Raises:
        FinalPipelineError:
            If any stage of the execution, evaluation, or archival process fails critically.
    """
    logger.info("Initializing Top-Level Research Pipeline Orchestrator.")

    # Define the path for diagnostic artifacts and the final archive
    base_path = Path(output_base_dir)

    # -------------------------------------------------------------------------
    # Step 1: Pipeline Execution and Diagnostic Persistence (Task 33)
    # -------------------------------------------------------------------------
    # This stage handles Tasks 1-32 internally, including validation and walk-forward loops.
    # It returns either the full reproduction package or a ValidationReport on failure.
    execution_result = task_33_orchestrator(
        raw_market_df=raw_market_df,
        risk_free_rate_series=risk_free_rate_series,
        master_trading_calendar=master_trading_calendar,
        asset_identifier_map=asset_identifier_map,
        study_config=study_config,
        output_directory=str(base_path / "diagnostics")
    )

    # Check for early-exit validation failures
    if not isinstance(execution_result, dict):
        logger.critical("Pipeline execution halted: Input validation failed.")
        # Return the ValidationReport directly to the researcher
        return {"validation_report": execution_result}

    reproduction_package = execution_result

    # -------------------------------------------------------------------------
    # Step 2: Performance Evaluation and Fidelity Classification (Task 34)
    # -------------------------------------------------------------------------
    # This stage computes annualized metrics, verifies qualitative claims from Appendix B,
    # and assigns the final fidelity status (e.g., 'Exact-number consistent').
    try:
        reproduction_package = task_34_orchestrator(
            reproduction_package=reproduction_package
        )
        logger.info(f"Performance evaluation complete. Fidelity: {reproduction_package['manifest']['fidelity_classification']}")
    except Exception as e:
        error_msg = f"Critical failure during performance evaluation: {e}"
        logger.error(error_msg)
        raise FinalPipelineError(error_msg)

    # -------------------------------------------------------------------------
    # Step 3: Immutable Archival and Cryptographic Fingerprinting (Task 35)
    # -------------------------------------------------------------------------
    # This stage serializes all artifacts to disk, writes the finalized manifest,
    # and generates the master SHA-256 fingerprint for the research archive.
    try:
        tarball_path, master_fingerprint = task_35_orchestrator(
            reproduction_package=reproduction_package,
            raw_market_df=raw_market_df,
            risk_free_rate_series=risk_free_rate_series,
            master_trading_calendar=master_trading_calendar,
            asset_identifier_map=asset_identifier_map,
            study_config=study_config,
            base_output_dir=str(base_path / "final_archive")
        )
        logger.info(f"Archival complete. Master Fingerprint: {master_fingerprint}")
    except Exception as e:
        error_msg = f"Critical failure during archival process: {e}"
        logger.error(error_msg)
        raise FinalPipelineError(error_msg)

    # -------------------------------------------------------------------------
    # Final Summary Construction
    # -------------------------------------------------------------------------
    # Construct the final implementation-grade summary dictionary
    pipeline_summary = {
        "status": "SUCCESS",
        "reproduction_package": reproduction_package,
        "archive_path": tarball_path,
        "master_fingerprint": master_fingerprint,
        "fidelity_classification": reproduction_package["manifest"]["fidelity_classification"],
        "total_execution_time": reproduction_package["manifest"].get("total_execution_time_seconds", 0.0)
    }

    logger.info("Top-Level Research Pipeline executed successfully. All artifacts frozen.")

    return pipeline_summary


#